# Brain tumour segmentation and multiclass classification on a Kaggle T4

This notebook trains and evaluates every model of the **Brain segmentation and classification**
project and exports artefacts the Flask application loads unchanged. It uses the two datasets the
project needs, for two different tasks:

| Task | Dataset | Kaggle |
| --- | --- | --- |
| Segmentation (plus the HGG/LGG grade experiment) | BraTS 2020 | `awsaf49/brats2020-training-data` |
| Multiclass tumour type (classes come from the dataset) | Brain Tumor MRI Dataset | `masoudnickparvar/brain-tumor-mri-dataset` |

The two datasets are never mixed: the label spaces are different, and the stacked ensemble accepts
a member only when its dataset and class list match the other members'.

Verified context for this notebook (see `docs/DELIVERY.md`):

* the shipped `segnet_model.pth` is degenerate -- its sigmoid output never crosses 0.5 on any
  available input, so SegNet is retrained here;
* `UNet-[16, 32, 64, 128, 256].pt` runs but was never evaluated against labelled masks, so the
  same architecture is retrained from scratch on real data;
* `raunet_brats_4ch.pt` does not exist yet;
* the shipped classifier expects 21 unnamed features and no scaler was saved, so neither
  classification artefact is trained to fit it: each one records its own ordered feature schema,
  scaler and label mapping, which is exactly what the API requires.

What each part produces is recorded next to it: every artefact carries its task, its dataset and
whether that dataset is real or a fixture, and `data/models/models_index.json` lists all of them.

## The data pipeline (why training is not CPU-bound)

Training reads a per-patient cache. That cache exists in two layouts, and the difference is
measured in cell 9 rather than assumed:

* **`memmap`** (default) -- `cache/<volume>.cache/{images,masks,singles}.npy`, one
  memory-mappable file per member: reading slice *i* reads slice *i*.
* **`npz`** (legacy) -- `cache/<volume>.npz`, a `savez_compressed` archive. Opening any member
  inflates that whole member, so reading one slice of one real BraTS volume means decompressing
  images + masks + singles (165 MiB).

The trainer reads through a bounded payload store: one read per patient per batch rather than one
per slice, only the members a batch needs (a 4-channel batch never opens the 512x512
single-channel member), slice counts from a `.npy` header instead of a payload load, and bounded
memory (`patient_cache_mb`, `patient_cache_handles`). An existing `.npz` cache is converted in
place by `upgrade_patient_cache()` -- written as `.part`, read back, compared value by value,
published, and only then is the archive deleted; a volume that fails keeps its archive and is
reported. Cell 18 then prints the five phases of a real training step (preparation, CPU->GPU
transfer, forward, backward + optimiser step, and the whole step) with
`torch.cuda.synchronize()` around the GPU phases and a real `nvidia-smi` utilisation sample.

Nothing about the model, the loss, the optimiser, the schedule, the checkpoints, the splits or the
normalisation changed: the test suite asserts that the new reader yields batches that are
bit-for-bit identical to the implementation it replaced. `docs/DELIVERY.md` §K has the measured
root cause and the before/after numbers.

## How to run it

1. **Settings -> Accelerator -> GPU T4 x2** (or any CUDA GPU). Cell 1 verifies the device and
   prints the memory before anything is trained.
2. **Add Data -> `awsaf49/brats2020-training-data`** and **Add Data ->
   `masoudnickparvar/brain-tumor-mri-dataset`.** Internet access is not required; the notebook
   reads both datasets from `/kaggle/input` and embeds all the code it needs. Each dataset is
   optional in the sense that a missing one is *reported* and its section skips -- with exact
   reasons and no invented results -- while the other still runs.
3. Run the cells in order. Cell 2 holds every size, epoch count and budget in one place.
4. A full run does **not** have to finish in one session. Checkpoints carry model, optimiser,
   scheduler, AMP scaler and RNG state; re-running the training cells continues where the last
   session stopped (see the runbook at the end).

## What it produces

Per-model checkpoints, training histories, held-out metrics in the application's own
`evaluation_results.json` / `model_selection.json` / `ensemble_calibration.json` formats, a
classifier artefact in the format `ClassificationService` requires, figures, and a
`brain_mri_artifacts.zip` containing everything plus a README and a provenance manifest.

## What it never does

* It never uses the test split for model selection, threshold tuning or ensemble weighting.
* It never fabricates a label, a metric or a model: a section that has no data says so and stops.
* It never overwrites an artefact that is already in the repository.


# Part 1. Environment, configuration and project code


## 1. Environment and GPU verification
`nbs_01_setup.py`


In [ ]:
# =============================================================================
# SECTION 1 - Environment, GPU and dependency verification
# =============================================================================
# Everything the notebook needs is checked here and reported, never assumed.
# The notebook works on a Kaggle T4 (preferred), on any other CUDA GPU, and on
# CPU for a tiny smoke run -- but the full run is sized for a T4.
# =============================================================================
import importlib
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

LOCK = {}
STARTED_AT = time.time()


def _pip_install(packages, optional=True):
    """Install only what is missing; never fail the notebook for an optional dep."""
    missing = []
    for name, module in packages:
        try:
            importlib.import_module(module)
        except Exception:
            missing.append(name)
    if not missing:
        return {"installed": [], "already_present": [p for p, _ in packages]}
    print(f"Installing {missing} ...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        message = (result.stderr or result.stdout or "").strip()[-400:]
        if optional:
            print(f"WARNING: could not install {missing}; optional features will be skipped.\n{message}")
            return {"installed": [], "failed": missing, "error": message}
        raise RuntimeError(f"Required packages {missing} could not be installed: {message}")
    return {"installed": missing}


INSTALL_REPORT = _pip_install(
    [
        ("h5py", "h5py"),
        ("scikit-learn", "sklearn"),
        ("joblib", "joblib"),
        ("matplotlib", "matplotlib"),
        ("pandas", "pandas"),
        ("tqdm", "tqdm"),
        ("nibabel", "nibabel"),
        ("SimpleITK", "SimpleITK"),
    ]
)

import numpy as np
import torch
import torch.nn as nn

CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device("cuda:0" if CUDA_AVAILABLE else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "none (CPU)"
GPU_TOTAL_GB = (
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if CUDA_AVAILABLE else 0.0
)
COMPUTE_CAPABILITY = torch.cuda.get_device_capability(0) if CUDA_AVAILABLE else None

AMP_SUPPORTED = bool(CUDA_AVAILABLE and torch.cuda.is_bf16_supported() is not None)
# T4 (SM 7.5) supports fp16 tensor-core maths; AMP is used when a CUDA device exists.
AMP_ENABLED = bool(CUDA_AVAILABLE)

print("=" * 78)
print("ENVIRONMENT")
print("=" * 78)
print(f"python           : {sys.version.split()[0]}")
print(f"platform         : {platform.platform()}")
print(f"torch            : {torch.__version__} (cuda build {torch.version.cuda})")
print(f"cuda available   : {CUDA_AVAILABLE}")
print(f"device           : {DEVICE}  |  {GPU_NAME}")
if CUDA_AVAILABLE:
    print(f"gpu memory       : {GPU_TOTAL_GB} GB  | capability {COMPUTE_CAPABILITY}")
    print(f"cuda runtime     : {torch.version.cuda}")
    print(f"cudnn            : {torch.backends.cudnn.version()}")
print(f"amp enabled      : {AMP_ENABLED}")
print(f"cpu count        : {os.cpu_count()}")
print(f"numpy            : {np.__version__}")
print(f"install report   : {INSTALL_REPORT}")

if not CUDA_AVAILABLE:
    print(
        "\nWARNING: no CUDA device. The notebook will still run, but only a small smoke "
        "run is realistic on CPU. Set RUN['smoke_test'] = True, or enable the GPU "
        "accelerator in Kaggle (Settings -> Accelerator -> GPU T4 x2) and rerun."
    )
elif GPU_TOTAL_GB < 12:
    print(
        f"\nWARNING: {GPU_TOTAL_GB} GB of GPU memory is below the 15 GB a Kaggle T4 offers. "
        "Reduce RUN['batch_size'] and RUN['cache_size'] before the full run."
    )

LOCK["environment"] = {
    "torch": torch.__version__,
    "cuda_available": CUDA_AVAILABLE,
    "gpu": GPU_NAME,
    "gpu_memory_gb": GPU_TOTAL_GB,
    "amp": AMP_ENABLED,
    "python": sys.version.split()[0],
}


## 2. Run configuration and resumable state
`nbs_01_setup.py`


In [ ]:
# =============================================================================
# SECTION 1b - Configuration: paths, sizes, epochs, memory budget, reproducibility
# =============================================================================
# Every number that affects a result lives here. Nothing downstream hard-codes a
# duration, a batch size or a metric.
# =============================================================================
import random

RUN = {
    # --- dataset -----------------------------------------------------------
    "dataset_slug": "awsaf49/brats2020-training-data",
    "dataset_roots": [  # searched in order; first match wins
        "/kaggle/input/brats2020-training-data",
        "/kaggle/input/brats2020-training-data/BraTS2020_training_data",
        "/kaggle/input/brats2020_training_data",
        "/kaggle/input/brats-2020-training-data",
    ],
    # --- segmentation ------------------------------------------------------
    "image_size": 256,          # 4-channel models (SegNet, RAUNet): 256x256
    "single_image_size": 512,   # single-channel model (DynamicUNet): 512x512
    "single_channel_index": 3,  # FLAIR is the channel the shipped U-Net was fed
    "batch_size": 16,           # 4-channel, 256x256, fp16 on a T4
    "batch_size_single": 4,     # 1-channel, 512x512
    "epochs_segnet": 40,
    "epochs_unet": 30,
    "epochs_raunet": 40,
    "max_steps_per_epoch": None,   # set an int to time-box an epoch (indices reshuffled each epoch)
    "min_steps_per_epoch": 200,
    "learning_rate": 3e-4,
    "learning_rate_single": 1e-4,
    "weight_decay": 1e-5,
    "dice_weight": 0.6,          # project convention: 0.6 * Dice + 0.4 * BCE
    "early_stopping_patience": 8,
    "validate_every_epochs": 1,
    "validation_cases": 64,      # patient-level validation cases per validation pass
    "grad_clip": 1.0,
    "num_workers": 2,          # unused: this pipeline feeds the GPU from cached arrays (see cell 17)
    # --- cached-array data pipeline ----------------------------------------
    # "memmap" writes one memory-mappable .npy per member (reading slice i reads slice i);
    # "npz" writes the older compressed archive (any access inflates the whole member).
    "cache_format": "memmap",
    "upgrade_cache_to_memmap": True,   # convert an existing .npz cache in place, verified per patient
    "patient_cache_mb": 1536,          # bound on materialised payloads held in RAM
    "patient_cache_handles": 32,       # bound on simultaneously open memory maps
    "prefetch_batches": False,         # prepare the next batch on a worker thread (measured, see cell 18)
    "benchmark_before_training": True,
    "benchmark_steps": 6,
    "benchmark_warmup": 2,
    # Set True to also measure the OLD per-slice reader at step level. It reads a whole
    # payload per slice, so it costs minutes on a full split; the cache-level before/after
    # in cell 9 is measured on the real archives instead, for free.
    "benchmark_legacy_pipeline": False,
    # --- time budget -------------------------------------------------------
    "session_budget_minutes": 480,   # stop cleanly before the Kaggle session ends
    "checkpoint_every_steps": 500,
    "checkpoint_every_minutes": 20,
    # --- classification ----------------------------------------------------
    "radiomics_slices_per_patient": 3,   # slices around the largest tumour slice
    "radiomics_max_patients": None,      # None = every labelled patient
    "classifier_improved": "hist_gradient_boosting",  # or "mlp"
    "image_classifier_epochs": 12,
    "image_classifier_enabled": True,
    "image_classifier_slices_per_patient": 6,
    # --- second dataset: multiclass image classification --------------------
    # Brain Tumor MRI Dataset (the dataset's own classes; never assumed to be HGG/LGG):
    #   https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
    "classification_dataset_slug": "masoudnickparvar/brain-tumor-mri-dataset",
    "classification_dataset_roots": [
        "/kaggle/input/brain-tumor-mri-dataset",
        "/kaggle/input/brain-tumor-mri-dataset/Training",
        "/kaggle/input/brain-tumor-mri-classification-dataset",
        "/kaggle/input/brain-tumor-classification-mri",
        "data/brain_tumor_mri_dataset",
    ],
    # "auto" checks for a fixture manifest first and accepts a tree with >= 1000 images as the
    # real release; anything else is recorded as unverified rather than claimed as real data.
    "classification_dataset_provenance": "auto",
    "classification_task_name": "brain tumour type (multiclass, classes defined by the dataset)",
    "classification_image_size": 224,
    "classification_validation_fraction": 0.15,
    # Near-duplicate audit. The hash only proposes candidates; the pixel correlation confirms them,
    # because an average hash cannot tell two similar-looking brain slices apart on its own.
    "classification_duplicate_hash_size": 16,             # 16x16 = 256-bit average hash
    "classification_duplicate_pixel_size": 64,             # size at which the pixels are compared
    "classification_duplicate_hamming_threshold": 24,      # <= this many differing bits = candidate
    "classification_duplicate_correlation_threshold": 0.995,  # Pearson r >= this = confirmed duplicate
    "duplicate_exclusion_max_fraction": 0.05,              # above this share the audit is inconclusive
    "exclude_duplicate_eval_images": True,                 # drop confirmed duplicates from validation/test
    "radiomics_image_size": 256,                       # the size the application's radiomics runs at
    "multiclass_radiomics_max_images": None,           # None = every image; an int caps it per class/split
    "cnn_channels": [32, 64, 128],
    "cnn_dropout": 0.3,
    "cnn_epochs": 30,
    "cnn_batch_size": 32,
    "cnn_learning_rate": 3e-4,
    "cnn_weight_decay": 1e-4,
    "cnn_max_steps_per_epoch": None,
    "cnn_validation_every_epochs": 1,
    "cnn_early_stopping_patience": 8,
    "skip_multiclass_cnn": False,
    # --- ensemble ----------------------------------------------------------
    "ensemble_members": ["segnet_brats_4ch", "raunet_brats_4ch"],
    "ensemble_weight_grid": 0.05,
    # --- reproducibility ---------------------------------------------------
    "seed": 20240101,
    "smoke_test": False,   # True -> tiny subsets, 1 epoch, 1 case per split
    # "auto" detects the synthetic fixture from its HDF5 marker and accepts a real BraTS
    # tree of >= 100 volumes as real. Anything else is recorded as "unknown" rather than
    # being claimed as real data. Override with "real_brats2020" / "synthetic_fixture".
    "dataset_provenance": "auto",
}

WORK = Path("/kaggle/working")
if not WORK.exists():  # running outside Kaggle
    WORK = Path("./kaggle_working").resolve()
PATHS = {
    "work": WORK,
    "cache": WORK / "cache",             # resized volumes + masks (resumable)
    "checkpoints": WORK / "checkpoints", # per model: last.pt / best.pt / history.json
    "artifacts": WORK / "artifacts",     # metrics, manifests, calibration
    "reports": WORK / "reports",         # figures, tables
    "logs": WORK / "logs",
    "export": WORK / "brain_mri_artifacts",
}
for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

SMOKE_OVERRIDES = {
    "epochs_segnet": 1,
    # the multiclass pipeline is exercised end to end too, on a handful of small images
    "classification_image_size": 64,
    "classification_validation_fraction": 0.34,
    "multiclass_radiomics_max_images": 4,
    "cnn_epochs": 1,
    "cnn_batch_size": 8,
    "cnn_max_steps_per_epoch": 3,
    "cnn_early_stopping_patience": 1,
    "epochs_unet": 1,
    "epochs_raunet": 1,
    "max_steps_per_epoch": 4,
    "min_steps_per_epoch": 1,
    "validation_cases": 2,
    "radiomics_max_patients": 4,
    "image_classifier_epochs": 1,
    "image_classifier_slices_per_patient": 2,
    "radiomics_slices_per_patient": 1,
    "session_budget_minutes": 30,
    "benchmark_steps": 2,
    "benchmark_warmup": 0,
    "batch_size": 8,           # the fixture has few slices; 8 keeps at least two batches
    "patient_cache_mb": 256,
    "patient_cache_handles": 8,
}


def apply_smoke_overrides() -> None:
    """Shrink every workload to a few minutes.

    Called here when RUN['smoke_test'] is True; the test suite flips the flag after this
    cell and calls the same function, so the tested configuration is the notebook's own.
    Idempotent: calling it twice changes nothing.
    """
    RUN["smoke_test"] = True
    RUN.update(SMOKE_OVERRIDES)


if RUN["smoke_test"]:
    apply_smoke_overrides()

SEED = int(RUN["seed"])


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False  # speed over bitwise determinism
    torch.backends.cudnn.benchmark = True


def set_deterministic_namespace() -> None:
    """Nothing random is allowed to depend on hidden state across sessions."""
    os.environ.setdefault("PYTHONHASHSEED", str(SEED))


seed_everything()
set_deterministic_namespace()


def write_json(path, payload) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")


def read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"{path} is not valid JSON: {exc}") from exc


def utc_now() -> str:
    from datetime import datetime, timezone

    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


LOCK["run"] = RUN
LOCK["paths"] = {key: str(value) for key, value in PATHS.items()}
write_json(PATHS["artifacts"] / "run_config.json", {"run": RUN, "paths": LOCK["paths"], "generated_at": utc_now()})

print("Work directory :", PATHS["work"])
print("Smoke test     :", RUN["smoke_test"])
print("Image size     :", RUN["image_size"], "| single-channel:", RUN["single_image_size"])
print("Batch sizes    :", RUN["batch_size"], "|", RUN["batch_size_single"])
print("Epochs         : segnet", RUN["epochs_segnet"], "| unet", RUN["epochs_unet"], "| raunet", RUN["epochs_raunet"])
print("Session budget :", RUN["session_budget_minutes"], "minutes (checkpoints let the next session continue)")
print("Seed           :", SEED)


## The project's inference code, embedded verbatim
`build_kaggle_notebook.embedded`


In [ ]:
# =============================================================================
# SECTION 1c - The project's own inference code, embedded verbatim
# =============================================================================
# Architectures, loss, metrics, Grad-CAM, the native radiomics extractor and the
# evaluation service are copied here *from the repository*, byte for byte. That is what
# makes the exports below loadable by the Flask application: the same builders build the
# same graphs, and the same extractor produces the same feature names.
#
# Each entry records the sha256 of the file it came from. `pytest tests/test_kaggle_notebook.py`
# recomputes those hashes against backend/app/**, so the notebook cannot silently drift
# away from the application: regenerate it with notebooks/build_kaggle_notebook.py.
# =============================================================================
EMBEDDED_SOURCES = json.loads(r'''{"backend/app/ml/architectures/segnet.py": {"sha256": "7e5069de43506b7a2e912307ae32c664af6cae012c710058c22ca86498109e80", "source": "from __future__ import annotations\nimport torch\nimport torch.nn as nn\nclass SegNet(nn.Module):\n    def __init__(self, in_channels: int = 4, out_channels: int = 1):\n        super(SegNet, self).__init__()\n        self.in_channels = in_channels\n        self.out_channels = out_channels\n        self.encoder = nn.Sequential(\n            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),\n            nn.BatchNorm2d(64),\n            nn.ReLU(inplace=True),\n            nn.MaxPool2d(kernel_size=2, stride=2, return_indices=False),\n            nn.Conv2d(64, 128, kernel_size=3, padding=1),\n            nn.BatchNorm2d(128),\n            nn.ReLU(inplace=True),\n            nn.MaxPool2d(kernel_size=2, stride=2, return_indices=False),\n        )\n        self.decoder = nn.Sequential(\n            nn.Upsample(scale_factor=2, mode=\"bilinear\", align_corners=True),\n            nn.Conv2d(128, 64, kernel_size=3, padding=1),\n            nn.BatchNorm2d(64),\n            nn.ReLU(inplace=True),\n            nn.Upsample(scale_factor=2, mode=\"bilinear\", align_corners=True),\n            nn.Conv2d(64, out_channels, kernel_size=3, padding=1),\n        )\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        for layer in self.encoder:\n            x = layer(x)\n        for layer in self.decoder:\n            x = layer(x)\n        return x\n    @property\n    def output_semantics(self) -> str:\n        return \"logit\"\n    def target_layer_name(self) -> str:\n        return \"encoder.4\"\n"}, "backend/app/ml/architectures/dynamic_unet.py": {"sha256": "ee848cb6420de3632a46157f991d3c81254d734443f641ec6e556b76e6834c6a", "source": "from __future__ import annotations\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nDEFAULT_FILTERS = [16, 32, 64, 128, 256]\nclass DynamicUNet(nn.Module):\n    def __init__(self, filters, input_channels: int = 1, output_channels: int = 1):\n        super(DynamicUNet, self).__init__()\n        if len(filters) != 5:\n            raise ValueError(f\"Filter list size {len(filters)}, expected 5!\")\n        self.filters = list(filters)\n        self.input_channels = input_channels\n        self.output_channels = output_channels\n        padding = 1\n        ks = 3\n        self.conv1_1 = nn.Conv2d(input_channels, filters[0], kernel_size=ks, padding=padding)\n        self.conv1_2 = nn.Conv2d(filters[0], filters[0], kernel_size=ks, padding=padding)\n        self.maxpool1 = nn.MaxPool2d(2)\n        self.conv2_1 = nn.Conv2d(filters[0], filters[1], kernel_size=ks, padding=padding)\n        self.conv2_2 = nn.Conv2d(filters[1], filters[1], kernel_size=ks, padding=padding)\n        self.maxpool2 = nn.MaxPool2d(2)\n        self.conv3_1 = nn.Conv2d(filters[1], filters[2], kernel_size=ks, padding=padding)\n        self.conv3_2 = nn.Conv2d(filters[2], filters[2], kernel_size=ks, padding=padding)\n        self.maxpool3 = nn.MaxPool2d(2)\n        self.conv4_1 = nn.Conv2d(filters[2], filters[3], kernel_size=ks, padding=padding)\n        self.conv4_2 = nn.Conv2d(filters[3], filters[3], kernel_size=ks, padding=padding)\n        self.maxpool4 = nn.MaxPool2d(2)\n        self.conv5_1 = nn.Conv2d(filters[3], filters[4], kernel_size=ks, padding=padding)\n        self.conv5_2 = nn.Conv2d(filters[4], filters[4], kernel_size=ks, padding=padding)\n        self.conv5_t = nn.ConvTranspose2d(filters[4], filters[3], 2, stride=2)\n        self.conv6_1 = nn.Conv2d(filters[4], filters[3], kernel_size=ks, padding=padding)\n        self.conv6_2 = nn.Conv2d(filters[3], filters[3], kernel_size=ks, padding=padding)\n        self.conv6_t = nn.ConvTranspose2d(filters[3], filters[2], 2, stride=2)\n        self.conv7_1 = nn.Conv2d(filters[3], filters[2], kernel_size=ks, padding=padding)\n        self.conv7_2 = nn.Conv2d(filters[2], filters[2], kernel_size=ks, padding=padding)\n        self.conv7_t = nn.ConvTranspose2d(filters[2], filters[1], 2, stride=2)\n        self.conv8_1 = nn.Conv2d(filters[2], filters[1], kernel_size=ks, padding=padding)\n        self.conv8_2 = nn.Conv2d(filters[1], filters[1], kernel_size=ks, padding=padding)\n        self.conv8_t = nn.ConvTranspose2d(filters[1], filters[0], 2, stride=2)\n        self.conv9_1 = nn.Conv2d(filters[1], filters[0], kernel_size=ks, padding=padding)\n        self.conv9_2 = nn.Conv2d(filters[0], filters[0], kernel_size=ks, padding=padding)\n        self.conv10 = nn.Conv2d(filters[0], output_channels, kernel_size=ks, padding=padding)\n    def forward(self, x: torch.Tensor, return_logits: bool = False) -> torch.Tensor:\n        conv1 = F.relu(self.conv1_1(x))\n        conv1 = F.relu(self.conv1_2(conv1))\n        pool1 = self.maxpool1(conv1)\n        conv2 = F.relu(self.conv2_1(pool1))\n        conv2 = F.relu(self.conv2_2(conv2))\n        pool2 = self.maxpool2(conv2)\n        conv3 = F.relu(self.conv3_1(pool2))\n        conv3 = F.relu(self.conv3_2(conv3))\n        pool3 = self.maxpool3(conv3)\n        conv4 = F.relu(self.conv4_1(pool3))\n        conv4 = F.relu(self.conv4_2(conv4))\n        pool4 = self.maxpool4(conv4)\n        conv5 = F.relu(self.conv5_1(pool4))\n        conv5 = F.relu(self.conv5_2(conv5))\n        up6 = torch.cat((self.conv5_t(conv5), conv4), dim=1)\n        conv6 = F.relu(self.conv6_1(up6))\n        conv6 = F.relu(self.conv6_2(conv6))\n        up7 = torch.cat((self.conv6_t(conv6), conv3), dim=1)\n        conv7 = F.relu(self.conv7_1(up7))\n        conv7 = F.relu(self.conv7_2(conv7))\n        up8 = torch.cat((self.conv7_t(conv7), conv2), dim=1)\n        conv8 = F.relu(self.conv8_1(up8))\n        conv8 = F.relu(self.conv8_2(conv8))\n        up9 = torch.cat((self.conv8_t(conv8), conv1), dim=1)\n        conv9 = F.relu(self.conv9_1(up9))\n        conv9 = F.relu(self.conv9_2(conv9))\n        logits = self.conv10(conv9)\n        if return_logits:\n            return logits\n        return torch.sigmoid(logits)\n    @property\n    def output_semantics(self) -> str:\n        return \"probability\"\n    def target_layer_name(self) -> str:\n        return \"conv9_2\"\n"}, "backend/app/ml/architectures/attention_unet.py": {"sha256": "9ffb6e9a364bf099d4708b9e19085812033f975d4f298ff215d5850334b390ab", "source": "from __future__ import annotations\nfrom dataclasses import dataclass, field\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n@dataclass\nclass RAUNetConfig:\n    in_channels: int = 4\n    out_channels: int = 1\n    base_filters: int = 16\n    depth: int = 4\n    use_attention: bool = True\n    use_residual: bool = True\n    deep_supervision: bool = True\n    dropout: float = 0.0\n    filters: list[int] = field(default_factory=list)\n    def channel_list(self) -> list[int]:\n        if self.filters:\n            return self.filters\n        return [self.base_filters * (2**i) for i in range(self.depth + 1)]\nclass ResidualBlock(nn.Module):\n    def __init__(self, in_channels: int, out_channels: int):\n        super().__init__()\n        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False)\n        self.bn1 = nn.BatchNorm2d(out_channels)\n        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)\n        self.bn2 = nn.BatchNorm2d(out_channels)\n        self.shortcut = (\n            nn.Identity()\n            if in_channels == out_channels\n            else nn.Conv2d(in_channels, out_channels, 1, bias=False)\n        )\n        self.relu = nn.ReLU(inplace=True)\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        residual = self.shortcut(x)\n        out = self.relu(self.bn1(self.conv1(x)))\n        out = self.bn2(self.conv2(out))\n        return self.relu(out + residual)\nclass PlainBlock(nn.Module):\n    def __init__(self, in_channels: int, out_channels: int):\n        super().__init__()\n        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False)\n        self.bn1 = nn.BatchNorm2d(out_channels)\n        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)\n        self.bn2 = nn.BatchNorm2d(out_channels)\n        self.relu = nn.ReLU(inplace=True)\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        x = self.relu(self.bn1(self.conv1(x)))\n        return self.relu(self.bn2(self.conv2(x)))\nclass AttentionGate(nn.Module):\n    def __init__(self, f_x: int, f_g: int, f_int: int | None = None):\n        super().__init__()\n        f_int = f_int or max(f_x // 2, 1)\n        self.theta_x = nn.Conv2d(f_x, f_int, kernel_size=1, bias=True)\n        self.phi_g = nn.Conv2d(f_g, f_int, kernel_size=1, bias=True)\n        self.psi = nn.Conv2d(f_int, 1, kernel_size=1, bias=True)\n    def forward(self, x: torch.Tensor, g: torch.Tensor) -> torch.Tensor:\n        if g.shape[-2:] != x.shape[-2:]:\n            g = F.interpolate(g, size=x.shape[-2:], mode=\"bilinear\", align_corners=False)\n        additive = F.relu(self.theta_x(x) + self.phi_g(g))\n        attention = torch.sigmoid(self.psi(additive))\n        return x * attention\nclass RAUNet(nn.Module):\n    def __init__(self, config: RAUNetConfig | None = None, **kwargs):\n        super().__init__()\n        if config is None:\n            config = RAUNetConfig(**kwargs)\n        self.config = config\n        channels = config.channel_list()\n        self.filters = channels\n        block = ResidualBlock if config.use_residual else PlainBlock\n        self.stem = block(config.in_channels, channels[0])\n        self.encoder_blocks = nn.ModuleList()\n        self.pools = nn.ModuleList()\n        for index in range(config.depth):\n            self.encoder_blocks.append(block(channels[index], channels[index + 1]))\n            self.pools.append(nn.MaxPool2d(2))\n        self.bottleneck = block(channels[-1], channels[-1])\n        self.upsamples = nn.ModuleList()\n        self.attention_gates = nn.ModuleList()\n        self.decoder_blocks = nn.ModuleList()\n        self.dropouts = nn.ModuleList()\n        self.heads = nn.ModuleList()\n        for index in reversed(range(config.depth)):\n            in_ch = channels[index + 1]\n            skip_ch = channels[index]\n            self.upsamples.append(nn.ConvTranspose2d(in_ch, skip_ch, kernel_size=2, stride=2))\n            self.attention_gates.append(\n                AttentionGate(skip_ch, skip_ch, max(skip_ch // 2, 1))\n                if config.use_attention\n                else nn.Identity()\n            )\n            self.decoder_blocks.append(block(skip_ch * 2, skip_ch))\n            self.dropouts.append(nn.Dropout2d(p=config.dropout) if config.dropout > 0 else nn.Identity())\n        for index in reversed(range(config.depth)):\n            self.heads.append(nn.Conv2d(channels[index], config.out_channels, kernel_size=1))\n    def forward(self, x: torch.Tensor, return_aux: bool = False, return_logits: bool = True):\n        skips = []\n        out = self.stem(x)\n        skips.append(out)\n        for block, pool in zip(self.encoder_blocks, self.pools):\n            out = pool(out)\n            out = block(out)\n            skips.append(out)\n        out = self.bottleneck(out)\n        decoder_outputs: list[torch.Tensor] = []\n        for index in range(len(self.upsamples)):\n            skip = skips[-1 - index - 1]\n            out = self.upsamples[index](out)\n            if out.shape[-2:] != skip.shape[-2:]:\n                out = F.interpolate(out, size=skip.shape[-2:], mode=\"bilinear\", align_corners=False)\n            gate = self.attention_gates[index]\n            skip_gated = gate(skip, out) if not isinstance(gate, nn.Identity) else skip\n            out = torch.cat([out, skip_gated], dim=1)\n            out = self.decoder_blocks[index](out)\n            out = self.dropouts[index](out)\n            decoder_outputs.append(out)\n        primary = self.heads[-1](decoder_outputs[-1])\n        logits = primary if return_logits else torch.sigmoid(primary)\n        if not return_aux:\n            return logits\n        aux = []\n        for index in range(len(decoder_outputs) - 1):\n            head_index = index\n            aux_map = self.heads[head_index](decoder_outputs[index])\n            if aux_map.shape[-2:] != primary.shape[-2:]:\n                aux_map = F.interpolate(\n                    aux_map, size=primary.shape[-2:], mode=\"bilinear\", align_corners=False\n                )\n            aux.append(aux_map)\n        return logits, aux\n    @property\n    def output_semantics(self) -> str:\n        return \"logit\"\n    def target_layer_name(self) -> str:\n        return f\"decoder_blocks.{len(self.decoder_blocks) - 1}\"\n    def has_dropout(self) -> bool:\n        return self.config.dropout > 0\n    def parameter_count(self) -> int:\n        return sum(p.numel() for p in self.parameters())\n    def trainable_parameter_count(self) -> int:\n        return sum(p.numel() for p in self.parameters() if p.requires_grad)\ndef build_raunet(**kwargs) -> RAUNet:\n    known = set(RAUNetConfig.__dataclass_fields__.keys())\n    config_kwargs = {k: v for k, v in kwargs.items() if k in known}\n    return RAUNet(RAUNetConfig(**config_kwargs))\n"}, "backend/app/ml/losses.py": {"sha256": "b4f60af831fc7373ebd37943e00dfecabdcabbe32438c2b17bdd5c85e52b56ba", "source": "from __future__ import annotations\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nclass DiceLoss(nn.Module):\n    def __init__(self, from_logits: bool = False, smooth: float = 1.0):\n        super().__init__()\n        self.from_logits = from_logits\n        self.smooth = smooth\n    def forward(self, predicted: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        if self.from_logits:\n            predicted = torch.sigmoid(predicted)\n        dims = tuple(range(1, predicted.ndim))\n        intersection = (predicted * target).sum(dim=dims)\n        denom = predicted.sum(dim=dims) + target.sum(dim=dims)\n        coefficient = (2 * intersection + self.smooth) / (denom + self.smooth)\n        return 1 - coefficient.mean()\nclass BCEDiceLoss(nn.Module):\n    def __init__(self, device=None, from_logits: bool = False, bce_weight: float = 1.0, dice_weight: float = 1.0):\n        super().__init__()\n        self.dice_loss = DiceLoss(from_logits=from_logits)\n        self.from_logits = from_logits\n        self.bce_weight = bce_weight\n        self.dice_weight = dice_weight\n        if device is not None:\n            self.to(device)\n    def forward(self, predicted: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        bce = F.binary_cross_entropy_with_logits(predicted, target) if self.from_logits else F.binary_cross_entropy(predicted, target)\n        return self.bce_weight * bce + self.dice_weight * self.dice_loss(predicted, target)\nclass FocalTverskyLoss(nn.Module):\n    def __init__(self, alpha: float = 0.3, beta: float = 0.7, gamma: float = 0.75, from_logits: bool = True):\n        super().__init__()\n        self.alpha = alpha\n        self.beta = beta\n        self.gamma = gamma\n        self.from_logits = from_logits\n    def forward(self, predicted: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        if self.from_logits:\n            predicted = torch.sigmoid(predicted)\n        dims = tuple(range(1, predicted.ndim))\n        tp = (predicted * target).sum(dim=dims)\n        fp = (predicted * (1 - target)).sum(dim=dims)\n        fn = ((1 - predicted) * target).sum(dim=dims)\n        tversky = (tp + 1.0) / (tp + self.alpha * fp + self.beta * fn + 1.0)\n        return ((1 - tversky) ** self.gamma).mean()\ndef distance_weight_map(target: torch.Tensor, beta: float = 5.0, sigma_ratio: float = 0.25) -> torch.Tensor:\n    from ...utils.metrics import _edt\n    array = target.detach().cpu().numpy()\n    if array.ndim == 4:\n        array = array[:, 0]\n    if array.ndim != 3:\n        raise ValueError(\"distance_weight_map expects a (B,H,W) or (B,1,H,W) tensor\")\n    weights = np.ones_like(array, dtype=np.float32)\n    for index in range(array.shape[0]):\n        mask = array[index] > 0.5\n        if not mask.any():\n            continue\n        distance_inside = _edt(mask)\n        distance_outside = _edt(~mask)\n        sigma = max(float(np.sqrt(mask.sum())) * sigma_ratio, 1.0)\n        w_in = 1.0 + beta * np.exp(-distance_inside / sigma)\n        w_out = 1.0 + beta * np.exp(-distance_outside / sigma)\n        weights[index] = np.where(mask, w_in, w_out)\n    tensor = torch.from_numpy(weights).to(target.device).unsqueeze(1)\n    return tensor\nclass BoundaryAwareLoss(nn.Module):\n    def __init__(\n        self,\n        from_logits: bool = True,\n        bce_weight: float = 1.0,\n        dice_weight: float = 1.0,\n        boundary_weight: float = 0.5,\n        beta: float = 5.0,\n    ):\n        super().__init__()\n        self.from_logits = from_logits\n        self.bce_weight = bce_weight\n        self.dice_weight = dice_weight\n        self.boundary_weight = boundary_weight\n        self.beta = beta\n        self.dice = DiceLoss(from_logits=from_logits)\n    def forward(self, predicted: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        if self.from_logits:\n            bce = F.binary_cross_entropy_with_logits(predicted, target)\n        else:\n            bce = F.binary_cross_entropy(predicted, target)\n        weights = distance_weight_map(target, beta=self.beta)\n        if self.from_logits:\n            boundary = F.binary_cross_entropy_with_logits(predicted, target, weight=weights)\n        else:\n            boundary = F.binary_cross_entropy(predicted, target, weight=weights)\n        return (\n            self.bce_weight * bce\n            + self.dice_weight * self.dice(predicted, target)\n            + self.boundary_weight * boundary\n        )\nclass DeepSupervisionLoss(nn.Module):\n    def __init__(self, base_loss: nn.Module, aux_weights: tuple[float, ...] = (0.4, 0.2, 0.1)):\n        super().__init__()\n        self.base_loss = base_loss\n        self.aux_weights = aux_weights\n    def forward(self, outputs, target: torch.Tensor) -> torch.Tensor:\n        if isinstance(outputs, (list, tuple)):\n            primary, *aux = outputs\n        else:\n            return self.base_loss(outputs, target)\n        total = self.base_loss(primary, target)\n        for weight, aux_output in zip(self.aux_weights, aux):\n            total = total + weight * self.base_loss(aux_output, target)\n        return total\ndef build_loss(\n    name: str = \"bce_dice\",\n    *,\n    from_logits: bool = True,\n    deep_supervision: bool = False,\n    aux_weights: tuple[float, ...] = (0.4, 0.2, 0.1),\n) -> nn.Module:\n    name = name.lower()\n    if name == \"dice\":\n        base: nn.Module = DiceLoss(from_logits=from_logits)\n    elif name == \"bce\":\n        base = nn.BCEWithLogitsLoss() if from_logits else nn.BCELoss()\n    elif name == \"bce_dice\":\n        base = BCEDiceLoss(from_logits=from_logits)\n    elif name == \"focal_tversky\":\n        base = FocalTverskyLoss(from_logits=from_logits)\n    elif name == \"boundary_aware\":\n        base = BoundaryAwareLoss(from_logits=from_logits)\n    else:\n        raise ValueError(f\"unknown loss '{name}'\")\n    if deep_supervision:\n        return DeepSupervisionLoss(base, aux_weights=aux_weights)\n    return base\ndef dice_score_from_logits(logits: torch.Tensor, target: torch.Tensor, threshold: float = 0.5) -> torch.Tensor:\n    probabilities = torch.sigmoid(logits)\n    predicted = (probabilities > threshold).float()\n    dims = tuple(range(1, predicted.ndim))\n    intersection = (predicted * target).sum(dim=dims)\n    denom = predicted.sum(dim=dims) + target.sum(dim=dims)\n    return ((2 * intersection + 1e-6) / (denom + 1e-6)).mean()\n"}, "backend/app/ml/gradcam.py": {"sha256": "9135125bedd246c7d1721d62697c981c7e6cd10fd89db404ee41a6b77001e217", "source": "from __future__ import annotations\nimport logging\nfrom dataclasses import dataclass\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom ..utils.images import resize_array\nlogger = logging.getLogger(__name__)\n@dataclass\nclass GradCAMResult:\n    heatmap: np.ndarray\n    model_id: str\n    target_layer: str\n    target_score: str\n    method: str\n    explanation: str\n    activation_shape: list[int]\n    warnings: list[str]\n    def summary(self) -> dict:\n        return {\n            \"model_id\": self.model_id,\n            \"target_layer\": self.target_layer,\n            \"target_score\": self.target_score,\n            \"method\": self.method,\n            \"explanation\": self.explanation,\n            \"activation_shape\": self.activation_shape,\n            \"heatmap_shape\": list(self.heatmap.shape),\n            \"heatmap_mean\": float(self.heatmap.mean()),\n            \"heatmap_max\": float(self.heatmap.max()),\n            \"warnings\": self.warnings,\n        }\nclass ActivationCapture:\n    def __init__(self, module: torch.nn.Module, layer_name: str):\n        self.module = module\n        self.layer_name = layer_name\n        self.activation: torch.Tensor | None = None\n        self._handle = None\n    def _resolve(self) -> torch.nn.Module:\n        target = self.module\n        for part in self.layer_name.split(\".\"):\n            if part.isdigit() and isinstance(target, (torch.nn.Sequential, torch.nn.ModuleList)):\n                target = target[int(part)]\n            elif hasattr(target, part):\n                target = getattr(target, part)\n            else:\n                raise ValueError(f\"target layer '{self.layer_name}' not found in the model\")\n        return target\n    def __enter__(self):\n        layer = self._resolve()\n        self._handle = layer.register_forward_hook(self._hook)\n        return self\n    def _hook(self, _module, _inputs, output):\n        self.activation = output\n    def __exit__(self, *exc_info):\n        if self._handle is not None:\n            self._handle.remove()\n            self._handle = None\ndef compute_gradcam(\n    module: torch.nn.Module,\n    tensor: torch.Tensor,\n    *,\n    model_id: str,\n    target_layer: str,\n    region_of_interest: np.ndarray | None = None,\n    output_is_logit: bool = True,\n    relu: bool = True,\n) -> GradCAMResult:\n    warnings: list[str] = []\n    module.eval()\n    tensor = tensor.detach().clone().requires_grad_(True)\n    with ActivationCapture(module, target_layer) as capture:\n        output = module(tensor)\n    logits = output[0] if isinstance(output, (list, tuple)) else output\n    activation = capture.activation\n    if activation is None:\n        raise RuntimeError(f\"no activation captured at layer '{target_layer}'\")\n    probabilities = torch.sigmoid(logits) if output_is_logit else logits\n    if region_of_interest is not None:\n        roi = torch.as_tensor(np.asarray(region_of_interest).astype(np.float32))\n        if roi.shape[-2:] != probabilities.shape[-2:]:\n            roi = F.interpolate(\n                roi[None, None], size=probabilities.shape[-2:], mode=\"nearest\"\n            )[0, 0]\n        roi = (roi > 0.5).to(probabilities.dtype)\n        denominator = roi.sum().clamp(min=1.0)\n        score = (probabilities * roi).sum() / denominator\n        target_score_name = \"mean predicted probability inside the predicted region\"\n        if float(roi.sum()) == 0:\n            warnings.append(\n                \"The predicted region is empty, so the explanation falls back to the whole image.\"\n            )\n            score = probabilities.mean()\n            target_score_name = \"mean predicted probability over the whole image\"\n    else:\n        score = probabilities.mean()\n        target_score_name = \"mean predicted probability over the whole image\"\n    gradient = torch.autograd.grad(\n        outputs=score,\n        inputs=activation,\n        retain_graph=False,\n        create_graph=False,\n        allow_unused=False,\n    )[0]\n    if gradient is None:\n        raise RuntimeError(\"Grad-CAM gradient is None; the target layer is not on the forward path\")\n    weights = gradient.mean(dim=(2, 3), keepdim=True)\n    cam = (weights * activation).sum(dim=1)\n    if relu:\n        cam = torch.relu(cam)\n    cam = cam[0].detach().cpu().numpy().astype(np.float32)\n    cam = cam - float(cam.min())\n    maximum = float(cam.max())\n    if maximum > 1e-8:\n        cam = cam / maximum\n    else:\n        warnings.append(\"The Grad-CAM map is constant; the model output does not depend on this layer.\")\n        cam = np.zeros_like(cam)\n    heatmap = resize_array(cam, tuple(probabilities.shape[-2:]))\n    return GradCAMResult(\n        heatmap=heatmap.astype(np.float32),\n        model_id=model_id,\n        target_layer=target_layer,\n        target_score=target_score_name,\n        method=\"Grad-CAM (gradient-weighted activation mapping, Selvaraju et al. 2017)\",\n        explanation=(\n            \"Warm colours mark image regions whose activation contributed most strongly to the \"\n            \"model's own predicted region for this case. The map explains the model's behaviour on \"\n            \"this input; it does not establish that a tumour is present, and it is not a \"\n            \"radiological finding.\"\n        ),\n        activation_shape=list(activation.shape),\n        warnings=warnings,\n    )\ndef layer_magnitude_map(module: torch.nn.Module, tensor: torch.Tensor, layer_name: str) -> np.ndarray:\n    with ActivationCapture(module, layer_name) as capture:\n        with torch.no_grad():\n            module(tensor)\n    activation = capture.activation\n    if activation is None:\n        raise RuntimeError(f\"no activation captured at layer '{layer_name}'\")\n    magnitude = activation.abs().mean(dim=1)[0].detach().cpu().numpy()\n    maximum = float(magnitude.max())\n    if maximum > 1e-8:\n        magnitude = magnitude / maximum\n    return magnitude.astype(np.float32)\ndef permutation_importance(\n    predict_proba,\n    features: np.ndarray,\n    feature_names: list[str],\n    *,\n    target_index: int,\n    n_repeats: int = 8,\n    seed: int = 42,\n) -> dict:\n    rng = np.random.default_rng(seed)\n    features = np.asarray(features, dtype=np.float32)\n    baseline = float(np.asarray(predict_proba(features.reshape(1, -1))).ravel()[target_index])\n    importances = []\n    for index, name in enumerate(feature_names):\n        deltas = []\n        for _ in range(n_repeats):\n            perturbed = features.copy()\n            perturbed[index] = float(rng.permutation(perturbed)[0]) if perturbed.size > 1 else perturbed[index]\n            score = float(np.asarray(predict_proba(perturbed.reshape(1, -1))).ravel()[target_index])\n            deltas.append(baseline - score)\n        importances.append(\n            {\n                \"feature\": name,\n                \"importance\": float(np.mean(deltas)),\n                \"std\": float(np.std(deltas)),\n                \"direction\": \"increases score\" if np.mean(deltas) > 0 else \"decreases score\",\n            }\n        )\n    importances.sort(key=lambda item: abs(item[\"importance\"]), reverse=True)\n    return {\n        \"method\": \"permutation importance (single case)\",\n        \"baseline_score\": baseline,\n        \"importances\": importances,\n        \"note\": (\n            \"Permutation importance measures how much the model's score for this case changes when \"\n            \"one feature value is shuffled. It describes the model, not the biology.\"\n        ),\n    }\n"}, "backend/app/ml/radiomics_native.py": {"sha256": "960db9b4ba72236ad101fb30a089b14580c3860e3e3b57a4ce2f6a1d0b8a4428", "source": "from __future__ import annotations\nimport logging\nimport math\nfrom dataclasses import dataclass\nfrom typing import Iterable\nimport numpy as np\nlogger = logging.getLogger(__name__)\nEPS = np.finfo(np.float32).eps\nFEATURE_CATEGORIES = {\n    \"shape\": \"Shape and size descriptors of the segmented region\",\n    \"firstorder\": \"First-order intensity statistics inside the segmented region\",\n    \"glcm\": \"Grey-level co-occurrence matrix texture features\",\n    \"glrlm\": \"Grey-level run-length matrix texture features\",\n    \"diagnostics\": \"Acquisition and segmentation diagnostics and global histogram descriptors\",\n}\nGLCM_DIRECTIONS_2D = ((0, 1), (1, 0), (1, 1), (-1, 1))\nGLRLM_DIRECTIONS_2D = ((0, 1), (1, 0), (1, 1), (-1, 1))\nGLCM_DIRECTIONS_3D = (\n    (0, 1, 0),\n    (1, 0, 0),\n    (0, 0, 1),\n    (1, 1, 0),\n    (1, -1, 0),\n    (1, 0, 1),\n    (1, 0, -1),\n    (0, 1, 1),\n    (0, 1, -1),\n    (1, 1, 1),\n    (1, 1, -1),\n    (1, -1, 1),\n    (1, -1, -1),\n)\nGLRLM_DIRECTIONS_3D = (\n    (0, 1, 0),\n    (1, 0, 0),\n    (0, 0, 1),\n    (1, 1, 0),\n    (1, -1, 0),\n    (1, 0, 1),\n    (1, 0, -1),\n    (0, 1, 1),\n    (0, 1, -1),\n    (1, 1, 1),\n    (1, 1, -1),\n    (1, -1, 1),\n    (1, -1, -1),\n)\n@dataclass\nclass DiscretisationSettings:\n    method: str = \"bin_count\"\n    bin_count: int = 32\n    bin_width: float = 25.0\n    def as_dict(self) -> dict:\n        return {\n            \"method\": self.method,\n            \"bin_count\": self.bin_count,\n            \"bin_width\": self.bin_width,\n            \"note\": (\n                \"Fixed bin count discretisation rescales every case to the same number of \"\n                \"bins, which makes texture values comparable across cases but dependent on \"\n                \"the intensity range. Fixed bin width is reported in the same intensity unit \"\n                \"as the image.\"\n            ),\n        }\ndef discretise(\n    intensities: np.ndarray, settings: DiscretisationSettings\n) -> tuple[np.ndarray, dict]:\n    values = np.asarray(intensities, dtype=np.float64).ravel()\n    if values.size == 0:\n        return np.zeros(0, dtype=np.int32), {\"levels\": 0}\n    if settings.method == \"bin_width\" and settings.bin_width > 0:\n        low = values.min()\n        levels = np.floor((values - low) / settings.bin_width).astype(np.int32) + 1\n        levels = np.clip(levels, 1, 256)\n        info = {\n            \"levels\": int(levels.max()),\n            \"method\": \"bin_width\",\n            \"bin_width\": settings.bin_width,\n            \"range\": [float(low), float(values.max())],\n        }\n        return levels, info\n    n_bins = max(2, int(settings.bin_count))\n    low, high = float(values.min()), float(values.max())\n    if high - low < 1e-12:\n        return np.ones_like(values, dtype=np.int32), {\n            \"levels\": 1,\n            \"method\": \"bin_count\",\n            \"bin_count\": n_bins,\n            \"range\": [low, high],\n        }\n    edges = np.linspace(low, high + 1e-9, n_bins + 1)\n    levels = np.digitize(values, edges, right=False).astype(np.int32)\n    levels = np.clip(levels, 1, n_bins)\n    return levels, {\n        \"levels\": int(levels.max()),\n        \"method\": \"bin_count\",\n        \"bin_count\": n_bins,\n        \"range\": [low, high],\n    }\ndef _perimeter_2d(mask: np.ndarray, spacing: tuple[float, float] = (1.0, 1.0)) -> float:\n    mask = np.asarray(mask).astype(bool)\n    if mask.ndim != 2:\n        raise ValueError(\"_perimeter_2d expects a 2-D mask\")\n    dy, dx = float(spacing[0]), float(spacing[1])\n    rows, cols = mask.shape\n    samples = np.pad(mask, 1)\n    east_west = samples[:, :-1] != samples[:, 1:]\n    north_south = samples[:-1, :] != samples[1:, :]\n    top = east_west[:-1, : cols + 1]\n    bottom = east_west[1:, : cols + 1]\n    left = north_south[:, : cols + 1]\n    right = north_south[:, 1 : cols + 2]\n    count = top.astype(np.int8) + bottom + left + right\n    vertical_segment = top & bottom\n    horizontal_segment = left & right\n    diagonal_segment = (count == 2) & ~(vertical_segment | horizontal_segment)\n    four_crossings = (count == 4) | (count % 2 == 1)\n    diagonal_length = math.hypot(dx / 2.0, dy / 2.0)\n    return float(\n        vertical_segment.sum() * dy\n        + horizontal_segment.sum() * dx\n        + diagonal_segment.sum() * diagonal_length\n        + four_crossings.sum() * 2.0 * diagonal_length\n    )\ndef shape_features_2d(mask: np.ndarray, spacing: tuple[float, float] | None = None) -> dict:\n    mask = np.asarray(mask).astype(bool)\n    area_px = float(mask.sum())\n    features: dict[str, float | None] = {}\n    if spacing is None:\n        features[\"pixel_area\"] = area_px\n        features[\"perimeter_px\"] = _perimeter_2d(mask, (1.0, 1.0))\n        features[\"equivalent_diameter_px\"] = float(2 * math.sqrt(area_px / math.pi)) if area_px else 0.0\n        features[\"circularity\"] = (\n            float(4 * math.pi * area_px / (features[\"perimeter_px\"] ** 2))\n            if features[\"perimeter_px\"] > 1e-9\n            else None\n        )\n        features[\"area_mm2\"] = None\n        features[\"perimeter_mm\"] = None\n        features[\"equivalent_diameter_mm\"] = None\n        features[\"units\"] = \"pixels\"\n    else:\n        area_mm2 = area_px * spacing[0] * spacing[1]\n        perimeter = _perimeter_2d(mask, spacing)\n        features[\"pixel_area\"] = area_px\n        features[\"perimeter_px\"] = None\n        features[\"equivalent_diameter_px\"] = None\n        features[\"area_mm2\"] = area_mm2\n        features[\"perimeter_mm\"] = perimeter\n        features[\"equivalent_diameter_mm\"] = float(2 * math.sqrt(area_mm2 / math.pi)) if area_mm2 else 0.0\n        features[\"circularity\"] = (\n            float(4 * math.pi * area_mm2 / (perimeter**2)) if perimeter > 1e-9 else None\n        )\n        features[\"units\"] = \"mm2\"\n    features[\"elongation\"] = _elongation(mask)\n    features[\"flatness\"] = None\n    extent = _bbox_extent(mask)\n    features[\"bbox_height_px\"] = extent[0] if extent else None\n    features[\"bbox_width_px\"] = extent[1] if extent else None\n    features[\"sphericity\"] = None\n    return features\ndef _bbox_extent(mask: np.ndarray) -> list[int] | None:\n    coordinates = np.argwhere(mask)\n    if coordinates.size == 0:\n        return None\n    return [int(coordinates[:, axis].max() - coordinates[:, axis].min() + 1) for axis in range(coordinates.shape[1])]\ndef shape_features_3d(mask: np.ndarray, spacing: tuple[float, float, float] | None = None) -> dict:\n    mask = np.asarray(mask).astype(bool)\n    voxels = float(mask.sum())\n    spacing = spacing or (1.0, 1.0, 1.0)\n    volume = voxels * float(np.prod(spacing))\n    axes = _principal_axes(mask, spacing)\n    surface = _surface_area(mask, spacing)\n    features = {\n        \"voxel_count\": voxels,\n        \"volume_mm3\": volume,\n        \"surface_area_mm2\": surface,\n        \"major_axis_mm\": axes[0] if axes else None,\n        \"minor_axis_mm\": axes[1] if len(axes) > 1 else None,\n        \"least_axis_mm\": axes[2] if len(axes) > 2 else None,\n        \"elongation\": (axes[1] / axes[0]) if axes and axes[0] > 0 else None,\n        \"flatness\": (axes[2] / axes[0]) if axes and len(axes) > 2 and axes[0] > 0 else None,\n        \"sphericity\": (\n            float((36 * math.pi * max(volume, 0.0) ** 2) ** (1 / 3) / surface)\n            if surface > 1e-9 and volume > 0\n            else None\n        ),\n        \"units\": \"mm3\",\n    }\n    features[\"equivalent_diameter_mm\"] = (\n        float((6 * volume / math.pi) ** (1 / 3)) if volume > 0 else None\n    )\n    return features\ndef _principal_axes(mask: np.ndarray, spacing: tuple[float, ...]) -> list[float]:\n    coordinates = np.argwhere(mask).astype(np.float64)\n    if coordinates.shape[0] < 3:\n        return []\n    coordinates *= np.asarray(spacing, dtype=np.float64)\n    centred = coordinates - coordinates.mean(axis=0)\n    covariance = np.cov(centred, rowvar=False)\n    try:\n        eigenvalues = np.linalg.eigvalsh(covariance)\n    except np.linalg.LinAlgError:\n        return []\n    eigenvalues = np.clip(np.sort(eigenvalues)[::-1], 0, None)\n    return [float(4 * math.sqrt(value)) for value in eigenvalues]\ndef _surface_area(mask: np.ndarray, spacing: tuple[float, ...]) -> float:\n    padded = np.pad(mask.astype(np.uint8), 1)\n    centre = padded[1:-1, 1:-1, 1:-1].astype(bool)\n    area = 0.0\n    for axis, step in enumerate(spacing):\n        for shift in (-1, 1):\n            shifted = np.roll(padded, shift, axis=axis)\n            if shift == 1:\n                sl = [slice(None)] * 3\n                sl[axis] = 0\n                shifted[tuple(sl)] = 0\n            else:\n                sl = [slice(None)] * 3\n                sl[axis] = -1\n                shifted[tuple(sl)] = 0\n            neighbour = shifted[1:-1, 1:-1, 1:-1].astype(bool)\n            faces = centre & ~neighbour\n            other = [spacing[i] for i in range(3) if i != axis]\n            area += float(faces.sum()) * float(np.prod(other))\n    return area\ndef _elongation(mask: np.ndarray) -> float | None:\n    coordinates = np.argwhere(mask).astype(np.float64)\n    if coordinates.shape[0] < 3:\n        return None\n    centred = coordinates - coordinates.mean(axis=0)\n    covariance = np.cov(centred, rowvar=False)\n    try:\n        eigenvalues = np.sort(np.clip(np.linalg.eigvalsh(covariance), 0, None))[::-1]\n    except np.linalg.LinAlgError:\n        return None\n    if eigenvalues[0] <= 0:\n        return None\n    return float(math.sqrt(eigenvalues[1] / eigenvalues[0]))\ndef first_order_features(values: np.ndarray) -> dict:\n    values = np.asarray(values, dtype=np.float64).ravel()\n    if values.size == 0:\n        return {}\n    mean = float(values.mean())\n    variance = float(values.var())\n    std = math.sqrt(variance)\n    sorted_values = np.sort(values)\n    percentiles = np.percentile(sorted_values, [10, 25, 50, 75, 90])\n    centred = values - mean\n    skewness = float((centred**3).mean() / (std**3)) if std > 1e-12 else 0.0\n    kurtosis = float((centred**4).mean() / (std**4) - 3.0) if std > 1e-12 else 0.0\n    return {\n        \"mean\": mean,\n        \"median\": float(np.median(sorted_values)),\n        \"minimum\": float(sorted_values[0]),\n        \"maximum\": float(sorted_values[-1]),\n        \"range\": float(sorted_values[-1] - sorted_values[0]),\n        \"variance\": variance,\n        \"standard_deviation\": std,\n        \"skewness\": skewness,\n        \"kurtosis\": kurtosis,\n        \"p10\": float(percentiles[0]),\n        \"p25\": float(percentiles[1]),\n        \"p50\": float(percentiles[2]),\n        \"p75\": float(percentiles[3]),\n        \"p90\": float(percentiles[4]),\n        \"interquartile_range\": float(percentiles[3] - percentiles[1]),\n        \"mean_absolute_deviation\": float(np.abs(centred).mean()),\n        \"root_mean_squared\": float(np.sqrt((values**2).mean())),\n        \"energy\": float((values**2).sum()),\n        \"pixel_count\": int(values.size),\n    }\ndef histogram_descriptors(probabilities: np.ndarray) -> dict:\n    values = np.asarray(probabilities, dtype=np.float64).ravel()\n    values = values[values > 0]\n    if values.size == 0:\n        return {\"entropy\": None, \"uniformity\": None, \"energy\": None, \"maximum_probability\": None}\n    return {\n        \"entropy\": float(-(values * np.log2(values)).sum()),\n        \"uniformity\": float((values**2).sum()),\n        \"energy\": float(np.sqrt((values**2).sum())),\n        \"maximum_probability\": float(values.max()),\n    }\ndef _offsets_inside(shape: tuple[int, ...], direction: tuple[int, ...]) -> tuple[slice, slice]:\n    src = []\n    dst = []\n    for size, step in zip(shape, direction):\n        if step >= 0:\n            src.append(slice(0, size - step))\n            dst.append(slice(step, size))\n        else:\n            src.append(slice(-step, size))\n            dst.append(slice(0, size + step))\n    return tuple(src), tuple(dst)\ndef glcm_matrix(levels: np.ndarray, n_levels: int, direction: tuple[int, ...]) -> np.ndarray:\n    src, dst = _offsets_inside(levels.shape, direction)\n    a = levels[src].astype(np.int32)\n    b = levels[dst].astype(np.int32)\n    valid = (a > 0) & (b > 0)\n    if not valid.any():\n        return np.zeros((n_levels, n_levels), dtype=np.float64)\n    index = (a[valid] - 1) * n_levels + (b[valid] - 1)\n    counts = np.bincount(index, minlength=n_levels * n_levels).astype(np.float64)\n    matrix = counts.reshape(n_levels, n_levels)\n    matrix = matrix + matrix.T\n    total = matrix.sum()\n    return matrix / total if total > 0 else matrix\ndef glcm_features(levels: np.ndarray, n_levels: int, directions: Iterable[tuple[int, ...]], spacing=None) -> dict:\n    matrices = [glcm_matrix(levels, n_levels, direction) for direction in directions]\n    matrices = [m for m in matrices if m.sum() > 0]\n    if not matrices:\n        return {}\n    i = np.arange(1, n_levels + 1, dtype=np.float64)[:, None]\n    j = np.arange(1, n_levels + 1, dtype=np.float64)[None, :]\n    contrast = []\n    dissimilarity = []\n    homogeneity = []\n    asm = []\n    energy = []\n    correlation = []\n    entropy = []\n    mean_i = []\n    variance_i = []\n    variance_j = []\n    for matrix in matrices:\n        p = matrix\n        diff = (i - j) ** 2\n        contrast.append(float((p * diff).sum()))\n        dissimilarity.append(float((p * np.abs(i - j)).sum()))\n        homogeneity.append(float((p / (1 + diff)).sum()))\n        asm.append(float((p**2).sum()))\n        energy.append(float(np.sqrt((p**2).sum())))\n        mu_i = float((p * i).sum())\n        mu_j = float((p * j).sum())\n        var_i = float((p * (i - mu_i) ** 2).sum())\n        var_j = float((p * (j - mu_j) ** 2).sum())\n        denom = math.sqrt(var_i * var_j)\n        correlation.append(float(((p * (i - mu_i) * (j - mu_j)).sum() / denom)) if denom > 1e-12 else 0.0)\n        entropy.append(float(-(p[p > 0] * np.log2(p[p > 0])).sum()))\n        mean_i.append(mu_i)\n        variance_i.append(var_i)\n        variance_j.append(var_j)\n    average = lambda values: float(np.mean(values)) if values else None\n    return {\n        \"contrast\": average(contrast),\n        \"dissimilarity\": average(dissimilarity),\n        \"homogeneity\": average(homogeneity),\n        \"asm\": average(asm),\n        \"energy\": average(energy),\n        \"correlation\": average(correlation),\n        \"entropy\": average(entropy),\n        \"mean_intensity\": average(mean_i),\n        \"variance\": average(variance_i),\n        \"n_directions\": len(matrices),\n    }\ndef _run_length_matrix(\n    levels: np.ndarray, n_levels: int, direction: tuple[int, ...]\n) -> np.ndarray:\n    shape = levels.shape\n    max_run = max(shape)\n    matrix = np.zeros((n_levels, max_run), dtype=np.float64)\n    steps = np.asarray(direction, dtype=np.int64)\n    for start in np.argwhere(levels > 0):\n        level = int(levels[tuple(start)])\n        predecessor = start - steps\n        if all(0 <= predecessor[k] < shape[k] for k in range(levels.ndim)):\n            if int(levels[tuple(predecessor)]) == level:\n                continue\n        length = 0\n        current = start.copy()\n        while all(0 <= current[k] < shape[k] for k in range(levels.ndim)):\n            if int(levels[tuple(current)]) != level:\n                break\n            length += 1\n            current = current + steps\n        if length:\n            matrix[level - 1, min(length, max_run) - 1] += 1\n    return matrix\ndef glrlm_features(levels: np.ndarray, n_levels: int, directions: Iterable[tuple[int, ...]]) -> dict:\n    max_run = max(levels.shape)\n    matrices = []\n    for direction in directions:\n        matrix = _run_length_matrix(levels, n_levels, direction)\n        total = matrix.sum()\n        if total > 0:\n            matrices.append(matrix / total)\n    if not matrices:\n        return {}\n    p = np.mean(matrices, axis=0)\n    p = p / p.sum()\n    j = np.arange(1, max_run + 1, dtype=np.float64)[None, :]\n    i = np.arange(1, n_levels + 1, dtype=np.float64)[:, None]\n    n_runs = p.sum()\n    sre = float((p / (j**2)).sum())\n    lre = float((p * (j**2)).sum())\n    gln = float((p.sum(axis=1) ** 2).sum())\n    rln = float((p.sum(axis=0) ** 2).sum())\n    rp = float(n_runs / p.size)\n    mean_level = float((p * i).sum())\n    return {\n        \"short_run_emphasis\": sre,\n        \"long_run_emphasis\": lre,\n        \"grey_level_non_uniformity\": gln,\n        \"run_length_non_uniformity\": rln,\n        \"run_percentage\": rp,\n        \"grey_level_variance\": float((p * (i - mean_level) ** 2).sum()),\n        \"low_grey_level_run_emphasis\": float((p / (i**2)).sum()),\n        \"high_grey_level_run_emphasis\": float((p * (i**2)).sum()),\n        \"n_directions\": len(matrices),\n    }\ndef extract_features(\n    image: np.ndarray,\n    mask: np.ndarray,\n    *,\n    spacing: tuple[float, ...] | None = None,\n    settings: DiscretisationSettings | None = None,\n    include_texture: bool = True,\n) -> dict:\n    image = np.asarray(image, dtype=np.float32)\n    mask = np.asarray(mask).astype(bool)\n    if image.shape != mask.shape:\n        raise ValueError(f\"image shape {image.shape} != mask shape {mask.shape}\")\n    if not mask.any():\n        raise ValueError(\"The region of interest is empty; no features can be extracted.\")\n    settings = settings or DiscretisationSettings()\n    dimensionality = \"3D\" if image.ndim == 3 else \"2D\"\n    roi_values = image[mask]\n    n_voxels = int(mask.sum())\n    levels, discretisation_info = discretise(roi_values, settings)\n    n_levels = max(1, int(discretisation_info.get(\"levels\", 1)))\n    result: dict = {}\n    result[\"diagnostics_dimensionality\"] = dimensionality\n    result[\"diagnostics_roi_voxel_count\"] = n_voxels\n    result[\"diagnostics_image_mean\"] = float(image.mean())\n    result[\"diagnostics_image_min\"] = float(image.min())\n    result[\"diagnostics_image_max\"] = float(image.max())\n    result[\"diagnostics_roi_mean\"] = float(roi_values.mean())\n    result[\"diagnostics_roi_min\"] = float(roi_values.min())\n    result[\"diagnostics_roi_max\"] = float(roi_values.max())\n    result[\"diagnostics_intensity_range_in_roi\"] = float(roi_values.max() - roi_values.min())\n    result[\"diagnostics_discretisation_levels\"] = n_levels\n    result[\"diagnostics_discretisation_method\"] = discretisation_info.get(\"method\")\n    result[\"diagnostics_spacing\"] = list(spacing) if spacing else None\n    result[\"diagnostics_spacing_available\"] = bool(spacing)\n    shape = shape_features_3d(mask, spacing) if image.ndim == 3 else shape_features_2d(mask, spacing)\n    for key, value in shape.items():\n        result[f\"shape_{key}\"] = value\n    for key, value in first_order_features(roi_values).items():\n        result[f\"firstorder_{key}\"] = value\n    counts, _ = np.histogram(roi_values, bins=max(2, n_levels))\n    probabilities = counts.astype(np.float64) / max(counts.sum(), 1)\n    for key, value in histogram_descriptors(probabilities).items():\n        result[f\"diagnostics_{key}\"] = value\n    if include_texture:\n        if n_voxels < 4 * n_levels:\n            result[\"diagnostics_texture_warning\"] = (\n                f\"The ROI contains {n_voxels} pixels for {n_levels} grey levels \"\n                f\"({n_voxels / n_levels:.2f} pixels per level). Texture matrices with fewer than \"\n                \"four pixels per grey level are not reliable; consider a fixed bin width or a \"\n                \"coarser bin count.\"\n            )\n        levels_map = np.zeros(mask.shape, dtype=np.int32)\n        levels_map[mask] = levels\n        directions = GLCM_DIRECTIONS_2D if image.ndim == 2 else GLCM_DIRECTIONS_3D\n        for key, value in glcm_features(levels_map, n_levels, directions).items():\n            result[f\"glcm_{key}\"] = value\n        run_directions = GLRLM_DIRECTIONS_2D if image.ndim == 2 else GLRLM_DIRECTIONS_3D\n        for key, value in glrlm_features(levels_map, n_levels, run_directions).items():\n            result[f\"glrlm_{key}\"] = value\n    numeric = {\n        key: (None if value is None or isinstance(value, str) or isinstance(value, bool) else float(value))\n        for key, value in result.items()\n        if not key.startswith(\"diagnostics_\") or key in {\n            \"diagnostics_roi_voxel_count\",\n            \"diagnostics_image_mean\",\n            \"diagnostics_image_min\",\n            \"diagnostics_image_max\",\n            \"diagnostics_roi_mean\",\n            \"diagnostics_roi_min\",\n            \"diagnostics_roi_max\",\n            \"diagnostics_intensity_range_in_roi\",\n            \"diagnostics_discretisation_levels\",\n            \"diagnostics_entropy\",\n            \"diagnostics_uniformity\",\n            \"diagnostics_energy\",\n            \"diagnostics_maximum_probability\",\n        }\n    }\n    for key, value in numeric.items():\n        if value is not None and not np.isfinite(value):\n            result[key] = None\n    return result\ndef categorise(feature_name: str) -> str:\n    if feature_name.startswith(\"shape_\"):\n        return \"shape\"\n    if feature_name.startswith(\"firstorder_\"):\n        return \"firstorder\"\n    if feature_name.startswith(\"glcm_\"):\n        return \"glcm\"\n    if feature_name.startswith(\"glrlm_\"):\n        return \"glrlm\"\n    return \"diagnostics\"\n"}, "backend/app/services/evaluation_service.py": {"sha256": "07aca8df5316c79ed6ac8783b36ca4af84e6969da5b0b0397bd3950566636466", "source": "from __future__ import annotations\nimport logging\nimport time\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Iterable, Sequence\nimport numpy as np\nfrom ..utils.io import read_json, to_jsonable, utc_now\nfrom ..utils.metrics import (\n    bootstrap_ci,\n    classification_report,\n    expected_calibration_error,\n    hausdorff_distance_95,\n    reliability_curve,\n    roc_auc_binary,\n    average_precision_binary,\n    segmentation_metrics,\n)\nlogger = logging.getLogger(__name__)\nNOT_EVALUATED = \"Not evaluated\"\nRESULT_STATE_KEYS = {\n    \"evaluated\": \"Metrics were computed against labelled reference data.\",\n    \"not_evaluated\": (\n        \"No labelled evaluation data is available in this deployment, so no metric was computed. \"\n        \"The project documentation must not report an accuracy, Dice or AUC value until the \"\n        \"reserved test set has been evaluated with this same protocol.\"\n    ),\n}\n@dataclass\nclass EvaluationReport:\n    state: str\n    reason: str\n    dataset: dict = field(default_factory=dict)\n    segmentation: dict = field(default_factory=dict)\n    classification: dict = field(default_factory=dict)\n    runtime: dict = field(default_factory=dict)\n    generated_at: str = utc_now()\n    def to_dict(self) -> dict:\n        return to_jsonable(\n            {\n                \"state\": self.state,\n                \"reason\": self.reason,\n                \"dataset\": self.dataset,\n                \"segmentation\": self.segmentation,\n                \"classification\": self.classification,\n                \"runtime\": self.runtime,\n                \"generated_at\": self.generated_at,\n                \"state_description\": RESULT_STATE_KEYS.get(self.state, \"\"),\n            }\n        )\nclass EvaluationService:\n    def __init__(self, registry, config, segmentation=None, ensemble=None):\n        self.registry = registry\n        self.config = config\n        self.segmentation = segmentation\n        self.ensemble = ensemble\n        data_dir = Path(config.DATA_DIR)\n        self.results_path = data_dir / \"models\" / \"evaluation_results.json\"\n        self.model_selection_path = data_dir / \"models\" / \"model_selection.json\"\n    def stored_results(self) -> dict:\n        payload = read_json(self.results_path, default=None)\n        if payload:\n            payload.setdefault(\"state\", \"evaluated\")\n            payload.setdefault(\"source\", str(self.results_path))\n            return payload\n        return {\n            \"state\": \"not_evaluated\",\n            \"reason\": RESULT_STATE_KEYS[\"not_evaluated\"],\n            \"expected_artifact\": str(self.results_path),\n            \"produced_by\": \"scripts/evaluate_models.py\",\n        }\n    def model_selection(self) -> dict:\n        payload = read_json(self.model_selection_path, default=None)\n        if payload:\n            return payload\n        return {\n            \"state\": \"not_evaluated\",\n            \"reason\": (\n                \"No model-selection record exists. Model selection should be performed on the \"\n                \"validation split and recorded (with the random seed, the configuration and the \"\n                \"selection metric) before the reserved test set is touched.\"\n            ),\n            \"expected_artifact\": str(self.model_selection_path),\n            \"produced_by\": \"scripts/evaluate_models.py --select\",\n        }\n    def segmentation_summary(self) -> dict:\n        stored = self.stored_results()\n        if stored.get(\"state\") != \"evaluated\":\n            return {\n                \"state\": \"not_evaluated\",\n                \"reason\": stored.get(\"reason\"),\n                \"models\": [\n                    {\n                        \"model_id\": descriptor.model_id,\n                        \"display_name\": descriptor.display_name,\n                        \"metrics\": None,\n                        \"status\": \"not_evaluated\",\n                    }\n                    for descriptor in self.registry.descriptors.values()\n                    if descriptor.task == \"segmentation\"\n                ],\n            }\n        return {\n            \"state\": \"evaluated\",\n            \"dataset\": stored.get(\"dataset\", {}),\n            \"models\": stored.get(\"segmentation\", {}).get(\"models\", []),\n            \"ensemble\": stored.get(\"segmentation\", {}).get(\"ensemble\"),\n            \"ablation\": stored.get(\"segmentation\", {}).get(\"ablation\", []),\n        }\n    def classification_summary(self) -> dict:\n        stored = self.stored_results()\n        if stored.get(\"state\") != \"evaluated\" or \"classification\" not in stored:\n            return {\n                \"state\": \"not_evaluated\",\n                \"reason\": (\n                    \"No classification evaluation artefact was found. It is produced by \"\n                    \"scripts/train_classifier.py, which requires a labelled radiomics table.\"\n                ),\n            }\n        return stored[\"classification\"]\n    def runtime_summary(self) -> dict:\n        stored = self.stored_results()\n        if stored.get(\"runtime\"):\n            return stored[\"runtime\"]\n        if self.segmentation is None:\n            return {\"state\": \"not_measured\"}\n        entries = []\n        from .model_health_service import brainai_reference_phantom_4ch, reference_phantom_1ch\n        for model_id in self.registry.segmentation_model_ids():\n            loaded = self.registry.load_model(model_id)\n            if not loaded.is_ready:\n                entries.append(\n                    {\n                        \"model_id\": model_id,\n                        \"state\": \"not_measured\",\n                        \"reason\": loaded.message,\n                    }\n                )\n                continue\n            spec = loaded.descriptor.input_spec\n            if spec.channels == 4:\n                tensor = brainai_reference_phantom_4ch(spec.height)[None]\n            else:\n                tensor = reference_phantom_1ch(spec.height)[None]\n            import torch\n            started = time.perf_counter()\n            with torch.no_grad():\n                loaded.module(torch.from_numpy(tensor).to(loaded.device))\n            elapsed = time.perf_counter() - started\n            entries.append(\n                {\n                    \"model_id\": model_id,\n                    \"state\": \"measured\",\n                    \"device\": str(loaded.device),\n                    \"input_shape\": list(tensor.shape),\n                    \"parameter_count\": loaded.parameter_count,\n                    \"seconds_per_image\": round(elapsed, 4),\n                    \"images_per_second\": round(1.0 / elapsed, 3) if elapsed > 0 else None,\n                    \"note\": (\n                        \"Measured on this machine with a single reference input; it is an inference \"\n                        \"cost, not a training cost.\"\n                    ),\n                }\n            )\n        return {\"state\": \"measured\", \"generated_at\": utc_now(), \"models\": entries}\n    @staticmethod\n    def evaluate_segmentation_predictions(\n        predictions: dict[str, Sequence[np.ndarray]],\n        references: Sequence[np.ndarray],\n        *,\n        threshold: float = 0.5,\n        spacing: Sequence[float] | None = None,\n        case_ids: Sequence[str] | None = None,\n    ) -> dict:\n        n_cases = len(references)\n        if n_cases == 0:\n            return {\"state\": \"not_evaluated\", \"reason\": \"No labelled cases were supplied.\"}\n        model_reports = {}\n        for model_id, maps in predictions.items():\n            if len(maps) != n_cases:\n                raise ValueError(\n                    f\"{model_id}: {len(maps)} predictions for {n_cases} references\"\n                )\n            per_case = []\n            for index, (probability, reference) in enumerate(zip(maps, references)):\n                metrics = segmentation_metrics(probability, reference, threshold)\n                metrics[\"hausdorff_95\"] = hausdorff_distance_95(\n                    np.asarray(probability) > threshold, np.asarray(reference) > 0.5, spacing\n                )\n                metrics[\"case_id\"] = (\n                    str(case_ids[index]) if case_ids is not None else f\"case-{index:04d}\"\n                )\n                metrics[\"reference_empty\"] = bool(not np.asarray(reference).astype(bool).any())\n                per_case.append(metrics)\n            model_reports[model_id] = {\n                \"model_id\": model_id,\n                \"threshold\": threshold,\n                \"n_cases\": n_cases,\n                \"per_case\": per_case,\n                \"aggregate\": _aggregate_segmentation(per_case, n_cases),\n            }\n        return {\n            \"state\": \"evaluated\",\n            \"threshold\": threshold,\n            \"n_cases\": n_cases,\n            \"models\": model_reports,\n            \"comparison_table\": _comparison_table(model_reports),\n            \"generated_at\": utc_now(),\n        }\n    @staticmethod\n    def evaluate_classification(\n        y_true: Sequence[int],\n        y_pred: Sequence[int],\n        probabilities: Sequence[Sequence[float]],\n        *,\n        n_classes: int,\n        label_mapping: dict | None = None,\n    ) -> dict:\n        if len(y_true) == 0:\n            return {\"state\": \"not_evaluated\", \"reason\": \"No labelled samples were supplied.\"}\n        report = classification_report(y_true, y_pred, n_classes)\n        report[\"label_mapping\"] = label_mapping or {}\n        report[\"state\"] = \"evaluated\"\n        if n_classes == 2:\n            positive_scores = [p[1] if len(p) > 1 else p[0] for p in probabilities]\n            report[\"roc_auc\"] = roc_auc_binary(y_true, positive_scores)\n            report[\"pr_auc\"] = average_precision_binary(y_true, positive_scores)\n            report[\"expected_calibration_error\"] = expected_calibration_error(\n                y_true, positive_scores\n            )\n            report[\"reliability_curve\"] = reliability_curve(y_true, positive_scores)\n        else:\n            report[\"roc_auc\"] = _macro_roc_auc(y_true, probabilities, n_classes)\n            report[\"pr_auc\"] = None\n            report[\"expected_calibration_error\"] = None\n            report[\"reliability_curve\"] = []\n        return report\ndef _aggregate_segmentation(per_case: list[dict], n_cases: int) -> dict:\n    keys = [\n        \"dice\",\n        \"iou\",\n        \"precision\",\n        \"recall\",\n        \"sensitivity\",\n        \"specificity\",\n        \"accuracy\",\n        \"hausdorff_95\",\n    ]\n    aggregate = {\"n_cases\": n_cases}\n    for key in keys:\n        values = [case.get(key) for case in per_case]\n        aggregate[key] = bootstrap_ci(values)\n    empty_references = sum(1 for case in per_case if case.get(\"reference_empty\"))\n    aggregate[\"cases_with_empty_reference\"] = empty_references\n    aggregate[\"note\"] = (\n        \"Dice and IoU are defined as 1.0 when both the prediction and the reference are empty. \"\n        f\"{empty_references} of {n_cases} reference masks are empty; report this explicitly when \"\n        \"comparing against published results.\"\n    )\n    return aggregate\ndef _comparison_table(model_reports: dict[str, dict]) -> list[dict]:\n    rows = []\n    for model_id, report in model_reports.items():\n        aggregate = report[\"aggregate\"]\n        rows.append(\n            {\n                \"model_id\": model_id,\n                \"n_cases\": aggregate[\"n_cases\"],\n                \"dice_mean\": (aggregate.get(\"dice\") or {}).get(\"mean\"),\n                \"dice_ci\": [\n                    (aggregate.get(\"dice\") or {}).get(\"ci_lower\"),\n                    (aggregate.get(\"dice\") or {}).get(\"ci_upper\"),\n                ],\n                \"iou_mean\": (aggregate.get(\"iou\") or {}).get(\"mean\"),\n                \"precision_mean\": (aggregate.get(\"precision\") or {}).get(\"mean\"),\n                \"recall_mean\": (aggregate.get(\"recall\") or {}).get(\"mean\"),\n                \"specificity_mean\": (aggregate.get(\"specificity\") or {}).get(\"mean\"),\n                \"hausdorff_95_mean\": (aggregate.get(\"hausdorff_95\") or {}).get(\"mean\"),\n            }\n        )\n    rows.sort(key=lambda row: (row[\"dice_mean\"] is None, -(row[\"dice_mean\"] or 0)))\n    for position, row in enumerate(rows, start=1):\n        row[\"rank\"] = position\n    return rows\ndef _macro_roc_auc(y_true: Sequence[int], probabilities: Sequence[Sequence[float]], n_classes: int):\n    aucs = []\n    for class_index in range(n_classes):\n        binary_truth = [1 if int(label) == class_index else 0 for label in y_true]\n        scores = [row[class_index] for row in probabilities]\n        value = roc_auc_binary(binary_truth, scores)\n        if value is not None:\n            aucs.append(value)\n    return float(np.mean(aucs)) if aucs else None\n"}, "backend/app/services/radiomics_service.py": {"sha256": "73040758e30eb969bd3bdefc6ce32d50df9e27da796d873fb0703a9e0b04b7b9", "source": "from __future__ import annotations\nimport logging\nfrom dataclasses import dataclass, field\nfrom typing import Any\nimport numpy as np\nfrom ..ml.radiomics_native import (\n    FEATURE_CATEGORIES,\n    DiscretisationSettings,\n    categorise,\n    extract_features,\n)\nfrom ..utils.io import to_jsonable\nlogger = logging.getLogger(__name__)\nFAMILY_ORDER = (\"shape\", \"firstorder\", \"glcm\", \"glrlm\", \"diagnostics\")\nFEATURE_DESCRIPTIONS = {\n    \"shape_pixel_area\": \"Number of pixels the predicted region covers in the 2-D image.\",\n    \"shape_area_mm2\": \"Physical area of the region, computed only when pixel spacing is known.\",\n    \"shape_perimeter_px\": \"Length of the region boundary measured in pixels.\",\n    \"shape_perimeter_mm\": \"Length of the region boundary in millimetres (requires spacing).\",\n    \"shape_compactness\": \"How close the region is to a disc (1 = perfect disc). Sensitive to boundary detail.\",\n    \"shape_sphericity\": \"Square root of compactness; 1 means the region is as round as possible.\",\n    \"shape_elongation\": \"Ratio of the second to the first principal axis length of the region.\",\n    \"shape_equivalent_diameter_px\": \"Diameter of the disc with the same area as the region, in pixels.\",\n    \"shape_voxel_count\": \"Number of voxels in the region of interest.\",\n    \"shape_volume_mm3\": \"Physical volume of the region, computed only when voxel spacing is known.\",\n    \"shape_surface_area_mm2\": \"Estimated surface area of the segmented region.\",\n    \"firstorder_mean\": \"Average intensity inside the predicted region.\",\n    \"firstorder_median\": \"Median intensity inside the predicted region; less sensitive to outliers than the mean.\",\n    \"firstorder_minimum\": \"Lowest intensity inside the region.\",\n    \"firstorder_maximum\": \"Highest intensity inside the region.\",\n    \"firstorder_variance\": \"Spread of intensities around the mean inside the region.\",\n    \"firstorder_standard_deviation\": \"Square root of the intensity variance.\",\n    \"firstorder_skewness\": \"Asymmetry of the intensity distribution inside the region.\",\n    \"firstorder_kurtosis\": \"Tail weight of the intensity distribution (excess kurtosis, 0 = Gaussian).\",\n    \"firstorder_energy\": \"Sum of squared intensities inside the region.\",\n    \"firstorder_root_mean_squared\": \"Root mean square intensity inside the region.\",\n    \"firstorder_interquartile_range\": \"Difference between the 75th and 25th intensity percentiles.\",\n    \"firstorder_pixel_count\": \"Number of pixels contributing to the intensity statistics.\",\n    \"glcm_contrast\": \"Grey-level co-occurrence contrast: how quickly intensities change between neighbouring pixels.\",\n    \"glcm_dissimilarity\": \"Average absolute intensity difference between neighbouring pixels.\",\n    \"glcm_homogeneity\": \"How uniform the local intensity pattern is (higher = more homogeneous texture).\",\n    \"glcm_asm\": \"Angular second moment of the co-occurrence matrix; a measure of order in the texture.\",\n    \"glcm_energy\": \"Square root of the angular second moment.\",\n    \"glcm_correlation\": \"Linear dependency between neighbouring pixel intensities.\",\n    \"glcm_entropy\": \"Information content of the co-occurrence matrix; higher means more heterogeneous texture.\",\n    \"glcm_mean_intensity\": \"Mean grey level of the co-occurrence matrix.\",\n    \"glrlm_short_run_emphasis\": \"Emphasis on short runs of identical intensity; higher in fine, granular textures.\",\n    \"glrlm_long_run_emphasis\": \"Emphasis on long runs of identical intensity; higher in smooth textures.\",\n    \"glrlm_run_percentage\": \"Fraction of the image covered by runs of each grey level.\",\n    \"diagnostics_entropy\": \"Entropy of the intensity histogram inside the region.\",\n    \"diagnostics_uniformity\": \"Uniformity of the intensity histogram; 1 means a single intensity value.\",\n    \"diagnostics_roi_voxel_count\": \"Size of the region of interest, in pixels (2-D) or voxels (3-D).\",\n    \"diagnostics_discretisation_levels\": \"Number of discrete grey levels used for the texture matrices.\",\n}\n@dataclass\nclass RadiomicsResult:\n    available: bool\n    backend: str\n    dimensionality: str\n    feature_count: int\n    features: dict[str, Any]\n    categories: dict[str, list[str]]\n    spacing_available: bool\n    warnings: list[str] = field(default_factory=list)\n    quality: dict = field(default_factory=dict)\n    unavailable_reason: str | None = None\n    backend_note: str = \"\"\n    def to_dict(self) -> dict:\n        return to_jsonable(\n            {\n                \"available\": self.available,\n                \"backend\": self.backend,\n                \"backend_note\": self.backend_note,\n                \"dimensionality\": self.dimensionality,\n                \"feature_count\": self.feature_count,\n                \"features\": self.features,\n                \"categories\": self.categories,\n                \"spacing_available\": self.spacing_available,\n                \"warnings\": self.warnings,\n                \"quality\": self.quality,\n                \"unavailable_reason\": self.unavailable_reason,\n            }\n        )\nclass RadiomicsService:\n    def __init__(self, config):\n        self.config = config\n        self._pyradiomics = None\n        self._pyradiomics_checked = False\n    def _try_import_pyradiomics(self):\n        if self._pyradiomics_checked:\n            return self._pyradiomics\n        self._pyradiomics_checked = True\n        try:\n            from radiomics import featureextractor\n            self._pyradiomics = featureextractor\n            logger.info(\"PyRadiomics is available and will be used as the primary backend\")\n        except Exception as exc:\n            self._pyradiomics = None\n            logger.info(\"PyRadiomics unavailable (%s); using the native extractor\", type(exc).__name__)\n        return self._pyradiomics\n    def backend_status(self) -> dict:\n        preference = (getattr(self.config, \"RADIOMICS_BACKEND\", \"auto\") or \"auto\").lower()\n        available = self._try_import_pyradiomics() is not None\n        selected = \"native\"\n        if preference == \"pyradiomics\" and available:\n            selected = \"pyradiomics\"\n        elif preference == \"auto\" and available:\n            selected = \"pyradiomics\"\n        return {\n            \"preference\": preference,\n            \"pyradiomics_available\": available,\n            \"selected_backend\": selected,\n            \"note\": (\n                \"The native extractor is a NumPy implementation of shape, first-order, GLCM and \"\n                \"GLRLM features. PyRadiomics, when installed, additionally provides GLDM, NGTDM \"\n                \"and a full IBSI implementation. Both report the same geometry diagnostics so \"\n                \"that results remain comparable.\"\n            ),\n        }\n    def extract(\n        self,\n        image: np.ndarray,\n        mask: np.ndarray,\n        *,\n        spacing: tuple[float, ...] | None = None,\n        mask_source: str = \"predicted\",\n    ) -> RadiomicsResult:\n        image = np.asarray(image, dtype=np.float32)\n        mask = np.asarray(mask).astype(bool)\n        dimensionality = \"3D\" if image.ndim == 3 else \"2D\"\n        if image.shape != mask.shape:\n            return RadiomicsResult(\n                available=False,\n                backend=\"none\",\n                dimensionality=dimensionality,\n                feature_count=0,\n                features={},\n                categories={},\n                spacing_available=bool(spacing),\n                unavailable_reason=(\n                    f\"Image shape {list(image.shape)} and mask shape {list(mask.shape)} differ; \"\n                    \"radiomics requires the mask to be defined on the image grid.\"\n                ),\n            )\n        if not mask.any():\n            return RadiomicsResult(\n                available=False,\n                backend=\"none\",\n                dimensionality=dimensionality,\n                feature_count=0,\n                features={},\n                categories={},\n                spacing_available=bool(spacing),\n                unavailable_reason=(\n                    \"The predicted region is empty, so no region-based features can be computed. \"\n                    \"An empty region is different from a measurement of zero.\"\n                ),\n            )\n        settings = DiscretisationSettings(\n            method=getattr(self.config, \"RADIOMICS_DISCRETIZATION\", \"bin_count\"),\n            bin_count=int(getattr(self.config, \"RADIOMICS_BIN_COUNT\", 32)),\n            bin_width=float(getattr(self.config, \"RADIOMICS_BIN_WIDTH\", 0.0) or 25.0),\n        )\n        backend = self.backend_status()[\"selected_backend\"]\n        warnings: list[str] = []\n        if backend == \"pyradiomics\":\n            try:\n                return self._extract_pyradiomics(\n                    image, mask, spacing=spacing, settings=settings, dimensionality=dimensionality,\n                    mask_source=mask_source, warnings=warnings,\n                )\n            except Exception as exc:\n                logger.warning(\"PyRadiomics extraction failed (%s); using the native extractor\", exc)\n                warnings.append(\n                    f\"PyRadiomics failed for this case ({type(exc).__name__}); the native extractor \"\n                    \"was used instead.\"\n                )\n        return self._extract_native(\n            image,\n            mask,\n            spacing=spacing,\n            settings=settings,\n            dimensionality=dimensionality,\n            mask_source=mask_source,\n            warnings=warnings,\n        )\n    def _extract_native(\n        self,\n        image: np.ndarray,\n        mask: np.ndarray,\n        *,\n        spacing,\n        settings: DiscretisationSettings,\n        dimensionality: str,\n        mask_source: str,\n        warnings: list[str],\n    ) -> RadiomicsResult:\n        features = extract_features(\n            image, mask, spacing=spacing, settings=settings, include_texture=True\n        )\n        if \"diagnostics_texture_warning\" in features:\n            warnings.append(features.pop(\"diagnostics_texture_warning\"))\n        if dimensionality == \"2D\":\n            warnings.append(\n                \"This is a 2-D slice analysis. Areas are reported in pixels and no physical \"\n                \"volume is computed, because a single slice has no voxel depth. Physical units \"\n                \"require a 3-D volume together with its spacing.\"\n            )\n        elif not spacing:\n            warnings.append(\n                \"No voxel spacing was supplied, so physical volume and surface area are not \"\n                \"reported; measurements are given in voxel units.\"\n            )\n        ordered = self._order_features(features)\n        categories = self._group_features(ordered)\n        quality = self._quality(features, mask, dimensionality, mask_source)\n        return RadiomicsResult(\n            available=True,\n            backend=\"native\",\n            dimensionality=dimensionality,\n            feature_count=len(ordered),\n            features=ordered,\n            categories=categories,\n            spacing_available=bool(spacing),\n            warnings=warnings,\n            quality=quality,\n            backend_note=(\n                \"Native NumPy implementation: shape, first-order, GLCM and GLRLM features with \"\n                \"IBSI-style naming.\"\n            ),\n        )\n    def _extract_pyradiomics(\n        self,\n        image: np.ndarray,\n        mask: np.ndarray,\n        *,\n        spacing,\n        settings: DiscretisationSettings,\n        dimensionality: str,\n        mask_source: str,\n        warnings: list[str],\n    ) -> RadiomicsResult:\n        import SimpleITK as sitk\n        from radiomics import featureextractor\n        params = {\n            \"binWidth\": settings.bin_width,\n            \"resampledPixelSpacing\": None,\n            \"interpolator\": \"sitkBSpline\",\n            \"normalize\": False,\n            \"geometryTolerance\": 1e-3,\n        }\n        extractor = featureextractor.RadiomicsFeatureExtractor(**params)\n        extractor.enableAllFeatures()\n        if spacing:\n            kwargs = {\"spacing\": tuple(spacing)}\n            image_sitk = sitk.GetImageFromArray(image.astype(np.float32))\n            image_sitk.SetSpacing(tuple(reversed(spacing))) if image.ndim == 3 else image_sitk.SetSpacing(tuple(reversed(spacing)))\n            mask_sitk = sitk.GetImageFromArray(mask.astype(np.uint8))\n            mask_sitk.CopyInformation(image_sitk)\n        else:\n            kwargs = {}\n            array = image[None, :, :] if image.ndim == 2 else image\n            mask_array = mask[None, :, :] if mask.ndim == 2 else mask\n            image_sitk = sitk.GetImageFromArray(array.astype(np.float32))\n            mask_sitk = sitk.GetImageFromArray(mask_array.astype(np.uint8))\n            mask_sitk.CopyInformation(image_sitk)\n            warnings.append(\n                \"No spacing was available, so a nominal 1 x 1 voxel spacing was used for the \"\n                \"PyRadiomics call; physical-size features are therefore expressed in pixels.\"\n            )\n        raw = extractor.execute(image_sitk, mask_sitk, **kwargs)\n        features: dict[str, Any] = {}\n        diagnostics: dict[str, Any] = {}\n        for key, value in raw.items():\n            if key.startswith(\"diagnostics_\"):\n                diagnostics[key] = value\n                continue\n            try:\n                features[key] = float(value)\n            except (TypeError, ValueError):\n                continue\n        features[\"diagnostics_dimensionality\"] = dimensionality\n        features[\"diagnostics_roi_voxel_count\"] = int(mask.sum())\n        features[\"diagnostics_spacing\"] = list(spacing) if spacing else None\n        features[\"diagnostics_spacing_available\"] = bool(spacing)\n        features[\"diagnostics_discretisation_method\"] = \"pyradiomics_bin_width\"\n        features[\"diagnostics_discretisation_levels\"] = None\n        for key, value in diagnostics.items():\n            if isinstance(value, (int, float, str)) and not key.endswith(\"_Hash\"):\n                features.setdefault(f\"pyradiomics_{key.replace('diagnostics_', '')}\", value)\n        ordered = self._order_features(features)\n        return RadiomicsResult(\n            available=True,\n            backend=\"pyradiomics\",\n            dimensionality=dimensionality,\n            feature_count=len(ordered),\n            features=ordered,\n            categories=self._group_features(ordered),\n            spacing_available=bool(spacing),\n            warnings=warnings,\n            quality=self._quality(features, mask, dimensionality, mask_source),\n            backend_note=f\"PyRadiomics {getattr(featureextractor, '__version__', '')} with all feature classes enabled.\",\n        )\n    @staticmethod\n    def _order_features(features: dict[str, Any]) -> dict[str, Any]:\n        buckets: dict[str, list[str]] = {family: [] for family in FAMILY_ORDER}\n        for name in features:\n            buckets.setdefault(categorise(name), []).append(name)\n        ordered: dict[str, Any] = {}\n        for family in FAMILY_ORDER:\n            for name in sorted(buckets.get(family, [])):\n                ordered[name] = features[name]\n        for family, names in buckets.items():\n            if family in FAMILY_ORDER:\n                continue\n            for name in sorted(names):\n                ordered[name] = features[name]\n        return to_jsonable(ordered)\n    @staticmethod\n    def _group_features(features: dict[str, Any]) -> dict[str, list[str]]:\n        groups: dict[str, list[str]] = {}\n        for name in features:\n            groups.setdefault(categorise(name), []).append(name)\n        return groups\n    @staticmethod\n    def _quality(features: dict, mask: np.ndarray, dimensionality: str, mask_source: str) -> dict:\n        voxel_count = int(mask.sum())\n        bbox = _bounding_box(mask)\n        extent = None\n        if bbox:\n            extents = [high - low + 1 for low, high in bbox]\n            extent = extents\n        touching_border = any(low == 0 or high == size - 1 for (low, high), size in zip(bbox, mask.shape)) if bbox else False\n        return {\n            \"mask_source\": mask_source,\n            \"roi_voxel_count\": voxel_count,\n            \"roi_bounding_box\": bbox,\n            \"roi_extent\": extent,\n            \"roi_touches_fov_border\": bool(touching_border),\n            \"relative_roi_size\": float(voxel_count / mask.size),\n            \"note\": (\n                \"Shape and texture features describe the region produced by the segmentation \"\n                \"model. If that region is inaccurate, every downstream value inherits the error, \"\n                \"which is why segmentation quality should be reviewed before the numbers are read.\"\n            ),\n        }\ndef _bounding_box(mask: np.ndarray) -> list[list[int]] | None:\n    coordinates = np.argwhere(mask)\n    if coordinates.size == 0:\n        return None\n    return [\n        [int(coordinates[:, axis].min()), int(coordinates[:, axis].max())]\n        for axis in range(coordinates.shape[1])\n    ]\ndef feature_tooltip(name: str) -> str:\n    if name in FEATURE_DESCRIPTIONS:\n        return FEATURE_DESCRIPTIONS[name]\n    family = categorise(name)\n    return FEATURE_CATEGORIES.get(family, \"Radiomics feature.\") + f\" (feature: {name})\"\n"}, "backend/app/utils/metrics.py": {"sha256": "ad10ddd9b766859d3727fb4201b4ed92188b8fa50d110476aa18b69bc5136e1a", "source": "from __future__ import annotations\nfrom typing import Iterable, Sequence\nimport numpy as np\nEPS = 1e-7\ndef _as_bool(array: np.ndarray, threshold: float = 0.5) -> np.ndarray:\n    return np.asarray(array) > threshold\ndef confusion_counts(pred: np.ndarray, target: np.ndarray, threshold: float = 0.5) -> dict:\n    p = _as_bool(pred, threshold).ravel()\n    t = _as_bool(target, threshold).ravel()\n    tp = float(np.logical_and(p, t).sum())\n    fp = float(np.logical_and(p, ~t).sum())\n    fn = float(np.logical_and(~p, t).sum())\n    tn = float(np.logical_and(~p, ~t).sum())\n    return {\"tp\": tp, \"fp\": fp, \"fn\": fn, \"tn\": tn}\ndef dice(pred: np.ndarray, target: np.ndarray, threshold: float = 0.5) -> float:\n    c = confusion_counts(pred, target, threshold)\n    denom = 2 * c[\"tp\"] + c[\"fp\"] + c[\"fn\"]\n    if denom == 0:\n        return 1.0\n    return float((2 * c[\"tp\"]) / denom)\ndef iou(pred: np.ndarray, target: np.ndarray, threshold: float = 0.5) -> float:\n    c = confusion_counts(pred, target, threshold)\n    denom = c[\"tp\"] + c[\"fp\"] + c[\"fn\"]\n    if denom == 0:\n        return 1.0\n    return float(c[\"tp\"] / denom)\ndef precision(pred: np.ndarray, target: np.ndarray, threshold: float = 0.5) -> float:\n    c = confusion_counts(pred, target, threshold)\n    denom = c[\"tp\"] + c[\"fp\"]\n    return float(c[\"tp\"] / denom) if denom else 0.0\ndef recall(pred: np.ndarray, target: np.ndarray, threshold: float = 0.5) -> float:\n    c = confusion_counts(pred, target, threshold)\n    denom = c[\"tp\"] + c[\"fn\"]\n    return float(c[\"tp\"] / denom) if denom else 0.0\ndef specificity(pred: np.ndarray, target: np.ndarray, threshold: float = 0.5) -> float:\n    c = confusion_counts(pred, target, threshold)\n    denom = c[\"tn\"] + c[\"fp\"]\n    return float(c[\"tn\"] / denom) if denom else 0.0\ndef accuracy(pred: np.ndarray, target: np.ndarray, threshold: float = 0.5) -> float:\n    c = confusion_counts(pred, target, threshold)\n    total = sum(c.values())\n    return float((c[\"tp\"] + c[\"tn\"]) / total) if total else 0.0\ndef segmentation_metrics(pred: np.ndarray, target: np.ndarray, threshold: float = 0.5) -> dict:\n    counts = confusion_counts(pred, target, threshold)\n    return {\n        \"dice\": dice(pred, target, threshold),\n        \"iou\": iou(pred, target, threshold),\n        \"precision\": precision(pred, target, threshold),\n        \"recall\": recall(pred, target, threshold),\n        \"sensitivity\": recall(pred, target, threshold),\n        \"specificity\": specificity(pred, target, threshold),\n        \"accuracy\": accuracy(pred, target, threshold),\n        \"confusion\": counts,\n        \"predicted_positive_pixels\": int(counts[\"tp\"] + counts[\"fp\"]),\n        \"reference_positive_pixels\": int(counts[\"tp\"] + counts[\"fn\"]),\n    }\ndef hausdorff_distance_95(pred: np.ndarray, target: np.ndarray, spacing: Sequence[float] | None = None) -> float | None:\n    p = _as_bool(pred)\n    t = _as_bool(target)\n    if not p.any() or not t.any():\n        return None\n    dt_pred = _edt(~p, spacing)\n    dt_target = _edt(~t, spacing)\n    surface_p = p & (_neighbour_min(dt_pred) > 0)\n    surface_t = t & (_neighbour_min(dt_target) > 0)\n    if not surface_p.any():\n        surface_p = p\n    if not surface_t.any():\n        surface_t = t\n    d1 = dt_target[surface_p]\n    d2 = dt_pred[surface_t]\n    return float(np.percentile(np.concatenate([d1, d2]), 95))\ndef _neighbour_min(array: np.ndarray) -> np.ndarray:\n    out = np.full_like(array, np.inf, dtype=np.float64)\n    for axis in range(array.ndim):\n        for shift in (-1, 1):\n            rolled = np.roll(array, shift, axis=axis)\n            if shift == 1:\n                sl = [slice(None)] * array.ndim\n                sl[axis] = 0\n                rolled[tuple(sl)] = np.inf\n            else:\n                sl = [slice(None)] * array.ndim\n                sl[axis] = -1\n                rolled[tuple(sl)] = np.inf\n            out = np.minimum(out, rolled)\n    return out\ndef _edt(mask: np.ndarray, spacing: Sequence[float] | None = None) -> np.ndarray:\n    mask = np.asarray(mask, dtype=bool)\n    if spacing is None:\n        spacing = [1.0] * mask.ndim\n    dist = np.where(mask, np.inf, 0.0).astype(np.float64)\n    for axis, sp in enumerate(spacing):\n        dist = np.apply_along_axis(lambda line: _dt_1d(line, sp), axis, dist)\n    return np.sqrt(np.maximum(dist, 0.0))\ndef _dt_1d(line: np.ndarray, spacing: float) -> np.ndarray:\n    n = line.shape[0]\n    f = line.astype(np.float64)\n    d = np.empty(n, dtype=np.float64)\n    v = np.zeros(n, dtype=int)\n    z = np.empty(n + 1, dtype=np.float64)\n    k = 0\n    v[0] = 0\n    z[0] = -np.inf\n    z[1] = np.inf\n    for q in range(1, n):\n        if not np.isfinite(f[q]):\n            continue\n        s = ((f[q] + (q * spacing) ** 2) - (f[v[k]] + (v[k] * spacing) ** 2)) / (\n            2 * spacing * (q - v[k])\n        )\n        while s <= z[k]:\n            k -= 1\n            if k < 0:\n                break\n            s = ((f[q] + (q * spacing) ** 2) - (f[v[k]] + (v[k] * spacing) ** 2)) / (\n                2 * spacing * (q - v[k])\n            )\n        if k < 0:\n            k = 0\n            v[k] = q\n            z[k] = -np.inf\n            z[k + 1] = np.inf\n            continue\n        k += 1\n        v[k] = q\n        z[k] = s\n        z[k + 1] = np.inf\n    k = 0\n    for q in range(n):\n        while z[k + 1] < q * spacing:\n            k += 1\n        d[q] = ((q * spacing) - (v[k] * spacing)) ** 2 + f[v[k]]\n    d[~np.isfinite(f)] = np.inf\n    return d\ndef agreement_metrics(mask_a: np.ndarray, mask_b: np.ndarray) -> dict:\n    return {\n        \"dice_agreement\": dice(mask_a, mask_b, 0.5),\n        \"iou_agreement\": iou(mask_a, mask_b, 0.5),\n        \"pixels_a\": int(_as_bool(mask_a).sum()),\n        \"pixels_b\": int(_as_bool(mask_b).sum()),\n    }\ndef multilabel_confusion(y_true: Sequence[int], y_pred: Sequence[int], n_classes: int) -> np.ndarray:\n    matrix = np.zeros((n_classes, n_classes), dtype=int)\n    for t, p in zip(y_true, y_pred):\n        matrix[int(t), int(p)] += 1\n    return matrix\ndef classification_report(y_true: Sequence[int], y_pred: Sequence[int], n_classes: int) -> dict:\n    y_true = np.asarray(y_true, dtype=int)\n    y_pred = np.asarray(y_pred, dtype=int)\n    matrix = multilabel_confusion(y_true, y_pred, n_classes)\n    per_class = {}\n    f1_scores = []\n    for index in range(n_classes):\n        tp = matrix[index, index]\n        fp = matrix[:, index].sum() - tp\n        fn = matrix[index, :].sum() - tp\n        prec = tp / (tp + fp) if (tp + fp) else 0.0\n        rec = tp / (tp + fn) if (tp + fn) else 0.0\n        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0\n        f1_scores.append(float(f1))\n        per_class[str(index)] = {\n            \"precision\": float(prec),\n            \"recall\": float(rec),\n            \"f1\": float(f1),\n            \"support\": int(matrix[index, :].sum()),\n        }\n    total = matrix.sum()\n    return {\n        \"accuracy\": float(np.trace(matrix) / total) if total else None,\n        \"macro_f1\": float(np.mean(f1_scores)) if f1_scores else None,\n        \"weighted_f1\": float(\n            np.average(f1_scores, weights=[matrix[i, :].sum() for i in range(n_classes)])\n        )\n        if total\n        else None,\n        \"per_class\": per_class,\n        \"confusion_matrix\": matrix.tolist(),\n        \"n_samples\": int(total),\n    }\ndef roc_auc_binary(y_true: Sequence[int], y_score: Sequence[float]) -> float | None:\n    y_true = np.asarray(y_true, dtype=int)\n    y_score = np.asarray(y_score, dtype=float)\n    positives = y_true == 1\n    negatives = ~positives\n    if positives.sum() == 0 or negatives.sum() == 0:\n        return None\n    order = np.argsort(y_score)\n    ranks = np.empty_like(order, dtype=float)\n    ranks[order] = np.arange(1, len(y_score) + 1)\n    sorted_scores = y_score[order]\n    i = 0\n    while i < len(sorted_scores):\n        j = i\n        while j + 1 < len(sorted_scores) and sorted_scores[j + 1] == sorted_scores[i]:\n            j += 1\n        if j > i:\n            ranks[order[i : j + 1]] = np.mean(ranks[order[i : j + 1]])\n        i = j + 1\n    auc = (ranks[positives].sum() - positives.sum() * (positives.sum() + 1) / 2) / (\n        positives.sum() * negatives.sum()\n    )\n    return float(auc)\ndef average_precision_binary(y_true: Sequence[int], y_score: Sequence[float]) -> float | None:\n    y_true = np.asarray(y_true, dtype=int)\n    y_score = np.asarray(y_score, dtype=float)\n    if y_true.sum() == 0:\n        return None\n    order = np.argsort(-y_score)\n    y_sorted = y_true[order]\n    tp = np.cumsum(y_sorted)\n    precision = tp / (np.arange(len(y_sorted)) + 1)\n    recall = tp / y_true.sum()\n    return float(np.trapz(precision, recall))\ndef expected_calibration_error(\n    y_true: Sequence[int], probabilities: Sequence[float], n_bins: int = 10\n) -> float | None:\n    y_true = np.asarray(y_true, dtype=int)\n    probs = np.asarray(probabilities, dtype=float)\n    if len(y_true) == 0:\n        return None\n    bins = np.linspace(0.0, 1.0, n_bins + 1)\n    ece = 0.0\n    for lower, upper in zip(bins[:-1], bins[1:]):\n        in_bin = (probs > lower) & (probs <= upper)\n        if not in_bin.any():\n            continue\n        confidence = probs[in_bin].mean()\n        accuracy_in_bin = y_true[in_bin].mean()\n        ece += (in_bin.sum() / len(probs)) * abs(confidence - accuracy_in_bin)\n    return float(ece)\ndef reliability_curve(\n    y_true: Sequence[int], probabilities: Sequence[float], n_bins: int = 10\n) -> list[dict]:\n    y_true = np.asarray(y_true, dtype=int)\n    probs = np.asarray(probabilities, dtype=float)\n    bins = np.linspace(0.0, 1.0, n_bins + 1)\n    curve = []\n    for lower, upper in zip(bins[:-1], bins[1:]):\n        in_bin = (probs > lower) & (probs <= upper)\n        curve.append(\n            {\n                \"bin_lower\": float(lower),\n                \"bin_upper\": float(upper),\n                \"count\": int(in_bin.sum()),\n                \"mean_confidence\": float(probs[in_bin].mean()) if in_bin.any() else None,\n                \"empirical_accuracy\": float(y_true[in_bin].mean()) if in_bin.any() else None,\n            }\n        )\n    return curve\ndef bootstrap_ci(\n    values: Iterable[float], n_resamples: int = 1000, alpha: float = 0.05, seed: int = 42\n) -> dict:\n    arr = np.asarray([v for v in values if v is not None and np.isfinite(v)], dtype=float)\n    if arr.size == 0:\n        return {\"mean\": None, \"ci_lower\": None, \"ci_upper\": None, \"n\": 0}\n    rng = np.random.default_rng(seed)\n    samples = rng.choice(arr, size=(n_resamples, arr.size), replace=True).mean(axis=1)\n    return {\n        \"mean\": float(arr.mean()),\n        \"ci_lower\": float(np.quantile(samples, alpha / 2)),\n        \"ci_upper\": float(np.quantile(samples, 1 - alpha / 2)),\n        \"n\": int(arr.size),\n    }\n"}, "backend/app/utils/images.py": {"sha256": "ee18163d200ccfdeec9c8cb8767692824621d7caa04b5a16f4fcd4011c78f582", "source": "from __future__ import annotations\nimport base64\nimport io\nfrom dataclasses import dataclass\nfrom typing import Iterable, Sequence\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont\nPNG_MIME = \"image/png\"\n@dataclass(frozen=True)\nclass DecodedImage:\n    array: np.ndarray\n    original_mode: str\n    original_size: tuple[int, int]\n    source_format: str\ndef decode_image_bytes(raw: bytes) -> DecodedImage:\n    with Image.open(io.BytesIO(raw)) as img:\n        source_format = (img.format or \"unknown\").upper()\n        original_mode = img.mode\n        original_size = img.size\n        if original_mode in (\"I;16\", \"I;16B\", \"I\", \"F\"):\n            arr = np.asarray(img).astype(np.float32)\n            finite = np.isfinite(arr)\n            if finite.any():\n                lo, hi = float(arr[finite].min()), float(arr[finite].max())\n                arr = (arr - lo) / (hi - lo) if hi > lo else np.zeros_like(arr)\n            else:\n                arr = np.zeros_like(arr)\n            arr = np.nan_to_num(arr, nan=0.0)\n        else:\n            arr = np.asarray(img.convert(\"L\")).astype(np.float32) / 255.0\n    return DecodedImage(\n        array=arr.astype(np.float32),\n        original_mode=original_mode,\n        original_size=original_size,\n        source_format=source_format,\n    )\ndef decode_gray(raw: bytes) -> np.ndarray:\n    return decode_image_bytes(raw).array\ndef resize_array(array: np.ndarray, size: tuple[int, int], *, nearest: bool = False) -> np.ndarray:\n    height, width = size\n    mode = Image.Resampling.NEAREST if nearest else Image.Resampling.BILINEAR\n    if array.dtype == np.bool_:\n        array = array.astype(np.uint8) * 255\n    if np.issubdtype(array.dtype, np.floating):\n        img = Image.fromarray(np.clip(array * 255.0, 0, 255).astype(np.uint8), mode=\"L\")\n        out = np.asarray(img.resize((width, height), mode), dtype=np.float32) / 255.0\n        return out\n    img = Image.fromarray(array.astype(np.uint8), mode=\"L\")\n    return np.asarray(img.resize((width, height), mode))\ndef to_uint8(array: np.ndarray, *, percentile_clip: tuple[float, float] | None = None) -> np.ndarray:\n    arr = np.asarray(array, dtype=np.float32)\n    finite = np.isfinite(arr)\n    if not finite.any():\n        return np.zeros(arr.shape, dtype=np.uint8)\n    arr = np.where(finite, arr, np.nan)\n    if percentile_clip is not None:\n        lo, hi = np.nanpercentile(arr, percentile_clip)\n    else:\n        lo, hi = float(np.nanmin(arr)), float(np.nanmax(arr))\n    if hi - lo < 1e-12:\n        return np.zeros(arr.shape, dtype=np.uint8)\n    scaled = (np.nan_to_num(arr, nan=lo) - lo) / (hi - lo)\n    return (np.clip(scaled, 0.0, 1.0) * 255.0).astype(np.uint8)\ndef apply_colormap(probability: np.ndarray, name: str = \"jet\") -> np.ndarray:\n    p = np.clip(np.asarray(probability, dtype=np.float32), 0.0, 1.0)\n    if name == \"viridis\":\n        stops = np.array(\n            [\n                [68, 1, 84],\n                [65, 68, 135],\n                [42, 120, 142],\n                [34, 168, 132],\n                [122, 209, 81],\n                [253, 231, 37],\n            ],\n            dtype=np.float32,\n        )\n    else:\n        stops = np.array(\n            [\n                [0, 0, 131],\n                [0, 60, 170],\n                [5, 255, 255],\n                [255, 255, 0],\n                [250, 0, 0],\n                [128, 0, 0],\n            ],\n            dtype=np.float32,\n        )\n    idx = p * (len(stops) - 1)\n    lo = np.floor(idx).astype(int)\n    hi = np.clip(lo + 1, 0, len(stops) - 1)\n    frac = (idx - lo)[..., None]\n    rgb = stops[lo] * (1 - frac) + stops[hi] * frac\n    return rgb.astype(np.uint8)\ndef mask_to_rgb(mask: np.ndarray, colour: Sequence[int] = (255, 64, 64)) -> np.ndarray:\n    rgb = np.zeros(mask.shape + (3,), dtype=np.uint8)\n    rgb[mask.astype(bool)] = np.array(colour, dtype=np.uint8)\n    return rgb\ndef make_overlay(\n    background: np.ndarray,\n    mask: np.ndarray,\n    *,\n    colour: Sequence[int] = (255, 64, 64),\n    alpha: float = 0.45,\n) -> np.ndarray:\n    gray = to_uint8(background)\n    if gray.shape != mask.shape:\n        mask = resize_array(mask.astype(np.float32), gray.shape, nearest=True) > 0.5\n    base = np.repeat(gray[..., None], 3, axis=2).astype(np.float32)\n    overlay = base.copy()\n    colour_arr = np.array(colour, dtype=np.float32)\n    hit = mask.astype(bool)\n    overlay[hit] = base[hit] * (1 - alpha) + colour_arr * alpha\n    return overlay.astype(np.uint8)\ndef make_heatmap_overlay(\n    background: np.ndarray,\n    heatmap: np.ndarray,\n    *,\n    alpha: float = 0.5,\n    colormap: str = \"jet\",\n) -> np.ndarray:\n    gray = to_uint8(background)\n    hm = np.asarray(heatmap, dtype=np.float32)\n    if hm.shape != gray.shape:\n        hm = resize_array(np.clip(hm, 0, 1), gray.shape)\n    rgb = apply_colormap(np.clip(hm, 0, 1), colormap).astype(np.float32)\n    base = np.repeat(gray[..., None], 3, axis=2).astype(np.float32)\n    blended = base * (1 - alpha) + rgb * alpha\n    return np.clip(blended, 0, 255).astype(np.uint8)\ndef encode_png(array: np.ndarray) -> bytes:\n    arr = np.asarray(array)\n    if arr.dtype != np.uint8:\n        arr = to_uint8(arr)\n    if arr.ndim == 2:\n        img = Image.fromarray(arr, mode=\"L\")\n    else:\n        img = Image.fromarray(arr.astype(np.uint8), mode=\"RGB\")\n    buffer = io.BytesIO()\n    img.save(buffer, format=\"PNG\", optimize=False)\n    return buffer.getvalue()\ndef encode_data_url(array: np.ndarray) -> str:\n    payload = base64.b64encode(encode_png(array)).decode(\"ascii\")\n    return f\"data:{PNG_MIME};base64,{payload}\"\ndef build_montage(\n    panels: Sequence[tuple[str, np.ndarray]],\n    *,\n    panel_size: tuple[int, int] = (256, 256),\n    padding: int = 8,\n    label_height: int = 20,\n    background: int = 18,\n) -> np.ndarray:\n    if not panels:\n        raise ValueError(\"build_montage requires at least one panel\")\n    height, width = panel_size\n    cols = len(panels)\n    canvas_h = height + label_height + 2 * padding\n    canvas_w = cols * width + (cols + 1) * padding\n    canvas = np.full((canvas_h, canvas_w, 3), background, dtype=np.uint8)\n    try:\n        font = ImageFont.load_default()\n    except Exception:\n        font = None\n    for index, (label, panel) in enumerate(panels):\n        arr = np.asarray(panel)\n        if arr.ndim == 2:\n            arr = np.repeat(arr[..., None], 3, axis=2)\n        if arr.shape[0] != height or arr.shape[1] != width:\n            img = Image.fromarray(arr.astype(np.uint8))\n            arr = np.asarray(img.resize((width, height), Image.Resampling.BILINEAR))\n            if arr.ndim == 2:\n                arr = np.repeat(arr[..., None], 3, axis=2)\n        x0 = padding + index * (width + padding)\n        y0 = padding\n        canvas[y0 : y0 + height, x0 : x0 + width] = arr.astype(np.uint8)\n        if font is not None:\n            pil = Image.fromarray(canvas)\n            draw = ImageDraw.Draw(pil)\n            draw.text((x0 + 4, y0 + height + 4), label, fill=(230, 230, 230), font=font)\n            canvas = np.asarray(pil)\n    return canvas\ndef tile_pair(left: np.ndarray, right: np.ndarray, *, gap: int = 8) -> np.ndarray:\n    a, b = np.asarray(left), np.asarray(right)\n    if a.ndim == 2:\n        a = np.repeat(a[..., None], 3, axis=2)\n    if b.ndim == 2:\n        b = np.repeat(b[..., None], 3, axis=2)\n    if a.shape[0] != b.shape[0]:\n        img = Image.fromarray(b.astype(np.uint8))\n        b = np.asarray(img.resize((a.shape[1], a.shape[0]), Image.Resampling.BILINEAR))\n        if b.ndim == 2:\n            b = np.repeat(b[..., None], 3, axis=2)\n    canvas = np.full((a.shape[0], a.shape[1] + b.shape[1] + gap, 3), 18, dtype=np.uint8)\n    canvas[:, : a.shape[1]] = a.astype(np.uint8)\n    canvas[:, a.shape[1] + gap :] = b.astype(np.uint8)\n    return canvas\ndef normalise_blockwise(array: np.ndarray) -> np.ndarray:\n    arr = np.asarray(array, dtype=np.float32)\n    mean = float(arr.mean())\n    std = float(arr.std())\n    if std < 1e-5:\n        return np.zeros_like(arr)\n    return (arr - mean) / std\n__all__ = [\n    \"DecodedImage\",\n    \"apply_colormap\",\n    \"build_montage\",\n    \"decode_gray\",\n    \"decode_image_bytes\",\n    \"encode_data_url\",\n    \"encode_png\",\n    \"make_heatmap_overlay\",\n    \"make_overlay\",\n    \"mask_to_rgb\",\n    \"normalise_blockwise\",\n    \"resize_array\",\n    \"tile_pair\",\n    \"to_uint8\",\n]\n"}, "backend/app/utils/io.py": {"sha256": "1ee9c51f8c559bb5dfa0a39b292785d247c2e9c8a648c94eb4bc381bae0cd21e", "source": "from __future__ import annotations\nimport json\nimport os\nimport re\nimport tempfile\nimport uuid\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any\nimport numpy as np\n_SAFE_NAME = re.compile(r\"[^A-Za-z0-9._-]+\")\nALLOWED_IMAGE_SUFFIXES = {\".png\", \".jpg\", \".jpeg\"}\nALLOWED_VOLUME_SUFFIXES = {\".npz\", \".npy\", \".h5\", \".hdf5\"}\ndef utc_now() -> str:\n    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()\ndef new_analysis_id() -> str:\n    return f\"an-{datetime.now(timezone.utc).strftime('%Y%m%d')}-{uuid.uuid4().hex[:10]}\"\ndef safe_filename(name: str, *, default: str = \"upload\") -> str:\n    base = os.path.basename(name or default).strip() or default\n    cleaned = _SAFE_NAME.sub(\"_\", base)\n    if len(cleaned) > 120:\n        stem, dot, suffix = cleaned.rpartition(\".\")\n        cleaned = f\"{stem[:100]}{dot}{suffix[:16]}\" if dot else cleaned[:120]\n    return cleaned or default\ndef ensure_within(base: Path, candidate: Path) -> Path:\n    base = Path(base).resolve()\n    resolved = (base / candidate).resolve()\n    if base not in resolved.parents and resolved != base:\n        raise ValueError(f\"path escapes storage root: {candidate}\")\n    return resolved\ndef write_bytes_atomic(path: Path, payload: bytes) -> Path:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(dir=str(path.parent), delete=False) as handle:\n        handle.write(payload)\n        temp_name = handle.name\n    os.replace(temp_name, path)\n    return path\ndef write_json(path: Path, payload: Any) -> Path:\n    return write_bytes_atomic(Path(path), json.dumps(to_jsonable(payload), indent=2).encode(\"utf-8\"))\ndef read_json(path: Path, default: Any = None) -> Any:\n    try:\n        with open(path, \"r\", encoding=\"utf-8\") as handle:\n            return json.load(handle)\n    except (OSError, json.JSONDecodeError):\n        return default\ndef to_jsonable(value: Any) -> Any:\n    if isinstance(value, dict):\n        return {str(k): to_jsonable(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple, set)):\n        return [to_jsonable(v) for v in value]\n    if isinstance(value, np.ndarray):\n        return to_jsonable(value.tolist())\n    if isinstance(value, (np.floating,)):\n        return float(value)\n    if isinstance(value, (np.integer,)):\n        return int(value)\n    if isinstance(value, (np.bool_,)):\n        return bool(value)\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, float) and (np.isnan(value) or np.isinf(value)):\n        return None\n    if hasattr(value, \"item\") and callable(getattr(value, \"item\")):\n        try:\n            return to_jsonable(value.item())\n        except Exception:\n            return str(value)\n    return value\ndef human_size(num_bytes: int) -> str:\n    size = float(num_bytes)\n    for unit in (\"B\", \"KB\", \"MB\", \"GB\"):\n        if size < 1024 or unit == \"GB\":\n            return f\"{size:.1f} {unit}\" if unit != \"B\" else f\"{int(size)} B\"\n        size /= 1024\n    return f\"{size:.1f} GB\"\ndef relative_to_root(path: Path, root: Path) -> str:\n    try:\n        return str(Path(path).resolve().relative_to(Path(root).resolve()))\n    except ValueError:\n        return str(path)\n"}, "backend/app/__init__.py": {"sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "source": ""}, "backend/app/ml/__init__.py": {"sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "source": ""}, "backend/app/ml/architectures/__init__.py": {"sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "source": ""}, "backend/app/services/__init__.py": {"sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "source": ""}, "backend/app/utils/__init__.py": {"sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "source": ""}}''')

EMBEDDED_ROOT = PATHS['work'] / '_embedded'
for relative, entry in EMBEDDED_SOURCES.items():
    target = EMBEDDED_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    current = target.read_text(encoding='utf-8') if target.exists() else None
    if current != entry['source']:
        target.write_text(entry['source'], encoding='utf-8')
if str(EMBEDDED_ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(EMBEDDED_ROOT / 'backend'))

from app.ml.architectures.segnet import SegNet
from app.ml.architectures.dynamic_unet import DEFAULT_FILTERS, DynamicUNet
from app.ml.architectures.attention_unet import build_raunet
from app.ml.losses import BCEDiceLoss, DiceLoss, build_loss
from app.utils import metrics as project_metrics

LOCK['embedded'] = {relative: entry['sha256'] for relative, entry in EMBEDDED_SOURCES.items()}
print(f'embedded {len(EMBEDDED_SOURCES)} project files under {EMBEDDED_ROOT}')
print('architectures:', SegNet.__module__, DynamicUNet.__module__, build_raunet.__module__)
print('loss:', build_loss.__module__, '| metrics:', project_metrics.__name__)

write_json(PATHS['artifacts'] / 'embedded_sources.json', LOCK['embedded'])

## 3. Dataset discovery
`nbs_02_data.py`


In [ ]:
# =============================================================================
# SECTION 2 - Dataset discovery
# =============================================================================
# The BraTS 2020 Kaggle mirror is mounted read-only at
# /kaggle/input/brats2020-training-data. Its layout is
#   BraTS2020_training_data/content/data/volume_<n>/volume_<n>_slice_<m>.h5
# plus name_mapping.csv with Grade (HGG/LGG) and BrATS_2020_subject_ID.
# Nothing is downloaded here: internet access is not required.
# =============================================================================
def discover_dataset(roots=RUN["dataset_roots"]):
    """Return the directory that holds the slice files and any grade table found."""
    candidates = []
    for root in roots:
        base = Path(root)
        if not base.exists():
            continue
        candidates.append(base)
        for nested in base.rglob("data"):
            if nested.is_dir():
                candidates.append(nested)
    for base in candidates:
        slice_files = list(base.glob("volume_*/volume_*_slice_*.h5")) or list(
            base.glob("*/volume_*_slice_*.h5")
        )
        if slice_files:
            return {
                "found": True,
                "root": str(base),
                "layout": "hdf5_slices",
                "slice_files": len(slice_files),
                "searched": [str(item) for item in roots],
            }
    # NIfTI release (one directory per patient)
    for base in candidates:
        nifti = list(base.glob("**/*_seg.nii*")) or list(base.glob("**/*_flair.nii*"))
        if nifti:
            return {
                "found": True,
                "root": str(base),
                "layout": "nifti",
                "files": len(nifti),
                "searched": [str(item) for item in roots],
            }
    return {
        "found": False,
        "searched": [str(item) for item in roots],
        "hint": (
            "Add the dataset to this notebook (Add Data -> awsaf49/brats2020-training-data). "
            "If it is mounted elsewhere, put the path first in RUN['dataset_roots']."
        ),
    }


DATASET = discover_dataset()
LOCK["dataset_discovery"] = DATASET

if not DATASET["found"]:
    message = [
        "BraTS 2020 data was not found in this environment.",
        f"searched: {DATASET['searched']}",
        DATASET["hint"],
        "",
        "Nothing else in this notebook can run without the dataset: no metrics are produced,",
        "no model is trained, and no synthetic substitute is used. Kaggle dataset:",
        "https://www.kaggle.com/datasets/awsaf49/brats2020-training-data",
    ]
    print("\n".join(message))
    raise SystemExit("Dataset missing -- attach it and re-run this cell.")

DATA_ROOT = Path(DATASET["root"])
GRADE_TABLE_CANDIDATES = [
    DATA_ROOT / "name_mapping.csv",
    DATA_ROOT.parent / "name_mapping.csv",
    DATA_ROOT.parent.parent / "name_mapping.csv",
    Path(DATASET["searched"][0]) / "name_mapping.csv",
    Path(DATASET["searched"][0]) / "BraTS2020_training_data" / "name_mapping.csv",
]
GRADE_TABLE = next((path for path in GRADE_TABLE_CANDIDATES if path.exists()), None)
SLICE_FILES = sorted(DATA_ROOT.glob("volume_*/volume_*_slice_*.h5")) or sorted(
    DATA_ROOT.glob("*/volume_*_slice_*.h5")
)

print(f"Dataset root   : {DATA_ROOT}")
print(f"Slice files    : {len(SLICE_FILES)}")
print(f"Grade table    : {GRADE_TABLE if GRADE_TABLE else 'NOT FOUND (labels unavailable -> classification section will refuse to run)'}")
if not SLICE_FILES:
    raise SystemExit("No HDF5 slice files matched the expected pattern.")
# --- what is this dataset? -----------------------------------------------------------
# Every artefact this notebook exports says whether it was trained on the real dataset or
# on a synthetic fixture. That claim is not decoration: it is what the application shows
# next to every prediction, so it is derived from the data and never assumed.
VOLUME_DIRECTORIES = sorted({path.parent.name for path in SLICE_FILES})
FIXTURE_MARKER = None
try:
    import h5py  # also imported by the structure-inspection cell below

    with h5py.File(SLICE_FILES[0], "r") as handle:
        FIXTURE_MARKER = dict(handle.attrs).get("source")
except Exception as exc:  # unreadable file -> not a provenance claim
    print("Could not read the HDF5 attributes of the first slice file:", exc)


def resolve_dataset_provenance(declared, marker, volumes):
    """Return (is_synthetic, note); None means "cannot be established, do not claim real data"."""
    declared = str(declared or "auto").lower()
    if declared in {"synthetic", "synthetic_fixture", "fixture"}:
        return True, "declared in RUN['dataset_provenance']"
    if declared in {"real", "real_brats2020", "brats2020"}:
        return False, "declared in RUN['dataset_provenance']"
    if marker == "synthetic-fixture":
        return True, "HDF5 file attribute source='synthetic-fixture' (scripts/make_brats_fixture.py)"
    if volumes >= 100:
        return False, f"{volumes} patient volumes with a name_mapping.csv, matching the BraTS 2020 release"
    return None, (
        f"neither the fixture marker nor a full BraTS release ({volumes} volumes); set "
        "RUN['dataset_provenance'] explicitly if this really is the real dataset"
    )


DATASET_IS_SYNTHETIC, DATASET_PROVENANCE_NOTE = resolve_dataset_provenance(
    RUN.get("dataset_provenance", "auto"), FIXTURE_MARKER, len(VOLUME_DIRECTORIES)
)
DATASET_NAME = (
    "Synthetic BraTS-format fixture (scripts/make_brats_fixture.py; the labels and masks are "
    "generated, any metric computed from them is meaningless)"
    if DATASET_IS_SYNTHETIC
    else "BraTS 2020 training data (Kaggle mirror awsaf49/brats2020-training-data)"
    if DATASET_IS_SYNTHETIC is False
    else f"Unverified HDF5 tree at {DATA_ROOT} (provenance not established)"
)
print("Patient volumes:", len(VOLUME_DIRECTORIES))
print(
    "Dataset provenance:",
    {True: "SYNTHETIC FIXTURE", False: "real BraTS 2020", None: "unknown"}[DATASET_IS_SYNTHETIC],
    "--",
    DATASET_PROVENANCE_NOTE,
)
if DATASET_IS_SYNTHETIC:
    print(
        "\n" + "!" * 78 + "\n"
        "WARNING: this is the synthetic fixture. Every number this notebook prints describes\n"
        "made-up images, made-up masks and made-up HGG/LGG labels. Nothing in the exported\n"
        "artefacts may be quoted as a result, and the application will label them\n"
        "'trained on a synthetic fixture'.\n" + "!" * 78
    )
LOCK["dataset"] = {
    "root": str(DATA_ROOT),
    "slice_files": len(SLICE_FILES),
    "volumes": len(VOLUME_DIRECTORIES),
    "grade_table": str(GRADE_TABLE) if GRADE_TABLE else None,
    "synthetic": DATASET_IS_SYNTHETIC,
    "provenance_note": DATASET_PROVENANCE_NOTE,
}


## 4. Dataset structure inspection
`nbs_02_data.py`


In [ ]:
# =============================================================================
# SECTION 3 - Structure inspection (keys, shapes, dtypes, channel order, masks)
# =============================================================================
import h5py

def inspect_slice_file(path):
    with h5py.File(path, "r") as handle:
        keys = list(handle.keys())
        info = {"path": str(path), "keys": keys, "file_attributes": dict(handle.attrs)}
        for key in keys:
            dataset = handle[key]
            info[key] = {
                "shape": list(dataset.shape),
                "dtype": str(dataset.dtype),
                "attributes": {name: str(value) for name, value in dataset.attrs.items()},
            }
    return info


SAMPLE_COUNT = min(8, len(SLICE_FILES))
STRUCTURE_SAMPLES = []
for path in SLICE_FILES[:SAMPLE_COUNT]:
    STRUCTURE_SAMPLES.append(inspect_slice_file(path))

first = STRUCTURE_SAMPLES[0]
shape_variants = {}
key_variants = {}
mask_shape_variants = {}
for sample in STRUCTURE_SAMPLES:
    key_variants[tuple(sorted(sample["keys"]))] = key_variants.get(tuple(sorted(sample["keys"])), 0) + 1
    if "image" in sample:
        shape_variants[tuple(sample["image"]["shape"])] = shape_variants.get(tuple(sample["image"]["shape"]), 0) + 1
    if "mask" in sample:
        mask_shape_variants[tuple(sample["mask"]["shape"])] = mask_shape_variants.get(tuple(sample["mask"]["shape"]), 0) + 1

with h5py.File(SLICE_FILES[0], "r") as handle:
    image0 = np.asarray(handle["image"][...])
    mask0 = np.asarray(handle["mask"][...])

STRUCTURE = {
    "sampled_files": SAMPLE_COUNT,
    "key_variants": {str(key): value for key, value in key_variants.items()},
    "image_shape_variants": {str(key): value for key, value in shape_variants.items()},
    "mask_shape_variants": {str(key): value for key, value in mask_shape_variants.items()},
    "image_dtype": str(image0.dtype),
    "mask_dtype": str(mask0.dtype),
    "mask_unique_values": [int(item) for item in np.unique(mask0)[:12]],
    "image_range": [float(image0.min()), float(image0.max())],
    "sample_attributes": STRUCTURE_SAMPLES[0],
}
write_json(PATHS["artifacts"] / "dataset_structure.json", STRUCTURE)

print("=" * 78)
print("STRUCTURE")
print("=" * 78)
print("HDF5 keys        :", STRUCTURE["key_variants"])
print("image shapes     :", STRUCTURE["image_shape_variants"])
print("mask shapes      :", STRUCTURE["mask_shape_variants"])
print("image dtype      :", STRUCTURE["image_dtype"], "| mask dtype:", STRUCTURE["mask_dtype"])
print("mask values      :", STRUCTURE["mask_unique_values"])
print("stored image range:", STRUCTURE["image_range"])

if tuple(sorted(STRUCTURE_SAMPLES[0]["keys"])) != ("image", "mask"):
    print(
        "\nWARNING: the HDF5 keys are not exactly ('image', 'mask'). The readers below look for "
        "these two names; adjust `IMAGE_KEY` / `MASK_KEY` if your copy differs."
    )
IMAGE_KEY, MASK_KEY = "image", "mask"

# The mirror stores one binary plane per tumour sub-region (NCR/NET, oedema, enhancing)
# or a single label map. Both are handled: the training target is always the union,
# because which plane means which sub-region is not declared inside the file.
MASK_KIND = "one_hot_channels" if mask0.ndim == 3 and mask0.shape[-1] == 3 else "single_or_label_map"
print("mask encoding    :", MASK_KIND, f"(mask shape {mask0.shape})")
print("channels         : T1, T1ce, T2, FLAIR in the stored order -- the notebook never reorders them")


## 5. Patient identifiers and label verification
`nbs_02_data.py`


In [ ]:
# =============================================================================
# SECTION 4 - Patient identifiers and HGG/LGG label verification
# =============================================================================
import csv
import re

VOLUME_PATTERN = re.compile(r"(volume_\d+)")


def volume_key_of(path):
    for part in reversed(path.parts):
        match = VOLUME_PATTERN.search(part)
        if match:
            return match.group(1)
    match = VOLUME_PATTERN.search(path.stem)
    return match.group(1) if match else path.parent.name


def slice_index_of(path):
    match = re.search(r"slice_(\d+)", path.stem)
    return int(match.group(1)) if match else None


def load_grade_table(path):
    """name_mapping.csv -> {identifier: {'grade': 'HGG'|'LGG'|None, 'raw': ...}}"""
    if path is None:
        return {}
    rows = list(csv.DictReader(path.open(encoding="utf-8-sig")))
    if not rows:
        return {}
    columns = list(rows[0].keys())
    id_column = next((c for c in columns if "brats" in c.lower() and "id" in c.lower()), None)
    id_column = id_column or next((c for c in columns if "id" in c.lower() or "name" in c.lower()), columns[0])
    grade_column = next((c for c in columns if "grade" in c.lower()), None)
    if grade_column is None:
        return {}
    table = {}
    for row in rows:
        identifier = str(row.get(id_column, "")).strip()
        if not identifier:
            continue
        raw = str(row.get(grade_column, "")).strip()
        grade = raw.upper() if raw.upper() in {"HGG", "LGG"} else None
        table[identifier] = {"grade": grade, "raw": raw, "id_column": id_column, "grade_column": grade_column}
    return table


GRADES = load_grade_table(GRADE_TABLE) if GRADE_TABLE else {}
GRADE_BY_VOLUME = {}
for identifier, entry in GRADES.items():
    GRADE_BY_VOLUME[identifier.lower()] = entry
    case_insensitive = identifier.strip().lower()
    GRADE_BY_VOLUME[case_insensitive] = entry
    # BraTS_2020_subject_ID / BraTS20_Training_001 style identifiers also map by number
    number_match = re.search(r"(\d{3,4})$", identifier)
    if number_match:
        GRADE_BY_VOLUME[f"volume_{int(number_match.group(1))}"] = entry

VOLUMES = {}
for path in SLICE_FILES:
    key = volume_key_of(path)
    entry = VOLUMES.setdefault(key, {"volume": key, "slices": [], "grade": None, "grade_raw": None})
    entry["slices"].append(path)
for key, entry in VOLUMES.items():
    label_entry = GRADE_BY_VOLUME.get(key.lower()) or GRADE_BY_VOLUME.get(key)
    entry["grade"] = (label_entry or {}).get("grade")
    entry["grade_raw"] = (label_entry or {}).get("raw")
    entry["slices"].sort(key=lambda item: (slice_index_of(item) is None, slice_index_of(item)))

LABELLED = {key: entry for key, entry in VOLUMES.items() if entry["grade"] in {"HGG", "LGG"}}
UNLABELLED = sorted(key for key, entry in VOLUMES.items() if entry["grade"] not in {"HGG", "LGG"})
CLASS_COUNTS = {}
for entry in LABELLED.values():
    CLASS_COUNTS[entry["grade"]] = CLASS_COUNTS.get(entry["grade"], 0) + 1

LABEL_REPORT = {
    "volumes": len(VOLUMES),
    "slices": len(SLICE_FILES),
    "grade_table": str(GRADE_TABLE) if GRADE_TABLE else None,
    "grade_table_rows": len(GRADES),
    "labelled_patients": len(LABELLED),
    "unlabelled_patients": len(UNLABELLED),
    "class_counts": CLASS_COUNTS,
    "label_level": "patient (a slice inherits its volume's grade; a mask can never carry a grade)",
    "slices_per_volume": {
        "min": min(len(entry["slices"]) for entry in VOLUMES.values()),
        "max": max(len(entry["slices"]) for entry in VOLUMES.values()),
    },
}
write_json(PATHS["artifacts"] / "label_verification.json", LABEL_REPORT)

print("=" * 78)
print("IDENTIFIERS AND LABELS")
print("=" * 78)
print("Volumes (patients):", len(VOLUMES), "| slices:", len(SLICE_FILES))
print("Grade table rows  :", len(GRADES), "-> labelled volumes:", len(LABELLED))
print("Class counts      :", CLASS_COUNTS)
print("Unlabelled        :", len(UNLABELLED), UNLABELLED[:5])
print("Slices per volume :", LABEL_REPORT["slices_per_volume"])
print(
    "\nHGG/LGG is the WHO grade group recorded by the dataset for a patient. It is not a\n"
    "malignancy verdict, not a stage and not a prognosis, and it is never derived from a mask."
)

if not GRADE_TABLE:
    print(
        "\nWARNING: no grade table was found, so classification cannot be trained here. "
        "Segmentation sections still run. Set RUN/GRADE_TABLE once the CSV is available."
    )
elif len(LABELLED) < 20:
    print("\nWARNING: fewer than 20 labelled patients -- classification metrics would not mean much.")

CLASSIFICATION_POSSIBLE = bool(GRADE_TABLE) and len(LABELLED) >= 2


## 6. Exploratory data analysis
`nbs_02_data.py`


In [ ]:
# =============================================================================
# SECTION 5 - Exploratory data analysis (per-patient slice counts, tumour slices,
# intensity distributions, class balance)
# =============================================================================
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


def tumour_fraction(path):
    with h5py.File(path, "r") as handle:
        mask = np.asarray(handle[MASK_KEY][...])
    return float((mask > 0).any(axis=-1).mean()) if mask.ndim == 3 else float((mask > 0).mean())


EDA_SAMPLE = min(400, len(SLICE_FILES))
step = max(1, len(SLICE_FILES) // EDA_SAMPLE)
EDA_FILES = SLICE_FILES[::step][:EDA_SAMPLE]
eda_rows = []
for path in EDA_FILES:
    with h5py.File(path, "r") as handle:
        image = np.asarray(handle[IMAGE_KEY][...], dtype=np.float32)
        mask = np.asarray(handle[MASK_KEY][...])
    whole = (mask > 0).any(axis=-1) if mask.ndim == 3 else (mask > 0)
    eda_rows.append(
        {
            "volume": volume_key_of(path),
            "slice": slice_index_of(path),
            "tumour_fraction": float(whole.mean()),
            "has_tumour": bool(whole.any()),
            "mean_intensity": float(image.mean()),
            "channel_means": [float(image[..., c].mean()) for c in range(image.shape[-1])],
        }
    )

tumour_rows = [row for row in eda_rows if row["has_tumour"]]
fractions = [row["tumour_fraction"] for row in tumour_rows] or [0.0]
per_volume = {}
for row in eda_rows:
    per_volume.setdefault(row["volume"], 0)
    per_volume[row["volume"]] += 1

EDA = {
    "sampled_slices": len(eda_rows),
    "sampled_patients": len(per_volume),
    "slices_with_tumour": len(tumour_rows),
    "slices_without_tumour": len(eda_rows) - len(tumour_rows),
    "tumour_fraction_mean": float(np.mean(fractions)),
    "tumour_fraction_median": float(np.median(fractions)),
    "slices_per_patient_percentiles": [
        float(item) for item in np.percentile(list(per_volume.values()) or [0], [0, 10, 50, 90, 100])
    ],
    "channel_mean_intensity": [float(np.mean([row["channel_means"][c] for row in eda_rows])) for c in range(4)],
    "class_counts": CLASS_COUNTS,
}
write_json(PATHS["artifacts"] / "eda.json", EDA)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].hist([row["tumour_fraction"] for row in eda_rows], bins=40, color="#0ea5e9")
axes[0, 0].set_title("Fraction of the slice covered by tumour")
axes[0, 0].set_xlabel("tumour pixels / total pixels")
axes[0, 1].hist(list(per_volume.values()), bins=30, color="#8b5cf6")
axes[0, 1].set_title("Slices per patient (sampled)")
axes[0, 1].set_xlabel("slices")
axes[1, 0].bar(list(CLASS_COUNTS.keys()) or ["no labels"], list(CLASS_COUNTS.values()) or [0], color="#14b8a6")
axes[1, 0].set_title("Patients per grade (HGG/LGG)")
channel_means = EDA["channel_mean_intensity"]
axes[1, 1].bar(["T1", "T1ce", "T2", "FLAIR"][: len(channel_means)], channel_means, color="#f59e0b")
axes[1, 1].set_title("Mean raw intensity per channel")
plt.tight_layout()
plt.savefig(PATHS["reports"] / "01_eda.png", dpi=130)
plt.close(fig)

print("=" * 78)
print("EXPLORATORY SUMMARY")
print("=" * 78)
for key in ("sampled_slices", "sampled_patients", "slices_with_tumour", "slices_without_tumour", "tumour_fraction_mean", "tumour_fraction_median"):
    print(f"{key:26s}: {EDA[key]}")
print("class counts              :", CLASS_COUNTS)
print(f"figure saved              : {PATHS['reports'] / '01_eda.png'}")


## 7. MRI visualisation
`nbs_02_data.py`


In [ ]:
# =============================================================================
# SECTION 6 - MRI visualisation (the four modalities of a patient)
# =============================================================================
def read_slice(path):
    with h5py.File(path, "r") as handle:
        image = np.asarray(handle[IMAGE_KEY][...], dtype=np.float32)
        mask = np.asarray(handle[MASK_KEY][...])
    channels_last = image.ndim == 3 and image.shape[-1] <= 8
    image = image if channels_last or image.ndim == 2 else np.moveaxis(image, 0, -1)
    if image.ndim == 2:
        image = image[..., None]
    whole = ((mask > 0).any(axis=-1) if mask.ndim == 3 else (mask > 0))
    return image, whole


EXAMPLE_VOLUME = sorted(LABELLED or VOLUMES)[0]
EXAMPLE_ENTRY = VOLUMES[EXAMPLE_VOLUME]
print(f"Example patient: {EXAMPLE_VOLUME} (grade {EXAMPLE_ENTRY['grade']}) with {len(EXAMPLE_ENTRY['slices'])} slices")

tumour_slice_path = None
for path in EXAMPLE_ENTRY["slices"]:
    _, whole = read_slice(path)
    if whole.any():
        tumour_slice_path = path
        break
if tumour_slice_path is None:
    tumour_slice_path = EXAMPLE_ENTRY["slices"][len(EXAMPLE_ENTRY["slices"]) // 2]
EXAMPLE_IMAGE, EXAMPLE_MASK = read_slice(tumour_slice_path)
print("Showing", tumour_slice_path)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for index, name in enumerate(["T1", "T1ce", "T2", "FLAIR"]):
    if index >= EXAMPLE_IMAGE.shape[-1]:
        axes[index].axis("off")
        continue
    axes[index].imshow(EXAMPLE_IMAGE[..., index], cmap="gray")
    axes[index].set_title(f"{name} (channel {index})")
    axes[index].axis("off")
plt.tight_layout()
plt.savefig(PATHS["reports"] / "02_modalities.png", dpi=130)
plt.close(fig)
print(f"figure saved: {PATHS['reports'] / '02_modalities.png'}")


## 8. Segmentation-mask visualisation
`nbs_02_data.py`


In [ ]:
# =============================================================================
# SECTION 7 - Segmentation-mask visualisation (sub-region planes and the union)
# =============================================================================
with h5py.File(tumour_slice_path, "r") as handle:
    raw_mask = np.asarray(handle[MASK_KEY][...])

fig, axes = plt.subplots(1, 5, figsize=(20, 4.4))
axes[0].imshow(EXAMPLE_IMAGE[..., 0], cmap="gray")
axes[0].set_title("T1")
axes[1].imshow(EXAMPLE_IMAGE[..., 3], cmap="gray")
axes[1].set_title("FLAIR")
if raw_mask.ndim == 3:
    for plane in range(min(3, raw_mask.shape[-1])):
        axes[2 + plane].imshow(raw_mask[..., plane], cmap="magma")
        axes[2 + plane].set_title(f"stored mask plane {plane}")
else:
    axes[2].imshow(raw_mask, cmap="magma")
    axes[2].set_title("stored mask")
axes[4].imshow(EXAMPLE_IMAGE[..., 3], cmap="gray")
axes[4].imshow(np.ma.masked_where(~EXAMPLE_MASK, EXAMPLE_MASK), cmap="autumn", alpha=0.45)
axes[4].set_title("FLAIR + whole tumour (union)")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.savefig(PATHS["reports"] / "03_masks.png", dpi=130)
plt.close(fig)

sub_region_note = (
    "The stored planes are treated as sub-region planes: they are kept separate for reporting, "
    "and only their union is used as the training target, because the file does not declare "
    "which plane is which sub-region."
)
MASK_REPORT = {
    "example_slice": str(tumour_slice_path),
    "stored_shape": list(raw_mask.shape),
    "stored_unique_values": [int(item) for item in np.unique(raw_mask)[:12]],
    "whole_tumour_pixels": int(EXAMPLE_MASK.sum()),
    "note": sub_region_note,
}
write_json(PATHS["artifacts"] / "mask_visualisation.json", MASK_REPORT)
print(sub_region_note)
print("stored mask values:", MASK_REPORT["stored_unique_values"], "| whole-tumour pixels:", MASK_REPORT["whole_tumour_pixels"])
print(f"figure saved: {PATHS['reports'] / '03_masks.png'}")


## 9. Data validation and preprocessing (resumable cache)
`nbs_02_data.py`


In [ ]:
# =============================================================================
# SECTION 8 - Validation and preprocessing
# =============================================================================
# For every patient volume:
#   * read each slice, build the whole-tumour mask (union of every non-zero label),
#   * resize to the size the model consumes (nearest for masks, bilinear for images),
#   * normalise exactly as the application does:
#       4-channel models : global z-score over the (H, W, C) block, then CHW
#       single-channel   : intensity / max-style scaling to [0, 1]  (the shipped
#                          U-Net contract is "image / 255")
#   * store float16 arrays so the whole cache fits comfortably on the Kaggle disk.
#
# Two on-disk layouts, one reader. The difference is measured at the bottom of this cell,
# not assumed:
#
#   "npy" (default)   cache/<volume>.cache/{images,masks,singles}.npy + meta.json
#                     One memory-mappable file per member: reading slice i reads slice i.
#   "npz" (legacy)    cache/<volume>.npz, a savez_compressed archive. ANY access to a
#                     member inflates that whole member, so reading a single slice of a
#                     real BraTS volume costs a full decompression of images + masks +
#                     singles (165 MiB). A cache written by an earlier version of this
#                     notebook still reads, and `upgrade_patient_cache()` converts it in
#                     place -- verifying every array before it deletes anything.
#
# Both layouts are read through `load_patient()` and `PATIENT_STORE`, so every other
# section of the notebook is layout-agnostic and an un-migrated cache still trains.
# =============================================================================
from PIL import Image as PILImage
from collections import OrderedDict
import shutil
import zipfile


def resize2d(array, size, nearest=False):
    if array.shape[0] == size and array.shape[1] == size:
        return array.astype(np.float32)
    resample = PILImage.NEAREST if nearest else PILImage.BILINEAR
    image = PILImage.fromarray(array.astype(np.float32), mode="F")
    resized = image.resize((size, size), resample=resample)
    return np.asarray(resized, dtype=np.float32)


def zscore_channels(block):
    """Global z-score over the (H, W, C) block, matching the application contract."""
    mean = float(block.mean())
    std = float(block.std()) or 1.0
    return (block - mean) / std


def resize2d_channels(image, size):
    return np.stack([resize2d(image[..., c], size) for c in range(image.shape[-1])], axis=-1)


def single_pair(flair, size):
    """Return the 512x512 single-channel input and the [0, 1] scaling used for it."""
    resized = resize2d(flair, size)
    low, high = float(resized.min()), float(resized.max())
    if high <= low:
        scaled = np.zeros_like(resized)
    elif high <= 1.0 + 1e-6:
        scaled = resized  # already [0, 1]
    elif high <= 255.1:
        scaled = resized / 255.0  # the / 255 contract of the shipped U-Net
    else:
        scaled = (resized - low) / (high - low)
    return scaled.astype(np.float32)


class CacheError(RuntimeError):
    pass


# =============================================================================
# Cache layout
# =============================================================================
CACHE_MEMBERS = ("images", "masks", "singles")
MEMORY_MAP_FORMAT = "npy"
ARCHIVE_FORMAT = "npz"
CACHE_INDEX_NAME = "_cache_index.json"   # the underscore keeps it out of the patient namespace
CACHE_DIRECTORY_SUFFIX = ".cache"


def cache_archive(volume_key):
    """The compressed single-file layout (what earlier versions of this notebook wrote)."""
    return PATHS["cache"] / f"{volume_key}.npz"


def cache_directory(volume_key):
    """The memory-mappable per-patient layout."""
    return PATHS["cache"] / f"{volume_key}{CACHE_DIRECTORY_SUFFIX}"


def cache_member_path(volume_key, member):
    return cache_directory(volume_key) / f"{member}.npy"


def cache_meta_path(volume_key):
    return cache_directory(volume_key) / "meta.json"


def cache_layout(volume_key):
    """`"npy"`, `"npz"` or None. The memory-mappable layout wins when both are present."""
    if cache_member_path(volume_key, "images").exists() and cache_member_path(volume_key, "masks").exists():
        return MEMORY_MAP_FORMAT
    if cache_archive(volume_key).exists():
        return ARCHIVE_FORMAT
    return None


def cache_entry_path(volume_key):
    """Where this patient lives now."""
    return cache_directory(volume_key) if cache_layout(volume_key) == MEMORY_MAP_FORMAT else cache_archive(volume_key)


def cached_volumes():
    """Every patient key present in the cache, whatever layout it is in."""
    keys = set()
    for path in PATHS["cache"].glob(f"*{CACHE_DIRECTORY_SUFFIX}"):
        if path.is_dir():
            keys.add(path.name[: -len(CACHE_DIRECTORY_SUFFIX)])
    for path in PATHS["cache"].glob(f"*.{ARCHIVE_FORMAT}"):
        keys.add(path.stem)
    return sorted(keys)


def round_mib(value):
    return round(value / 1024 ** 2, 1)


def layout_bytes(volume_key):
    """What the cached payload of one patient costs on disk."""
    layout = cache_layout(volume_key)
    if layout == MEMORY_MAP_FORMAT:
        return sum(path.stat().st_size for path in cache_directory(volume_key).glob("*.npy"))
    if layout == ARCHIVE_FORMAT:
        return cache_archive(volume_key).stat().st_size
    return 0


# =============================================================================
# Slice counts without reading the arrays
# =============================================================================
# The old code learned how many slices a patient has by loading the whole payload:
#
#     for volume_key in patient_ids:
#         payload = load_patient(volume_key)              # a whole archive inflation
#         index.extend(... for position in range(payload["masks"].shape[0]))
#
# For the 369-volume BraTS 2020 release that is 369 archive inflations per epoch, for
# training AND for every validation pass, to read one integer per patient. Both layouts
# expose the shape in a header, so it costs a header read instead.


def read_npy_header(path):
    """(shape, dtype) of a .npy file, without reading its data."""
    with open(path, "rb") as handle:
        version = np.lib.format.read_magic(handle)
        reader = np.lib.format.read_array_header_1_0 if version == (1, 0) else np.lib.format.read_array_header_2_0
        shape, fortran_order, dtype = reader(handle)
    return tuple(shape), np.dtype(dtype)


def read_archive_member_header(path, member):
    """(shape, dtype) of one member of an .npz archive, inflating only its header."""
    with zipfile.ZipFile(path) as archive:
        with archive.open(f"{member}.npy") as handle:
            version = np.lib.format.read_magic(handle)
            reader = np.lib.format.read_array_header_1_0 if version == (1, 0) else np.lib.format.read_array_header_2_0
            shape, fortran_order, dtype = reader(handle)
    return tuple(shape), np.dtype(dtype)


def probe_patient(volume_key):
    """Slice count, member shapes and dtypes of a cached patient. Touches headers only."""
    layout = cache_layout(volume_key)
    if layout is None:
        return None
    members = {}
    if layout == MEMORY_MAP_FORMAT:
        for member in CACHE_MEMBERS:
            path = cache_member_path(volume_key, member)
            if path.exists():
                members[member] = read_npy_header(path)
    else:
        archive = cache_archive(volume_key)
        with zipfile.ZipFile(archive) as handle:
            available = set(handle.namelist())
        for member in CACHE_MEMBERS:
            if f"{member}.npy" in available:
                members[member] = read_archive_member_header(archive, member)
    if not members:
        return None
    return {
        "volume": volume_key,
        "layout": layout,
        "slices": int(members.get("masks", members["images"])[0][0]),
        "members": {member: list(shape) for member, (shape, _) in members.items()},
        "dtypes": {member: str(dtype) for member, (_, dtype) in members.items()},
        "bytes": layout_bytes(volume_key),
    }


CACHE_INDEX: dict = {}
CACHE_INDEX_PATH = PATHS["cache"] / CACHE_INDEX_NAME
CACHE_INDEX_PROBES = {"header_reads": 0, "arrays_read": 0}


def load_cache_index(rebuild=False):
    """Per-patient slice counts: from the index file, else one header read per patient."""
    global CACHE_INDEX
    if CACHE_INDEX and not rebuild:
        return CACHE_INDEX
    stored = read_json(CACHE_INDEX_PATH, default=None) or {}
    entries = {}
    for volume_key in cached_volumes():
        entry = stored.get(volume_key)
        if entry and entry.get("layout") == cache_layout(volume_key):
            entries[volume_key] = entry
            continue
        probed = probe_patient(volume_key)
        CACHE_INDEX_PROBES["header_reads"] += 1
        if probed is not None:
            entries[volume_key] = probed
    CACHE_INDEX = entries
    return CACHE_INDEX


def write_cache_index():
    write_json(CACHE_INDEX_PATH, {key: value for key, value in sorted(CACHE_INDEX.items())})
    return CACHE_INDEX


def cache_index_entry(volume_key, refresh=False):
    if not CACHE_INDEX:
        load_cache_index()
    entry = CACHE_INDEX.get(volume_key)
    if entry and not refresh and entry.get("layout") == cache_layout(volume_key):
        return entry
    probed = probe_patient(volume_key)
    CACHE_INDEX_PROBES["header_reads"] += 1
    if probed is not None:
        CACHE_INDEX[volume_key] = probed
    return probed


def patient_slice_count(volume_key):
    entry = cache_index_entry(volume_key)
    if entry is None:
        raise CacheError(f"{volume_key} is not cached; run the preprocessing cell first")
    return int(entry["slices"])


# =============================================================================
# Writing the cache
# =============================================================================
def write_npy_part(path, array):
    """Write `<path>.part` and return it. Nothing at `path` changes until it is published."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    part = path.with_name(path.name + ".part")
    if part.exists():
        part.unlink()
    with open(part, "wb") as handle:
        np.save(handle, array, allow_pickle=False)
    return part


def parts_of(volume_key):
    return {member: cache_member_path(volume_key, member).with_name(f"{member}.npy.part") for member in CACHE_MEMBERS}


def clean_parts(volume_key):
    for part in parts_of(volume_key).values():
        if part.exists():
            part.unlink()


def write_cache_members(volume_key, arrays, *, verify=True):
    """Write every member of one patient, and only publish them once they are all complete.

    `cache_layout()` reports this patient as cached as soon as `images.npy` and `masks.npy`
    exist, so those two are published last: a session that dies (or a conversion that fails)
    can leave `.part` files or a partial set behind, but it can never leave a cache that
    looks complete while a member is missing. With `verify=True` every part file is read
    back and compared value by value before it is published.
    """
    parts = {}
    for member in ("singles", "masks", "images"):        # `images` is published last on purpose
        if member in arrays:
            parts[member] = write_npy_part(cache_member_path(volume_key, member), arrays[member])
    if verify:
        for member, part in parts.items():
            written = np.load(part, mmap_mode="r")
            same = (
                written.shape == np.shape(arrays[member])
                and written.dtype == np.asarray(arrays[member]).dtype
                and np.array_equal(np.asarray(written), arrays[member])
            )
            del written
            if not same:
                clean_parts(volume_key)
                raise CacheError(f"{volume_key}: the {member} member did not survive the write")
    for member in ("singles", "masks", "images"):
        if member in parts:
            parts[member].replace(cache_member_path(volume_key, member))
    return {member: cache_member_path(volume_key, member) for member in parts}


def build_patient_cache(volume_key, entry, size, single_size, single_channel, force=False):
    layout = cache_layout(volume_key)
    if layout is not None and not force:
        return {"volume": volume_key, "cached": True, "layout": layout, "path": str(cache_entry_path(volume_key))}
    channels, singles, masks = [], [], []
    for path in entry["slices"]:
        image, whole = read_slice(path)
        channels.append(zscore_channels(resize2d_channels(image, size)))
        masks.append(resize2d(whole.astype(np.float32), size, nearest=True) > 0.5)
        flair = image[..., single_channel] if image.shape[-1] > single_channel else image[..., 0]
        pairs = single_pair(flair, single_size)
        singles.append(pairs)
    arrays = {
        "images": np.stack(channels).astype(np.float16),
        "masks": np.stack(masks).astype(np.uint8),
        "singles": np.stack(singles).astype(np.float16),
    }
    slice_paths = [str(path) for path in entry["slices"]]
    grade = entry["grade"] or ""

    if RUN.get("cache_format", MEMORY_MAP_FORMAT) == ARCHIVE_FORMAT:
        target = cache_archive(volume_key)
        np.savez_compressed(target, grade=np.array([grade]), slices=np.array(slice_paths), **arrays)
        written, layout = target, ARCHIVE_FORMAT
    else:
        try:
            write_cache_members(volume_key, arrays)
        except CacheError as exc:
            return {"volume": volume_key, "cached": False, "layout": MEMORY_MAP_FORMAT, "error": str(exc)}
        write_json(cache_meta_path(volume_key), {"volume": volume_key, "grade": grade, "slices": slice_paths})
        written, layout = cache_directory(volume_key), MEMORY_MAP_FORMAT

    CACHE_INDEX[volume_key] = {
        "volume": volume_key,
        "layout": layout,
        "slices": len(masks),
        "members": {member: list(array.shape) for member, array in arrays.items()},
        "dtypes": {member: str(array.dtype) for member, array in arrays.items()},
        "bytes": layout_bytes(volume_key),
    }
    return {
        "volume": volume_key,
        "cached": False,
        "layout": layout,
        "path": str(written),
        "slices": len(masks),
        "size_bytes": layout_bytes(volume_key),
    }


def upgrade_patient_cache(volume_keys=None, *, headroom_bytes=512 * 1024 ** 2, verbose=True):
    """Convert legacy .npz archives to the memory-mappable layout, in place and verified.

    One-time, and it costs one archive inflation per patient -- the same read the old
    training loop paid for EVERY slice and for every patient just to count slices. Nothing
    is deleted before it has been verified:

      1. the arrays are written to `<member>.npy.part`;
      2. each written file is memory-mapped and compared with the archive member by shape,
         dtype and every value;
      3. only then is the part file renamed, i.e. the member becomes visible to the reader;
      4. only when every member is complete is the archive deleted.

    A patient whose conversion fails keeps its archive and is listed in `failed`; the reader
    keeps serving it from the archive. Free space is checked per patient first, so the peak
    cost is one patient's payload rather than the whole cache.
    """
    report = {"converted": [], "already": [], "failed": [], "bytes_before": 0, "bytes_after": 0, "seconds": 0.0}
    candidates = [
        volume_key
        for volume_key in (volume_keys if volume_keys is not None else cached_volumes())
        if cache_layout(volume_key) == ARCHIVE_FORMAT
    ]
    if not candidates:
        return report
    started = time.time()
    for index, volume_key in enumerate(candidates, start=1):
        archive = cache_archive(volume_key)
        try:
            size = archive.stat().st_size
            free = shutil.disk_usage(PATHS["cache"]).free
            if free < size + headroom_bytes:
                report["failed"].append(
                    {"volume": volume_key, "reason": f"only {free / 1024 ** 2:.0f} MiB free, the archive needs {size / 1024 ** 2:.0f} MiB"}
                )
                continue
            with np.load(archive, allow_pickle=False) as payload:
                sources = {member: np.asarray(payload[member]) for member in CACHE_MEMBERS if member in payload.files}
                meta = {
                    "volume": volume_key,
                    "grade": str(payload["grade"][0]) if "grade" in payload.files else "",
                    "slices": [str(item) for item in payload["slices"]] if "slices" in payload.files else [],
                }
            # Every member is written, verified against the archive, and only then published;
            # `images.npy` is published last, so the patient only becomes "memory-mappable"
            # once all of it is on disk.
            write_cache_members(volume_key, sources, verify=True)
            write_json(cache_meta_path(volume_key), meta)
            archive.unlink()
            report["converted"].append(volume_key)
            report["bytes_before"] += size
            report["bytes_after"] += layout_bytes(volume_key)
        except Exception as exc:  # noqa: BLE001 - a failed conversion must never lose the archive
            clean_parts(volume_key)
            report["failed"].append({"volume": volume_key, "reason": f"{type(exc).__name__}: {exc}"})
        if verbose and (index % 25 == 0 or index == len(candidates)):
            done = time.time() - started
            print(
                f"  cache upgrade {index}/{len(candidates)} ({done:.0f}s, {done / index:.2f}s per volume) -- "
                f"{len(report['converted'])} converted, {len(report['failed'])} failed"
            )
    report["seconds"] = round(time.time() - started, 2)
    report["bytes_before"] = int(report["bytes_before"])
    report["bytes_after"] = int(report["bytes_after"])
    if report["failed"]:
        print(
            f"WARNING: {len(report['failed'])} volumes kept their .npz archive "
            f"({report['failed'][0]['reason']}). They still load, just more slowly."
        )
    return report


# =============================================================================
# Reading the cache: one payload per patient, one member per request
# =============================================================================
class PatientPayloadStore:
    """Bounded reader for the cached patient payloads.

    What went wrong before this class existed: `iterate_batches()` called `load_patient()`
    once per patient (to count slices) and once per SLICE (to build the batch), and every
    call inflated that patient's whole archive -- images + masks + singles, 165 MiB for a
    real BraTS volume -- to look at one slice. A batch of 8 therefore cost 20 archive
    inflations. On the reported run that measured 109.7 s for the first batch; the profile
    put 100% of it inside `zipfile.read` (deflate), and the steady-state cost was 7.7 s per
    batch of 8 -- about 1.6 hours of CPU per epoch, with the GPU waiting the whole time.

    What this class does instead:

      * reads are counted per PATIENT, not per slice -- a batch of 8 slices from 8 patients
        costs at most 8 archive reads, and slices that share a patient share one read;
      * only the members the caller asks for are touched: a 4-channel batch never opens
        `singles` (half the payload), a single-channel batch never opens `images`;
      * the memory-mappable layout decodes nothing -- slice i is read directly from disk;
      * memory stays bounded: `patient_cache_mb` caps materialised arrays and
        `patient_cache_handles` caps open memory maps, so nothing loads the dataset into RAM.

    `PATIENT_STORE.stats_report()` is what the benchmark prints.
    """

    def __init__(self, *, budget_bytes=None, max_handles=None):
        self.budget_bytes = (
            int(budget_bytes) if budget_bytes is not None else int(RUN.get("patient_cache_mb", 1536)) * 1024 ** 2
        )
        self.max_handles = (
            int(max_handles) if max_handles is not None else int(RUN.get("patient_cache_handles", 32))
        )
        self._handles = OrderedDict()        # volume -> {member: memory map}
        self._entries = OrderedDict()        # volume -> {"layout", "members", "meta", "bytes", "slices"}
        self._materialised_bytes = 0
        self.stats = {
            "payload_reads": 0,
            "member_reads": 0,
            "member_hits": 0,
            "slice_requests": 0,
            "slices_returned": 0,
            "archive_reads": 0,
            "evictions": 0,
            "bytes_materialised": 0,
        }

    # -- bookkeeping ------------------------------------------------------
    def _entry(self, volume_key):
        entry = self._entries.get(volume_key)
        if entry is not None:
            self._entries.move_to_end(volume_key)
        return entry

    def _new_entry(self, volume_key, layout):
        entry = {"layout": layout, "members": {}, "meta": None, "bytes": 0, "slices": None}
        self._entries[volume_key] = entry
        self._trim_materialised()
        return entry

    def _trim_materialised(self):
        while self._materialised_bytes > self.budget_bytes and len(self._entries) > 1:
            _, entry = self._entries.popitem(last=False)
            self._materialised_bytes -= entry["bytes"]
            self.stats["evictions"] += 1

    def _trim_handles(self):
        while len(self._handles) > self.max_handles:
            self._handles.popitem(last=False)

    def _read_member(self, volume_key, member):
        """The member as it is stored: a memory map for the npy layout, an array for an archive.

        Memory maps live only in `_handles`, never in the entry: each open map holds a file
        descriptor, and with 369 patients x 3 members a dict that keeps them all would hit the
        process file-descriptor limit. `_handles` is capped (`patient_cache_handles`), so an
        evicted map is closed by refcounting as soon as its slices have been copied out.
        Materialised arrays (the archive layout, and `load_patient`) live in the entry instead,
        where they are counted against the byte budget.
        """
        layout = cache_layout(volume_key)
        if layout is None:
            raise CacheError(f"{volume_key} is not cached; run the preprocessing cell first")
        entry = self._entry(volume_key) or self._new_entry(volume_key, layout)
        current = entry["members"].get(member)
        if current is not None:
            self.stats["member_hits"] += 1
            return current
        handles = self._handles.get(volume_key)
        if handles and member in handles:
            self.stats["member_hits"] += 1
            self._handles.move_to_end(volume_key)
            return handles[member]
        self.stats["member_reads"] += 1
        if layout == MEMORY_MAP_FORMAT:
            path = cache_member_path(volume_key, member)
            if not path.exists():
                raise CacheError(f"{volume_key}: the cache has no {member} member")
            array = np.load(path, mmap_mode="r")
            self._handles.setdefault(volume_key, {})[member] = array
            self._handles.move_to_end(volume_key)
            self._trim_handles()
            return array
        with np.load(cache_archive(volume_key), allow_pickle=False) as payload:
            self.stats["archive_reads"] += 1
            if member not in payload.files:
                raise CacheError(f"{volume_key}: the archive has no {member} member")
            array = np.asarray(payload[member])
        entry["members"][member] = array
        entry["bytes"] += int(array.nbytes)
        self._materialised_bytes += int(array.nbytes)
        self.stats["bytes_materialised"] += int(array.nbytes)
        self._trim_materialised()
        return array

    def _materialised_member(self, volume_key, member):
        """The member as an in-memory array (what `load_patient()` promises callers).

        The array is returned directly, so it stays valid even if the eviction below or a
        later request pushes this patient out of the store.
        """
        array = self._read_member(volume_key, member)
        if isinstance(array, np.memmap):
            array = np.array(array, copy=True)          # detach from the file
            entry = self._entry(volume_key)
            if entry is not None:
                entry["members"][member] = array
                entry["bytes"] += int(array.nbytes)
                self._materialised_bytes += int(array.nbytes)
                self.stats["bytes_materialised"] += int(array.nbytes)
            handles = self._handles.get(volume_key)
            if handles is not None:                      # this member no longer needs its map
                handles.pop(member, None)
                if not handles:
                    self._handles.pop(volume_key, None)
            self._trim_materialised()
        return array

    def _meta(self, volume_key, layout):
        entry = self._entry(volume_key) or self._new_entry(volume_key, layout)
        if entry["meta"] is not None:
            return entry["meta"]
        if layout == MEMORY_MAP_FORMAT:
            stored = read_json(cache_meta_path(volume_key), default=None) or {}
            meta = {"grade": str(stored.get("grade") or "") or None,
                    "slices": [str(item) for item in stored.get("slices", [])]}
        else:
            with np.load(cache_archive(volume_key), allow_pickle=False) as payload:
                self.stats["archive_reads"] += 1
                grade = str(payload["grade"][0]) if "grade" in payload.files else ""
                slice_paths = [str(item) for item in payload["slices"]] if "slices" in payload.files else []
            meta = {"grade": grade or None, "slices": slice_paths}
        entry["meta"] = meta
        return meta

    # -- public API -------------------------------------------------------
    def release(self, volume_key=None):
        """Forget cached state (the benchmark uses this to measure a cold read)."""
        if volume_key is None:
            self._handles.clear()
            self._entries.clear()
            self._materialised_bytes = 0
            return
        entry = self._entries.pop(volume_key, None)
        if entry is not None:
            self._materialised_bytes -= entry["bytes"]
        self._handles.pop(volume_key, None)

    def slice_count(self, volume_key):
        entry = self._entry(volume_key)
        if entry is not None and entry["slices"] is not None:
            return entry["slices"]
        count = patient_slice_count(volume_key)
        entry = self._entry(volume_key) or self._new_entry(volume_key, cache_layout(volume_key) or MEMORY_MAP_FORMAT)
        entry["slices"] = count
        return count

    def slices(self, volume_key, positions, *, member):
        """Rows `positions` of one member, in the order requested. Nothing else is read."""
        array = self._read_member(volume_key, member)
        self.stats["slice_requests"] += 1
        if len(positions) == 0:
            return np.empty((0,) + tuple(array.shape[1:]), dtype=array.dtype)
        self.stats["slices_returned"] += len(positions)
        rows = array[np.asarray(positions, dtype=np.int64)]
        return np.array(rows, copy=True)      # always a fresh array, never a view into a map

    def payload(self, volume_key):
        """The full payload dict `load_patient()` has always returned.

        The arrays are detached copies: a caller may keep them after the store evicted the
        patient, without holding a file descriptor open.
        """
        self.stats["payload_reads"] += 1
        layout = cache_layout(volume_key)
        if layout is None:
            raise CacheError(f"{volume_key} is not cached; run the preprocessing cell first")
        images = self._materialised_member(volume_key, "images")
        masks = self._materialised_member(volume_key, "masks")
        singles = self._materialised_member(volume_key, "singles")
        meta = self._meta(volume_key, layout)
        return {
            "volume": volume_key,
            "images": images,     # (S, H, W, 4) float16, channels T1, T1ce, T2, FLAIR
            "masks": masks,       # (S, H, W) uint8, whole tumour
            "singles": singles,   # (S, 512, 512) float16, scaled to [0, 1]
            "grade": meta["grade"],
            "slices": meta["slices"],
            "layout": layout,
        }

    def stats_report(self):
        report = dict(self.stats)
        report["budget_mib"] = round_mib(self.budget_bytes)
        report["materialised_mib"] = round_mib(self._materialised_bytes)
        report["open_memory_maps"] = len(self._handles)
        report["max_handles"] = self.max_handles
        return report


PATIENT_STORE = None


def patient_store(*, budget_bytes=None, max_handles=None, reset=False):
    """The process-wide payload store (created on first use, recreatable for tests)."""
    global PATIENT_STORE
    if PATIENT_STORE is None or reset:
        PATIENT_STORE = PatientPayloadStore(budget_bytes=budget_bytes, max_handles=max_handles)
    return PATIENT_STORE


def load_patient(volume_key, split_guard=True, legacy_archive=False):
    """One patient's cached payload.

    `legacy_archive=True` reads the compressed archive directly, which is what this function
    did before the store existed: one full inflation per call. It is kept so the benchmark
    can measure the old behaviour on the same cache, and so an un-migrated cache stays
    readable even if its index cannot be built.
    """
    if legacy_archive:
        path = cache_archive(volume_key)
        if not path.exists():
            raise CacheError(f"{volume_key} has no .npz archive to read in legacy mode")
        with np.load(path, allow_pickle=False) as payload:
            return {
                "volume": volume_key,
                "images": np.asarray(payload["images"]),
                "masks": np.asarray(payload["masks"]),
                "singles": np.asarray(payload["singles"]),
                "grade": str(payload["grade"][0]) or None,
                "slices": [str(item) for item in payload["slices"]],
                "layout": ARCHIVE_FORMAT,
            }
    return patient_store().payload(volume_key)


def measure_cache_io(volume_keys, *, slices=4, repeats=3, legacy_repeats=1):
    """Cost of preparing a batch, of reading one slice, and of building the slice index.

    Every row is measured on the same volumes, the same slice positions and the same cache
    state, so the layouts and the two readers can be compared like for like:

      * `legacy_*`   what the old batch loop did -- `load_patient()` per slice, i.e. one
                     whole-archive inflation per slice. Reported only for volumes that
                     still have their `.npz` archive, and `None` otherwise, because it
                     cannot be measured honestly once the archive is gone.
      * `batch_*`    what `iterate_batches()` does now -- one member read per patient plus a
                     direct slice read (or one inflation per patient on an archive).
      * `index_*`    building the slice index from the in-memory index (per epoch).
      * `cold_index_*` the first-time cost: one header probe per patient instead of one
                     payload load per patient.
    """
    store = patient_store()
    keys = [key for key in volume_keys if cache_layout(key) is not None]
    if not keys:
        return {"volumes": 0, "note": "no cached volume to measure"}
    positions = list(range(min(slices, max(1, patient_slice_count(keys[0])))))

    def best(fn, repeats=repeats):
        values = []
        for _ in range(repeats):
            started = time.perf_counter()
            fn()
            values.append(time.perf_counter() - started)
        return min(values)

    def optimised_batch():
        for volume_key in keys:
            store.release(volume_key)                     # cold patient, as a shuffled batch sees it
            store.slices(volume_key, positions, member="images")
            store.slices(volume_key, positions, member="masks")

    def legacy_batch():
        # one archive read per slice: that is what the old loop did (`load_patient` per
        # slice, inflating images + masks + singles each time), so this row is the
        # like-for-like "before" number rather than a cheaper approximation of it.
        for volume_key in keys:
            for position in positions:
                with np.load(cache_archive(volume_key), allow_pickle=False) as payload:
                    _ = payload["images"][position], payload["masks"][position]

    def optimised_index():
        return sum(patient_slice_count(key) for key in keys)

    def cold_index():
        for key in keys:
            probe_patient(key)
        CACHE_INDEX_PROBES["header_reads"] += len(keys)
        return len(keys)

    def legacy_index():
        total = 0
        for key in keys:
            total += load_patient(key, legacy_archive=True)["masks"].shape[0]
        return total

    archives_only = all(cache_archive(key).exists() for key in keys)
    layout = cache_layout(keys[0])
    optimised = best(optimised_batch)
    # The legacy reader is the slow one by design: once is enough to measure it.
    legacy = best(legacy_batch, repeats=legacy_repeats) if archives_only else None
    store.release()
    optimised_index_seconds = best(optimised_index, repeats=1)
    cold_index_seconds = best(cold_index, repeats=1)
    legacy_index_seconds = best(legacy_index, repeats=legacy_repeats) if archives_only else None
    store.release()
    total_slices = len(keys) * len(positions)
    return {
        "volumes": len(keys),
        "layout": layout,
        "slices_per_volume": len(positions),
        "batch_seconds": round(optimised, 4),
        "batch_seconds_per_volume": round(optimised / len(keys), 5),
        "batch_seconds_per_slice": round(optimised / total_slices, 6),
        "legacy_batch_seconds": None if legacy is None else round(legacy, 4),
        "legacy_batch_seconds_per_slice": None if legacy is None else round(legacy / total_slices, 6),
        "index_seconds": round(optimised_index_seconds, 4),
        "index_seconds_per_volume": round(optimised_index_seconds / len(keys), 6),
        "cold_index_seconds_per_volume": round(cold_index_seconds / len(keys), 6),
        "legacy_index_seconds_per_volume": None if legacy_index_seconds is None
        else round(legacy_index_seconds / len(keys), 6),
        "legacy_measured": archives_only,
        "store": store.stats_report(),
    }


# =============================================================================
# Build the cache (skipped for every volume that is already cached)
# =============================================================================
CACHE_REPORT = []
CACHE_STARTED = time.time()
for index, (volume_key, entry) in enumerate(sorted(VOLUMES.items()), start=1):
    CACHE_REPORT.append(
        build_patient_cache(volume_key, entry, RUN["image_size"], RUN["single_image_size"], RUN["single_channel_index"])
    )
    if index % 25 == 0 or index == len(VOLUMES):
        elapsed = time.time() - CACHE_STARTED
        print(f"cached {index}/{len(VOLUMES)} volumes ({elapsed:.0f}s, {elapsed / index:.2f}s per volume)")

CACHED = [item for item in CACHE_REPORT if cache_layout(item["volume"]) is not None]
total_bytes = sum(layout_bytes(item["volume"]) for item in CACHED)
write_json(
    PATHS["artifacts"] / "cache_report.json",
    {
        "generated_at": utc_now(),
        "volumes": len(VOLUMES),
        "cached": len(CACHED),
        "preferred_layout": RUN.get("cache_format", MEMORY_MAP_FORMAT),
        "bytes": int(total_bytes),
        "per_volume": CACHE_REPORT,
    },
)

# --- the old behaviour, measured on this cache before it is migrated -------------------
# `load_patient()` inflated the whole compressed archive once per slice read, and once per
# patient just to count slices. Both are measured here on the same volumes, the same slices
# and the same cache state as the optimised reader below, so the before/after comparison is
# a measurement rather than an expectation.
LEGACY_VOLUMES = [item["volume"] for item in CACHED if cache_layout(item["volume"]) == ARCHIVE_FORMAT]
# Four volumes x four slices, read in both layouts. A batch of RUN["batch_size"] is that
# many reads; the per-slice column scales to it without re-measuring.
SAMPLE = LEGACY_VOLUMES[:4] or [item["volume"] for item in CACHED][:4]
CACHE_IO_BEFORE = measure_cache_io(SAMPLE) if LEGACY_VOLUMES else None

# --- one-time, verified conversion of a legacy cache ----------------------------------
CACHE_UPGRADE = {"converted": [], "already": [], "failed": [], "seconds": 0.0}
if RUN.get("upgrade_cache_to_memmap", True) and LEGACY_VOLUMES:
    print(
        f"{len(LEGACY_VOLUMES)} volumes are still in the compressed layout; converting them in "
        "place (verified array by array, one patient at a time)..."
    )
    CACHE_UPGRADE = upgrade_patient_cache()
load_cache_index(rebuild=True)
write_cache_index()

# Same patient IDs, same slices, same batch size: only the cache layout changed.
CACHE_IO_AFTER = measure_cache_io(SAMPLE)
write_json(
    PATHS["artifacts"] / "cache_io_benchmark.json",
    {"sample": SAMPLE, "before": CACHE_IO_BEFORE, "after": CACHE_IO_AFTER, "upgrade": CACHE_UPGRADE, "generated_at": utc_now()},
)

# =============================================================================
# Summary
# =============================================================================
# Sizes are re-read after the conversion, so the numbers describe the cache as it is now.
DISK = shutil.disk_usage(PATHS["cache"])
total_bytes = sum(layout_bytes(item["volume"]) for item in CACHED)
per_volume_bytes = total_bytes / max(1, len(CACHED))
print("=" * 78)
print("PREPROCESSING")
print("=" * 78)
print(f"cached volumes   : {len(CACHED)}")
print(f"cache size       : {total_bytes / 1024 ** 2:.1f} MiB ({per_volume_bytes / 1024 ** 2:.0f} MiB per volume)")
print(f"cache layout     : {sum(1 for item in CACHED if cache_layout(item['volume']) == MEMORY_MAP_FORMAT)} memory-mappable, "
      f"{sum(1 for item in CACHED if cache_layout(item['volume']) == ARCHIVE_FORMAT)} compressed")
if per_volume_bytes:
    print(f"disk space       : {DISK.free / 1024 ** 3:.1f} GiB free of {DISK.total / 1024 ** 3:.1f} GiB; at the measured "
          f"{per_volume_bytes / 1024 ** 3:.3f} GiB per volume the full 369-volume release needs about "
          f"{369 * per_volume_bytes / 1024 ** 3:.0f} GiB")
print(f"4-channel tensors: {len(CACHED)} x [N_slices, {RUN['image_size']}, {RUN['image_size']}, 4] (z-scored, CHW at load time)")
print(f"1-channel tensors: {len(CACHED)} x [N_slices, {RUN['single_image_size']}, {RUN['single_image_size']}] (scaled to [0, 1])")
print("A rerun skips existing cache entries, so this cell is cheap the second time.")
print()
print("CACHE READ BENCHMARK -- same volumes, same slices, same cache state")
print("-" * 78)
if CACHE_IO_BEFORE:
    print(f"  before  archive   : {CACHE_IO_BEFORE['legacy_batch_seconds_per_slice'] * 1000:9.3f} ms per slice "
          f"({CACHE_IO_BEFORE['legacy_batch_seconds']:.3f} s to read "
          f"{CACHE_IO_BEFORE['slices_per_volume']} slices x {CACHE_IO_BEFORE['volumes']} volumes; "
          f"{CACHE_IO_BEFORE['legacy_batch_seconds_per_slice'] * RUN['batch_size'] * 1000:.0f} ms for a batch of {RUN['batch_size']})")
    print(f"                      index build: {CACHE_IO_BEFORE['legacy_index_seconds_per_volume'] * 1000:.1f} ms per volume")
else:
    print("  before  archive   : not measured -- this cache was already converted (or built new)")
print(f"  after   memory map: {CACHE_IO_AFTER['batch_seconds_per_slice'] * 1000:9.4f} ms per slice "
      f"({CACHE_IO_AFTER['batch_seconds']:.4f} s for {CACHE_IO_AFTER['slices_per_volume']} slices x "
      f"{CACHE_IO_AFTER['volumes']} volumes, cold patient each time; "
      f"{CACHE_IO_AFTER['batch_seconds_per_slice'] * RUN['batch_size'] * 1000:.2f} ms for a batch of {RUN['batch_size']})")
print(f"                      index build: {CACHE_IO_AFTER['cold_index_seconds_per_volume'] * 1000:.3f} ms per volume "
      f"for the first probe, then {CACHE_IO_AFTER['index_seconds_per_volume'] * 1e6:.1f} us per volume from the index")
if CACHE_IO_BEFORE and CACHE_IO_BEFORE["legacy_batch_seconds_per_slice"] and CACHE_IO_AFTER["batch_seconds_per_slice"]:
    print(f"  batch preparation is {CACHE_IO_BEFORE['legacy_batch_seconds_per_slice'] / CACHE_IO_AFTER['batch_seconds_per_slice']:,.0f}x "
          f"cheaper per slice; the index no longer reads any array data")
print(f"  in a batch of {RUN['batch_size']}: about "
      f"{CACHE_IO_AFTER['batch_seconds_per_slice'] * RUN['batch_size'] * 1000:.1f} ms of CPU preparation, "
      "one page-cache read per patient, no decompression")
print("=" * 78)


def validate_cache(sample=5):
    """Fail early and loudly on a corrupt cache instead of training on garbage.

    Works for both layouts and reads only a couple of slices per patient, plus the index
    header, so validating the cache does not itself cost a payload load.
    """
    problems = []
    store = patient_store()
    checked = 0
    for item in CACHED[:sample]:
        volume_key = item["volume"]
        entry = cache_index_entry(volume_key, refresh=True)
        if entry is None:
            problems.append(f"{volume_key}: no cache entry")
            continue
        shapes = entry["members"]
        if len(shapes.get("images", [])) != 4 or shapes["images"][-1] != 4:
            problems.append(f"{volume_key}: image tensor {shapes.get('images')} is not (S, H, W, 4)")
        if len(shapes.get("masks", [])) != 3:
            problems.append(f"{volume_key}: mask tensor {shapes.get('masks')} is not (S, H, W)")
        if shapes.get("images", [None])[:2] != shapes.get("masks", [None])[:2]:
            problems.append(
                f"{volume_key}: mask resolution {shapes.get('masks', [])[:2]} does not match images {shapes.get('images', [])[:2]}"
            )
        positions = sorted({0, max(0, int(entry["slices"]) - 1)})
        images = np.asarray(store.slices(volume_key, positions, member="images"), dtype=np.float32)
        masks = np.asarray(store.slices(volume_key, positions, member="masks"))
        if not np.isfinite(images).all():
            problems.append(f"{volume_key}: non-finite image values")
        if masks.max(initial=0) > 1:
            problems.append(f"{volume_key}: mask is not binary (values above 1)")
        if masks.shape[-2:] != images.shape[-3:-1]:
            problems.append(f"{volume_key}: mask and image resolutions disagree ({masks.shape} vs {images.shape})")
        checked += 1
    store.release()
    if problems:
        raise CacheError("cache validation failed: " + "; ".join(problems[:5]))
    return {"checked": checked, "problems": problems, "index_probes": CACHE_INDEX_PROBES["header_reads"]}


CACHE_VALIDATION = validate_cache()
print("cache validation :", CACHE_VALIDATION)
print("payload store    :", patient_store().stats_report())


## 10. Classification dataset discovery (Brain Tumor MRI)
`nbs_02b_classification_data.py`


In [ ]:
# =============================================================================
# SECTION 2b - Second dataset: Brain Tumor MRI Dataset (multiclass classification)
# =============================================================================
# Segmentation and classification use *different* datasets, deliberately:
#
#   * BraTS 2020 (HDF5 slices + masks)  -> segmentation, and the HGG/LGG grade
#     experiment that follows the project's own radiomics schema;
#   * Brain Tumor MRI Dataset           -> the multiclass image classification
#     task, with the classes the dataset itself defines.
#
#   https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
#
# The classes are read from the directory structure and are never assumed. This
# notebook does not know in advance whether the mounted copy is the balanced v2
# release (1400/400 per class) or an older composition -- so it counts what is
# there and reports it.
#
# If the dataset is not attached, every cell below reports that and skips: the
# BraTS segmentation pipeline still runs. Nothing is substituted for it.
# =============================================================================

CLASSIFICATION_IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}
CLASSIFICATION_SPLIT_NAMES = {"train": "train", "training": "train", "test": "test", "testing": "test", "val": "validation", "valid": "validation", "validation": "validation"}


def _image_files(directory):
    try:
        entries = sorted(directory.iterdir())
    except OSError:
        return []
    return [path for path in entries if path.is_file() and path.suffix.lower() in CLASSIFICATION_IMAGE_EXTENSIONS]


def _split_directories(base):
    """Return {split_name: [class directories]} if `base` looks like an ImageFolder tree."""
    if not base.is_dir():
        return {}
    found = {}
    for child in sorted(base.iterdir()):
        if not child.is_dir() or child.name.startswith("."):
            continue
        key = CLASSIFICATION_SPLIT_NAMES.get(child.name.strip().lower())
        if key is None:
            continue
        class_dirs = [item for item in sorted(child.iterdir()) if item.is_dir() and _image_files(item)]
        if len(class_dirs) >= 2:
            found.setdefault(key, []).extend(class_dirs)
    return found


def discover_classification_dataset(roots=None):
    """Find an image-classification tree: class sub-directories holding image files.

    Returns a dict with the split -> class -> files mapping. `found: False` carries the
    directories that were searched and the exact reason, so a missing dataset is reported
    instead of silently producing an empty run.
    """
    roots = list(roots if roots is not None else RUN["classification_dataset_roots"])
    searched = []
    candidates = []
    for root in roots:
        base = Path(root).expanduser()
        searched.append(str(base))
        if not base.exists():
            continue
        candidates.append(base)
        # Kaggle mounts a dataset as /kaggle/input/<slug>/<inner folder>; walk one or two
        # levels so the tree is found wherever the archive was unpacked.
        for first in sorted((item for item in base.iterdir() if item.is_dir()), key=lambda p: p.name):
            candidates.append(first)
            for second in sorted((item for item in first.iterdir() if item.is_dir()), key=lambda p: p.name):
                candidates.append(second)

    for base in candidates:
        splits = _split_directories(base)
        if splits:
            return {
                "found": True,
                "root": str(base),
                "layout": "image_folder_with_split_directories",
                "splits": {name: {directory.name: [str(path) for path in _image_files(directory)] for directory in directories} for name, directories in splits.items()},
                "searched": searched,
            }
    # No explicit split directories: accept a plain class-per-directory tree.
    for base in candidates:
        class_dirs = [item for item in sorted(base.iterdir()) if item.is_dir() and _image_files(item)] if base.is_dir() else []
        if len(class_dirs) >= 2:
            return {
                "found": True,
                "root": str(base),
                "layout": "image_folder_single_tree",
                "splits": {"all": {directory.name: [str(path) for path in _image_files(directory)] for directory in class_dirs}},
                "searched": searched,
            }
    return {
        "found": False,
        "searched": searched,
        "hint": (
            "Add Data -> Dataset -> masoudnickparvar/brain-tumor-mri-dataset. It mounts under "
            "/kaggle/input/brain-tumor-mri-dataset; if your copy is elsewhere, put its path first "
            "in RUN['classification_dataset_roots']."
        ),
    }


CLASSIFICATION_DATASET = discover_classification_dataset()
CLASSIFICATION_IMAGES_AVAILABLE = bool(CLASSIFICATION_DATASET["found"])

if not CLASSIFICATION_IMAGES_AVAILABLE:
    print("Classification dataset: NOT FOUND")
    print("  searched:", CLASSIFICATION_DATASET["searched"])
    print("  ", CLASSIFICATION_DATASET["hint"])
    print("  The multiclass classification cells below will report this and skip. The BraTS")
    print("  segmentation pipeline is unaffected, and no labels are invented for it.")
    CLASSIFICATION_CLASSES: list = []
    CLASSIFICATION_SPLIT_FILES: dict = {}
    CLASSIFICATION_IS_SYNTHETIC = None
    CLASSIFICATION_DATASET_NAME = "not attached"
    CLASSIFICATION_PROVENANCE_NOTE = "the dataset was not found in this environment"
else:
    CLASSIFICATION_ROOT = Path(CLASSIFICATION_DATASET["root"])
    CLASSIFICATION_SPLIT_FILES = CLASSIFICATION_DATASET["splits"]
    # Classes come from the directories, in the dataset's own spelling.
    CLASSIFICATION_CLASSES = sorted({name for groups in CLASSIFICATION_SPLIT_FILES.values() for name in groups})
    CLASSIFICATION_TOTAL_IMAGES = sum(
        len(paths) for groups in CLASSIFICATION_SPLIT_FILES.values() for paths in groups.values()
    )

    # --- what is this dataset? -------------------------------------------------
    # Same rule as for the segmentation data: the claim is derived from the files, never
    # assumed. A fixture writes `fixture_manifest.json` next to its classes; a real release
    # has thousands of images; anything else is recorded as unverified rather than "real".
    FIXTURE_MANIFEST = None
    for candidate in (CLASSIFICATION_ROOT / "fixture_manifest.json", CLASSIFICATION_ROOT.parent / "fixture_manifest.json"):
        if candidate.exists():
            FIXTURE_MANIFEST = read_json(candidate, default=None)
            if FIXTURE_MANIFEST:
                break


    def resolve_classification_provenance(declared, fixture, images, classes):
        declared = str(declared or "auto").lower()
        if declared in {"synthetic", "synthetic_fixture", "fixture"}:
            return True, "declared in RUN['classification_dataset_provenance']"
        if declared in {"real", "real_brain_tumor_mri", "brain_tumor_mri"}:
            return False, "declared in RUN['classification_dataset_provenance']"
        if fixture and fixture.get("synthetic") is True:
            return True, f"fixture manifest ({fixture.get('generator', 'generator not recorded')})"
        if images >= 1000 and len(classes) >= 2:
            return False, f"{images} images in {len(classes)} classes, matching the released Kaggle dataset"
        return None, (
            f"neither a fixture manifest nor a full release ({images} images in {len(classes)} classes); "
            "set RUN['classification_dataset_provenance'] if this really is the real dataset"
        )


    CLASSIFICATION_IS_SYNTHETIC, CLASSIFICATION_PROVENANCE_NOTE = resolve_classification_provenance(
        RUN.get("classification_dataset_provenance", "auto"), FIXTURE_MANIFEST, CLASSIFICATION_TOTAL_IMAGES, CLASSIFICATION_CLASSES
    )
    CLASSIFICATION_DATASET_NAME = (
        "Synthetic image-classification fixture (scripts/make_classification_fixture.py; the classes are "
        "generated, any metric computed from them is meaningless)"
        if CLASSIFICATION_IS_SYNTHETIC
        else "Brain Tumor MRI Dataset (Kaggle: masoudnickparvar/brain-tumor-mri-dataset)"
        if CLASSIFICATION_IS_SYNTHETIC is False
        else f"Unverified image tree at {CLASSIFICATION_ROOT} (provenance not established)"
    )
    print(f"Classification dataset : {CLASSIFICATION_ROOT} ({CLASSIFICATION_DATASET['layout']})")
    print(f"Images                 : {CLASSIFICATION_TOTAL_IMAGES}")
    for split_name, groups in sorted(CLASSIFICATION_SPLIT_FILES.items()):
        counts = ", ".join(f"{name}: {len(paths)}" for name, paths in sorted(groups.items()))
        print(f"  {split_name:<12} {counts}")
    print(f"Classes ({len(CLASSIFICATION_CLASSES)})      : {', '.join(CLASSIFICATION_CLASSES)}")
    print(
        "Dataset provenance    :",
        {True: "SYNTHETIC FIXTURE", False: "real Brain Tumor MRI Dataset", None: "unknown"}[CLASSIFICATION_IS_SYNTHETIC],
        "--",
        CLASSIFICATION_PROVENANCE_NOTE,
    )
    if CLASSIFICATION_IS_SYNTHETIC:
        print(
            "\n" + "!" * 78 + "\n"
            "WARNING: the classification data is a synthetic fixture. Every number printed for\n"
            "the multiclass task below describes generated images with generated labels. It must\n"
            "not be quoted as a result.\n" + "!" * 78
        )

LOCK["classification_dataset"] = {
    "root": CLASSIFICATION_DATASET.get("root"),
    "layout": CLASSIFICATION_DATASET.get("layout"),
    "classes": CLASSIFICATION_CLASSES,
    "images": CLASSIFICATION_TOTAL_IMAGES if CLASSIFICATION_IMAGES_AVAILABLE else 0,
    "synthetic": CLASSIFICATION_IS_SYNTHETIC,
    "provenance_note": CLASSIFICATION_PROVENANCE_NOTE,
}
write_json(PATHS["artifacts"] / "classification_dataset_discovery.json", {"dataset": CLASSIFICATION_DATASET, "lock": LOCK["classification_dataset"]})


## 11. Classification dataset structure inspection
`nbs_02b_classification_data.py`


In [ ]:
# =============================================================================
# SECTION 3b - Structure inspection of the classification images
# =============================================================================
# What is actually inside the tree: file formats, colour modes, pixel sizes, per-class
# counts, unreadable files, and byte-identical duplicates. All of it is written to disk so
# the preprocessing decisions (grayscale, resize) are traceable to the data.
import hashlib
from collections import Counter

STRUCTURE = {"state": "not_available", "reason": "the classification dataset was not attached"}
if CLASSIFICATION_IMAGES_AVAILABLE:
    from PIL import Image as PILImage

    def inspect_image(path, max_side=None):
        with PILImage.open(path) as handle:
            size = handle.size
            mode = handle.mode
            fmt = handle.format
            array = np.asarray(handle.convert("L"), dtype=np.float32)
        return {"path": path, "size": size, "mode": mode, "format": fmt, "array": array}

    extension_counts = Counter()
    mode_counts = Counter()
    size_counts = Counter()
    digest_to_paths = {}
    unreadable = []
    per_class = {}
    for split_name, groups in sorted(CLASSIFICATION_SPLIT_FILES.items()):
        for class_name, paths in sorted(groups.items()):
            entry = per_class.setdefault(
                class_name,
                {"split": split_name, "images": 0, "min_side": None, "max_side": None, "extensions": Counter(), "modes": Counter()},
            )
            for path in paths:
                path = Path(path)
                extension_counts[path.suffix.lower()] += 1
                entry["extensions"][path.suffix.lower()] += 1
                try:
                    with PILImage.open(path) as handle:
                        width, height = handle.size
                        mode_counts[handle.mode] += 1
                        entry["modes"][handle.mode] += 1
                except Exception as exc:  # unreadable file -> reported, never assumed away
                    unreadable.append({"path": str(path), "error": f"{type(exc).__name__}: {exc}"})
                    continue
                entry["images"] += 1
                smallest, largest = min(width, height), max(width, height)
                entry["min_side"] = smallest if entry["min_side"] is None else min(entry["min_side"], smallest)
                entry["max_side"] = largest if entry["max_side"] is None else max(entry["max_side"], largest)
                size_counts[(width, height)] += 1
                digest = hashlib.sha256(path.read_bytes()).hexdigest()
                digest_to_paths.setdefault(digest, []).append((split_name, str(path)))

    duplicate_groups = [
        {"sha256": digest, "files": [{"split": split, "path": path} for split, path in paths]}
        for digest, paths in digest_to_paths.items()
        if len(paths) > 1
    ]
    cross_split_duplicates = [
        group
        for group in duplicate_groups
        if len({entry["split"] for entry in group["files"]}) > 1
    ]
    STRUCTURE = {
        "state": "inspected",
        "root": CLASSIFICATION_DATASET["root"],
        "layout": CLASSIFICATION_DATASET["layout"],
        "split_counts": {name: sum(len(paths) for paths in groups.values()) for name, groups in CLASSIFICATION_SPLIT_FILES.items()},
        "class_counts": {
            name: sum(len(CLASSIFICATION_SPLIT_FILES[split].get(name, [])) for split in CLASSIFICATION_SPLIT_FILES)
            for name in CLASSIFICATION_CLASSES
        },
        "classes_in_each_split": {name: sorted(groups) for name, groups in CLASSIFICATION_SPLIT_FILES.items()},
        "per_class": {name: {**entry, "extensions": dict(entry["extensions"]), "modes": dict(entry["modes"])} for name, entry in per_class.items()},
        "extensions": dict(extension_counts),
        "modes": dict(mode_counts),
        "distinct_pixel_sizes": len(size_counts),
        "most_common_sizes": [[list(size), count] for size, count in size_counts.most_common(5)],
        "unreadable_files": unreadable,
        "byte_identical_duplicate_groups": len(duplicate_groups),
        "byte_identical_duplicates_across_splits": len(cross_split_duplicates),
        "duplicate_examples": cross_split_duplicates[:5],
        "synthetic": CLASSIFICATION_IS_SYNTHETIC,
        "provenance_note": CLASSIFICATION_PROVENANCE_NOTE,
    }
    print("Split sizes    :", STRUCTURE["split_counts"])
    print("Class counts   :", STRUCTURE["class_counts"])
    print("Extensions     :", STRUCTURE["extensions"])
    print("Colour modes   :", STRUCTURE["modes"])
    print("Distinct sizes :", STRUCTURE["distinct_pixel_sizes"], "(most common:", STRUCTURE["most_common_sizes"], ")")
    print("Unreadable     :", len(unreadable))
    print("Byte-identical duplicate groups:", len(duplicate_groups), "of which across splits:", len(cross_split_duplicates))
    if unreadable:
        print("  first unreadable:", unreadable[0])
    write_json(PATHS["artifacts"] / "classification_structure.json", STRUCTURE)


## 12. Classification dataset EDA and visualisation
`nbs_02b_classification_data.py`


In [ ]:
# =============================================================================
# SECTION 4b - Class distribution and sample images
# =============================================================================
CLASSIFICATION_EDA = {"state": "not_available", "reason": "the classification dataset was not attached"}
if CLASSIFICATION_IMAGES_AVAILABLE:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    distribution = {
        split_name: {name: len(paths) for name, paths in sorted(groups.items())}
        for split_name, groups in sorted(CLASSIFICATION_SPLIT_FILES.items())
    }
    totals = {
        name: sum(len(groups.get(name, [])) for groups in CLASSIFICATION_SPLIT_FILES.values())
        for name in CLASSIFICATION_CLASSES
    }

    # --- class distribution figure ------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    split_labels = list(distribution)
    positions = np.arange(len(CLASSIFICATION_CLASSES))
    width = 0.8 / max(1, len(split_labels))
    for index, split_name in enumerate(split_labels):
        counts = [distribution[split_name].get(name, 0) for name in CLASSIFICATION_CLASSES]
        axes[0].bar(positions + index * width, counts, width, label=split_name)
    axes[0].set_xticks(positions + width * (len(split_labels) - 1) / 2)
    axes[0].set_xticklabels(CLASSIFICATION_CLASSES, rotation=20, ha="right")
    axes[0].set_ylabel("images")
    axes[0].set_title("Class distribution per split")
    axes[0].legend()
    axes[1].bar(list(totals), [totals[name] for name in totals], color="tab:blue")
    axes[1].set_title("Total images per class")
    axes[1].tick_params(axis="x", rotation=20)
    plt.tight_layout()
    distribution_figure = PATHS["reports"] / "07_classification_class_distribution.png"
    plt.savefig(distribution_figure, dpi=130)
    plt.close(fig)

    # --- sample grid ----------------------------------------------------------
    preview_split = "train" if "train" in CLASSIFICATION_SPLIT_FILES else split_labels[0]
    columns = max(len(CLASSIFICATION_CLASSES), 1)
    fig, axes = plt.subplots(1, columns, figsize=(3.1 * columns, 3.4))
    axes = np.atleast_1d(axes)
    examples = {}
    for index, class_name in enumerate(CLASSIFICATION_CLASSES):
        paths = CLASSIFICATION_SPLIT_FILES[preview_split].get(class_name, [])
        axes[index].axis("off")
        if not paths:
            axes[index].set_title(f"{class_name}\n(no image)")
            continue
        sample = inspect_image(Path(paths[0]))
        examples[class_name] = sample["path"].name
        axes[index].imshow(sample["array"], cmap="gray")
        axes[index].set_title(f"{class_name}\n{sample['size'][0]}x{sample['size'][1]} {sample['format']}", fontsize=9)
    plt.tight_layout()
    samples_figure = PATHS["reports"] / "08_classification_samples.png"
    plt.savefig(samples_figure, dpi=130)
    plt.close(fig)

    CLASSIFICATION_EDA = {
        "state": "generated",
        "split": preview_split,
        "distribution": distribution,
        "totals": totals,
        "classes": CLASSIFICATION_CLASSES,
        "example_filenames": examples,
        "figures": {"distribution": str(distribution_figure), "samples": str(samples_figure)},
        "synthetic": CLASSIFICATION_IS_SYNTHETIC,
    }
    print("distribution:", distribution)
    print("figures     :", CLASSIFICATION_EDA["figures"])
    write_json(PATHS["artifacts"] / "classification_eda.json", CLASSIFICATION_EDA)


## 13. Classification preprocessing, splits and leakage audit
`nbs_02b_classification_data.py`


In [ ]:
# =============================================================================
# SECTION 9b - Images, splits and the leakage audit
# =============================================================================
# Three separate concerns, all of them recorded:
#
# 1. Preprocessing: grayscale, resized to RUN['classification_image_size'] with Lanczos,
#    stored as uint8 arrays in the resumable cache. The classifier sees exactly these.
# 2. Splitting: the dataset ships its own train/test directories. This notebook does not
#    trust that blindly -- it keeps the shipped test directory as the held-out test set and
#    carves a stratified validation set out of the shipped training directory.
#    **There are no patient identifiers in this dataset** (it is a curated collection of
#    individual images from figshare / SARTAJ / Br35H), so a patient-level split is
#    impossible here and is not claimed. What can be done is done: a near-duplicate audit
#    that flags test images whose content matches a training image, and those images are
#    excluded from the reported test set by default.
# 3. Audit: 8x8 average-hash (64-bit) per image; Hamming distance to the nearest training
#    image is reported per test image. The threshold and the counts are recorded.
# =============================================================================
import random as _random

CLASSIFICATION_IMAGES: dict = {"train": {}, "validation": {}, "test": {}}
CLASSIFICATION_AUDIT = {"state": "not_available", "reason": "the classification dataset was not attached"}
CLASSIFICATION_SPLIT_SOURCE = None

if CLASSIFICATION_IMAGES_AVAILABLE:
    from PIL import Image as PILImage

    IMAGE_SIZE = int(RUN["classification_image_size"])
    CACHE_ROOT = PATHS["cache"] / "classification_images"
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)


    def load_image_array(path, size=IMAGE_SIZE, cache=True):
        """Grayscale uint8 array at `size`, cached on disk so a re-run does not re-decode."""
        path = Path(path)
        digest = hashlib.sha256(f"{path}|{size}|grayscale".encode("utf-8")).hexdigest()[:16]
        target = CACHE_ROOT / f"{digest}.npy"
        if cache and target.exists():
            return np.load(target)
        with PILImage.open(path) as handle:
            array = handle.convert("L").resize((size, size), PILImage.LANCZOS)
        array = np.asarray(array, dtype=np.uint8)
        if cache:
            np.save(target, array)
        return array


    AUDIT_PIXEL_SIZE = int(RUN["classification_duplicate_pixel_size"])
    AUDIT_HASH_SIZE = int(RUN["classification_duplicate_hash_size"])


    def average_hash(array, hash_size=AUDIT_HASH_SIZE):
        """Coarse perceptual hash, used only as a *candidate filter* for the duplicate audit."""
        with PILImage.fromarray(np.asarray(array, dtype=np.uint8)) as handle:
            small = np.asarray(handle.resize((hash_size, hash_size), PILImage.BILINEAR), dtype=np.float32)
        return (small > small.mean()).ravel()


    def pixel_correlation(first, second):
        """Pearson correlation of two same-sized grayscale arrays, used to confirm a candidate.

        An average hash is very coarse: images that share a global structure (every one of these
        is a centred brain slice) can land within a few bits of each other while showing clearly
        different content. A hash match therefore only *proposes* a duplicate; this correlation
        decides it, so the audit cannot quietly throw away a large share of the test set.
        """
        a = np.asarray(first, dtype=np.float32).ravel()
        b = np.asarray(second, dtype=np.float32).ravel()
        a = a - a.mean()
        b = b - b.mean()
        denominator = float(np.sqrt((a * a).sum() * (b * b).sum()))
        return float((a * b).sum() / denominator) if denominator > 0 else 0.0


    # --- assign splits -------------------------------------------------------
    RAW = CLASSIFICATION_SPLIT_FILES
    if {"train", "test"} <= set(RAW):
        train_pool = {name: list(paths) for name, paths in RAW["train"].items()}
        test_pool = {name: list(paths) for name, paths in RAW["test"].items()}
        CLASSIFICATION_SPLIT_SOURCE = "the dataset's own Training/ and Testing/ directories"
    else:
        rng_split = np.random.default_rng(SEED)
        train_pool, test_pool = {}, {}
        for name, paths in RAW.get("all", {}).items():
            shuffled = list(paths)
            rng_split.shuffle(shuffled)
            cut = int(round(len(shuffled) * 0.7))
            train_pool[name] = shuffled[:cut]
            test_pool[name] = shuffled[cut:]
        CLASSIFICATION_SPLIT_SOURCE = "assigned by this notebook (70/30, stratified by class, seed %d)" % SEED

    # stratified validation carve-out from the training pool
    rng_val = np.random.default_rng(SEED + 1)
    validation_pool = {}
    for name, paths in train_pool.items():
        shuffled = list(paths)
        rng_val.shuffle(shuffled)
        cuts = int(round(len(shuffled) * float(RUN["classification_validation_fraction"])))
        cuts = min(max(cuts, 1), max(len(shuffled) - 1, 0))
        validation_pool[name] = shuffled[:cuts]
        train_pool[name] = shuffled[cuts:]

    for split_name, pool in (("train", train_pool), ("validation", validation_pool), ("test", test_pool)):
        for name, paths in sorted(pool.items()):
            CLASSIFICATION_IMAGES[split_name][name] = [Path(path) for path in paths]

    split_sizes = {name: sum(len(paths) for paths in pool.values()) for name, pool in CLASSIFICATION_IMAGES.items()}
    print("split source :", CLASSIFICATION_SPLIT_SOURCE)
    print("split sizes  :", split_sizes)

    # --- leakage audit: validation and test against the training images --------
    # No patient identifiers exist in this dataset, so grouping by patient is impossible. The
    # next best thing is measurable: perceptual near-duplicates of a training image inside the
    # validation or test split would inflate the reported scores, so they are counted and, by
    # default, removed from the evaluation splits. The count is reported either way.
    hash_threshold = int(RUN["classification_duplicate_hamming_threshold"])
    correlation_threshold = float(RUN["classification_duplicate_correlation_threshold"])
    max_exclusion_fraction = float(RUN["duplicate_exclusion_max_fraction"])

    train_paths_flat = [Path(path) for paths in CLASSIFICATION_IMAGES["train"].values() for path in paths]
    train_arrays = np.stack(
        [load_image_array(path, size=AUDIT_PIXEL_SIZE) for path in train_paths_flat]
    ) if train_paths_flat else np.zeros((0, AUDIT_PIXEL_SIZE, AUDIT_PIXEL_SIZE), dtype=np.uint8)
    train_hashes = np.stack([average_hash(array) for array in train_arrays]) if len(train_arrays) else np.zeros((0, AUDIT_HASH_SIZE * AUDIT_HASH_SIZE), dtype=bool)
    print(
        f"leakage audit: {len(train_arrays)} training image(s) indexed, "
        f"{AUDIT_HASH_SIZE}x{AUDIT_HASH_SIZE} hash (candidate filter) + pixel correlation >= {correlation_threshold} (confirmation)"
    )

    def audit_split(split_name):
        """Two-stage duplicate detection: hash candidates, then confirm with correlation."""
        total = 0
        candidates = []
        confirmed = []
        for class_name, paths in sorted(CLASSIFICATION_IMAGES[split_name].items()):
            for path in paths:
                total += 1
                array = load_image_array(path, size=AUDIT_PIXEL_SIZE)
                digest = average_hash(array)
                if not train_hashes.shape[0]:
                    continue
                distances = np.count_nonzero(train_hashes != digest, axis=1)
                nearest = int(distances.min())
                if nearest > hash_threshold:
                    continue
                nearest_index = int(distances.argmin())
                correlation = pixel_correlation(array, train_arrays[nearest_index])
                entry = {
                    "path": str(path),
                    "class": class_name,
                    "hamming_distance": nearest,
                    "pixel_correlation": correlation,
                    "nearest_train_image": str(train_paths_flat[nearest_index]),
                }
                candidates.append(entry)
                if correlation >= correlation_threshold:
                    confirmed.append(entry)
        return total, candidates, confirmed

    audit_report = {}
    for split_name in ("validation", "test"):
        total, candidates, confirmed = audit_split(split_name)
        excluded = []
        suppression = None
        if confirmed and len(confirmed) / max(1, total) > max_exclusion_fraction:
            # A calibrated audit removes a handful of images, not a share of the split. If the
            # fraction is this large the threshold is wrong for this data, so nothing is removed
            # and the situation is reported instead of silently shrinking the evaluation set.
            suppression = (
                f"{len(confirmed)} of {total} images ({len(confirmed) / total:.1%}) matched the duplicate "
                f"criteria, above the {max_exclusion_fraction:.0%} limit; the audit is treated as "
                "inconclusive and no image is removed. Raise RUN['classification_duplicate_correlation_threshold'] "
                "or inspect the flagged pairs."
            )
            confirmed = []
        if RUN["exclude_duplicate_eval_images"] and confirmed:
            confirmed_paths = {entry["path"] for entry in confirmed}
            for class_name, paths in list(CLASSIFICATION_IMAGES[split_name].items()):
                kept = [path for path in paths if str(path) not in confirmed_paths]
                excluded.extend({"path": str(path), "class": class_name} for path in paths if str(path) not in kept)
                CLASSIFICATION_IMAGES[split_name][class_name] = kept
        audit_report[split_name] = {
            "images_before": total,
            "hash_candidates": len(candidates),
            "confirmed_near_duplicates": len(confirmed),
            "confirmed_fraction": len(confirmed) / max(1, total),
            "confirmed_by_class": {
                name: sum(1 for entry in confirmed if entry["class"] == name) for name in CLASSIFICATION_CLASSES
            },
            "excluded": excluded,
            "images_after": total - len(excluded),
            "exclusion_suppressed": suppression,
            "examples": (confirmed or candidates)[:5],
        }

    CLASSIFICATION_AUDIT = {
        "state": "measured_and_enforced" if RUN["exclude_duplicate_eval_images"] else "measured",
        "method": (
            f"{AUDIT_HASH_SIZE}x{AUDIT_HASH_SIZE} average hash (Hamming <= {hash_threshold}) as a candidate "
            f"filter, confirmed by Pearson correlation >= {correlation_threshold} of the "
            f"{AUDIT_PIXEL_SIZE}x{AUDIT_PIXEL_SIZE} grayscale images"
        ),
        "hash_threshold": hash_threshold,
        "correlation_threshold": correlation_threshold,
        "max_exclusion_fraction": max_exclusion_fraction,
        "split_audits": audit_report,
        "patient_identifiers_available": False,
        "patient_level_split_possible": False,
        "note": (
            "The dataset carries no patient identifiers -- it is a curated collection of individual "
            "images -- so a patient-level split cannot be made and is not claimed here. The dataset's "
            "own train/test directories are kept, and images confirmed to duplicate a training image "
            "are removed from the validation and test splits by default, so a reported score is not "
            "earned by a duplicated image."
        ),
    }
    print(
        "audit        :",
        {
            split: {
                "images_before": entry["images_before"],
                "hash_candidates": entry["hash_candidates"],
                "confirmed": entry["confirmed_near_duplicates"],
                "kept": entry["images_after"],
            }
            for split, entry in audit_report.items()
        },
    )
    for split_name, entry in audit_report.items():
        if entry["exclusion_suppressed"]:
            print(f"  WARNING {split_name}:", entry["exclusion_suppressed"])

    # --- materialise the arrays ------------------------------------------------
    t0 = time.time()
    load_counts = {}
    for split_name, groups in CLASSIFICATION_IMAGES.items():
        for name, paths in groups.items():
            for path in paths:
                load_image_array(path)
            load_counts[f"{split_name}/{name}"] = len(paths)
    print(f"decoded/cached {sum(load_counts.values())} images in {time.time() - t0:.1f}s -> {CACHE_ROOT}")

    CLASSIFICATION_SPLIT_RECORD = {
        "state": "ready",
        "root": CLASSIFICATION_DATASET["root"],
        "classes": CLASSIFICATION_CLASSES,
        "split_source": CLASSIFICATION_SPLIT_SOURCE,
        "validation_fraction": float(RUN["classification_validation_fraction"]),
        "seed": SEED,
        "image_size": IMAGE_SIZE,
        "splits": {
            name: {class_name: len(paths) for class_name, paths in sorted(groups.items())}
            for name, groups in CLASSIFICATION_IMAGES.items()
        },
        "split_sizes": {name: sum(len(paths) for paths in groups.values()) for name, groups in CLASSIFICATION_IMAGES.items()},
        "audit": CLASSIFICATION_AUDIT,
        "synthetic": CLASSIFICATION_IS_SYNTHETIC,
        "provenance_note": CLASSIFICATION_PROVENANCE_NOTE,
    }
    write_json(PATHS["artifacts"] / "classification_splits.json", CLASSIFICATION_SPLIT_RECORD)


## 14. Classification dataset readiness
`nbs_02b_classification_data.py`


In [ ]:
# =============================================================================
# SECTION 10b - What the second dataset gives us, stated plainly
# =============================================================================
if not CLASSIFICATION_IMAGES_AVAILABLE:
    CLASSIFICATION_READY = False
    print("Classification task: skipped --", CLASSIFICATION_DATASET["hint"])
    print("No multiclass model is trained, no metric is reported, and nothing is fabricated.")
else:
    CLASSIFICATION_READY = all(
        CLASSIFICATION_IMAGES["train"] and CLASSIFICATION_IMAGES["validation"] and CLASSIFICATION_IMAGES["test"]
    ) and all(
        sum(len(paths) for paths in groups.values()) > 0 for groups in CLASSIFICATION_IMAGES.values()
    )
    summary_rows = []
    for split_name in ("train", "validation", "test"):
        groups = CLASSIFICATION_IMAGES[split_name]
        for class_name in CLASSIFICATION_CLASSES:
            summary_rows.append((split_name, class_name, len(groups.get(class_name, []))))
    print(f"{'split':<12}{'class':<14}images")
    for split_name, class_name, count in summary_rows:
        print(f"{split_name:<12}{class_name:<14}{count}")
    print()
    print("task            :", RUN["classification_task_name"])
    print("classes         :", CLASSIFICATION_CLASSES, "(from the dataset's own directories)")
    print("image size      :", RUN["classification_image_size"], "px, grayscale")
    print("split source    :", CLASSIFICATION_SPLIT_SOURCE)
    print(
        "leakage audit   :",
        CLASSIFICATION_AUDIT["state"],
        "(test: {before} -> {after} images, {flagged} confirmed near-duplicate(s) of training "
        "images)".format(
            before=CLASSIFICATION_AUDIT["split_audits"]["test"]["images_before"],
            after=CLASSIFICATION_AUDIT["split_audits"]["test"]["images_after"],
            flagged=CLASSIFICATION_AUDIT["split_audits"]["test"]["confirmed_near_duplicates"],
        ),
    )
    print("dataset is the  :", {True: "synthetic fixture", False: "real dataset", None: "unverified tree"}[CLASSIFICATION_IS_SYNTHETIC])
    if not CLASSIFICATION_READY:
        empty = [name for name, groups in CLASSIFICATION_IMAGES.items() if not sum(len(paths) for paths in groups.values())]
        partial = {
            name: {class_name: len(paths) for class_name, paths in groups.items() if not paths}
            for name, groups in CLASSIFICATION_IMAGES.items()
            if name not in empty
        }
        print(f"Classification task: no multiclass model is trained -- empty split(s): {empty}; classes with no image: {partial}")

LOCK["classification_splits"] = {
    "classes": CLASSIFICATION_CLASSES,
    "splits": {name: sum(len(paths) for paths in groups.values()) for name, groups in CLASSIFICATION_IMAGES.items()},
    "audit": (
        CLASSIFICATION_AUDIT["split_audits"]["test"]["confirmed_near_duplicates"] if CLASSIFICATION_IMAGES_AVAILABLE else None
    ),
}


## 15. Patient-level train / validation / test splitting
`nbs_03_splits_trainer.py`


In [ ]:
# =============================================================================
# SECTION 9 - Patient-level splits (HGG/LGG stratified, leakage checked, frozen)
# =============================================================================
# BraTS slices from one patient are near-duplicates of each other. Splitting by
# slice would leak the same tumour into train and test, so the split is made on
# the patient (volume) and every slice follows its patient. The split is written
# to disk and reused by every later section; nothing reselects it.
# =============================================================================
SPLIT_PATH = PATHS["artifacts"] / "patient_splits.json"


def stratified_patient_splits(volumes, *, train=0.70, val=0.15, test=0.15, seed=SEED):
    """Stratify on the grade when grades exist, otherwise fall back to plain random."""
    rng = random.Random(seed)
    groups = {}
    for volume_key, entry in volumes.items():
        key = entry["grade"] if entry["grade"] in {"HGG", "LGG"} else "unlabelled"
        groups.setdefault(key, []).append(volume_key)
    splits = {"train": [], "validation": [], "test": []}
    warnings = []
    for key, patient_ids in sorted(groups.items()):
        ordered = sorted(patient_ids)
        rng.shuffle(ordered)
        count = len(ordered)
        if count < 3:
            splits["train"].extend(ordered)
            warnings.append(f"stratum '{key}' has {count} patient(s); kept in training (no metric can come from it)")
            continue
        n_val = max(1, int(round(count * val)))
        n_test = max(1, int(round(count * test)))
        n_train = count - n_val - n_test
        if n_train < 1:
            n_train, n_val, n_test = 1, max(1, (count - 1) // 2), max(0, count - 1 - max(1, (count - 1) // 2))
        splits["train"].extend(ordered[:n_train])
        splits["validation"].extend(ordered[n_train : n_train + n_val])
        splits["test"].extend(ordered[n_train + n_val : n_train + n_val + n_test])
    return splits, warnings


def split_summary(splits, volumes):
    placement = {}
    summary = {}
    for name in ("train", "validation", "test"):
        grades = {}
        slices = 0
        for volume_key in splits[name]:
            grades[volumes[volume_key]["grade"] or "unlabelled"] = grades.get(volumes[volume_key]["grade"] or "unlabelled", 0) + 1
            slices += len(volumes[volume_key]["slices"])
            placement.setdefault(volume_key, []).append(name)
        summary[name] = {"patients": len(splits[name]), "slices": slices, "grade_counts": grades}
    overlap = sorted(key for key, names in placement.items() if len(names) > 1)
    summary["patient_overlap"] = overlap
    summary["leakage_free"] = not overlap
    summary["patients_assigned"] = len(placement)
    summary["patients_total"] = len(volumes)
    return summary


if SPLIT_PATH.exists() and not RUN.get("refreeze_splits"):
    SPLIT_RECORD = read_json(SPLIT_PATH)
    SPLITS = SPLIT_RECORD["splits"]
    print(f"Reusing the frozen split from {SPLIT_PATH}")
else:
    SPLITS, SPLIT_WARNINGS = stratified_patient_splits(VOLUMES, seed=SEED)
    SPLIT_RECORD = {
        "created_at": utc_now(),
        "method": "patient_level_stratified_by_grade",
        "seed": SEED,
        "ratios": {"train": 0.70, "validation": 0.15, "test": 0.15},
        "splits": SPLITS,
        "warnings": SPLIT_WARNINGS,
        "summary": split_summary(SPLITS, VOLUMES),
    }
    write_json(SPLIT_PATH, SPLIT_RECORD)

SPLIT_SUMMARY = split_summary(SPLITS, VOLUMES)
assert SPLIT_SUMMARY["leakage_free"], f"patients appear in more than one split: {SPLIT_SUMMARY['patient_overlap']}"

print("=" * 78)
print("SPLITS (patient level, leakage checked)")
print("=" * 78)
for name in ("train", "validation", "test"):
    entry = SPLIT_SUMMARY[name]
    print(f"{name:11s}: {entry['patients']:4d} patients | {entry['slices']:6d} slices | grades {entry['grade_counts']}")
print("leakage-free      :", SPLIT_SUMMARY["leakage_free"])
print("frozen at         :", SPLIT_PATH)
print(
    "\nThe test split is written once here and is not touched until Section 19. Model\n"
    "selection, threshold tuning and ensemble weights use the validation split only."
)

TRAIN_PATIENTS = list(SPLITS["train"])
VAL_PATIENTS = list(SPLITS["validation"])
TEST_PATIENTS = list(SPLITS["test"])
if RUN["smoke_test"]:
    TRAIN_PATIENTS, VAL_PATIENTS, TEST_PATIENTS = TRAIN_PATIENTS[:2], VAL_PATIENTS[:1], TEST_PATIENTS[:1]
    print("smoke test: reduced to", len(TRAIN_PATIENTS), len(VAL_PATIENTS), len(TEST_PATIENTS), "patients")


## 16. Reproducible baselines
`nbs_03_splits_trainer.py`


In [ ]:
# =============================================================================
# SECTION 10 - Reproducible baselines
# =============================================================================
# Two reference points so every later number has something to be compared with:
#   * an all-foreground predictor (what Dice a trivial mask gets),
#   * the training-split prevalence of tumour pixels.
# They are computed on the validation split, never on the test split.
# =============================================================================
def validation_prevalence():
    total, foreground, slices = 0, 0, 0
    for volume_key in VAL_PATIENTS:
        payload = load_patient(volume_key)
        masks = payload["masks"].astype(np.float32)
        foreground += float(masks.sum())
        total += float(masks.size)
        slices += masks.shape[0]
    return {"slices": slices, "pixels": total, "foreground_pixels": foreground, "prevalence": foreground / max(1.0, total)}


VAL_PREVALENCE = validation_prevalence()
# Dice of an all-foreground prediction against the reference masks.
dice_all_foreground = 2 * VAL_PREVALENCE["foreground_pixels"] / max(
    1.0, VAL_PREVALENCE["pixels"] + VAL_PREVALENCE["foreground_pixels"]
)
BASELINES = {
    "all_foreground_dice": dice_all_foreground,
    "validation_prevalence": VAL_PREVALENCE["prevalence"],
    "validation_slices": VAL_PREVALENCE["slices"],
    "note": (
        "Any trained model has to beat the all-foreground Dice on the same split to be worth "
        "anything; both numbers are reported next to the model results."
    ),
}
write_json(PATHS["artifacts"] / "baselines.json", BASELINES)
print("=" * 78)
print("REFERENCE POINTS (validation split)")
print("=" * 78)
print(f"validation slices        : {VAL_PREVALENCE['slices']}")
print(f"tumour pixel prevalence  : {VAL_PREVALENCE['prevalence']:.4f}")
print(f"all-foreground Dice      : {dice_all_foreground:.4f}")


## 17. Trainer core (AMP, resume, progress, early stopping, scheduling)
`nbs_03_splits_trainer.py`


In [ ]:
# =============================================================================
# SECTION 11 - Trainer
# =============================================================================
# One trainer is used by every segmentation model, so the comparison is fair:
# same loss (0.6 * Dice + 0.4 * BCE), same optimiser, same schedule, same
# validation protocol, same checkpoint format.
#
# Checkpoints contain everything needed to continue in a later Kaggle session:
# model, optimiser, scheduler, AMP scaler, epoch, step, best metric, RNG state.
# =============================================================================
from tqdm.auto import tqdm

import contextlib
import copy
import queue
import shutil as _shutil
import subprocess as _subprocess
import threading

from app.ml.losses import BCEDiceLoss  # embedded copy of the project's own loss class


class TimeBudgetExceeded(RuntimeError):
    pass


class TimeBudget:
    """Stops training cleanly before the Kaggle session dies, so the run can resume."""

    def __init__(self, minutes):
        self.deadline = time.time() + minutes * 60

    @property
    def remaining_minutes(self):
        return max(0.0, (self.deadline - time.time()) / 60)

    @property
    def expired(self):
        return self.remaining_minutes <= 0

    def require(self, minutes_needed=5.0):
        if self.remaining_minutes < minutes_needed:
            raise TimeBudgetExceeded(
                f"only {self.remaining_minutes:.1f} min left of the session budget; "
                "checkpoints are saved and the next session continues from them"
            )


def dice_score(prediction, target, eps=1e-7):
    prediction = (np.asarray(prediction) > 0.5).astype(np.float64)
    target = (np.asarray(target) > 0.5).astype(np.float64)
    intersection = float((prediction * target).sum())
    return float((2 * intersection + eps) / (prediction.sum() + target.sum() + eps))


def upsample_mask_nearest(mask, size):
    """Nearest-neighbour resize of a binary mask.

    The cache stores masks at the 4-channel size (256); the single-channel U-Net consumes
    512x512 inputs, so its masks are resized the same way the images are -- nearest, so a
    pixel is either tumour or not, and never a half-labelled boundary invented by bilinear
    interpolation.
    """
    tensor = torch.from_numpy(np.asarray(mask, dtype=np.float32))
    if tensor.dim() == 3:
        tensor = tensor[:, None]
    resized = torch.nn.functional.interpolate(tensor, size=(size, size), mode="nearest")
    return (resized[:, 0].numpy() > 0.5).astype(np.float32)


def iterate_batches_legacy(patient_ids, *, batch_size, size, single_channel=False, shuffle=True, rng=None, max_steps=None):
    """The original generator, kept verbatim for the before/after benchmark.

    It called `load_patient()` once per patient (to count slices) and once per slice (to
    build the batch). Every one of those calls inflated the patient's whole cached archive.
    `iterate_batches(..., legacy=True)` runs this; nothing else does.
    """
    rng = rng or np.random.default_rng(SEED)
    index = []
    for volume_key in patient_ids:
        payload = load_patient(volume_key)
        count = payload["masks"].shape[0]
        index.extend((volume_key, position) for position in range(count))
    if shuffle:
        rng.shuffle(index)
    steps = 0
    for start in range(0, len(index), batch_size):
        chunk = index[start : start + batch_size]
        images, masks = [], []
        for volume_key, position in chunk:
            payload = load_patient(volume_key)
            mask = payload["masks"][position][None, :, :].astype(np.float32)
            if single_channel:
                images.append(payload["singles"][position].astype(np.float32)[None, :, :])
                if mask.shape[-1] != size:
                    mask = upsample_mask_nearest(mask, size)
            else:
                images.append(np.moveaxis(payload["images"][position].astype(np.float32), -1, 0))
            masks.append(mask)
        yield np.stack(images), np.stack(masks)
        steps += 1
        if max_steps is not None and steps >= max_steps:
            return


def prepare_batch(chunk, *, store, size, single_channel):
    """Assemble one batch from `chunk` = [(volume_key, slice position), ...].

    One read per member per patient that appears in the chunk -- not one read per slice:
    the positions of that patient are requested together and indexed in a single step. The
    arrays, their order, the channel order (T1, T1ce, T2, FLAIR) and the mask handling are
    exactly what the previous implementation produced, so a batch is bit-for-bit the same
    (tests/test_data_pipeline.py asserts it against `iterate_batches_legacy`).
    """
    grouped: dict = {}
    for slot, (volume_key, position) in enumerate(chunk):
        grouped.setdefault(volume_key, []).append((slot, position))
    images: list = [None] * len(chunk)
    masks: list = [None] * len(chunk)
    image_member = "singles" if single_channel else "images"
    for volume_key, entries in grouped.items():
        positions = [position for _, position in entries]
        mask_rows = store.slices(volume_key, positions, member="masks")
        image_rows = store.slices(volume_key, positions, member=image_member)
        for row, (slot, _) in enumerate(entries):
            mask = mask_rows[row][None, :, :].astype(np.float32)
            if single_channel:
                image = image_rows[row].astype(np.float32)[None, :, :]
                if mask.shape[-1] != size:
                    mask = upsample_mask_nearest(mask, size)
            else:
                image = np.moveaxis(image_rows[row].astype(np.float32), -1, 0)
            images[slot] = image
            masks[slot] = mask
    return np.stack(images), np.stack(masks)


def iterate_batches(
    patient_ids,
    *,
    batch_size,
    size,
    single_channel=False,
    shuffle=True,
    rng=None,
    max_steps=None,
    store=None,
    legacy=False,
    prefetch=None,
):
    """Yield (images, masks) batches read from the cache on the fly.

    The batch composition, the slice ordering, the channel order (T1, T1ce, T2, FLAIR), the
    normalisation already baked into the cache, the mask resizing and the yielded dtypes are
    unchanged -- `legacy=True` runs the old implementation next to this one and the test
    suite asserts the two produce identical batches for the same seed.

    What changed is the number of times a cached patient is read:

      * the slice index is built from the cache index, i.e. from a header, so it costs no
        array reads at all. The old loop loaded a whole payload per patient for this.
      * each batch reads every member it needs once per patient appearing in it, and indexes
        the slices it wants from that. The old loop read the patient's entire archive for
        every single slice.
      * a 4-channel batch never opens `singles` and a single-channel batch never opens
        `images`; on a real BraTS volume that halves the bytes read.
    """
    if legacy:
        yield from iterate_batches_legacy(
            patient_ids,
            batch_size=batch_size,
            size=size,
            single_channel=single_channel,
            shuffle=shuffle,
            rng=rng,
            max_steps=max_steps,
        )
        return
    store = store or patient_store()
    rng = rng or np.random.default_rng(SEED)
    index = []
    for volume_key in patient_ids:
        count = store.slice_count(volume_key)
        index.extend((volume_key, position) for position in range(count))
    if shuffle:
        rng.shuffle(index)
    limit = len(index) if max_steps is None else min(len(index), int(max_steps) * batch_size)
    chunks = [index[start : start + batch_size] for start in range(0, limit, batch_size)]
    def make_batch(chunk):
        return prepare_batch(chunk, store=store, size=size, single_channel=single_channel)

    if prefetch is None:
        prefetch = bool(RUN.get("prefetch_batches", False))
    if prefetch:
        yield from prefetched_batches(make_batch, chunks)
        return
    for chunk in chunks:
        yield make_batch(chunk)


def prefetched_batches(make_batch, chunks, *, maxsize=2, join_timeout=2.0):
    """Prepare batches on one worker thread while the caller trains on the previous one.

    Threads rather than DataLoader processes: the arrays are read from memory maps, the GIL
    is released during the read, and nothing is pickled, so a batch costs no IPC and needs
    no worker start-up. It is only worth using when preparing a batch is a noticeable share
    of a training step; cell 18 measures both settings instead of assuming (`RUN[
    "prefetch_batches"]` is the default for the training loop).
    """
    end = object()
    slots = queue.Queue(maxsize=maxsize)
    stop = threading.Event()

    def produce():
        try:
            for chunk in chunks:
                if stop.is_set():
                    break
                slots.put(make_batch(chunk))
        except BaseException as exc:  # noqa: BLE001 - handed to the consumer, not swallowed
            with contextlib.suppress(Exception):
                slots.put(exc, timeout=5)
        finally:
            with contextlib.suppress(Exception):
                slots.put(end, timeout=5)

    worker = threading.Thread(target=produce, name="prefetch-batches", daemon=True)
    worker.start()
    try:
        while True:
            item = slots.get()
            if item is end:
                break
            if isinstance(item, BaseException):
                raise item
            yield item
    finally:
        stop.set()
        while True:                      # let a worker blocked on a full queue finish
            try:
                slots.get_nowait()
            except queue.Empty:
                break
        worker.join(timeout=join_timeout)


def augment_batch(images, masks):
    """Left/right flips only: safe for anatomy, cheap, and never invents lesion tissue.

    Works on torch tensors (the training loop moves batches to the device first) and on numpy
    arrays. Torch rejects negative strides, so the flip goes through `torch.flip`.
    """
    for index in range(images.shape[0]):
        if random.random() >= 0.5:
            continue
        if isinstance(images, torch.Tensor):
            images[index] = torch.flip(images[index], dims=[2])
            masks[index] = torch.flip(masks[index], dims=[2])
        else:
            images[index] = np.ascontiguousarray(images[index][:, :, ::-1])
            masks[index] = np.ascontiguousarray(masks[index][:, :, ::-1])
    return images, masks


class SegmentationTrainer:
    """AMP training loop with resumable checkpoints and validation-based model selection.

    `from_logits` mirrors the architecture's own `output_semantics`: SegNet and RAUNet return
    logits, DynamicUNet returns probabilities (it applies the sigmoid inside `forward`). The
    loss and the Dice computation follow that flag instead of assuming one convention.
    """

    def __init__(
        self,
        model_id,
        model,
        *,
        loss_weights,
        learning_rate,
        weight_decay,
        from_logits=True,
        device=DEVICE,
        amp=AMP_ENABLED,
    ):
        self.model_id = model_id
        self.model = model.to(device)
        self.device = device
        self.from_logits = bool(from_logits)
        self.criterion = BCEDiceLoss(
            from_logits=self.from_logits,
            bce_weight=loss_weights[1],
            dice_weight=loss_weights[0],
        ).to(device)
        self.amp = bool(amp and device.type == "cuda")
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        self.optimizer_zero_grad = getattr(self.optimizer, "zero_grad", None)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=50, eta_min=learning_rate / 20)
        try:
            self.scaler = torch.amp.GradScaler("cuda", enabled=self.amp)
            self.autocast = lambda: torch.amp.autocast("cuda", enabled=self.amp)
            self.amp_api = "torch.amp"
        except (AttributeError, TypeError):
            self.scaler = torch.cuda.amp.GradScaler(enabled=self.amp)
            self.autocast = lambda: torch.cuda.amp.autocast(enabled=self.amp)
            self.amp_api = "torch.cuda.amp"
        self.loss_weights = loss_weights
        self.directory = PATHS["checkpoints"] / model_id
        self.directory.mkdir(parents=True, exist_ok=True)
        self.history_path = self.directory / "history.json"
        self.state = {
            "epoch": 0,
            "step": 0,
            "best_dice": -1.0,
            "best_epoch": None,
            "history": read_json(self.history_path, default=[]) or [],
        }

    # -- checkpointing ----------------------------------------------------
    def checkpoint_payload(self, *, metrics=None, config=None, epochs_planned=None):
        return {
            "state_dict": {key: value.detach().cpu() for key, value in self.model.state_dict().items()},
            "model_id": self.model_id,
            "epoch": self.state["epoch"],
            "step": self.state["step"],
            "best_dice": self.state["best_dice"],
            "optimizer": self.optimizer.state_dict(),
            "scheduler": self.scheduler.state_dict(),
            "scaler": self.scaler.state_dict() if self.amp else None,
            "python_rng": random.getstate(),
            "numpy_rng": np.random.get_state(),
            "torch_rng": torch.get_rng_state(),
            "config": config or {},
            "epochs_planned": epochs_planned,
            "metrics": metrics or {},
            "saved_at": utc_now(),
            "amp_api": self.amp_api,
        }

    def save(self, *, metrics=None, config=None, epochs_planned=None, best=False):
        payload = self.checkpoint_payload(metrics=metrics, config=config, epochs_planned=epochs_planned)
        torch.save(payload, self.directory / "last.pt")
        if best:
            torch.save(payload, self.directory / "best.pt")
        write_json(self.history_path, self.state["history"])
        return self.directory / ("best.pt" if best else "last.pt")

    def resume(self, *, config=None, allow=False):
        path = self.directory / "last.pt"
        if not allow or not path.exists():
            return {"resumed": False, "path": str(path) if path.exists() else None}
        payload = torch.load(path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(payload["state_dict"])
        try:
            self.optimizer.load_state_dict(payload["optimizer"])
            self.scheduler.load_state_dict(payload["scheduler"])
            if payload.get("scaler") and self.amp:
                self.scaler.load_state_dict(payload["scaler"])
        except (KeyError, ValueError) as exc:
            print(f"WARNING: optimiser state from {path.name} could not be restored ({exc}); continuing with fresh optimiser state")
        self.state["epoch"] = int(payload.get("epoch", 0))
        self.state["step"] = int(payload.get("step", 0))
        self.state["best_dice"] = float(payload.get("best_dice", -1.0))
        if isinstance(payload.get("history"), list):
            self.state["history"] = payload["history"]
        elif self.history_path.exists():
            self.state["history"] = read_json(self.history_path, default=self.state["history"])
        print(
            f"resumed {self.model_id} from {path} at epoch {self.state['epoch']} "
            f"(best validation Dice {self.state['best_dice']:.4f})"
        )
        return {"resumed": True, "epoch": self.state["epoch"], "best_dice": self.state["best_dice"], "path": str(path)}

    # -- training ---------------------------------------------------------
    def train(
        self,
        *,
        train_patients,
        val_patients,
        epochs,
        batch_size,
        size,
        single_channel=False,
        max_steps_per_epoch=None,
        validation_cases=None,
        early_stopping_patience=8,
        validate_every=1,
        budget=None,
        resume=True,
        config=None,
        max_steps_per_batch=None,
    ):
        self.resume(allow=resume, config=config)
        rng = np.random.default_rng(SEED + self.state["epoch"])
        checkpoint_every_steps = int(RUN["checkpoint_every_steps"])
        last_save = time.time()
        started = time.time()
        stop_reason = None
        for epoch in range(self.state["epoch"] + 1, epochs + 1):
            self.state["epoch"] = epoch
            self.model.train()
            epoch_losses = []
            batch_iterator = iterate_batches(
                train_patients,
                batch_size=batch_size,
                size=size,
                single_channel=single_channel,
                shuffle=True,
                rng=rng,
                max_steps=max_steps_per_epoch,
            )
            progress = tqdm(batch_iterator, desc=f"{self.model_id} epoch {epoch}/{epochs}", leave=False)
            epoch_started = time.time()
            for step_in_epoch, (images, masks) in enumerate(progress, start=1):
                images_t = torch.from_numpy(images).to(self.device, non_blocking=True)
                masks_t = torch.from_numpy(masks).to(self.device, non_blocking=True)
                if not single_channel:
                    images_t, masks_t = augment_batch(images_t, masks_t)
                self.optimizer.zero_grad(set_to_none=True)
                with self.autocast():
                    outputs = self.model(images_t)
                    loss = self.criterion(outputs, masks_t)
                self.scaler.scale(loss).backward()
                if RUN["grad_clip"]:
                    self.scaler.unscale_(self.optimizer)
                    nn.utils.clip_grad_norm_(self.model.parameters(), RUN["grad_clip"])
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.state["step"] += 1
                epoch_losses.append(float(loss.detach()))
                progress.set_postfix(loss=f"{np.mean(epoch_losses[-20:]):.4f}")
                if budget is not None and budget.expired:
                    stop_reason = "session time budget reached"
                    break
                if checkpoint_every_steps and self.state["step"] % checkpoint_every_steps == 0:
                    self.save(config=config, epochs_planned=epochs)
                    last_save = time.time()
                    write_json(
                        PATHS["artifacts"] / f"{self.model_id}_progress.json",
                        {"epoch": epoch, "step": self.state["step"], "elapsed_minutes": round((time.time() - started) / 60, 2), "stop_reason": None},
                    )
            self.scheduler.step()
            epoch_seconds = time.time() - epoch_started
            entry = {
                "epoch": epoch,
                "train_loss": float(np.mean(epoch_losses)) if epoch_losses else None,
                "lr": float(self.optimizer.param_groups[0]["lr"]),
                "epoch_seconds": round(epoch_seconds, 2),
                "elapsed_minutes": round((time.time() - started) / 60, 2),
            }
            if validate_every and epoch % validate_every == 0 and val_patients:
                val_dice, val_loss = self.validate(
                    val_patients,
                    batch_size=batch_size,
                    size=size,
                    single_channel=single_channel,
                    max_cases=validation_cases,
                    budget=budget,
                )
                entry["val_dice"] = val_dice
                entry["val_loss"] = val_loss
                improved = val_dice > self.state["best_dice"]
                if improved:
                    self.state["best_dice"] = val_dice
                    self.state["best_epoch"] = epoch
                    self.state["epochs_without_improvement"] = 0
                else:
                    self.state["epochs_without_improvement"] = int(self.state.get("epochs_without_improvement", 0)) + 1
                self.state["history"].append(entry)
                self.save(metrics=entry, config=config, epochs_planned=epochs, best=improved)
                last_save = time.time()
                remaining = len(train_patients) * 155 // max(1, batch_size)  # rough remaining work
                print(
                    f"epoch {epoch:3d}/{epochs}  train_loss {entry['train_loss']:.4f}  "
                    f"val_dice {val_dice:.4f}  val_loss {val_loss:.4f}  "
                    f"best {self.state['best_dice']:.4f} @ {self.state['best_epoch']}  "
                    f"{epoch_seconds:.0f}s/epoch"
                )
                if self.state["epochs_without_improvement"] >= early_stopping_patience:
                    stop_reason = f"early stopping after {early_stopping_patience} epochs without validation improvement"
                    break
            else:
                self.state["history"].append(entry)
                self.save(metrics=entry, config=config, epochs_planned=epochs)
                print(f"epoch {epoch:3d}/{epochs}  train_loss {entry['train_loss']}  (no validation this epoch)")
            if budget is not None and budget.expired:
                stop_reason = "session time budget reached"
            if time.time() - last_save > RUN["checkpoint_every_minutes"] * 60:
                self.save(config=config, epochs_planned=epochs)
                last_save = time.time()
            if stop_reason:
                break
        self.save(config=config, epochs_planned=epochs)
        summary = {
            "model_id": self.model_id,
            "epochs_completed": self.state["epoch"],
            "epochs_planned": epochs,
            "best_dice": self.state["best_dice"],
            "best_epoch": self.state["best_epoch"],
            "stop_reason": stop_reason,
            "history": self.state["history"],
            "amp": self.amp,
            "device": str(self.device),
            "resumable": (self.directory / "last.pt").exists(),
        }
        write_json(self.directory / "training_summary.json", summary)
        return summary

    # -- validation -------------------------------------------------------
    def probabilities(self, outputs):
        """Detach a network output and turn it into probabilities, whatever it returned."""
        tensor = outputs[0] if isinstance(outputs, (list, tuple)) else outputs
        tensor = tensor.float()
        if self.from_logits:
            tensor = torch.sigmoid(tensor)
        return tensor.cpu().numpy()

    def validate(self, patient_ids, *, batch_size, size, single_channel=False, max_cases=None, budget=None):
        self.model.eval()
        scores, losses = [], []
        cases = 0
        with torch.no_grad():
            for images, masks in iterate_batches(
                patient_ids,
                batch_size=batch_size,
                size=size,
                single_channel=single_channel,
                shuffle=False,
            ):
                images_t = torch.from_numpy(images).to(self.device)
                masks_t = torch.from_numpy(masks).to(self.device)
                with self.autocast():
                    outputs = self.model(images_t)
                    loss = self.criterion(outputs, masks_t)
                probabilities = self.probabilities(outputs)
                targets = masks_t.cpu().numpy()
                for index in range(probabilities.shape[0]):
                    scores.append(dice_score(probabilities[index, 0] > 0.5, targets[index, 0] > 0.5))
                    losses.append(float(loss))
                    cases += 1
                    if max_cases and cases >= max_cases:
                        break
                if max_cases and cases >= max_cases:
                    break
                if budget is not None and budget.expired:
                    break
        self.model.train()
        return (float(np.mean(scores)) if scores else float("nan"), float(np.mean(losses)) if losses else float("nan"))


# =============================================================================
# SECTION 11c - Measuring the pipeline (where a training step's time goes)
# =============================================================================
# The trainer above is unchanged: same loss, same optimiser, same schedule, same AMP, same
# checkpoint format, same validation protocol. What is added here is measurement, because
# the reason the GPU was idle (0% utilisation observed, CPU at 100%) was never the model or
# the device placement -- it was 109.7 s of CPU work to prepare one batch of 8.
#
# Every GPU phase is timed with torch.cuda.synchronize() on both sides. Without the
# synchronise, a "fast" measurement only covers the kernel launch.
# =============================================================================


def synchronise(device=None):
    """`torch.cuda.synchronize()` on a CUDA device, a no-op anywhere else."""
    target = DEVICE if device is None else device
    kind = getattr(target, "type", None) or str(target).split(":")[0]
    if str(kind) == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize(torch.device(target))


def store_activity(store, before):
    """The counters a single measurement added, not the process-wide totals."""
    report = store.stats_report()
    for key, value in before.items():
        if isinstance(value, int) and isinstance(report.get(key), int):
            report[key] = report[key] - value
    return report


def summarise_seconds(values):
    """Median / min / max of a list of durations, in seconds."""
    if not values:
        return {"n": 0, "median": None, "min": None, "max": None}
    array = np.asarray(values, dtype=np.float64)
    return {
        "n": int(array.size),
        "median": round(float(np.median(array)), 5),
        "min": round(float(array.min()), 5),
        "max": round(float(array.max()), 5),
    }


def autocast_dtype():
    """The dtype autocast actually uses, reported instead of assumed."""
    try:
        for candidate in ("cuda", "cpu"):
            try:
                return str(torch.get_autocast_dtype(candidate))
            except (AttributeError, TypeError, RuntimeError):
                continue
    except Exception:  # noqa: BLE001 - diagnostics must never break a run
        pass
    return "unknown"


class gpu_utilization_probe:
    """Sample the real `nvidia-smi` utilisation while the body of the `with` block runs.

    On CPU-only machines (and in CI) this reports `available: False` with the reason; it
    never invents a number.
    """

    def __init__(self, interval=0.2):
        self.interval = float(interval)
        self.samples: list = []
        self.reason = None
        self._stop = threading.Event()
        self._thread = None

    def start(self):
        if not _shutil.which("nvidia-smi"):
            self.reason = "nvidia-smi is not on PATH"
            return self
        self._thread = threading.Thread(target=self._run, name="gpu-utilisation", daemon=True)
        self._thread.start()
        return self

    def _run(self):
        while not self._stop.is_set():
            try:
                result = _subprocess.run(
                    ["nvidia-smi", "--query-gpu=utilization.gpu", "--format=csv,noheader,nounits"],
                    capture_output=True,
                    text=True,
                    timeout=5,
                )
                if result.returncode == 0:
                    self.samples.append([int(item) for item in result.stdout.split()])
            except Exception as exc:  # noqa: BLE001 - report, never raise from a probe
                self.reason = f"{type(exc).__name__}: {exc}"
                return
            self._stop.wait(self.interval)

    def stop(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=3)
        return self.report()

    def report(self):
        if not self.samples:
            return {"available": False, "reason": self.reason or "no sample collected"}
        values = np.asarray(self.samples, dtype=np.float64)
        return {
            "available": True,
            "samples": len(self.samples),
            "interval_seconds": self.interval,
            "mean_percent": round(float(values.mean()), 1),
            "max_percent": round(float(values.max()), 1),
            "first_gpu_mean_percent": round(float(values[:, 0].mean()), 1),
            "per_sample_first_gpu": [int(row[0]) for row in self.samples],
        }

    def __enter__(self):
        return self.start()

    def __exit__(self, *exc_info):
        self.stop()
        return False


def benchmark_data_pipeline(
    patient_ids,
    *,
    batch_size,
    size,
    single_channel=False,
    steps=None,
    warmup=None,
    seed=None,
    legacy=False,
    prefetch=None,
    store=None,
):
    """Time batch preparation alone: no model, no GPU work, no other variable.

    `legacy=True` runs the previous implementation (one whole-archive read per slice) so the
    two can be compared on the same patients, the same cache state and the same batch size.
    """
    steps = int(RUN.get("benchmark_steps", 6) if steps is None else steps)
    warmup = int(RUN.get("benchmark_warmup", 2) if warmup is None else warmup)
    store = store or patient_store()
    seeds = int(SEED if seed is None else seed)
    activity_before = dict(store.stats)

    # The slice index the trainer rebuilds at the start of every epoch.
    started = time.perf_counter()
    if legacy:
        total_slices = sum(int(load_patient(key)["masks"].shape[0]) for key in patient_ids)
    else:
        total_slices = sum(patient_slice_count(key) for key in patient_ids)
    index_seconds = time.perf_counter() - started

    store.release()
    iterator = iterate_batches(
        patient_ids,
        batch_size=batch_size,
        size=size,
        single_channel=single_channel,
        shuffle=True,
        rng=np.random.default_rng(seeds),
        max_steps=warmup + steps,
        legacy=legacy,
        prefetch=prefetch,
        store=store,
    )
    durations, first_batch = [], None
    while True:
        started = time.perf_counter()
        try:
            images, masks = next(iterator)
        except StopIteration:
            break
        durations.append(time.perf_counter() - started)
        if first_batch is None:
            first_batch = durations[-1]
    # Drop the warm-up batches -- unless there were not enough of them, in which case every
    # batch that could be prepared is reported and `warmup_dropped` says so.
    warmup_dropped = warmup if len(durations) > warmup else 0
    measured = durations[warmup_dropped:]
    store = store or patient_store()
    report = {
        "pipeline": "legacy" if legacy else "cached-array",
        "prefetch": bool(prefetch if prefetch is not None else RUN.get("prefetch_batches", False)),
        "patients": len(patient_ids),
        "slices_available": int(total_slices),
        "batch_size": int(batch_size),
        "input_size": int(size),
        "channels": 1 if single_channel else 4,
        "index_seconds": round(index_seconds, 4),
        "index_seconds_per_patient": round(index_seconds / max(1, len(patient_ids)), 6),
        "first_batch_seconds": None if first_batch is None else round(first_batch, 4),
        "batch_seconds": summarise_seconds(measured),
        "warmup_dropped": warmup_dropped,
        "note": None if len(durations) > warmup else f"only {len(durations)} batch(es) were available; none dropped as warm-up",
        "slices_per_second": None
        if not measured or max(measured) == 0
        else round(batch_size * len(measured) / sum(measured), 1),
        "store": store_activity(store, activity_before),
    }
    return report


def benchmark_training_step(
    trainer,
    patient_ids,
    *,
    batch_size,
    size,
    single_channel=False,
    steps=None,
    warmup=None,
    seed=None,
    legacy=False,
    prefetch=None,
    probe_gpu=True,
    store=None,
):
    """Time the five phases of a real training step, separately and end to end.

    1. batch preparation (CPU, cache -> stacked numpy arrays)
    2. host-to-device transfer
    3. forward pass
    4. backward pass + optimiser step
    5. the whole step

    The measurement runs on an identical COPY of the model with a fresh AdamW, so the
    weights, the optimiser state and the step counter training is about to use are untouched.
    AMP, the loss, the gradient clipping and the autocast context are the trainer's own.
    """
    steps = int(RUN.get("benchmark_steps", 6) if steps is None else steps)
    warmup = int(RUN.get("benchmark_warmup", 2) if warmup is None else warmup)
    device = trainer.device
    model = copy.deepcopy(trainer.model).to(device)
    model.train()
    reference = trainer.optimizer.param_groups[0]
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=float(reference["lr"]), weight_decay=float(reference.get("weight_decay", 0.0))
    )
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=trainer.amp)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=trainer.amp)

    def one_step(images, masks):
        started = time.perf_counter()
        images_t = torch.from_numpy(images).to(device, non_blocking=True)
        masks_t = torch.from_numpy(masks).to(device, non_blocking=True)
        synchronise(device)
        transferred = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        with trainer.autocast():
            outputs = model(images_t)
            loss = trainer.criterion(outputs, masks_t)
        synchronise(device)
        forwarded = time.perf_counter()
        scaler.scale(loss).backward()
        if RUN["grad_clip"]:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), RUN["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        synchronise(device)
        finished = time.perf_counter()
        return (transferred - started, forwarded - transferred, finished - forwarded, finished - started), float(loss.detach())

    store = store or patient_store()
    activity_before = dict(store.stats)
    total_slices = sum(patient_slice_count(key) for key in patient_ids)
    store.release()
    iterator = iterate_batches(
        patient_ids,
        batch_size=batch_size,
        size=size,
        single_channel=single_channel,
        shuffle=True,
        rng=np.random.default_rng(int(SEED if seed is None else seed)),
        max_steps=warmup + steps,
        legacy=legacy,
        prefetch=prefetch,
        store=store,
    )
    seen_prepare, seen_transfer, seen_forward, seen_backward, seen_totals, seen_losses = [], [], [], [], [], []
    first_batch = None
    probe = gpu_utilization_probe() if probe_gpu else None
    if probe is not None:
        probe.start()
    try:
        while True:
            started = time.perf_counter()
            try:
                images, masks = next(iterator)
            except StopIteration:
                break
            prepared = time.perf_counter() - started
            phases, loss = one_step(images, masks)
            if first_batch is None:
                first_batch = prepared + phases[3]
            seen_prepare.append(prepared)
            seen_transfer.append(phases[0])
            seen_forward.append(phases[1])
            seen_backward.append(phases[2])
            seen_totals.append(phases[3])
            seen_losses.append(loss)
    finally:
        gpu = probe.stop() if probe is not None else None

    # Drop the warm-up steps -- unless there were not enough of them (a small smoke run or a
    # capped epoch), in which case every step that ran is reported instead of nothing.
    warmup_dropped = warmup if len(seen_totals) > warmup else 0
    prepare = seen_prepare[warmup_dropped:]
    transfer = seen_transfer[warmup_dropped:]
    forward = seen_forward[warmup_dropped:]
    backward = seen_backward[warmup_dropped:]
    totals = seen_totals[warmup_dropped:]
    losses = seen_losses[warmup_dropped:]

    report = {
        "device": str(device),
        "cuda_available": bool(torch.cuda.is_available()),
        "gpu_name": GPU_NAME,
        "amp": bool(trainer.amp),
        "autocast_dtype": autocast_dtype(),
        "model_id": trainer.model_id,
        "batch_size": int(batch_size),
        "input_size": int(size),
        "channels": 1 if single_channel else 4,
        "patients": len(patient_ids),
        "slices_available": int(total_slices),
        "pipeline": "legacy" if legacy else "cached-array",
        "prefetch": bool(prefetch if prefetch is not None else RUN.get("prefetch_batches", False)),
        "steps_measured": len(totals),
        "steps_seen": len(seen_totals),
        "steps_warmup": warmup,
        "warmup_dropped": warmup_dropped,
        "first_batch_seconds": None if first_batch is None else round(first_batch, 4),
        "phases_seconds": {
            "prepare": summarise_seconds(prepare),
            "cpu_to_gpu": summarise_seconds(transfer),
            "forward": summarise_seconds(forward),
            "backward_and_step": summarise_seconds(backward),
            "step_total": summarise_seconds(totals),
        },
        "loss_first": None if not losses else round(losses[0], 6),
        "loss_last": None if not losses else round(losses[-1], 6),
        "gpu_utilization": gpu,
        "store": store_activity(store, activity_before),
    }
    median_total = report["phases_seconds"]["step_total"]["median"]
    median_prepare = report["phases_seconds"]["prepare"]["median"]
    if median_total:
        report["prepare_share_of_step"] = round(median_prepare / median_total, 3)
        report["gpu_share_of_step"] = round(1 - median_prepare / median_total, 3)
    report["notes"] = []
    if not torch.cuda.is_available():
        report["notes"].append(
            "no CUDA device here: the forward/backward numbers are CPU timings. The GPU phases "
            "can only be judged on the T4 run."
        )
    elif report.get("prepare_share_of_step", 0) > 0.5:
        report["notes"].append(
            "the CPU still spends most of the step preparing batches, so the GPU cannot be busy: "
            f"{report['prepare_share_of_step'] * 100:.0f}% of the step is preparation"
        )
    else:
        report["notes"].append(
            f"preparation is {report.get('prepare_share_of_step', 0) * 100:.0f}% of a step, so the GPU is now the "
            "limiting factor -- utilisation can only be as high as the GPU work allows"
        )
    report["notes"].append(
        "measured on a copy of the model with a fresh optimiser: the real training state (weights, "
        "optimiser, step counter) is untouched by the benchmark"
    )
    if len(seen_totals) <= warmup:
        report["notes"].append(
            f"only {len(seen_totals)} step(s) could be prepared from the available slices, so no step was "
            "discarded as warm-up; the numbers include a cold first step"
        )
    if not totals:
        report["notes"].append("no step could be measured: no cached patient was available")
    return report


def verify_training_step(trainer, patient_ids, *, batch_size, size, single_channel=False, seed=None, store=None):
    """One batch through the real pipeline: shapes, dtypes, finite loss, real gradients, and a
    checkpoint that reloads.

    The forward/backward runs on a copy of the model WITHOUT an optimiser step, so nothing
    the training loop depends on moves. The checkpoint is written to `work/selftest/`, never
    to `checkpoints/<model_id>/`.
    """
    device = trainer.device
    model = copy.deepcopy(trainer.model).to(device)
    model.train()
    iterator = iterate_batches(
        patient_ids,
        batch_size=batch_size,
        size=size,
        single_channel=single_channel,
        shuffle=False,
        rng=np.random.default_rng(int(SEED if seed is None else seed)),
        max_steps=1,
        store=store or patient_store(),
    )
    images, masks = next(iterator)
    images_t = torch.from_numpy(images).to(device)
    masks_t = torch.from_numpy(masks).to(device)
    model.zero_grad(set_to_none=True)
    with trainer.autocast():
        outputs = model(images_t)
        loss = trainer.criterion(outputs, masks_t)
    loss.backward()
    logits = outputs[0] if isinstance(outputs, (list, tuple)) else outputs
    gradients = [parameter.grad for parameter in model.parameters() if parameter.grad is not None]
    gradient_norm = float(torch.sqrt(sum((gradient.detach().float() ** 2).sum() for gradient in gradients)))

    payload = trainer.checkpoint_payload(metrics={"self_test": True}, config={"self_test": True}, epochs_planned=0)
    target = PATHS["work"] / "selftest" / f"{trainer.model_id}_selftest.pt"
    target.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, target)
    reloaded = torch.load(target, map_location="cpu", weights_only=False)
    fresh = copy.deepcopy(trainer.model).to("cpu")
    load_report = fresh.load_state_dict(reloaded["state_dict"], strict=True)
    identical = all(
        torch.equal(fresh.state_dict()[key].cpu(), trainer.model.state_dict()[key].detach().cpu()) for key in fresh.state_dict()
    )
    resume_state = trainer.resume(allow=True, config={"self_test": True})
    model.zero_grad(set_to_none=True)

    return {
        "model_id": trainer.model_id,
        "device": str(device),
        "model_parameter_device": str(next(model.parameters()).device),
        "images_shape": list(images.shape),
        "masks_shape": list(masks.shape),
        "images_dtype": str(np.asarray(images).dtype),
        "masks_dtype": str(np.asarray(masks).dtype),
        "mask_values": sorted(int(value) for value in np.unique(masks)[:4]),
        "images_finite": bool(np.isfinite(np.asarray(images, dtype=np.float32)).all()),
        "masks_binary": bool(np.isin(np.unique(masks), [0, 1]).all()),
        "channel_order": "T1, T1ce, T2, FLAIR (cache contract; channel c is read as-is)",
        "output_shape": list(logits.shape),
        "output_device": str(logits.device),
        "loss": round(float(loss.detach()), 6),
        "loss_finite": bool(torch.isfinite(loss)),
        "amp_enabled": bool(trainer.amp),
        "autocast_dtype": autocast_dtype(),
        "gradients": len(gradients),
        "gradient_norm": round(gradient_norm, 6),
        "gradients_finite": bool(all(torch.isfinite(gradient).all() for gradient in gradients)),
        "gradients_nonzero": bool(any(float(gradient.abs().max()) > 0 for gradient in gradients)),
        "checkpoint_path": str(target),
        "checkpoint_keys": sorted(payload),
        "checkpoint_reload_ok": not load_report.missing_keys and not load_report.unexpected_keys and identical,
        "resume_state": resume_state,
    }


def load_trained_model(model_id, builder, path=None):
    """Load a checkpoint written by this notebook (or by the project's scripts)."""
    candidate = Path(path) if path else PATHS["checkpoints"] / model_id / "best.pt"
    if not candidate.exists():
        candidate = PATHS["checkpoints"] / model_id / "last.pt"
    if not candidate.exists():
        raise FileNotFoundError(
            f"no checkpoint for {model_id} at {PATHS['checkpoints'] / model_id}; run its training section first"
        )
    payload = torch.load(candidate, map_location=DEVICE, weights_only=False)
    model = builder()
    report = model.load_state_dict(payload["state_dict"], strict=True)
    model.eval()
    return model.to(DEVICE), payload, candidate


print(
    "Trainer ready. Loss =",
    f"{RUN['dice_weight']}*Dice + {1 - RUN['dice_weight']:.1f}*BCE",
    "(BCEDiceLoss, the project's own class; from_logits set per architecture)",
    "| AMP:",
    AMP_ENABLED,
)


## 18. SegNet training / retraining (4-channel)
`nbs_04_models.py`


In [ ]:
# =============================================================================
# SECTION 11b - SegNet
# =============================================================================
# This is the model that ships as `segnet_model.pth` and that was found to be
# degenerate (its sigmoid output never crossed 0.5 on any available input).
# This section retrains it on real BraTS data with the architecture the
# application loads: SegNet(in_channels=4, out_channels=1), 256x256, z-scored
# channels, logits out. A checkpoint trained here is written to the notebook's
# own directory; the repository's original file is never overwritten.
# =============================================================================
from app.ml.architectures.segnet import SegNet

SEGNET_CONFIG = {
    "model_id": "segnet_brats_4ch",
    "architecture": "SegNet(in_channels=4, out_channels=1) -- application-compatible state dict",
    "input_size": RUN["image_size"],
    "channels": 4,
    "normalisation": "global z-score over (H,W,C), then CHW",
    "output_semantics": "logit (BCE with logits; sigmoid applied for Dice and at inference)",
    "loss": f"{RUN['dice_weight']}*Dice + {1 - RUN['dice_weight']:.1f}*BCE",
    "optimiser": f"AdamW lr={RUN['learning_rate']} weight_decay={RUN['weight_decay']}",
    "scheduler": "cosine annealing",
    "batch_size": RUN["batch_size"],
    "epochs": RUN["epochs_segnet"],
    "seed": SEED,
    "dataset": LOCK.get("dataset", {}),
    "splits": {"train": len(TRAIN_PATIENTS), "validation": len(VAL_PATIENTS), "test": len(TEST_PATIENTS)},
    "amp": AMP_ENABLED,
}

segnet = SegNet(in_channels=4, out_channels=1)
SEGNET_PARAMETERS = sum(parameter.numel() for parameter in segnet.parameters())
segnet_trainer = SegmentationTrainer(
    "segnet_brats_4ch",
    segnet,
    loss_weights=(RUN["dice_weight"], 1 - RUN["dice_weight"]),
    learning_rate=RUN["learning_rate"],
    weight_decay=RUN["weight_decay"],
    from_logits=SegNet.output_semantics.fget(segnet) == "logit",
)
print(f"SegNet parameters: {SEGNET_PARAMETERS:,} (the application's builder produces the same graph)")

# --- pre-flight checks and a real benchmark, BEFORE the first epoch -----------
# A one-batch forward/backward, the tensors' shapes and dtypes, a finite loss and real
# gradients, a checkpoint that saves and reloads, and the five phases of a training step
# measured separately. Nothing here is estimated: the numbers below are what this machine
# and this cache actually do, which is what decides whether the epoch loop is worth starting.
if RUN.get("benchmark_before_training", True):
    print("=" * 78)
    print("PRE-TRAINING CHECKS -- SegNet")
    print("=" * 78)
    SEGNET_SELF_TEST = verify_training_step(
        segnet_trainer, TRAIN_PATIENTS, batch_size=RUN["batch_size"], size=RUN["image_size"]
    )
    print(f"  batch                : images {SEGNET_SELF_TEST['images_shape']} {SEGNET_SELF_TEST['images_dtype']} | "
          f"masks {SEGNET_SELF_TEST['masks_shape']} {SEGNET_SELF_TEST['masks_dtype']} values {SEGNET_SELF_TEST['mask_values']}")
    print(f"  forward output       : {SEGNET_SELF_TEST['output_shape']} on {SEGNET_SELF_TEST['output_device']}")
    print(f"  loss                 : {SEGNET_SELF_TEST['loss']} (finite: {SEGNET_SELF_TEST['loss_finite']}), "
          f"amp {SEGNET_SELF_TEST['amp_enabled']} ({SEGNET_SELF_TEST['autocast_dtype']})")
    print(f"  gradients            : {SEGNET_SELF_TEST['gradients']} tensors, norm {SEGNET_SELF_TEST['gradient_norm']}, "
          f"finite {SEGNET_SELF_TEST['gradients_finite']}, nonzero {SEGNET_SELF_TEST['gradients_nonzero']}")
    print(f"  inputs finite        : {SEGNET_SELF_TEST['images_finite']} | masks binary: {SEGNET_SELF_TEST['masks_binary']}")
    print(f"  checkpoint           : {SEGNET_SELF_TEST['checkpoint_path']}")
    print(f"                         keys: {', '.join(SEGNET_SELF_TEST['checkpoint_keys'])}")
    print(f"                         reloads byte-identically: {SEGNET_SELF_TEST['checkpoint_reload_ok']}")
    print(f"  resume state         : {SEGNET_SELF_TEST['resume_state']}")
    write_json(PATHS["artifacts"] / "segnet_self_test.json", SEGNET_SELF_TEST)

    SEGNET_BENCHMARK = benchmark_training_step(
        segnet_trainer,
        TRAIN_PATIENTS,
        batch_size=RUN["batch_size"],
        size=RUN["image_size"],
        steps=RUN["benchmark_steps"],
        warmup=RUN["benchmark_warmup"],
        legacy=bool(RUN.get("benchmark_legacy_pipeline", False)),
    )
    SEGNET_BENCHMARK["cache_io"] = {"before": CACHE_IO_BEFORE, "after": CACHE_IO_AFTER, "upgrade": CACHE_UPGRADE}
    print()
    print(f"  BENCHMARK -- {SEGNET_BENCHMARK['steps_measured']} measured steps after "
          f"{SEGNET_BENCHMARK['steps_warmup']} warm-up, batch {SEGNET_BENCHMARK['batch_size']}, "
          f"{SEGNET_BENCHMARK['channels']}x{SEGNET_BENCHMARK['input_size']}x{SEGNET_BENCHMARK['input_size']}, "
          f"{SEGNET_BENCHMARK['device']}, pipeline '{SEGNET_BENCHMARK['pipeline']}'")
    for label, key in (
        ("1. batch preparation", "prepare"),
        ("2. CPU -> GPU transfer", "cpu_to_gpu"),
        ("3. forward pass", "forward"),
        ("4. backward + optimiser step", "backward_and_step"),
        ("5. end-to-end training step", "step_total"),
    ):
        entry = SEGNET_BENCHMARK["phases_seconds"][key]
        if entry["n"]:
            print(f"     {label:30s} median {entry['median'] * 1000:9.2f} ms   "
                  f"min {entry['min'] * 1000:9.2f} ms   max {entry['max'] * 1000:9.2f} ms")
        else:
            print(f"     {label:30s} not measured")
    print(f"     {'first batch (cold start)':30s} {SEGNET_BENCHMARK['first_batch_seconds']} s")
    print(f"     {'preparation share of a step':30s} {SEGNET_BENCHMARK.get('prepare_share_of_step')}")
    print(f"     {'GPU utilisation (nvidia-smi)':30s} {SEGNET_BENCHMARK['gpu_utilization']}")
    if CACHE_IO_BEFORE and CACHE_IO_BEFORE.get("legacy_batch_seconds_per_slice"):
        print(f"     {'cache read before (archive)':30s} "
              f"{CACHE_IO_BEFORE['legacy_batch_seconds_per_slice'] * 1000:.3f} ms per slice")
    print(f"     {'cache read after (memory map)':30s} "
          f"{CACHE_IO_AFTER['batch_seconds_per_slice'] * 1000:.3f} ms per slice")
    for note in SEGNET_BENCHMARK["notes"]:
        print("     note:", note)
    write_json(PATHS["artifacts"] / "segnet_benchmark.json", SEGNET_BENCHMARK)
    print("=" * 78)

if not RUN.get("skip_segnet_training", False):
    SEGNET_BUDGET = TimeBudget(RUN["session_budget_minutes"])
    SEGNET_SUMMARY = segnet_trainer.train(
        train_patients=TRAIN_PATIENTS,
        val_patients=VAL_PATIENTS,
        epochs=RUN["epochs_segnet"],
        batch_size=RUN["batch_size"],
        size=RUN["image_size"],
        single_channel=False,
        max_steps_per_epoch=RUN["max_steps_per_epoch"],
        validation_cases=RUN["validation_cases"],
        early_stopping_patience=RUN["early_stopping_patience"],
        budget=SEGNET_BUDGET,
        config=SEGNET_CONFIG,
    )
    SEGNET_SUMMARY["parameters"] = SEGNET_PARAMETERS
    write_json(PATHS["artifacts"] / "segnet_training_summary.json", SEGNET_SUMMARY)
    print("=" * 78)
    print("SEGNET TRAINING SUMMARY")
    print("=" * 78)
    for key in ("epochs_completed", "epochs_planned", "best_dice", "best_epoch", "stop_reason", "amp", "device"):
        print(f"{key:18s}: {SEGNET_SUMMARY[key]}")
    if SEGNET_SUMMARY["stop_reason"]:
        print(
            "\nThe run did not reach its planned epoch count. Re-run this cell in a new session "
            "with RESUME=True (default): training continues from checkpoints/segnet_brats_4ch/last.pt."
        )
else:
    print("skip_segnet_training = True -> not training SegNet in this session")


## 19. U-Net baseline (single-channel 512x512)
`nbs_04_models.py`


In [ ]:
# =============================================================================
# SECTION 12 - U-Net baseline on real data
# =============================================================================
# The shipped `UNet-[16, 32, 64, 128, 256].pt` is the only artefact that already
# worked, but it was never evaluated on labelled data. This section trains the
# same architecture from scratch on real BraTS FLAIR slices resized to 512x512
# with the input contract the application uses (1 channel, [0, 1] scaling), so
# the reported Dice belongs to a checkpoint whose provenance is known.
# =============================================================================
from app.ml.architectures.dynamic_unet import DEFAULT_FILTERS, DynamicUNet

UNET_CONFIG = {
    "model_id": "unet_dynamic_512",
    "architecture": f"DynamicUNet filters {DEFAULT_FILTERS}, in_channels=1, out_channels=1",
    "input_size": RUN["single_image_size"],
    "channels": 1,
    "channel_used": {0: "T1", 1: "T1ce", 2: "T2", 3: "FLAIR"}[RUN["single_channel_index"]],
    "normalisation": "[0, 1] intensity scaling (the application's single-channel contract)",
    "output_semantics": "probability (sigmoid inside the network; loss uses the probability branch)",
    "loss": f"{RUN['dice_weight']}*Dice + {1 - RUN['dice_weight']:.1f}*BCE",
    "optimiser": f"AdamW lr={RUN['learning_rate_single']} weight_decay={RUN['weight_decay']}",
    "batch_size": RUN["batch_size_single"],
    "epochs": RUN["epochs_unet"],
    "seed": SEED,
    "splits": {"train": len(TRAIN_PATIENTS), "validation": len(VAL_PATIENTS), "test": len(TEST_PATIENTS)},
    "amp": AMP_ENABLED,
}

unet = DynamicUNet(DEFAULT_FILTERS, input_channels=1, output_channels=1)
UNET_PARAMETERS = sum(parameter.numel() for parameter in unet.parameters())
unet_trainer = SegmentationTrainer(
    "unet_dynamic_512_real_brats",
    unet,
    loss_weights=(RUN["dice_weight"], 1 - RUN["dice_weight"]),
    learning_rate=RUN["learning_rate_single"],
    weight_decay=RUN["weight_decay"],
    from_logits=DynamicUNet.output_semantics.fget(unet) == "logit",
)
print(f"DynamicUNet parameters: {UNET_PARAMETERS:,}")

if not RUN.get("skip_unet_training", False):
    UNET_BUDGET = TimeBudget(RUN["session_budget_minutes"])
    UNET_SUMMARY = unet_trainer.train(
        train_patients=TRAIN_PATIENTS,
        val_patients=VAL_PATIENTS,
        epochs=RUN["epochs_unet"],
        batch_size=RUN["batch_size_single"],
        size=RUN["single_image_size"],
        single_channel=True,
        max_steps_per_epoch=RUN["max_steps_per_epoch"],
        validation_cases=max(4, RUN["validation_cases"] // 4),
        early_stopping_patience=RUN["early_stopping_patience"],
        budget=UNET_BUDGET,
        config=UNET_CONFIG,
    )
    UNET_SUMMARY["parameters"] = UNET_PARAMETERS
    write_json(PATHS["artifacts"] / "unet_training_summary.json", UNET_SUMMARY)
    print("=" * 78)
    print("U-NET TRAINING SUMMARY")
    print("=" * 78)
    for key in ("epochs_completed", "epochs_planned", "best_dice", "best_epoch", "stop_reason", "amp", "device"):
        print(f"{key:18s}: {UNET_SUMMARY[key]}")
else:
    print("skip_unet_training = True -> not training the U-Net in this session")


## 20. Residual Attention U-Net (RAUNet, 4-channel)
`nbs_04_models.py`


In [ ]:
# =============================================================================
# SECTION 13 - RAUNet (residual attention U-Net with deep supervision)
# =============================================================================
# Same input contract as SegNet (4 x 256 x 256, z-score), so the two can be
# fused later. It is the heaviest model here: if a session ends mid-run, rerun
# this cell and it continues from checkpoints/raunet_brats_4ch/last.pt.
# =============================================================================
from app.ml.architectures.attention_unet import build_raunet

RAUNET_CONFIG = {
    "model_id": "raunet_brats_4ch",
    "architecture": "RAUNet, base_filters=16, depth=4, attention gates, deep supervision",
    "input_size": RUN["image_size"],
    "channels": 4,
    "normalisation": "global z-score over (H,W,C), then CHW (identical to SegNet)",
    "output_semantics": "logit",
    "loss": f"{RUN['dice_weight']}*Dice + {1 - RUN['dice_weight']:.1f}*BCE",
    "optimiser": f"AdamW lr={RUN['learning_rate']} weight_decay={RUN['weight_decay']}",
    "batch_size": RUN["batch_size"],
    "epochs": RUN["epochs_raunet"],
    "seed": SEED,
    "splits": {"train": len(TRAIN_PATIENTS), "validation": len(VAL_PATIENTS), "test": len(TEST_PATIENTS)},
    "amp": AMP_ENABLED,
}

raunet = build_raunet(in_channels=4, out_channels=1, base_filters=16, depth=4)
RAUNET_PARAMETERS = sum(parameter.numel() for parameter in raunet.parameters())
raunet_trainer = SegmentationTrainer(
    "raunet_brats_4ch",
    raunet,
    loss_weights=(RUN["dice_weight"], 1 - RUN["dice_weight"]),
    learning_rate=RUN["learning_rate"],
    weight_decay=RUN["weight_decay"],
    from_logits=type(raunet).output_semantics.fget(raunet) == "logit",
)
print(f"RAUNet parameters: {RAUNET_PARAMETERS:,}")

if not RUN.get("skip_raunet_training", False):
    RAUNET_BUDGET = TimeBudget(RUN["session_budget_minutes"])
    RAUNET_SUMMARY = raunet_trainer.train(
        train_patients=TRAIN_PATIENTS,
        val_patients=VAL_PATIENTS,
        epochs=RUN["epochs_raunet"],
        batch_size=RUN["batch_size"],
        size=RUN["image_size"],
        single_channel=False,
        max_steps_per_epoch=RUN["max_steps_per_epoch"],
        validation_cases=RUN["validation_cases"],
        early_stopping_patience=RUN["early_stopping_patience"],
        budget=RAUNET_BUDGET,
        config=RAUNET_CONFIG,
    )
    RAUNET_SUMMARY["parameters"] = RAUNET_PARAMETERS
    write_json(PATHS["artifacts"] / "raunet_training_summary.json", RAUNET_SUMMARY)
    print("=" * 78)
    print("RAUNET TRAINING SUMMARY")
    print("=" * 78)
    for key in ("epochs_completed", "epochs_planned", "best_dice", "best_epoch", "stop_reason", "amp", "device"):
        print(f"{key:18s}: {RAUNET_SUMMARY[key]}")
    if RAUNET_SUMMARY["stop_reason"]:
        print("\nResume in the next session by re-running this cell: the checkpoint carries optimiser, scheduler and RNG state.")
else:
    print("skip_raunet_training = True -> not training RAUNet in this session")


## 21. Training curves
`nbs_04_models.py`


In [ ]:
# =============================================================================
# Loss and validation curves for every model trained above
# =============================================================================
def plot_history(model_id, axis_pair):
    history = read_json(PATHS["checkpoints"] / model_id / "history.json", default=[]) or []
    if not history:
        return False
    epochs = [entry["epoch"] for entry in history]
    losses = [entry.get("train_loss") for entry in history]
    dice = [entry.get("val_dice") for entry in history]
    axis_pair[0].plot(epochs, losses, marker="o", label=f"{model_id} train loss")
    axis_pair[0].set_ylabel("loss")
    axis_pair[1].plot(epochs, dice, marker="o", label=f"{model_id} val Dice")
    axis_pair[1].set_ylabel("Dice")
    return True


fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
plotted = False
for model_id in ("segnet_brats_4ch", "unet_dynamic_512_real_brats", "raunet_brats_4ch"):
    plotted = plot_history(model_id, axes) or plotted
axes[0].set_title("Training loss"); axes[1].set_title("Validation Dice")
axes[0].set_xlabel("epoch"); axes[1].set_xlabel("epoch")
if plotted:
    axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(PATHS["reports"] / "04_training_curves.png", dpi=130)
plt.close(fig)
print("training curves ->", PATHS["reports"] / "04_training_curves.png", "| histories found:", plotted)


## 22. Classification dataset: radiomics features in the application's schema
`nbs_05_classification.py`


In [ ]:
# =============================================================================
# SECTION 14 / 16 - Classification data (radiomics features in the application's schema)
# =============================================================================
# Order note: the request lists classification training before radiomics
# extraction. Radiomics features ARE the classification input here, so extraction
# has to happen first; Section 16 later adds the feature-quality analysis
# (reference mask vs predicted mask, slice-to-slice stability).
#
# The feature names come from the application's own extractor (embedded verbatim
# from backend/app/ml/radiomics_native.py), so the table a model is trained on has
# exactly the schema the API computes at serving time. That is the fix for the
# original "21 features expected, 11 available, no scaler" mismatch.
#
# What the serving path actually does (read from the code, not assumed):
#   * analysis_service._run_radiomics() extracts from `display_reference`, which is
#     preprocessed.display_stack[0][1] -- the FIRST channel of a four-channel upload;
#   * preprocessing_service.MODALITY_ORDER_BRATS = ("t1", "t1ce", "t2", "flair"),
#     so that plane is T1;
#   * the ROI is one slice's mask, and the feature names carry no channel prefix.
# The tables below therefore hold bare feature names on the T1 plane of the slice
# with the largest tumour area, which is the closest training-side equivalent of a
# served analysis. Features are computed for two regions of interest so the effect
# of the mask source can be measured instead of guessed:
#   * "reference" - the BraTS ground-truth mask (what the annotation says),
#   * "predicted" - the mask of the best segmentation checkpoint chosen on the
#     validation split (what the API will actually use).
# =============================================================================
import csv
from types import SimpleNamespace

from app.services.radiomics_service import RadiomicsService

RADIOMICS_CONFIG = SimpleNamespace(
    RADIOMICS_BACKEND="native",       # PyRadiomics is used when installed; the schema is identical
    RADIOMICS_BIN_COUNT=32,
    RADIOMICS_BIN_WIDTH=0.0,
    RADIOMICS_DISCRETIZATION="bin_count",
)
radiomics_service = RadiomicsService(RADIOMICS_CONFIG)
print("radiomics backend:", radiomics_service.backend_status()["selected_backend"])

SERVING_CHANNEL = 0
SERVING_CHANNEL_NAME = ("t1", "t1ce", "t2", "flair")[SERVING_CHANNEL]
MASK_SOURCES = ("reference", "predicted")

FEATURE_TABLE_PATHS = {
    "reference": PATHS["artifacts"] / "radiomics_features.csv",
    "predicted": PATHS["artifacts"] / "radiomics_features_predicted.csv",
}
FEATURE_TABLE_PATH = FEATURE_TABLE_PATHS["reference"]
ROW_METADATA_COLUMNS = (
    "case_id",
    "patient_id",
    "label",
    "split",
    "mask_source",
    "foreground_pixels",
    "radiomics_backend",
    "image_channel",
    "slice_index",
)
# The extractor also reports diagnostics next to the features ('shape_units': 'pixels',
# 'diagnostics_dimensionality': '2D', ...). They are not measurements, so they are kept out
# of the schema -- and the names are recorded so the exclusion is visible in the manifest.
NON_NUMERIC_FEATURE_NAMES: set[str] = set()


def numeric_features(outcome):
    """Yield (name, value) for the numeric, finite features only."""
    for name, value in outcome.features.items():
        if isinstance(value, bool) or not isinstance(value, (int, float, np.integer, np.floating)):
            NON_NUMERIC_FEATURE_NAMES.add(name)
            continue
        number = float(value)
        if not np.isfinite(number):
            continue
        yield name, number


def focus_slice_index(masks):
    """Index and pixel count of the slice with the largest tumour area."""
    counts = masks.reshape(masks.shape[0], -1).sum(axis=1)
    return int(np.argmax(counts)), int(counts.max())


def segmentation_for_classification():
    """Segmentation model with the best *validation* Dice (validation split only)."""
    ranking = []
    for model_id in ("segnet_brats_4ch", "unet_dynamic_512_real_brats", "raunet_brats_4ch"):
        summary = read_json(PATHS["checkpoints"] / model_id / "training_summary.json", default=None)
        if not summary or summary.get("best_dice") is None:
            continue
        ranking.append((float(summary["best_dice"]), model_id))
    ranking.sort(reverse=True)
    return ranking[0][1] if ranking else None


SEGMENTATION_FOR_FEATURES = segmentation_for_classification()
print(
    "segmentation checkpoint used for the predicted-mask table (best validation Dice):",
    SEGMENTATION_FOR_FEATURES,
)

SEGMENTATION_BUILDERS = {
    "segnet_brats_4ch": lambda: SegNet(in_channels=4, out_channels=1),
    "unet_dynamic_512_real_brats": lambda: DynamicUNet(
        DEFAULT_FILTERS, input_channels=1, output_channels=1
    ),
    "raunet_brats_4ch": lambda: build_raunet(in_channels=4, out_channels=1, base_filters=16, depth=4),
}
_SEGMENTATION_CACHE: dict = {}


def predict_mask_probability(volume_key, index, model_id):
    """Probability map of `model_id` for one cached slice, at the cached mask resolution."""
    if model_id not in _SEGMENTATION_CACHE:
        model, _, path = load_trained_model(model_id, SEGMENTATION_BUILDERS[model_id])
        is_logit = str(getattr(model, "output_semantics", "logit")).lower().startswith("logit")
        _SEGMENTATION_CACHE[model_id] = (model, is_logit, path)
        print(f"loaded {model_id} from {path} for the predicted-mask features")
    model, is_logit, _ = _SEGMENTATION_CACHE[model_id]
    payload = load_patient(volume_key)
    if model_id == "unet_dynamic_512_real_brats":
        image = payload["singles"][index].astype(np.float32)[None, None, :, :]
    else:
        image = np.moveaxis(payload["images"][index].astype(np.float32), -1, 0)[None]
    with torch.no_grad():
        output = model(torch.from_numpy(image).to(DEVICE))
        if isinstance(output, (list, tuple)):
            output = output[0]
        tensor = output.float()
        probability = torch.sigmoid(tensor) if is_logit else tensor
        probability = probability.cpu().numpy()[0, 0]
    target = tuple(payload["masks"].shape[1:])
    if probability.shape != target:
        probability = np.asarray(
            PILImage.fromarray(probability.astype(np.float32), mode="F").resize(
                (target[1], target[0]), resample=PILImage.BILINEAR
            ),
            dtype=np.float32,
        )
    return probability, model_id


def patient_feature_row(volume_key, *, mask_source):
    """One feature row per patient: the focus slice on the serving channel (T1).

    The application extracts radiomics from a single analysed plane, so the training row is
    built from a single plane too. Averaging several slices would produce a schema the API can
    never feed.
    """
    payload = load_patient(volume_key)
    images = payload["images"].astype(np.float32)      # (S, 256, 256, 4)
    masks = payload["masks"].astype(np.float32)        # (S, 256, 256)
    if masks.ndim != 3 or masks.shape[0] == 0:
        return None, {"volume": volume_key, "mask_source": mask_source, "reason": "no cached slice"}
    focus, _ = focus_slice_index(masks)
    extra = {}
    if mask_source == "reference":
        mask = masks[focus] > 0.5
    else:
        if not SEGMENTATION_FOR_FEATURES:
            return None, {
                "volume": volume_key,
                "mask_source": mask_source,
                "reason": "no segmentation checkpoint is available for a predicted mask",
            }
        probability, model_id = predict_mask_probability(volume_key, focus, SEGMENTATION_FOR_FEATURES)
        mask = probability > 0.5
        extra = {
            "roi_mask_dice": dice_score(mask, masks[focus] > 0.5),
            "segmentation_model": model_id,
        }
    if not mask.any():
        return None, {
            "volume": volume_key,
            "mask_source": mask_source,
            "reason": "the region of interest is empty, so no region-based feature exists",
        }
    outcome = radiomics_service.extract(
        images[focus, :, :, SERVING_CHANNEL],
        mask,
        spacing=None,          # the Kaggle mirror carries no voxel spacing: units stay pixels
        mask_source=mask_source,
    )
    if not outcome.available:
        return None, {
            "volume": volume_key,
            "mask_source": mask_source,
            "reason": outcome.unavailable_reason,
        }
    row = {name: value for name, value in numeric_features(outcome)}
    row.update(
        {
            "case_id": f"{volume_key}_focus{focus}",
            "patient_id": volume_key,
            "label": payload["grade"],
            "split": next(
                (name for name in ("train", "validation", "test") if volume_key in SPLITS[name]),
                "unassigned",
            ),
            "mask_source": mask_source,
            "foreground_pixels": int(mask.sum()),
            "radiomics_backend": outcome.backend,
            "image_channel": SERVING_CHANNEL_NAME,
            "slice_index": focus,
        }
    )
    row.update(extra)
    return row, {
        "volume": volume_key,
        "mask_source": mask_source,
        "focus_slice": focus,
        "foreground_pixels": int(mask.sum()),
    }


FEATURE_TABLES: dict = {}
FEATURE_NAMES_BY_SOURCE: dict = {}
FEATURE_SKIPPED = {source: [] for source in MASK_SOURCES}
FEATURE_ROWS: list = []
FEATURE_NAMES: list = []

if CLASSIFICATION_POSSIBLE:
    if RUN["smoke_test"]:
        # Smoke run: the classifier needs rows in train, validation AND test, so extract for the
        # patients of those splits instead of the first N sorted volumes (which could all sit in
        # one split and make the comparison impossible).
        candidates = sorted(
            {key for key in (*TRAIN_PATIENTS, *VAL_PATIENTS, *TEST_PATIENTS) if key in LABELLED}
        )
    else:
        candidates = sorted(LABELLED)
        if RUN["radiomics_max_patients"]:
            candidates = candidates[: RUN["radiomics_max_patients"]]
    print(f"extracting features for {len(candidates)} labelled patient(s) x {len(MASK_SOURCES)} mask source(s)")
    for source in MASK_SOURCES:
        rows: list = []
        skipped: list = []
        started = time.time()
        for index, volume_key in enumerate(candidates, start=1):
            row, note = patient_feature_row(volume_key, mask_source=source)
            if row is None:
                skipped.append(note)
            else:
                rows.append(row)
            if index % 20 == 0 or index == len(candidates):
                elapsed = time.time() - started
                print(f"  {source:9s} {index}/{len(candidates)} patients ({elapsed:.0f}s, {elapsed / index:.3f}s per patient)")
        # A schema is only usable if every row carries the value: a name that is missing for
        # one patient would make the API refuse that case instead of scoring it.
        all_names = sorted({name for row in rows for name in row} - set(ROW_METADATA_COLUMNS) - {"roi_mask_dice", "segmentation_model"})
        schema = [
            name
            for name in all_names
            if all(isinstance(row.get(name), float) and np.isfinite(row[name]) for row in rows)
        ]
        dropped = [name for name in all_names if name not in schema]
        metadata = [column for column in ROW_METADATA_COLUMNS if column not in {"image_channel"}] + ["image_channel"]
        if source == "predicted":
            metadata = metadata + ["roi_mask_dice", "segmentation_model"]
        path = FEATURE_TABLE_PATHS[source]
        with path.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=[*metadata, *schema])
            writer.writeheader()
            for row in rows:
                writer.writerow({key: row.get(key, "") if row.get(key) is not None else "" for key in writer.fieldnames})
        FEATURE_TABLES[source] = rows
        FEATURE_NAMES_BY_SOURCE[source] = schema
        FEATURE_SKIPPED[source] = skipped
        write_json(
            PATHS["artifacts"] / f"radiomics_features_{source}_report.json",
            {
                "mask_source": source,
                "patients_requested": len(candidates),
                "rows": len(rows),
                "feature_count": len(schema),
                "features_dropped_for_missing_values": dropped,
                "non_numeric_extractor_outputs": sorted(NON_NUMERIC_FEATURE_NAMES),
                "skipped": skipped[:20],
                "table": str(path),
                "image_channel": SERVING_CHANNEL_NAME,
                "serving_note": (
                    "Schema matches analysis_service._run_radiomics(): one plane "
                    "(display_stack[0], T1), bare feature names, no channel prefix."
                ),
            },
        )

    FEATURE_ROWS = FEATURE_TABLES.get("reference", [])
    FEATURE_NAMES = FEATURE_NAMES_BY_SOURCE.get("reference", [])

    print("=" * 78)
    print("CLASSIFICATION FEATURES")
    print("=" * 78)
    for source in MASK_SOURCES:
        rows = FEATURE_TABLES.get(source, [])
        print(
            f"{source:9s} mask: {len(rows):4d} row(s) | {len(FEATURE_NAMES_BY_SOURCE.get(source, [])):3d} feature(s) "
            f"| skipped {len(FEATURE_SKIPPED[source])} | {FEATURE_TABLE_PATHS[source].name}"
        )
        by_split: dict = {}
        for row in rows:
            by_split.setdefault(row["split"], {}).setdefault(row["label"], 0)
            by_split[row["split"]][row["label"]] += 1
        print(f"            rows per split/class: {by_split}")
    print(f"serving channel: {SERVING_CHANNEL} ({SERVING_CHANNEL_NAME}) | non-numeric extractor outputs excluded: {sorted(NON_NUMERIC_FEATURE_NAMES)}")
else:
    print(
        "No grade table was found, so no labelled feature table could be built. Classification is "
        "skipped: the platform must not train on an invented label."
    )


## 23. Baseline and improved classification
`nbs_05_classification.py`


In [ ]:
# =============================================================================
# SECTION 15 - Baseline and improved classifiers on the same held-out split
# =============================================================================
# For each mask source (reference, predicted):
#   * same patient-level split as segmentation (train / validation / test),
#   * imputation, scaling and constant-feature removal fitted on the training split only,
#   * baseline = logistic regression, improved = gradient boosting (or MLP),
#   * selection by validation macro-F1; test metrics reported once, for every candidate.
# The exported artefact carries the ordered feature schema, the scaler, the label mapping and
# the metrics -- exactly what ClassificationService requires to serve a prediction.
# =============================================================================
LABELS: list = []
LABEL_TO_INDEX: dict = {}
INDEX_TO_LABEL: dict = {}
train_only_imputer = None
train_only_scaler = None
ACTIVE_FEATURES: list = []
CONSTANT_FEATURES: list = []
SELECTED_MODEL = None
train_rows: list = []
CLASSIFICATION_EVALUATION = None

if CLASSIFICATION_POSSIBLE and FEATURE_ROWS:
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        accuracy_score,
        confusion_matrix,
        f1_score,
        precision_recall_fscore_support,
        roc_auc_score,
    )
    from sklearn.neural_network import MLPClassifier
    from sklearn.preprocessing import StandardScaler

    LABELS = sorted({row["label"] for row in FEATURE_ROWS})
    LABEL_TO_INDEX = {label: index for index, label in enumerate(LABELS)}
    INDEX_TO_LABEL = {str(index): label for label, index in LABEL_TO_INDEX.items()}
    def evaluate_classifier(name, model, X_train, y_train, X_val, y_val, X_test, y_test):
        model.fit(X_train, y_train)
        report = {"model": name}
        for split_name, X, y in (("validation", X_val, y_val), ("test", X_test, y_test)):
            probabilities = model.predict_proba(X)
            predicted = probabilities.argmax(axis=1)
            precision, recall, f1, support = precision_recall_fscore_support(
                y, predicted, labels=list(range(len(LABELS))), zero_division=0
            )
            entry = {
                "accuracy": float(accuracy_score(y, predicted)),
                "macro_f1": float(f1_score(y, predicted, average="macro", zero_division=0)),
                "weighted_f1": float(f1_score(y, predicted, average="weighted", zero_division=0)),
                "per_class": {
                    INDEX_TO_LABEL[str(index)]: {
                        "precision": float(precision[index]),
                        "recall": float(recall[index]),
                        "f1": float(f1[index]),
                        "support": int(support[index]),
                    }
                    for index in range(len(LABELS))
                },
                "confusion_matrix": confusion_matrix(y, predicted, labels=list(range(len(LABELS)))).tolist(),
                "confusion_matrix_order": [INDEX_TO_LABEL[str(index)] for index in range(len(LABELS))],
                "samples": int(len(y)),
                "class_distribution": {INDEX_TO_LABEL[str(index)]: int((y == index).sum()) for index in range(len(LABELS))},
            }
            try:
                if len(LABELS) == 2:
                    entry["roc_auc"] = float(roc_auc_score(y, probabilities[:, 1]))
                    entry["roc_auc_average"] = f"binary (positive class = {LABELS[1]})"
                else:
                    entry["roc_auc"] = float(roc_auc_score(y, probabilities, multi_class="ovr", average="macro"))
                    entry["roc_auc_average"] = "macro one-vs-rest"
            except ValueError as exc:
                entry["roc_auc"] = None
                entry["roc_auc_error"] = str(exc)
            report[split_name] = entry
        return report

    def train_feature_source(source, rows, feature_names):
        """Fit both candidates on one feature table; return everything needed to export."""
        source_train = [row for row in rows if row["split"] == "train"]
        source_val = [row for row in rows if row["split"] == "validation"]
        source_test = [row for row in rows if row["split"] == "test"]
        missing = [
            name
            for name, subset in (("train", source_train), ("validation", source_val), ("test", source_test))
            if not subset
        ]
        if missing:
            raise SystemExit(
                f"these splits have no radiomics rows for the {source} mask: {missing}. Without a "
                "held-out patient-level test split no classification metric can be reported. Increase "
                "the number of patients, or check that the grade table covers enough volumes."
            )
        matrix = lambda subset: np.asarray(  # noqa: E731 - short local helper
            [[row.get(name, np.nan) for name in feature_names] for row in subset], dtype=np.float64
        )
        X_train_raw, X_val_raw, X_test_raw = matrix(source_train), matrix(source_val), matrix(source_test)
        y_train = np.asarray([LABEL_TO_INDEX[row["label"]] for row in source_train])
        y_val = np.asarray([LABEL_TO_INDEX[row["label"]] for row in source_val])
        y_test = np.asarray([LABEL_TO_INDEX[row["label"]] for row in source_test])

        imputer = SimpleImputer(strategy="median").fit(X_train_raw)
        scaler = StandardScaler().fit(imputer.transform(X_train_raw))
        X_train = scaler.transform(imputer.transform(X_train_raw))
        X_val = scaler.transform(imputer.transform(X_val_raw))
        X_test = scaler.transform(imputer.transform(X_test_raw))

        # ClassificationService applies median imputation and standardisation from the manifest and
        # then calls predict_proba on a vector of exactly len(feature_schema) values. Dropping
        # columns here would produce a model the API could never feed, so features that are constant
        # in the training split are *reported*, not removed. Standardisation turns them into zeros,
        # so they cannot influence the linear model either way.
        variances = np.nanstd(X_train, axis=0)
        if np.all(variances <= 1e-9):
            raise SystemExit(f"every {source}-mask feature is constant in the training split; check the extraction step")
        constant = [name for name, variance in zip(feature_names, variances) if variance <= 1e-9]
        active = list(feature_names)
        print(
            f"[{source}] features: {len(active)} | constant in the training split (kept: zero after "
            f"standardisation, no signal): {len(constant)}"
        )

        baseline_model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
        if RUN["classifier_improved"] == "mlp":
            improved_model = MLPClassifier(
                hidden_layer_sizes=(64, 32), max_iter=1500, random_state=SEED, early_stopping=True
            )
        else:
            improved_model = HistGradientBoostingClassifier(random_state=SEED, max_iter=400, learning_rate=0.06)

        reports = {
            "baseline_logistic_regression": evaluate_classifier(
                "baseline_logistic_regression", baseline_model, X_train, y_train, X_val, y_val, X_test, y_test
            ),
            f"improved_{RUN['classifier_improved']}": evaluate_classifier(
                f"improved_{RUN['classifier_improved']}", improved_model, X_train, y_train, X_val, y_val, X_test, y_test
            ),
        }
        estimators = {
            "baseline_logistic_regression": baseline_model,
            f"improved_{RUN['classifier_improved']}": improved_model,
        }
        return {
            "source": source,
            "rows": {"train": source_train, "validation": source_val, "test": source_test},
            "imputer": imputer,
            "scaler": scaler,
            "features": active,
            "constant_features": constant,
            "reports": reports,
            "estimators": estimators,
        }

    FITTED = {}
    for source in MASK_SOURCES:
        rows = FEATURE_TABLES.get(source) or []
        names = FEATURE_NAMES_BY_SOURCE.get(source) or []
        if not rows or not names:
            continue
        FITTED[source] = train_feature_source(source, rows, names)

    if not FITTED:
        raise SystemExit("no feature table had rows and a usable schema; classification cannot be trained")

    comparison = {}
    for source, fitted in FITTED.items():
        for model_name, report in fitted["reports"].items():
            comparison[f"{source}::{model_name}"] = {
                "feature_source": source,
                "model": model_name,
                "validation": report["validation"],
                "test": report["test"],
            }

    selection_metric = "macro_f1"
    selected_key = max(comparison, key=lambda key: comparison[key]["validation"][selection_metric])
    selected_source = comparison[selected_key]["feature_source"]
    selected_fitted = FITTED[selected_source]

    # Bind the exported artefacts to the selected candidate.
    train_only_imputer = selected_fitted["imputer"]
    train_only_scaler = selected_fitted["scaler"]
    ACTIVE_FEATURES = selected_fitted["features"]
    CONSTANT_FEATURES = selected_fitted["constant_features"]
    SELECTED_MODEL = selected_fitted["estimators"][comparison[selected_key]["model"]]
    train_rows = selected_fitted["rows"]["train"]
    SELECTED_FEATURE_SOURCE = selected_source
    SELECTED_ESTIMATOR_NAME = comparison[selected_key]["model"]

    CLASSIFICATION_EVALUATION = {
        "target": "BraTS 2020 HGG vs LGG (WHO grade group recorded in name_mapping.csv)",
        "label_mapping": INDEX_TO_LABEL,
        "selection_metric": selection_metric,
        "selection_split": "validation",
        "selected_model": selected_key,
        "selected_estimator_name": SELECTED_ESTIMATOR_NAME,
        "selected_feature_source": SELECTED_FEATURE_SOURCE,
        "comparison": comparison,
        "feature_sources": {
            source: {
                "feature_count": len(fitted["features"]),
                "schema": fitted["features"],
                "constant_features_in_training_split": fitted["constant_features"],
                "table": str(FEATURE_TABLE_PATHS[source]),
                "mask_source": source,
                "rows": {split: len(rows) for split, rows in fitted["rows"].items()},
            }
            for source, fitted in FITTED.items()
        },
        "features": ACTIVE_FEATURES,
        "constant_features_in_training_split": CONSTANT_FEATURES,
        "non_numeric_extractor_outputs": sorted(NON_NUMERIC_FEATURE_NAMES),
        "image_channel": SERVING_CHANNEL_NAME,
        "splits": {
            split: {
                "rows": len(selected_fitted["rows"][split]),
                "patients": len({row["patient_id"] for row in selected_fitted["rows"][split]}),
            }
            for split in ("train", "validation", "test")
        },
        "class_balance": {
            split: {
                label: sum(1 for row in selected_fitted["rows"][split] if row["label"] == label)
                for label in LABELS
            }
            for split in ("train", "validation", "test")
        },
        "caveats": [
            "The label is the dataset's own grade annotation (WHO grade group). It is not a malignancy verdict, a stage or a prognosis.",
            "Features are computed on slices resized to 256x256 with no voxel spacing, so they describe the stored image, not physical anatomy.",
            "The number of patients per class bounds how much these numbers can mean; one split with few patients has wide intervals.",
            "Selection uses the validation split only. The test split is scored once here, after selection.",
        ],
        "generated_at": utc_now(),
    }
    write_json(PATHS["artifacts"] / "classification_evaluation.json", CLASSIFICATION_EVALUATION)

    print("=" * 78)
    print("CLASSIFICATION (patient-level split, features from the app's own extractor)")
    print("=" * 78)
    for key, entry in sorted(comparison.items()):
        validation, test = entry["validation"], entry["test"]
        print(
            f"{key:52s} val macro-F1 {validation['macro_f1']:.4f} (acc {validation['accuracy']:.4f}) | "
            f"test macro-F1 {test['macro_f1']:.4f} (acc {test['accuracy']:.4f}, AUC {test['roc_auc']})"
        )
    print(f"selected by validation {selection_metric}:", selected_key)
    print("test confusion matrix (rows reference, columns predicted):")
    print(comparison[selected_key]["test"]["confusion_matrix"])
    print("classes:", LABELS, "| test rows:", len(selected_fitted["rows"]["test"]))
else:
    print("Classification skipped: no labelled feature table (see the section above for the reason).")


## 24. Image-based classifier (optional, second view on the same split)
`nbs_05_classification.py`


In [ ]:
# =============================================================================
# SECTION 15b - Image-based classifier on the same patient-level split
# =============================================================================
# A compact 2-D CNN classifier on slices, pooled per patient at inference. It exists so the
# radiomics numbers can be compared with an image-based model on identical splits; it is
# deliberately small, trained only when a GPU is present, and it is not exported as a servable
# artefact (the API's classification path consumes radiomics features).
# =============================================================================
class SliceClassifier(nn.Module):
    def __init__(self, channels=4, width=24, n_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(channels, width, 3, padding=1), nn.BatchNorm2d(width), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(width, width * 2, 3, padding=1), nn.BatchNorm2d(width * 2), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(width * 2, width * 4, 3, padding=1), nn.BatchNorm2d(width * 4), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(width * 4, n_classes)

    def forward(self, x):
        return self.head(self.features(x).flatten(1))


IMAGE_CLASSIFIER_SUMMARY = None
if CLASSIFICATION_POSSIBLE and FEATURE_ROWS and RUN["image_classifier_enabled"] and CUDA_AVAILABLE:
    from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score

    def classifier_batches(patient_ids, *, slices_per_patient, batch_size, shuffle, rng=None):
        items = []
        for volume_key in patient_ids:
            payload = load_patient(volume_key)
            masks = payload["masks"]
            counts = masks.reshape(masks.shape[0], -1).sum(axis=1)
            focus = int(np.argmax(counts))
            half = slices_per_patient // 2
            for index in range(max(0, focus - half), min(len(counts), focus + half + 1)):
                if counts[index] == 0:
                    continue
                items.append((volume_key, index))
        if shuffle:
            rng = rng or np.random.default_rng(SEED)
            rng.shuffle(items)
        for start in range(0, len(items), batch_size):
            chunk = items[start : start + batch_size]
            images, labels = [], []
            for volume_key, index in chunk:
                payload = load_patient(volume_key)
                images.append(np.moveaxis(payload["images"][index].astype(np.float32), -1, 0))
                labels.append(LABEL_TO_INDEX[payload["grade"]])
            yield np.stack(images), np.asarray(labels, dtype=np.int64), [item[0] for item in chunk]

    def image_autocast():
        try:
            return torch.amp.autocast("cuda", enabled=AMP_ENABLED)
        except (AttributeError, TypeError):
            return torch.cuda.amp.autocast(enabled=AMP_ENABLED)

    image_classifier = SliceClassifier(channels=4).to(DEVICE)
    image_optimizer = torch.optim.AdamW(image_classifier.parameters(), lr=3e-4)
    try:
        image_scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    except (AttributeError, TypeError):
        image_scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)
    image_loss = nn.CrossEntropyLoss()
    IMAGE_HISTORY = []
    image_started = time.time()
    for epoch in range(1, RUN["image_classifier_epochs"] + 1):
        image_classifier.train()
        losses = []
        for images, labels, _ in classifier_batches(
            TRAIN_PATIENTS, slices_per_patient=RUN["image_classifier_slices_per_patient"], batch_size=RUN["batch_size"], shuffle=True
        ):
            batch_images = torch.from_numpy(images).to(DEVICE).to(torch.float32)
            batch_labels = torch.from_numpy(labels).to(DEVICE)
            image_optimizer.zero_grad(set_to_none=True)
            with image_autocast():
                loss = image_loss(image_classifier(batch_images), batch_labels)
            image_scaler.scale(loss).backward()
            image_scaler.step(image_optimizer)
            image_scaler.update()
            losses.append(float(loss.detach()))
        IMAGE_HISTORY.append({"epoch": epoch, "train_loss": float(np.mean(losses)) if losses else None})
        print(f"image classifier epoch {epoch}/{RUN['image_classifier_epochs']}  loss {IMAGE_HISTORY[-1]['train_loss']:.4f}")

    def patient_predictions(patient_ids):
        image_classifier.eval()
        per_patient = {}
        with torch.no_grad():
            for images, _, volume_keys in classifier_batches(
                patient_ids, slices_per_patient=RUN["image_classifier_slices_per_patient"], batch_size=RUN["batch_size"], shuffle=False
            ):
                with image_autocast():
                    logits = image_classifier(torch.from_numpy(images).to(DEVICE).to(torch.float32))
                probabilities = torch.softmax(logits.float(), dim=1).cpu().numpy()
                for row, volume_key in enumerate(volume_keys):
                    per_patient.setdefault(volume_key, []).append(probabilities[row])
        return {volume_key: np.mean(np.stack(stack), axis=0) for volume_key, stack in per_patient.items()}

    image_metrics = {}
    for split_name, patient_ids in (("validation", VAL_PATIENTS), ("test", TEST_PATIENTS)):
        probabilities = patient_predictions(patient_ids)
        usable = [key for key in patient_ids if key in probabilities and VOLUMES[key]["grade"] in LABEL_TO_INDEX]
        if not usable:
            image_metrics[split_name] = {"state": "not_evaluated", "reason": "no patient had both a prediction and a label"}
            continue
        truth = np.asarray([LABEL_TO_INDEX[VOLUMES[key]["grade"]] for key in usable])
        scores = np.stack([probabilities[key] for key in usable])
        predicted = scores.argmax(axis=1)
        image_metrics[split_name] = {
            "accuracy": float(accuracy_score(truth, predicted)),
            "macro_f1": float(f1_score(truth, predicted, average="macro", zero_division=0)),
            "patients": len(usable),
            "roc_auc": float(roc_auc_score(truth, scores[:, 1]))
            if len(LABELS) == 2 and len(set(truth.tolist())) > 1
            else None,
            "confusion_matrix": confusion_matrix(truth, predicted, labels=list(range(len(LABELS)))).tolist(),
        }

    torch.save(
        {
            "state_dict": {key: value.detach().cpu() for key, value in image_classifier.state_dict().items()},
            "architecture": "SliceClassifier(channels=4, width=24)",
            "config": {"epochs": RUN["image_classifier_epochs"]},
            "label_mapping": INDEX_TO_LABEL,
            "history": IMAGE_HISTORY,
        },
        PATHS["checkpoints"] / "slice_classifier_image_based.pt",
    )
    IMAGE_CLASSIFIER_SUMMARY = {
        "architecture": "SliceClassifier(channels=4, width=24), slice predictions averaged per patient",
        "split": "same patient-level split as the radiomics classifiers",
        "history": IMAGE_HISTORY,
        "metrics": image_metrics,
        "elapsed_minutes": round((time.time() - image_started) / 60, 2),
        "exported": False,
        "note": (
            "Comparison only: the API's classification path consumes radiomics features, so this "
            "checkpoint is stored in the notebook output but not exported as a servable classifier."
        ),
    }
    write_json(PATHS["artifacts"] / "image_classifier_summary.json", IMAGE_CLASSIFIER_SUMMARY)
    print("=" * 78)
    print("IMAGE-BASED CLASSIFIER (same patient-level split)")
    print("=" * 78)
    for split_name, entry in image_metrics.items():
        print(f"{split_name:11s}: {entry}")
else:
    print(
        "Image-based classifier skipped "
        f"(labels={CLASSIFICATION_POSSIBLE}, rows={len(FEATURE_ROWS)}, "
        f"enabled={RUN['image_classifier_enabled']}, cuda={CUDA_AVAILABLE})."
    )
    print(
        "It needs a GPU to be worth running; the radiomics comparison in Section 15 still holds, "
        "and no image-based number is reported in its place."
    )


## 25. Radiomics quality: reference mask vs predicted mask
`nbs_05_classification.py`


In [ ]:
# =============================================================================
# SECTION 16 - Radiomics extraction detail and quality
# =============================================================================
# Three things matter for honesty here:
#   1. features computed on a predicted mask inherit the segmentation error,
#   2. features computed on resized 256x256 slices without voxel spacing are
#      pixel-domain numbers, not physical ones,
#   3. a single-slice region is a small sample: neighbouring slices disagree.
# This cell measures 1 and 3 on the data instead of asserting anything about them.
# =============================================================================
def radiomics_quality_report(reference_rows, predicted_rows, schema, *, comparison_patients=8):
    """Measure the mask-source drift and the slice-to-slice stability of the feature table.

    Written as a function over the two tables rather than as inline code so the test suite can
    exercise it on tables built by the notebook itself.
    """
    reference_by_patient = {row["patient_id"]: row for row in reference_rows}
    predicted_by_patient = {row["patient_id"]: row for row in predicted_rows}
    shared_patients = sorted(set(reference_by_patient) & set(predicted_by_patient))
    shared_schema = [name for name in schema if name in set(predicted_rows[0])] if predicted_rows else []
    per_feature = {}
    for name in shared_schema:
        relative = []
        for patient in shared_patients:
            reference_value = float(reference_by_patient[patient][name])
            predicted_value = float(predicted_by_patient[patient][name])
            relative.append(abs(predicted_value - reference_value) / (abs(reference_value) + 1e-9))
        per_feature[name] = float(np.median(relative))

    # Slice-to-slice variability of the reference-mask features around the focus slice.
    slice_variability = {}
    for patient in shared_patients[:comparison_patients]:
        payload = load_patient(patient)
        masks = payload["masks"].astype(np.float32)
        images = payload["images"].astype(np.float32)
        focus, _ = focus_slice_index(masks)
        focus_features = dict(
            numeric_features(
                radiomics_service.extract(
                    images[focus, :, :, SERVING_CHANNEL], masks[focus] > 0.5, spacing=None, mask_source="reference"
                )
            )
        )
        for offset in (-1, 1):
            neighbour = focus + offset
            if not 0 <= neighbour < masks.shape[0] or not (masks[neighbour] > 0.5).any():
                continue
            neighbour_features = dict(
                numeric_features(
                    radiomics_service.extract(
                        images[neighbour, :, :, SERVING_CHANNEL],
                        masks[neighbour] > 0.5,
                        spacing=None,
                        mask_source="reference",
                    )
                )
            )
            for name, value in neighbour_features.items():
                if name in focus_features:
                    slice_variability.setdefault(name, []).append(
                        abs(value - focus_features[name]) / (abs(focus_features[name]) + 1e-9)
                    )
    per_feature_variability = {name: float(np.median(values)) for name, values in slice_variability.items()}

    mask_dice_values = [
        float(row["roi_mask_dice"]) for row in predicted_rows if isinstance(row.get("roi_mask_dice"), float)
    ]
    return {
        "state": "measured",
        "patients_compared": len(shared_patients),
        "shared_feature_count": len(shared_schema),
        "reference_vs_predicted_mask": {
            "median_relative_difference_over_features": float(np.median(list(per_feature.values()))) if per_feature else None,
            "p90_relative_difference_over_features": float(np.percentile(list(per_feature.values()), 90)) if per_feature else None,
            "per_feature_median_relative_difference": per_feature,
            "roi_mask_dice": {
                "median": float(np.median(mask_dice_values)) if mask_dice_values else None,
                "min": float(np.min(mask_dice_values)) if mask_dice_values else None,
                "max": float(np.max(mask_dice_values)) if mask_dice_values else None,
                "patients": len(mask_dice_values),
                "model": SEGMENTATION_FOR_FEATURES,
                "note": "Dice of the predicted mask against the BraTS reference mask on the focus slice.",
            },
        },
        "slice_to_slice_variability": {
            "patients": min(len(shared_patients), comparison_patients),
            "features_compared": len(per_feature_variability),
            "median_relative_difference_over_features": float(np.median(list(per_feature_variability.values())))
            if per_feature_variability
            else None,
            "per_feature_median_relative_difference": per_feature_variability,
            "note": "Focus slice against its immediate neighbours with the reference mask, same extractor and settings.",
        },
        "spacing_available": False,
        "dimensionality": "2D (single slice) features on 256x256 resized images",
        "caveats": [
            "Physical-unit features (the *_mm entries) are None everywhere: the Kaggle slice mirror carries no voxel spacing, so those are excluded from the schema rather than approximated.",
            "A classifier trained on reference regions and served on predicted regions inherits the difference measured above; the measured number is the size of that inheritance, not a correction for it.",
            "Slice-to-slice variability is reported because the training row is a single slice, like a served analysis.",
        ],
        "generated_at": utc_now(),
    }


RADIOMICS_QUALITY: dict = {"state": "not_evaluated"}
if CLASSIFICATION_POSSIBLE and FEATURE_TABLES.get("reference") and FEATURE_TABLES.get("predicted"):
    RADIOMICS_QUALITY = radiomics_quality_report(
        FEATURE_TABLES["reference"],
        FEATURE_TABLES["predicted"],
        FEATURE_NAMES_BY_SOURCE["reference"],
    )
    write_json(PATHS["artifacts"] / "radiomics_quality.json", RADIOMICS_QUALITY)
    print("=" * 78)
    print("RADIOMICS QUALITY (measured, not assumed)")
    print("=" * 78)
    print("patients compared                        :", RADIOMICS_QUALITY["patients_compared"])
    print("mask Dice (predicted vs reference, focus):", RADIOMICS_QUALITY["reference_vs_predicted_mask"]["roi_mask_dice"]["median"])
    print("mask source drift (median over features) :", RADIOMICS_QUALITY["reference_vs_predicted_mask"]["median_relative_difference_over_features"])
    print("slice-to-slice drift (median)            :", RADIOMICS_QUALITY["slice_to_slice_variability"]["median_relative_difference_over_features"])
else:
    reasons = []
    if not CLASSIFICATION_POSSIBLE:
        reasons.append("no grade table, so no labelled row exists")
    if not FEATURE_TABLES.get("reference"):
        reasons.append("the reference-mask table is empty")
    if not FEATURE_TABLES.get("predicted"):
        reasons.append(
            "the predicted-mask table is empty ("
            + "; ".join(sorted({str(note.get("reason")) for note in FEATURE_SKIPPED.get("predicted", []) if note.get("reason")})[:3])
            + ")"
        )
    print("Radiomics quality comparison skipped: " + "; ".join(reasons) + ".")
    print(
        "Nothing is reported in its place: an unmeasured difference is not a measured one. Re-run "
        "after the segmentation models converge and the predicted regions stop being empty."
    )


## 26. Multiclass dataset: ROI, radiomics features and schema check
`nbs_05b_multiclass.py`


In [ ]:
# =============================================================================
# SECTION 12b - Radiomics for the second dataset (automatic brain-region ROI)
# =============================================================================
# The image-classification dataset has no lesion masks: each file is a single 2-D MRI slice
# with a class label. Radiomics therefore needs a region of interest that is *computed*, not
# annotated -- so one is derived per image (Otsu threshold, largest connected component,
# filling, one opening/closing pair) and every ROI is recorded together with its size. An ROI
# that ends up empty or degenerate disqualifies that image, and it is listed: no feature row is
# invented for it.
#
# The features come from the application's own extractor, in the same bare-named schema the
# serving path uses, so "extracted here" and "computed by the API" mean the same numbers.
# =============================================================================

MULTICLASS_FEATURE_TABLE = PATHS["artifacts"] / "classification_radiomics_features.csv"
MULTICLASS_FEATURES: dict = {"train": [], "validation": [], "test": []}
MULTICLASS_FEATURE_NAMES: list = []
MULTICLASS_ROI_REJECTS: list = []
MULTICLASS_EXTRACTED = False


def otsu_threshold(array):
    """Otsu's threshold on a uint8 image, in numpy only."""
    values = np.asarray(array, dtype=np.uint8).ravel()
    histogram = np.bincount(values, minlength=256).astype(np.float64)
    total = histogram.sum()
    if total <= 0:
        return 0
    probabilities = histogram / total
    omega = np.cumsum(probabilities)
    mu = np.cumsum(probabilities * np.arange(256))
    mu_total = mu[-1]
    denominator = omega * (1.0 - omega)
    denominator[denominator <= 0] = 1e-12
    between = (mu_total * omega - mu) ** 2 / denominator
    return int(np.argmax(between))


def largest_component(mask):
    """Largest 4-connected component of a boolean mask (scipy when available)."""
    try:
        from scipy import ndimage

        labels, count = ndimage.label(mask)
        if count <= 1:
            return mask
        sizes = ndimage.sum(mask, labels, index=range(1, count + 1))
        return labels == (int(np.argmax(sizes)) + 1)
    except Exception:  # pragma: no cover - scipy is a hard requirement in requirements.txt
        rows, columns = np.where(mask)
        if rows.size == 0:
            return mask
        return mask


def brain_roi(array, *, min_fraction=0.02):
    """Region of interest for a single grayscale slice, with its diagnostics.

    Returns (mask, diagnostics). The mask is the Otsu-foreground brain region, reduced to its
    largest connected component and morphologically cleaned. `min_fraction` guards against
    degenerate ROIs (a mostly-empty or almost-full mask).
    """
    array = np.asarray(array, dtype=np.uint8)
    try:
        from scipy import ndimage

        smooth = ndimage.gaussian_filter(array.astype(np.float32), sigma=1.2)
    except Exception:  # pragma: no cover
        smooth = array.astype(np.float32)
    threshold = otsu_threshold(smooth.astype(np.uint8))
    foreground = smooth > threshold
    # Keep the brain: drop the thin bright rim that some of these images carry.
    foreground = largest_component(foreground)
    try:
        from scipy import ndimage

        foreground = ndimage.binary_fill_holes(foreground)
        foreground = ndimage.binary_opening(foreground, iterations=2)
        foreground = ndimage.binary_closing(foreground, iterations=2)
        foreground = largest_component(foreground)
    except Exception:  # pragma: no cover
        pass
    fraction = float(foreground.mean())
    diagnostics = {"otsu_threshold": threshold, "roi_fraction": fraction, "roi_pixels": int(foreground.sum())}
    if fraction < min_fraction or fraction > 0.98:
        diagnostics["reason"] = f"degenerate ROI (foreground fraction {fraction:.4f})"
        diagnostics["accepted"] = False
        return None, diagnostics
    diagnostics["accepted"] = True
    return foreground.astype(bool), diagnostics


def multiclass_feature_row(path, class_name, split_name):
    """One feature row for one image, or (None, reason) when the extraction is impossible."""
    array = load_image_array(path, size=int(RUN["radiomics_image_size"]))
    mask, diagnostics = brain_roi(array)
    if mask is None:
        return None, {"image": str(path), "class": class_name, "reason": diagnostics.get("reason", "no ROI")}
    outcome = radiomics_service.extract(
        array.astype(np.float32),
        mask,
        spacing=None,  # no voxel spacing exists for these images: units stay pixels
        mask_source="auto_roi",
    )
    if not outcome.available:
        return None, {"image": str(path), "class": class_name, "reason": outcome.unavailable_reason}
    row = {name: value for name, value in numeric_features(outcome)}
    row.update(
        {
            "image_id": Path(path).stem,
            "image_relative_path": str(Path(path).name),
            "class_label": class_name,
            "split": split_name,
            "roi_pixels": diagnostics["roi_pixels"],
            "roi_fraction": diagnostics["roi_fraction"],
            "otsu_threshold": diagnostics["otsu_threshold"],
            "radiomics_backend": outcome.backend,
            "image_channel": "grayscale",
        }
    )
    return row, None


if not CLASSIFICATION_READY:
    print("Multiclass radiomics: skipped --", "the second dataset is not attached" if not CLASSIFICATION_IMAGES_AVAILABLE else "a split is empty")
else:
    limit = RUN["multiclass_radiomics_max_images"]
    rng_features = np.random.default_rng(SEED + 7)
    print(f"extracting radiomics on {RUN['radiomics_image_size']}px images with an automatic brain ROI")
    for split_name in ("train", "validation", "test"):
        for class_name in CLASSIFICATION_CLASSES:
            paths = [Path(path) for path in CLASSIFICATION_IMAGES[split_name].get(class_name, [])]
            if limit and len(paths) > limit:
                keep = rng_features.permutation(len(paths))[: int(limit)]
                paths = [paths[index] for index in sorted(keep.tolist())]
            for path in paths:
                row, rejection = multiclass_feature_row(path, class_name, split_name)
                if row is None:
                    MULTICLASS_ROI_REJECTS.append(rejection)
                    continue
                MULTICLASS_FEATURES[split_name].append(row)
    MULTICLASS_FEATURE_NAMES = list(MULTICLASS_FEATURES["train"][0]) if MULTICLASS_FEATURES["train"] else []
    MULTICLASS_FEATURE_NAMES = [
        name
        for name in MULTICLASS_FEATURE_NAMES
        if name
        not in {
            "image_id",
            "image_relative_path",
            "class_label",
            "split",
            "roi_pixels",
            "roi_fraction",
            "otsu_threshold",
            "radiomics_backend",
            "image_channel",
        }
    ]
    MULTICLASS_EXTRACTED = bool(MULTICLASS_FEATURES["train"] and MULTICLASS_FEATURES["validation"] and MULTICLASS_FEATURES["test"])

    rows_for_csv = [
        {**{name: row.get(name) for name in MULTICLASS_FEATURE_NAMES}, **{key: row[key] for key in (
            "image_id", "image_relative_path", "class_label", "split", "roi_pixels", "roi_fraction",
            "otsu_threshold", "radiomics_backend", "image_channel",
        )}}
        for split_name in ("train", "validation", "test")
        for row in MULTICLASS_FEATURES[split_name]
    ]
    if rows_for_csv:
        import csv as _csv

        with open(MULTICLASS_FEATURE_TABLE, "w", encoding="utf-8", newline="") as handle:
            writer = _csv.DictWriter(handle, fieldnames=list(rows_for_csv[0]))
            writer.writeheader()
            writer.writerows(rows_for_csv)

    roi_fraction_by_split = {
        split_name: (
            float(np.median([row["roi_fraction"] for row in rows])) if rows else None
        )
        for split_name, rows in MULTICLASS_FEATURES.items()
    }
    print("feature rows  :", {split: len(rows) for split, rows in MULTICLASS_FEATURES.items()})
    print("features      :", len(MULTICLASS_FEATURE_NAMES))
    print("median ROI fraction:", roi_fraction_by_split)
    print("rejected images:", len(MULTICLASS_ROI_REJECTS), "(first:", MULTICLASS_ROI_REJECTS[0] if MULTICLASS_ROI_REJECTS else "none", ")")

    # --- schema contract -------------------------------------------------------
    # The application's extractor defines the schema. Two things are checked here, both of them
    # about the mismatch that kept the shipped classifier disabled (21 expected features, 11 in
    # features_df.csv, no scaler, no feature order):
    #   1. the multiclass table and the BraTS table use the *same* feature names, because both
    #      come from the same extractor on the same kind of input, and
    #   2. every exported artefact records this exact ordered schema plus its scaler, so a
    #      served vector can never be silently padded or reordered.
    BRATS_SCHEMA = set(FEATURE_NAMES) if isinstance(FEATURE_NAMES, (list, tuple)) and FEATURE_NAMES else set()
    MULTICLASS_SCHEMA = set(MULTICLASS_FEATURE_NAMES)
    SCHEMA_CONTRACT = {
        "extractor": f"{radiomics_service.backend_status().get('selected_backend')} (app/ml/radiomics_native.py)",
        "multiclass_feature_count": len(MULTICLASS_FEATURE_NAMES),
        "brats_feature_count": len(BRATS_SCHEMA),
        "identical_to_brats_schema": bool(BRATS_SCHEMA) and BRATS_SCHEMA == MULTICLASS_SCHEMA,
        "missing_versus_brats": sorted(BRATS_SCHEMA - MULTICLASS_SCHEMA),
        "extra_versus_brats": sorted(MULTICLASS_SCHEMA - BRATS_SCHEMA),
        "legacy_checkpoint_note": (
            "radiomics_cnn_model.pth expects 21 unnamed features and shipped without a scaler or "
            "feature order, while the application's extractor produces the features listed above. "
            "It is never padded or reordered to fit: every artefact trained here carries its own "
            "ordered schema, scaler and label mapping, which is what makes it servable."
        ),
    }
    print("schema contract:", {key: SCHEMA_CONTRACT[key] for key in ("multiclass_feature_count", "identical_to_brats_schema")})
    write_json(PATHS["artifacts"] / "classification_schema_contract.json", SCHEMA_CONTRACT)

    # --- ROI figure ------------------------------------------------------------
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    preview_classes = CLASSIFICATION_CLASSES[:4]
    fig, axes = plt.subplots(2, len(preview_classes), figsize=(3.1 * max(1, len(preview_classes)), 6.4))
    axes = np.atleast_2d(axes)
    for index, class_name in enumerate(preview_classes):
        rows = [row for row in MULTICLASS_FEATURES["train"] if row["class_label"] == class_name]
        axes[0][index].set_title(class_name, fontsize=10)
        axes[0][index].axis("off")
        axes[1][index].axis("off")
        if not rows:
            continue
        source = next(
            (Path(path) for path in CLASSIFICATION_IMAGES["train"].get(class_name, []) if Path(path).stem == rows[0]["image_id"]),
            None,
        )
        if source is None:
            continue
        display = load_image_array(source, size=int(RUN["radiomics_image_size"]))
        mask, _ = brain_roi(display)
        axes[0][index].imshow(display, cmap="gray")
        axes[1][index].imshow(display, cmap="gray")
        if mask is not None:
            overlay = np.zeros(display.shape + (4,), dtype=np.float32)
            overlay[..., 0] = 1.0
            overlay[..., 3] = mask.astype(np.float32) * 0.45
            axes[1][index].imshow(overlay)
        axes[1][index].set_title(f"ROI {rows[0]['roi_fraction']:.2f} of the slice", fontsize=9)
    plt.tight_layout()
    roi_figure = PATHS["reports"] / "09_classification_roi.png"
    plt.savefig(roi_figure, dpi=130)
    plt.close(fig)
    print("figure saved  :", roi_figure)


## 27. Multiclass CNN training (from scratch, AMP, resumable)
`nbs_05b_multiclass.py`


In [ ]:
# =============================================================================
# SECTION 13b - Image classifier for the second dataset
# =============================================================================
# A small convolutional network trained from scratch -- no pretrained weights, because the
# notebook has no internet access and weight downloads that cannot happen are not part of a
# reproducible run. Everything a long run needs is here: mixed precision, checkpoint/resume,
# progress with elapsed and ETA, early stopping, LR scheduling and validation-based selection.
# =============================================================================
MULTICLASS_CNN_TRAINED = False
MULTICLASS_CNN_HISTORY: list = []
MULTICLASS_CNN_SUMMARY: dict = {}


def classifier_architecture(channels, num_classes, dropout):
    """Small VGG-style 2-D classifier: three blocks, global average pooling, linear head."""
    import torch.nn as nn

    blocks = []
    in_channels = 1
    for width in channels:
        blocks += [
            nn.Conv2d(in_channels, width, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(width),
            nn.ReLU(inplace=True),
            nn.Conv2d(width, width, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(width),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        ]
        in_channels = width
    return nn.Sequential(
        *blocks,
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Dropout(dropout),
        nn.Linear(in_channels, num_classes),
    )


class ImageClassificationTrainer:
    """Checkpointed training loop for the multiclass CNN (AMP, resume, ETA, early stopping)."""

    def __init__(self, model_id, model, *, num_classes, learning_rate, weight_decay, device=DEVICE, amp=AMP_ENABLED, dropout=0.3):
        import torch.nn as nn

        self.model_id = model_id
        self.model = model.to(device)
        self.device = device
        self.num_classes = int(num_classes)
        self.dropout = float(dropout)
        self.criterion = nn.CrossEntropyLoss()
        self.amp = bool(amp and device.type == "cuda")
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=50, eta_min=learning_rate / 20)
        try:
            self.scaler = torch.amp.GradScaler("cuda", enabled=self.amp)
            self.autocast = lambda: torch.amp.autocast("cuda", enabled=self.amp)
            self.amp_api = "torch.amp"
        except (AttributeError, TypeError):  # pragma: no cover - older torch
            self.scaler = torch.cuda.amp.GradScaler(enabled=self.amp)
            self.autocast = lambda: torch.cuda.amp.autocast(enabled=self.amp)
            self.amp_api = "torch.cuda.amp"
        self.directory = PATHS["checkpoints"] / model_id
        self.directory.mkdir(parents=True, exist_ok=True)
        self.history_path = self.directory / "history.json"
        self.state = {"epoch": 0, "step": 0, "best_macro_f1": -1.0, "best_epoch": None, "history": read_json(self.history_path, default=[]) or []}

    def save(self, *, metrics=None, config=None, epochs_planned=None, best=False):
        payload = {
            "state_dict": {key: value.detach().cpu() for key, value in self.model.state_dict().items()},
            "model_id": self.model_id,
            "task": "multiclass_classification",
            "classes": list(CLASSIFICATION_CLASSES),
            "architecture": "classifier_architecture(channels, num_classes, dropout)",
            "config": config or {},
            "epoch": self.state["epoch"],
            "step": self.state["step"],
            "best_macro_f1": self.state["best_macro_f1"],
            "history": self.state["history"],
            "optimizer": self.optimizer.state_dict(),
            "scheduler": self.scheduler.state_dict(),
            "scaler": self.scaler.state_dict() if self.amp else None,
            "numpy_rng": np.random.get_state(),
            "torch_rng": torch.get_rng_state(),
            "metrics": metrics or {},
            "epochs_planned": epochs_planned,
            "saved_at": utc_now(),
            "amp_api": self.amp_api,
        }
        torch.save(payload, self.directory / "last.pt")
        if best:
            torch.save(payload, self.directory / "best.pt")
        write_json(self.history_path, self.state["history"])
        return self.directory / ("best.pt" if best else "last.pt")

    def resume(self, *, allow=True):
        path = self.directory / "last.pt"
        if not allow or not path.exists():
            return {"resumed": False, "path": str(path) if path.exists() else None}
        payload = torch.load(path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(payload["state_dict"])
        try:
            self.optimizer.load_state_dict(payload["optimizer"])
            self.scheduler.load_state_dict(payload["scheduler"])
            if payload.get("scaler") and self.amp:
                self.scaler.load_state_dict(payload["scaler"])
        except (KeyError, ValueError) as exc:
            print(f"WARNING: optimiser state from {path.name} could not be restored ({exc}); continuing fresh")
        self.state["epoch"] = int(payload.get("epoch", 0))
        self.state["step"] = int(payload.get("step", 0))
        self.state["best_macro_f1"] = float(payload.get("best_macro_f1", -1.0))
        if isinstance(payload.get("history"), list):
            self.state["history"] = payload["history"]
        print(f"resumed {self.model_id} at epoch {self.state['epoch']} (best validation macro-F1 {self.state['best_macro_f1']:.4f})")
        return {"resumed": True, "epoch": self.state["epoch"], "best_macro_f1": self.state["best_macro_f1"]}

    # -- data ----------------------------------------------------------------
    @staticmethod
    def _tensor(array):
        """Per-image standardisation, then a channel dimension. Recorded in the manifest."""
        array = np.asarray(array, dtype=np.float32)
        array = (array - array.mean()) / max(float(array.std()), 1e-6)
        return torch.from_numpy(array[None])

    def batches(self, index, batch_size, rng, augment, max_steps=None):
        order = list(index)
        if augment:
            rng.shuffle(order)
        steps = 0
        for start in range(0, len(order), batch_size):
            chunk = order[start : start + batch_size]
            images, labels = [], []
            for path, class_index in chunk:
                images.append(self._tensor(load_image_array(path)))
                labels.append(class_index)
            batch = torch.stack(images)
            if augment:
                if rng.random() < 0.5:
                    batch = torch.flip(batch, dims=[3])
                angles = torch.from_numpy(rng.uniform(-10, 10, size=len(chunk)).astype(np.float32))
                scales = torch.from_numpy(rng.uniform(0.9, 1.1, size=len(chunk)).astype(np.float32))
                theta = torch.zeros(len(chunk), 2, 3)
                theta[:, 0, 0] = torch.cos(angles * np.pi / 180) / scales
                theta[:, 0, 1] = -torch.sin(angles * np.pi / 180) / scales
                theta[:, 1, 0] = torch.sin(angles * np.pi / 180) / scales
                theta[:, 1, 1] = torch.cos(angles * np.pi / 180) / scales
                grid = torch.nn.functional.affine_grid(theta, list(batch.shape), align_corners=False)
                batch = torch.nn.functional.grid_sample(batch, grid, align_corners=False, padding_mode="border")
            yield batch.to(self.device), torch.tensor(labels, dtype=torch.long, device=self.device)
            steps += 1
            if max_steps is not None and steps >= max_steps:
                return

    def predict_probabilities(self, paths, class_indices, batch_size):
        """Softmax probabilities for a list of images (used for the stacked ensemble)."""
        self.model.eval()
        outputs = []
        with torch.no_grad():
            for start in range(0, len(paths), batch_size):
                chunk = paths[start : start + batch_size]
                batch = torch.stack([self._tensor(load_image_array(path)) for path in chunk]).to(self.device)
                with self.autocast():
                    logits = self.model(batch)
                outputs.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
        return np.concatenate(outputs) if outputs else np.zeros((0, self.num_classes), dtype=np.float32)

    def evaluate(self, paths, class_indices, batch_size):
        from sklearn.metrics import accuracy_score, f1_score

        probabilities = self.predict_probabilities(paths, class_indices, batch_size)
        if not len(probabilities):
            return {"macro_f1": None, "accuracy": None, "samples": 0}
        predicted = probabilities.argmax(axis=1)
        return {
            "macro_f1": float(f1_score(class_indices, predicted, average="macro", zero_division=0)),
            "accuracy": float(accuracy_score(class_indices, predicted)),
            "samples": int(len(class_indices)),
        }

    # -- training -------------------------------------------------------------
    def train(self, *, train_index, val_index, epochs, batch_size, max_steps_per_epoch=None, early_stopping_patience=6, budget=None, config=None, validation_every=1):
        from tqdm.auto import tqdm

        rng = np.random.default_rng(SEED + 11)
        self.resume(allow=True)
        started = time.time()
        history = list(self.state["history"])
        best = self.state["best_macro_f1"]
        stale = 0
        epochs_planned = epochs
        last_path = None
        for epoch in range(self.state["epoch"] + 1, epochs + 1):
            self.model.train()
            if budget is not None and budget.expired:
                print("session budget reached; stopping cleanly (the next session resumes from last.pt)")
                break
            epoch_started = time.time()
            running, seen = 0.0, 0
            progress = tqdm(
                self.batches(train_index, batch_size, rng, augment=True, max_steps=max_steps_per_epoch),
                total=(min(max_steps_per_epoch, int(np.ceil(len(train_index) / batch_size))) if max_steps_per_epoch else int(np.ceil(len(train_index) / batch_size))),
                desc=f"{self.model_id} epoch {epoch}/{epochs}",
                leave=False,
            )
            for images, labels in progress:
                self.optimizer.zero_grad(set_to_none=True)
                with self.autocast():
                    loss = self.criterion(self.model(images), labels)
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.state["step"] += 1
                running += float(loss.detach().cpu()) * len(labels)
                seen += len(labels)
                progress.set_postfix(loss=f"{running / max(seen, 1):.4f}", lr=f"{self.scheduler.get_last_lr()[0]:.2e}")
                if RUN["checkpoint_every_steps"] and self.state["step"] % int(RUN["checkpoint_every_steps"]) == 0:
                    self.save(metrics={"running_loss": running / max(seen, 1)}, config=config, epochs_planned=epochs_planned)
            progress.close()
            self.scheduler.step()
            self.state["epoch"] = epoch
            epoch_seconds = time.time() - epoch_started
            metrics = {"epoch_seconds": epoch_seconds, "train_loss": running / max(seen, 1), "samples": seen}
            if epoch % max(1, validation_every) == 0 or epoch == epochs:
                validation = self.evaluate([path for path, _ in val_index], np.asarray([index for _, index in val_index]), batch_size)
                metrics.update({f"validation_{key}": value for key, value in validation.items()})
            record = {"epoch": epoch, "lr": self.scheduler.get_last_lr()[0], **metrics, "elapsed_minutes": (time.time() - started) / 60}
            history.append(record)
            self.state["history"] = history
            improved = metrics.get("validation_macro_f1") is not None and metrics["validation_macro_f1"] > best
            if improved:
                best = metrics["validation_macro_f1"]
                self.state["best_macro_f1"] = best
                self.state["best_epoch"] = epoch
                stale = 0
            else:
                stale += 1
            last_path = self.save(metrics=metrics, config=config, epochs_planned=epochs_planned, best=improved)
            eta = ""
            if metrics.get("validation_macro_f1") is not None:
                eta = f" | val macro-F1 {metrics['validation_macro_f1']:.4f}{' (best)' if improved else ''}"
            print(
                f"{self.model_id} epoch {epoch}/{epochs}: loss {metrics['train_loss']:.4f} | "
                f"{epoch_seconds:.1f}s/epoch | elapsed {(time.time() - started) / 60:.1f} min | "
                f"step {self.state['step']}{eta}"
            )
            if early_stopping_patience and stale >= early_stopping_patience:
                print(f"early stopping: no validation improvement for {stale} epoch(s)")
                break
            if budget is not None and budget.require(5.0) is None and budget.expired:
                break
        return {
            "model_id": self.model_id,
            "epochs_completed": self.state["epoch"],
            "epochs_planned": epochs_planned,
            "steps": self.state["step"],
            "best_macro_f1": self.state["best_macro_f1"] if self.state["best_macro_f1"] >= 0 else None,
            "best_epoch": self.state["best_epoch"],
            "history": history,
            "minutes": (time.time() - started) / 60,
            "checkpoint": str(self.directory / ("best.pt" if self.state["best_epoch"] else "last.pt")),
            "resumable_from": str(self.directory / "last.pt"),
            "stopped_early": bool(early_stopping_patience and stale >= early_stopping_patience),
        }


if not (CLASSIFICATION_READY and MULTICLASS_EXTRACTED):
    print("Multiclass CNN: skipped --", "dataset missing" if not CLASSIFICATION_READY else "no radiomics rows")
else:
    MULTICLASS_LABEL_TO_INDEX = {name: index for index, name in enumerate(CLASSIFICATION_CLASSES)}
    MULTICLASS_INDEX_TO_LABEL = {str(index): name for name, index in MULTICLASS_LABEL_TO_INDEX.items()}

    def _index_for(split_name):
        return [
            (Path(path), MULTICLASS_LABEL_TO_INDEX[class_name])
            for class_name in CLASSIFICATION_CLASSES
            for path in CLASSIFICATION_IMAGES[split_name].get(class_name, [])
        ]

    TRAIN_INDEX = _index_for("train")
    VALIDATION_INDEX = _index_for("validation")
    TEST_INDEX = _index_for("test")
    print("train/validation/test images:", len(TRAIN_INDEX), len(VALIDATION_INDEX), len(TEST_INDEX))

    MULTICLASS_CNN_CONFIG = {
        "model_id": "brain_tumor_mri_cnn",
        "task": "multiclass_classification",
        "classes": CLASSIFICATION_CLASSES,
        "dataset": CLASSIFICATION_DATASET_NAME,
        "input": f"grayscale {RUN['classification_image_size']}x{RUN['classification_image_size']}, per-image standardisation",
        "channels": list(RUN["cnn_channels"]),
        "dropout": RUN["cnn_dropout"],
        "optimiser": f"AdamW lr={RUN['cnn_learning_rate']} weight_decay={RUN['cnn_weight_decay']}",
        "scheduler": "cosine annealing",
        "batch_size": RUN["cnn_batch_size"],
        "epochs": RUN["cnn_epochs"],
        "augmentation": "horizontal flip, rotation +/-10 deg, scale 0.9-1.1",
        "selection": "best validation macro-F1",
        "seed": SEED,
        "amp": AMP_ENABLED,
        "pretrained_weights": None,
    }

    cnn_model = classifier_architecture(tuple(RUN["cnn_channels"]), len(CLASSIFICATION_CLASSES), float(RUN["cnn_dropout"]))
    MULTICLASS_CNN_PARAMETERS = sum(parameter.numel() for parameter in cnn_model.parameters())
    print(f"CNN parameters: {MULTICLASS_CNN_PARAMETERS:,}")

    if not RUN.get("skip_multiclass_cnn", False):
        cnn_trainer = ImageClassificationTrainer(
            "brain_tumor_mri_cnn",
            cnn_model,
            num_classes=len(CLASSIFICATION_CLASSES),
            learning_rate=RUN["cnn_learning_rate"],
            weight_decay=RUN["cnn_weight_decay"],
            dropout=RUN["cnn_dropout"],
        )
        MULTICLASS_CNN_SUMMARY = cnn_trainer.train(
            train_index=TRAIN_INDEX,
            val_index=VALIDATION_INDEX,
            epochs=RUN["cnn_epochs"],
            batch_size=RUN["cnn_batch_size"],
            max_steps_per_epoch=RUN["cnn_max_steps_per_epoch"],
            early_stopping_patience=RUN["cnn_early_stopping_patience"],
            validation_every=RUN["cnn_validation_every_epochs"],
            budget=TimeBudget(RUN["session_budget_minutes"]),
            config=MULTICLASS_CNN_CONFIG,
        )
        MULTICLASS_CNN_SUMMARY["parameters"] = MULTICLASS_CNN_PARAMETERS
        MULTICLASS_CNN_HISTORY = MULTICLASS_CNN_SUMMARY["history"]
        MULTICLASS_CNN_TRAINED = True
        write_json(PATHS["artifacts"] / "classification_cnn_training_summary.json", MULTICLASS_CNN_SUMMARY)
        with open(PATHS["reports"] / "10_classification_cnn_curves.json", "w", encoding="utf-8") as handle:
            json.dump(MULTICLASS_CNN_HISTORY, handle, indent=2)
    else:
        print("CNN training skipped by configuration")


## 28. Radiomics classifiers on the same images
`nbs_05b_multiclass.py`


In [ ]:
# =============================================================================
# SECTION 14b - Feature-based members for the second dataset
# =============================================================================
# Two feature-based classifiers on the radiomics table, fitted exactly like the segmentation-side
# classifiers: imputation and scaling from the training split only, validation macro-F1 for
# selection, test metrics reported once. Their probabilities are what the stacked ensemble
# consumes further down.
# =============================================================================
MULTICLASS_FEATURE_MODELS: dict = {}
MULTICLASS_PREPROCESSING: dict = {}
MULTICLASS_FEATURE_REPORTS: dict = {}

if not MULTICLASS_EXTRACTED:
    print("Radiomics classifiers: skipped -- no feature rows")
else:
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score
    from sklearn.preprocessing import StandardScaler

    def _matrix(rows):
        return np.asarray([[row.get(name, np.nan) for name in MULTICLASS_FEATURE_NAMES] for row in rows], dtype=np.float64)

    X_train_raw = _matrix(MULTICLASS_FEATURES["train"])
    X_val_raw = _matrix(MULTICLASS_FEATURES["validation"])
    X_test_raw = _matrix(MULTICLASS_FEATURES["test"])
    y_train = np.asarray([MULTICLASS_LABEL_TO_INDEX[row["class_label"]] for row in MULTICLASS_FEATURES["train"]])
    y_val = np.asarray([MULTICLASS_LABEL_TO_INDEX[row["class_label"]] for row in MULTICLASS_FEATURES["validation"]])
    y_test = np.asarray([MULTICLASS_LABEL_TO_INDEX[row["class_label"]] for row in MULTICLASS_FEATURES["test"]])

    multiclass_imputer = SimpleImputer(strategy="median").fit(X_train_raw)
    multiclass_scaler = StandardScaler().fit(multiclass_imputer.transform(X_train_raw))
    X_train = multiclass_scaler.transform(multiclass_imputer.transform(X_train_raw))
    X_val = multiclass_scaler.transform(multiclass_imputer.transform(X_val_raw))
    X_test = multiclass_scaler.transform(multiclass_imputer.transform(X_test_raw))
    MULTICLASS_PREPROCESSING = {"imputer": multiclass_imputer, "scaler": multiclass_scaler}

    constant_features = [name for name, spread in zip(MULTICLASS_FEATURE_NAMES, np.nanstd(X_train, axis=0)) if spread <= 1e-9]
    print(f"features: {len(MULTICLASS_FEATURE_NAMES)} | constant in the training split (kept, zeroed by scaling): {len(constant_features)}")

    multiclass_candidates = {
        "radiomics_logistic_regression": LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED),
        "radiomics_hist_gradient_boosting": HistGradientBoostingClassifier(random_state=SEED, max_iter=300, learning_rate=0.08),
    }
    for name, estimator in multiclass_candidates.items():
        estimator.fit(X_train, y_train)
        entry = {"model": name, "feature_source": "auto_roi"}
        for split_name, X, y in (("validation", X_val, y_val), ("test", X_test, y_test)):
            probabilities = estimator.predict_proba(X)
            predicted = probabilities.argmax(axis=1)
            entry[split_name] = {
                "accuracy": float(accuracy_score(y, predicted)),
                "macro_f1": float(f1_score(y, predicted, average="macro", zero_division=0)),
                "samples": int(len(y)),
            }
        MULTICLASS_FEATURE_MODELS[name] = estimator
        MULTICLASS_FEATURE_REPORTS[name] = entry
    for name, entry in MULTICLASS_FEATURE_REPORTS.items():
        print(f"{name:<34} validation macro-F1 {entry['validation']['macro_f1']:.4f} | test {entry['test']['macro_f1']:.4f}")


## 29. Stacked ensemble: compatibility guard, stacking layer, honest comparison
`nbs_05b_multiclass.py`


In [ ]:
# =============================================================================
# SECTION 17b - Stacked ensemble for the multiclass task
# =============================================================================
# Members are combined only when they are actually compatible: same dataset, same label space,
# same class order, same images, and outputs that are proper probability distributions. Members
# that fail any of those checks are excluded *with the reason printed* -- which is why the
# BraTS HGG/LGG classifier, a different dataset with a different label space, can never end up
# in this ensemble.
#
# The stacking layer (multinomial logistic regression) is fitted on the validation split's
# member probabilities and scored once on the held-out test split. The comparison against the
# best single member is reported as measured: no superiority is claimed beyond what the numbers
# on that split show.
# =============================================================================
MULTICLASS_STACK_RESULT: dict = {"state": "not_available", "reason": "the multiclass pipeline did not run"}


def stack_member(name, classes, dataset_id, probabilities, *, fitted_on):
    return {
        "name": name,
        "classes": tuple(classes),
        "dataset_id": dataset_id,
        "probabilities": probabilities,
        "fitted_on": fitted_on,
    }


def check_stackable(members):
    """Return (compatible_members, rejections). A member is rejected with a reason, never patched."""
    accepted, rejections = [], []
    if not members:
        return accepted, [{"name": None, "reason": "no members supplied"}]
    reference = members[0]
    for member in members:
        problems = []
        if member["classes"] != reference["classes"]:
            problems.append(f"label space differs: {member['classes']} vs {reference['classes']}")
        if member["dataset_id"] != reference["dataset_id"]:
            problems.append(f"dataset differs: {member['dataset_id']!r} vs {reference['dataset_id']!r}")
        probabilities = member["probabilities"]
        expected_classes = len(reference["classes"])
        for split_name, matrix in probabilities.items():
            if matrix.ndim != 2 or matrix.shape[1] != expected_classes:
                problems.append(f"{split_name}: shape {matrix.shape} does not match {expected_classes} classes")
                continue
            if not np.all(np.isfinite(matrix)):
                problems.append(f"{split_name}: contains NaN or infinite values")
            sums = matrix.sum(axis=1)
            if matrix.size and not np.allclose(sums, 1.0, atol=1e-3):
                problems.append(f"{split_name}: rows do not sum to 1 (max deviation {float(np.abs(sums - 1).max()):.4f})")
            if matrix.min() < -1e-6 or matrix.max() > 1 + 1e-6:
                problems.append(f"{split_name}: values outside [0, 1]")
        if problems:
            rejections.append({"name": member["name"], "reason": "; ".join(problems)})
        else:
            accepted.append(member)
    # every accepted member must describe the same images in the same order
    if len(accepted) > 1:
        sizes = {name: matrix.shape[0] for name, matrix in accepted[0]["probabilities"].items()}
        for member in accepted[1:]:
            for split_name, matrix in member["probabilities"].items():
                if matrix.shape[0] != sizes.get(split_name):
                    rejections.append(
                        {"name": member["name"], "reason": f"{split_name}: {matrix.shape[0]} rows, expected {sizes.get(split_name)}"}
                    )
                    accepted = [entry for entry in accepted if entry["name"] != member["name"]]
    return accepted, rejections


if not (MULTICLASS_CNN_TRAINED and MULTICLASS_FEATURE_MODELS):
    print("Stacked ensemble: skipped -- need the CNN and the feature-based members")
    MULTICLASS_STACK_RESULT = {"state": "skipped", "reason": "a required member was not trained"}
else:
    from sklearn.linear_model import LogisticRegression as _StackingLogisticRegression
    from sklearn.metrics import accuracy_score, f1_score

    MULTICLASS_DATASET_ID = "brain_tumor_mri_dataset::" + ",".join(CLASSIFICATION_CLASSES)

    def _paths_for(split_name):
        return [path for path, _ in _index_for(split_name)]

    def _labels_for(split_name):
        return np.asarray([index for _, index in _index_for(split_name)])

    cnn_probabilities = {
        split_name: cnn_trainer.predict_probabilities(_paths_for(split_name), _labels_for(split_name), RUN["cnn_batch_size"])
        for split_name in ("train", "validation", "test")
    }
    candidates = [
        stack_member("cnn", CLASSIFICATION_CLASSES, MULTICLASS_DATASET_ID, cnn_probabilities, fitted_on="train"),
        stack_member(
            "radiomics_logistic_regression",
            CLASSIFICATION_CLASSES,
            MULTICLASS_DATASET_ID,
            {split: MULTICLASS_FEATURE_MODELS["radiomics_logistic_regression"].predict_proba(X) for split, X in (("train", X_train), ("validation", X_val), ("test", X_test))},
            fitted_on="train",
        ),
        stack_member(
            "radiomics_hist_gradient_boosting",
            CLASSIFICATION_CLASSES,
            MULTICLASS_DATASET_ID,
            {split: MULTICLASS_FEATURE_MODELS["radiomics_hist_gradient_boosting"].predict_proba(X) for split, X in (("train", X_train), ("validation", X_val), ("test", X_test))},
            fitted_on="train",
        ),
    ]
    # Deliberately offered as a candidate so the guard can demonstrate the refusal: the BraTS
    # HGG/LGG classifier is a different dataset with a different label space.
    if CLASSIFICATION_EVALUATION and CLASSIFICATION_EVALUATION.get("comparison"):
        candidates.append(
            stack_member(
                "brats_hgg_lgg_classifier",
                ("HGG", "LGG"),
                "brats2020::HGG,LGG",
                {split: np.zeros((len(_paths_for("validation") if split == "validation" else _paths_for("test")), 2), dtype=np.float32) for split in ("validation", "test")},
                fitted_on="brats patient-level train split",
            )
        )
    accepted, rejections = check_stackable(candidates)
    print("members accepted :", [member["name"] for member in accepted])
    for rejection in rejections:
        print(f"member rejected  : {rejection['name']} -- {rejection['reason']}")

    if len(accepted) < 2:
        MULTICLASS_STACK_RESULT = {"state": "skipped", "reason": "fewer than two compatible members", "rejections": rejections}
        print("Not enough compatible members to stack: reporting the single member instead.")
    else:
        y_validation = _labels_for("validation")
        y_test = _labels_for("test")
        stack_features_validation = np.concatenate([member["probabilities"]["validation"] for member in accepted], axis=1)
        stack_features_test = np.concatenate([member["probabilities"]["test"] for member in accepted], axis=1)
        stacker = _StackingLogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED)
        stacker.fit(stack_features_validation, y_validation)
        stack_probabilities = {
            "validation": stacker.predict_proba(stack_features_validation),
            "test": stacker.predict_proba(stack_features_test),
        }
        average_probabilities = {
            "validation": np.mean([member["probabilities"]["validation"] for member in accepted], axis=0),
            "test": np.mean([member["probabilities"]["test"] for member in accepted], axis=0),
        }

        def _score(probabilities, labels):
            predicted = probabilities.argmax(axis=1)
            return {
                "accuracy": float(accuracy_score(labels, predicted)),
                "macro_f1": float(f1_score(labels, predicted, average="macro", zero_division=0)),
                "samples": int(len(labels)),
            }

        comparison = {
            member["name"]: {
                "validation": _score(member["probabilities"]["validation"], y_validation),
                "test": _score(member["probabilities"]["test"], y_test),
            }
            for member in accepted
        }
        comparison["simple_average"] = {
            "validation": _score(average_probabilities["validation"], y_validation),
            "test": _score(average_probabilities["test"], y_test),
        }
        comparison["stacked_logistic_regression"] = {
            "validation": _score(stack_probabilities["validation"], y_validation),
            "test": _score(stack_probabilities["test"], y_test),
        }
        best_single = max(
            (name for name in comparison if name not in {"simple_average", "stacked_logistic_regression"}),
            key=lambda name: comparison[name]["test"]["macro_f1"],
        )
        stacked_test = comparison["stacked_logistic_regression"]["test"]["macro_f1"]
        best_test = comparison[best_single]["test"]["macro_f1"]
        MULTICLASS_STACK_RESULT = {
            "state": "fitted_and_evaluated",
            "members": [member["name"] for member in accepted],
            "rejections": rejections,
            "stacking_layer": "multinomial logistic regression",
            "fitted_on": "validation split member probabilities (the test split is never used to fit it)",
            "selection_split": "validation",
            "test_split_used_for_tuning": False,
            "comparison": comparison,
            "best_single_member": best_single,
            "stacked_minus_best_single_macro_f1": float(stacked_test - best_test),
            "statement": (
                f"On the held-out test split the stacking layer scores macro-F1 {stacked_test:.4f} and the best "
                f"single member ({best_single}) scores {best_test:.4f}. "
                + (
                    "The stacking layer is better on this split."
                    if stacked_test > best_test
                    else "The stacking layer is not better on this split; the ensemble is reported as measured, "
                    "not as an improvement."
                )
            ),
            "stacker": stacker,
            "probabilities": {"validation": stack_probabilities["validation"], "test": stack_probabilities["test"]},
            "average_probabilities": average_probabilities,
            "labels": {"validation": y_validation, "test": y_test},
            "classes": CLASSIFICATION_CLASSES,
        }
        write_json(
            PATHS["artifacts"] / "classification_multiclass_stack.json",
            {key: value for key, value in MULTICLASS_STACK_RESULT.items() if key not in {"stacker", "probabilities", "average_probabilities", "labels"}},
        )
        print("\nheld-out comparison (macro-F1 on the test split):")
        for name, entry in sorted(comparison.items(), key=lambda item: -item[1]["test"]["macro_f1"]):
            print(f"  {name:<34} validation {entry['validation']['macro_f1']:.4f} | test {entry['test']['macro_f1']:.4f}")
        print(MULTICLASS_STACK_RESULT["statement"])


## 30. Multiclass held-out evaluation (metrics, confusion matrix, ROC-AUC)
`nbs_05b_multiclass.py`


In [ ]:
# =============================================================================
# SECTION 18b - Everything the report needs, on data the models never saw
# =============================================================================
MULTICLASS_EVALUATION: dict = {"state": "not_available", "reason": "the multiclass pipeline did not run"}

if MULTICLASS_STACK_RESULT.get("state") != "fitted_and_evaluated":
    print("Multiclass evaluation: skipped --", MULTICLASS_STACK_RESULT.get("reason", "no fitted ensemble"))
else:
    from sklearn.metrics import (
        accuracy_score,
        confusion_matrix,
        f1_score,
        precision_recall_fscore_support,
        roc_auc_score,
    )

    y_test = MULTICLASS_STACK_RESULT["labels"]["test"]
    model_probabilities = {
        **{member: None for member in MULTICLASS_STACK_RESULT["members"]},
        "simple_average": MULTICLASS_STACK_RESULT["average_probabilities"]["test"],
        "stacked_logistic_regression": MULTICLASS_STACK_RESULT["probabilities"]["test"],
    }
    for member in accepted:
        model_probabilities[member["name"]] = member["probabilities"]["test"]

    reports = {}
    for name, probabilities in model_probabilities.items():
        if probabilities is None:
            continue
        predicted = probabilities.argmax(axis=1)
        precision, recall, f1, support = precision_recall_fscore_support(
            y_test, predicted, labels=list(range(len(CLASSIFICATION_CLASSES))), zero_division=0
        )
        entry = {
            "accuracy": float(accuracy_score(y_test, predicted)),
            "macro_f1": float(f1_score(y_test, predicted, average="macro", zero_division=0)),
            "weighted_f1": float(f1_score(y_test, predicted, average="weighted", zero_division=0)),
            "macro_precision": float(precision.mean()),
            "macro_recall": float(recall.mean()),
            "per_class": {
                name_class: {
                    "precision": float(precision[index]),
                    "recall": float(recall[index]),
                    "f1": float(f1[index]),
                    "support": int(support[index]),
                }
                for index, name_class in enumerate(CLASSIFICATION_CLASSES)
            },
            "confusion_matrix": confusion_matrix(y_test, predicted, labels=list(range(len(CLASSIFICATION_CLASSES)))).tolist(),
            "confusion_matrix_order": list(CLASSIFICATION_CLASSES),
            "samples": int(len(y_test)),
            "class_distribution": {
                name_class: int((y_test == index).sum()) for index, name_class in enumerate(CLASSIFICATION_CLASSES)
            },
        }
        if len(CLASSIFICATION_CLASSES) == 2:
            entry["roc_auc"] = float(roc_auc_score(y_test, probabilities[:, 1]))
            entry["roc_auc_average"] = f"binary (positive class = {CLASSIFICATION_CLASSES[1]})"
        else:
            try:
                entry["roc_auc"] = float(roc_auc_score(y_test, probabilities, multi_class="ovr", average="macro"))
                entry["roc_auc_average"] = "macro one-vs-rest"
            except ValueError as exc:
                entry["roc_auc"] = None
                entry["roc_auc_error"] = str(exc)
        reports[name] = entry

    MULTICLASS_EVALUATION = {
        "state": "evaluated",
        "task": "multiclass_classification",
        "evaluated_at": utc_now(),
        "dataset": {
            "name": CLASSIFICATION_DATASET_NAME,
            "root": CLASSIFICATION_DATASET.get("root"),
            "classes": CLASSIFICATION_CLASSES,
            "images": {name: len(groups) and sum(len(paths) for paths in groups.values()) for name, groups in CLASSIFICATION_IMAGES.items()},
            "synthetic": CLASSIFICATION_IS_SYNTHETIC,
            "provenance_note": CLASSIFICATION_PROVENANCE_NOTE,
        },
        "protocol": {
            "split_source": CLASSIFICATION_SPLIT_SOURCE,
            "split_method": "stratified by class; no patient identifiers exist in this dataset",
            "patient_level_split_possible": False,
            "selection_split": "validation",
            "test_split_used_for_tuning": False,
            "near_duplicate_audit": CLASSIFICATION_AUDIT,
            "preprocessing": "grayscale, resize to RUN['classification_image_size'], per-image standardisation (CNN) or train-fit median imputation + standardisation (radiomics)",
        },
        "models": reports,
        "stacked_ensemble": {
            "members": MULTICLASS_STACK_RESULT["members"],
            "rejected_members": MULTICLASS_STACK_RESULT["rejections"],
            "best_single_member": MULTICLASS_STACK_RESULT["best_single_member"],
            "statement": MULTICLASS_STACK_RESULT["statement"],
        },
        "caveats": [
            "The classes are the dataset's own annotations (glioma / meningioma / pituitary / no tumour). "
            "They are not a malignancy verdict, a stage or a prognosis.",
            "The dataset carries no patient identifiers, so a patient-level split is impossible; "
            "near-duplicates of training images are detected and removed from the evaluation splits instead.",
            "Radiomics here is measured in an automatically derived brain-region ROI without voxel spacing, "
            "so the units are pixel-domain and the ROI is an estimate, not an annotation.",
        ],
    }
    write_json(PATHS["artifacts"] / "classification_multiclass_evaluation.json", MULTICLASS_EVALUATION)

    # --- figures ---------------------------------------------------------------
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    stacked = reports["stacked_logistic_regression"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    matrix = np.asarray(stacked["confusion_matrix"])
    image = axes[0].imshow(matrix, cmap="Blues")
    axes[0].set_xticks(range(len(CLASSIFICATION_CLASSES)))
    axes[0].set_xticklabels(CLASSIFICATION_CLASSES, rotation=20, ha="right")
    axes[0].set_yticks(range(len(CLASSIFICATION_CLASSES)))
    axes[0].set_yticklabels(CLASSIFICATION_CLASSES)
    axes[0].set_xlabel("predicted")
    axes[0].set_ylabel("actual")
    axes[0].set_title(f"Stacked ensemble, held-out test (n={stacked['samples']})")
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            axes[0].text(column, row, str(matrix[row, column]), ha="center", va="center", fontsize=9)
    fig.colorbar(image, ax=axes[0], shrink=0.8)
    names = list(reports)
    axes[1].bar(names, [reports[name]["macro_f1"] for name in names], color="tab:green")
    axes[1].set_ylabel("macro-F1 (test split)")
    axes[1].tick_params(axis="x", rotation=20)
    axes[1].set_title("Members and ensemble on the same held-out images")
    plt.tight_layout()
    evaluation_figure = PATHS["reports"] / "11_classification_evaluation.png"
    plt.savefig(evaluation_figure, dpi=130)
    plt.close(fig)

    print("held-out metrics (test split):")
    for name, entry in sorted(reports.items(), key=lambda item: -item[1]["macro_f1"]):
        print(
            f"  {name:<34} accuracy {entry['accuracy']:.4f} | macro-F1 {entry['macro_f1']:.4f} | "
            f"ROC-AUC {entry['roc_auc'] if entry.get('roc_auc') is None else round(entry['roc_auc'], 4)} | n={entry['samples']}"
        )
    print("confusion matrix (stacked):", stacked["confusion_matrix"], "order", stacked["confusion_matrix_order"])
    print("figure saved:", evaluation_figure)


## 31. Ensemble construction and validation-only calibration
`nbs_06_eval_export.py`


In [ ]:
# =============================================================================
# SECTION 17 - Ensemble (probability fusion, weights from the validation split only)
# =============================================================================
# Only models that share an input contract can be fused. SegNet and RAUNet both
# take 4 x 256 x 256 z-scored slices; the single-channel U-Net takes a different
# input and is therefore never averaged in. Weights are searched on the
# validation split; the test split is not touched here.
# =============================================================================
ENSEMBLE_PATH = PATHS["artifacts"] / "ensemble_calibration.json"
ENSEMBLE_REPORT = {"members": [], "state": "not_available"}


def model_probability_maps(model_id, builder, patient_ids, *, size, batch_size, single_channel=False, max_cases=None):
    model, payload, path = load_trained_model(model_id, builder)
    maps = {}
    seen = 0
    with torch.no_grad():
        for images, masks in iterate_batches(patient_ids, batch_size=batch_size, size=size, single_channel=single_channel, shuffle=False):
            tensor = torch.from_numpy(images).to(DEVICE)
            autocast = torch.amp.autocast("cuda", enabled=AMP_ENABLED) if hasattr(torch, "amp") else torch.cuda.amp.autocast(enabled=AMP_ENABLED)
            with autocast:
                logits = model(tensor)
                if isinstance(logits, (list, tuple)):
                    logits = logits[0]
            probabilities = torch.sigmoid(logits.float()).cpu().numpy()
            for index in range(probabilities.shape[0]):
                key = f"case-{seen:05d}"
                maps[key] = (probabilities[index, 0], masks[index, 0])
                seen += 1
                if max_cases and seen >= max_cases:
                    return maps, path
    return maps, path


available_members = []
for model_id, builder in (
    ("segnet_brats_4ch", lambda: SegNet(in_channels=4, out_channels=1)),
    ("raunet_brats_4ch", lambda: build_raunet(in_channels=4, out_channels=1, base_filters=16, depth=4)),
):
    if (PATHS["checkpoints"] / model_id / "best.pt").exists() or (PATHS["checkpoints"] / model_id / "last.pt").exists():
        available_members.append((model_id, builder))

if len(available_members) >= 2:
    validation_case_limit = max(16, RUN["validation_cases"])
    member_maps = {}
    for model_id, builder in available_members:
        maps, path = model_probability_maps(
            model_id, builder, VAL_PATIENTS, size=RUN["image_size"], batch_size=RUN["batch_size"], max_cases=validation_case_limit
        )
        member_maps[model_id] = maps
        print(f"{model_id}: {len(maps)} validation slices from {path}")

    shared_keys = sorted(set.intersection(*(set(maps.keys()) for maps in member_maps.values())))
    stacked = {model_id: np.stack([member_maps[model_id][key][0] for key in shared_keys]) for model_id, _ in available_members}
    references = [member_maps[available_members[0][0]][key][1] for key in shared_keys]

    def dice_for_weights(weights):
        fused = np.zeros_like(next(iter(stacked.values())))
        total = sum(weights.values()) or 1.0
        for model_id, weight in weights.items():
            fused += (weight / total) * stacked[model_id]
        return float(np.mean([dice_score(fused[index] > 0.5, references[index] > 0.5) for index in range(len(shared_keys))]))

    grid = np.arange(0.0, 1.0001, RUN["ensemble_weight_grid"])
    best = {"dice": -1.0, "weights": None}
    for weight in grid:
        candidate = {available_members[0][0]: float(weight), available_members[1][0]: float(1 - weight)}
        value = dice_for_weights(candidate)
        if value > best["dice"]:
            best = {"dice": value, "weights": candidate}
    equal_weights = {model_id: 0.5 for model_id, _ in available_members}
    member_dice = {model_id: float(np.mean([dice_score(stacked[model_id][index] > 0.5, references[index] > 0.5) for index in range(len(shared_keys))])) for model_id, _ in available_members}
    calibrated_dice = best["dice"]
    equal_dice = dice_for_weights(equal_weights)

    ENSEMBLE_REPORT = {
        "state": "calibrated",
        "split": "validation",
        "test_split_used": False,
        "metric": "dice",
        "members": [model_id for model_id, _ in available_members],
        "member_validation_dice": member_dice,
        "equal_weight_dice": equal_dice,
        "calibrated_weights": best["weights"],
        "calibrated_dice": calibrated_dice,
        "validation_slices": len(shared_keys),
        "weights": best["weights"],
        "generated_at": utc_now(),
        "note": (
            "Weights were chosen on the validation split only. Whether the ensemble actually beats "
            "the best single member is decided in Section 19 on the reserved test split; this cell "
            "does not claim it."
        ),
    }
    write_json(ENSEMBLE_PATH, ENSEMBLE_REPORT)
    print("=" * 78)
    print("ENSEMBLE (validation split)")
    print("=" * 78)
    for model_id, value in member_dice.items():
        print(f"member {model_id:18s}: Dice {value:.4f}")
    print(f"equal weights              : Dice {equal_dice:.4f}")
    print(f"calibrated {best['weights']}: Dice {calibrated_dice:.4f}")
    print("calibration written to", ENSEMBLE_PATH)
    if best["weights"] and min(best["weights"].values()) < 0.05:
        print(
            "\nThe search put almost all weight on one member. That is a legitimate outcome: it means "
            "the other member adds nothing on validation data, and the test evaluation will report it."
        )
else:
    print(
        "Only one 4-channel model is available on this run, so no fusion can be built. "
        "An ensemble needs at least two models with the same input contract."
    )


## 32. Validation-based checkpoint selection
`nbs_06_eval_export.py`


In [ ]:
# =============================================================================
# SECTION 18 - Checkpoint selection (validation split only)
# =============================================================================
SELECTION = {"state": "not_available", "candidates": {}, "selected_model": None}
candidates = {}
for model_id in ("segnet_brats_4ch", "unet_dynamic_512_real_brats", "raunet_brats_4ch"):
    summary_path = PATHS["checkpoints"] / model_id / "training_summary.json"
    summary = read_json(summary_path, default=None)
    if summary:
        candidates[model_id] = {
            "best_validation_dice": summary.get("best_dice"),
            "best_epoch": summary.get("best_epoch"),
            "epochs_completed": summary.get("epochs_completed"),
            "stop_reason": summary.get("stop_reason"),
        }
if candidates:
    comparable = {key: value for key, value in candidates.items() if value["best_validation_dice"] is not None and value["best_validation_dice"] >= 0}
    selected = max(comparable, key=lambda key: comparable[key]["best_validation_dice"]) if comparable else None
    SELECTION = {
        "state": "selected" if selected else "not_available",
        "metric": "best validation Dice during training",
        "split": "validation",
        "test_split_used": False,
        "candidates": candidates,
        "selected_model": selected,
        "rationale": (
            "Selection reads only the validation Dice recorded during training. The reserved test "
            "split is scored once in Section 19 for every model, including the ones that were not selected."
        ),
        "generated_at": utc_now(),
    }
write_json(PATHS["artifacts"] / "model_selection.json", SELECTION)
print("=" * 78)
print("MODEL SELECTION (validation only)")
print("=" * 78)
for model_id, entry in SELECTION["candidates"].items():
    print(f"{model_id:32s} best val Dice {entry['best_validation_dice']} @ epoch {entry['best_epoch']} ({entry['epochs_completed']} epochs)")
print("selected:", SELECTION["selected_model"])


## 33. Held-out test evaluation
`nbs_06_eval_export.py`


In [ ]:
# =============================================================================
# SECTION 19 - Reserved test split: Dice, IoU, precision, recall, HD95, ASD
# =============================================================================
# This is the only cell that reads the test split. Every model is evaluated on
# exactly the same patients and slices with the same 0.5 threshold, and the
# project's own metric code is used (embedded from backend/app/utils/metrics.py
# and backend/app/services/evaluation_service.py) so the numbers match what the
# Flask application would report for the same predictions.
# =============================================================================
from app.services.evaluation_service import EvaluationService

TEST_EVALUATION_PATH = PATHS["artifacts"] / "evaluation_results.json"
THRESHOLD = 0.5
# Every model is scored on ONE grid: the 4-channel reference grid (RUN["image_size"]). The
# single-channel U-Net works at 512x512, and Dice/HD95 computed on different grids would not be
# comparable, so its probability map is resized to the scoring grid before thresholding. The
# reference masks come straight from the cache, so all models see identical arrays.
SCORING_SIZE = int(RUN["image_size"])


def test_reference_masks():
    """Reference masks of the test split, in the exact order `iterate_batches` yields them."""
    references = []
    for volume_key in TEST_PATIENTS:
        payload = load_patient(volume_key)
        for position in range(payload["masks"].shape[0]):
            references.append(payload["masks"][position] > 0.5)
    return references


def resize_to_scoring_grid(probability):
    if probability.shape == (SCORING_SIZE, SCORING_SIZE):
        return np.asarray(probability, dtype=np.float32)
    image = PILImage.fromarray(np.asarray(probability, dtype=np.float32), mode="F")
    return np.asarray(image.resize((SCORING_SIZE, SCORING_SIZE), resample=PILImage.BILINEAR), dtype=np.float32)


def test_predictions(model_id, builder, *, single_channel=False, size=None, batch_size=None, max_cases=None):
    size = size or (RUN["single_image_size"] if single_channel else RUN["image_size"])
    batch_size = batch_size or (RUN["batch_size_single"] if single_channel else RUN["batch_size"])
    model, payload, path = load_trained_model(model_id, builder)
    probabilities_map = []
    case_ids = []
    started = time.perf_counter()
    with torch.no_grad():
        for images, _ in iterate_batches(TEST_PATIENTS, batch_size=batch_size, size=size, single_channel=single_channel, shuffle=False):
            autocast = torch.amp.autocast("cuda", enabled=AMP_ENABLED) if hasattr(torch, "amp") else torch.cuda.amp.autocast(enabled=AMP_ENABLED)
            with autocast:
                outputs = model(torch.from_numpy(images).to(DEVICE))
            is_logit = str(getattr(model, "output_semantics", "logit")).lower().startswith("logit")
            tensor = outputs[0] if isinstance(outputs, (list, tuple)) else outputs
            tensor = tensor.float()
            probability = (torch.sigmoid(tensor) if is_logit else tensor).cpu().numpy()
            for index in range(probability.shape[0]):
                probabilities_map.append(resize_to_scoring_grid(probability[index, 0]))
                case_ids.append(f"{model_id}-case-{len(case_ids):05d}")
            if max_cases and len(probabilities_map) >= max_cases:
                break
    elapsed = time.perf_counter() - started
    return {
        "maps": probabilities_map,
        "case_ids": case_ids,
        "checkpoint": str(path),
        "seconds_per_slice": round(elapsed / max(1, len(probabilities_map)), 4),
        "payload_epoch": payload.get("epoch"),
        "best_dice": payload.get("best_dice"),
    }


if TEST_PATIENTS:
    SCORING_REFERENCES = test_reference_masks()
    prediction_sets = {}
    runtime_entries = []
    for model_id, builder, single in (
        ("segnet_brats_4ch", lambda: SegNet(in_channels=4, out_channels=1), False),
        ("raunet_brats_4ch", lambda: build_raunet(in_channels=4, out_channels=1, base_filters=16, depth=4), False),
        ("unet_dynamic_512_real_brats", lambda: DynamicUNet(DEFAULT_FILTERS, input_channels=1, output_channels=1), True),
    ):
        directory = PATHS["checkpoints"] / model_id
        if not ((directory / "best.pt").exists() or (directory / "last.pt").exists()):
            print(f"{model_id}: no checkpoint, skipped")
            continue
        try:
            payload = test_predictions(model_id, builder, single_channel=single)
        except Exception as exc:
            print(f"{model_id}: evaluation failed ({type(exc).__name__}: {exc})")
            continue
        prediction_sets[model_id] = payload["maps"]
        runtime_entries.append(
            {
                "model_id": model_id,
                "state": "measured",
                "device": str(DEVICE),
                "input_shape": [1, 1 if single else 4, RUN["single_image_size"] if single else RUN["image_size"], RUN["single_image_size"] if single else RUN["image_size"]],
                "seconds_per_image": payload["seconds_per_slice"],
                "images_per_second": round(1 / payload["seconds_per_slice"], 3) if payload["seconds_per_slice"] else None,
                "checkpoint": payload["checkpoint"],
                "best_validation_dice": payload["best_dice"],
                "note": "Measured on the reserved test split on this machine.",
            }
        )
        print(f"{model_id}: {len(payload['maps'])} test slices, {payload['seconds_per_slice']:.4f} s per slice")

    if prediction_sets:
        # Every model iterates the same test patients, in the same order, without shuffling, so
        # each prediction maps onto the same reference slice -- which is checked, not assumed.
        lengths = {name: len(maps) for name, maps in prediction_sets.items()}
        if len(set(lengths.values())) > 1:
            raise SystemExit(f"models produced different numbers of test slices: {lengths}")
        reference_count = len(SCORING_REFERENCES)
        for name, maps in prediction_sets.items():
            if len(maps) != reference_count:
                raise SystemExit(
                    f"{name} produced {len(maps)} slice predictions but the test split has {reference_count} slices"
                )
            wrong_shape = [
                index for index, item in enumerate(maps) if item.shape != SCORING_REFERENCES[index].shape
            ]
            if wrong_shape:
                raise SystemExit(f"{name}: {len(wrong_shape)} prediction(s) are not on the scoring grid")
        references = SCORING_REFERENCES

        # Fuse the two 4-channel models on the same slices when both are present and calibrated.
        if "segnet_brats_4ch" in prediction_sets and "raunet_brats_4ch" in prediction_sets and ENSEMBLE_REPORT.get("state") == "calibrated":
            weights = ENSEMBLE_REPORT["calibrated_weights"]
            length = min(len(prediction_sets["segnet_brats_4ch"]), len(prediction_sets["raunet_brats_4ch"]))
            total = sum(weights.values()) or 1.0
            fused = [
                (weights["segnet_brats_4ch"] / total) * prediction_sets["segnet_brats_4ch"][index]
                + (weights["raunet_brats_4ch"] / total) * prediction_sets["raunet_brats_4ch"][index]
                for index in range(length)
            ]
            prediction_sets["ensemble_segnet_raunet"] = fused
            runtime_entries.append(
                {
                    "model_id": "ensemble_segnet_raunet",
                    "state": "measured",
                    "device": str(DEVICE),
                    "weights": weights,
                    "weights_source": "validation split calibration",
                    "note": "Fusion of two 4-channel models; weights estimated on validation data only.",
                }
            )

        report = EvaluationService.evaluate_segmentation_predictions(
            prediction_sets, references, threshold=THRESHOLD, case_ids=None
        )
        dataset_summary = {
            "name": DATASET_NAME,
            "root": str(DATA_ROOT),
            "volumes": len(VOLUMES),
            "slices": len(SLICE_FILES),
            "synthetic": DATASET_IS_SYNTHETIC,
            "provenance_note": DATASET_PROVENANCE_NOTE,
            "cases": sum(len(VOLUMES[key]["slices"]) for key in TEST_PATIENTS),
            "patients": len(TEST_PATIENTS),
            "split": "test",
            "split_method": SPLIT_RECORD.get("method", "patient_level_stratified_by_grade"),
            "split_seed": SEED,
            "labels_used": False,
            "note": "Patient-level split; no slice of a test patient appears in training or validation.",
        }
        results = {
            "state": "evaluated",
            "evaluated_at": utc_now(),
            "split": "test",
            "dataset": dataset_summary,
            "segmentation": report,
            "runtime": {"state": "measured", "generated_at": utc_now(), "models": runtime_entries},
            "baselines": BASELINES,
            "protocol": {
                "seed": SEED,
                "threshold": THRESHOLD,
                "bootstrap_samples": 1000,
                "selection_split": "validation",
                "test_split_used_for_tuning": False,
                "source": "notebooks/brain_tumor_training_kaggle_t4.ipynb",
                "amp": AMP_ENABLED,
                "device": str(DEVICE),
                "scoring_grid": [SCORING_SIZE, SCORING_SIZE],
                "scoring_note": (
                    "All models are scored on the 256x256 reference grid; the single-channel "
                    "U-Net's 512x512 probability map is resized (bilinear) before thresholding."
                ),
            },
        }
        if CLASSIFICATION_EVALUATION:
            results["classification"] = {
                "state": "evaluated",
                "model_id": "radiomics_grade_classifier",
                "target": CLASSIFICATION_EVALUATION["target"],
                "label_mapping": CLASSIFICATION_EVALUATION["label_mapping"],
                "comparison": CLASSIFICATION_EVALUATION["comparison"],
                "selected_model": CLASSIFICATION_EVALUATION["selected_model"],
                "splits": CLASSIFICATION_EVALUATION["splits"],
                "feature_count": len(CLASSIFICATION_EVALUATION["features"]),
                "produced_by": "notebooks/brain_tumor_training_kaggle_t4.ipynb",
                "generated_at": utc_now(),
            }
        write_json(TEST_EVALUATION_PATH, results)
        print("=" * 78)
        print(f"HELD-OUT TEST RESULTS ({DATASET_NAME}, patient-level split)")
        print("=" * 78)
        for model_id, model_report in report["models"].items():
            aggregate = model_report.get("aggregate", {})

            def mean_of(key, _aggregate=aggregate):
                """Every aggregate entry is a bootstrap_ci dict (or None when unmeasurable)."""
                entry = _aggregate.get(key) or {}
                value = entry.get("mean")
                return "not available" if value is None else f"{value:.4f}"

            dice = aggregate.get("dice") or {}
            print(
                f"{model_id:32s} Dice {mean_of('dice')} [{dice.get('ci_lower')} .. {dice.get('ci_upper')}]  "
                f"IoU {mean_of('iou')}  precision {mean_of('precision')}  recall {mean_of('recall')}  "
                f"HD95 {mean_of('hausdorff_95')}  cases {model_report.get('n_cases')}"
            )
        print("\nall-foreground reference Dice:", round(BASELINES["all_foreground_dice"], 4))
        print("written to", TEST_EVALUATION_PATH)
    else:
        print("No model produced test predictions; nothing to evaluate.")
else:
    print("The test split is empty (fewer than 3 patients per stratum), so no test evaluation was run.")


## 34. Explainability and visualisation
`nbs_06_eval_export.py`


In [ ]:
# =============================================================================
# SECTION 20 - Explainable AI on real data
# =============================================================================
# Grad-CAM computed with the project's own implementation (embedded from
# backend/app/ml/gradcam.py) on a test patient, for whichever model is selected.
# =============================================================================
from app.ml.gradcam import compute_gradcam

XAI_REPORT = {"state": "not_available"}
if TEST_PATIENTS and (PATHS["checkpoints"] / "segnet_brats_4ch" / "best.pt").exists():
    xai_patient = TEST_PATIENTS[0]
    xai_payload = load_patient(xai_patient)
    counts = xai_payload["masks"].reshape(xai_payload["masks"].shape[0], -1).sum(axis=1)
    xai_index = int(np.argmax(counts))
    xai_image = np.moveaxis(xai_payload["images"][xai_index].astype(np.float32), -1, 0)[None]
    xai_model, xai_meta, xai_path = load_trained_model("segnet_brats_4ch", lambda: SegNet(in_channels=4, out_channels=1))
    xai_tensor = torch.from_numpy(xai_image).to(DEVICE)
    with torch.no_grad():
        xai_logits = xai_model(xai_tensor)
        if isinstance(xai_logits, (list, tuple)):
            xai_logits = xai_logits[0]
        probability = torch.sigmoid(xai_logits.float()).cpu().numpy()[0, 0]
    result = compute_gradcam(
        xai_model,
        xai_tensor,
        model_id="segnet_brats_4ch",
        target_layer="encoder.4",
        region_of_interest=(probability > 0.5).astype(np.float32),
        output_is_logit=True,
    )
    heat = np.asarray(result.heatmap, dtype=np.float32)
    reference = xai_payload["masks"][xai_index].astype(np.float32)
    fig, axes = plt.subplots(1, 4, figsize=(17, 4.4))
    axes[0].imshow(xai_image[0, 3], cmap="gray"); axes[0].set_title(f"{xai_patient} FLAIR slice {xai_index}")
    axes[1].imshow(reference, cmap="magma"); axes[1].set_title("reference mask (whole tumour)")
    axes[2].imshow(probability, cmap="viridis"); axes[2].set_title("predicted probability")
    axes[3].imshow(xai_image[0, 3], cmap="gray"); axes[3].imshow(heat, cmap="jet", alpha=0.45)
    axes[3].set_title(f"Grad-CAM ({result.target_layer})")
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.savefig(PATHS["reports"] / "05_gradcam.png", dpi=130)
    plt.close(fig)
    XAI_REPORT = {
        "state": "generated",
        "model_id": "segnet_brats_4ch",
        "checkpoint": str(xai_path),
        "patient": xai_patient,
        "slice_index": xai_index,
        "target_layer": result.target_layer,
        "target_score": result.target_score,
        "warnings": result.warnings,
        "dice_on_this_slice": dice_score(probability > 0.5, reference > 0.5),
        "note": "A heat map shows which pixels the network is sensitive to. It is not a lesion outline and not a diagnosis.",
    }
    write_json(PATHS["artifacts"] / "explainability.json", XAI_REPORT)
    print("Grad-CAM written:", XAI_REPORT)
else:
    print("Explainability skipped: it needs a trained SegNet checkpoint and a non-empty test split.")


## 35. Comparison charts and tables
`nbs_06_eval_export.py`


In [ ]:
# =============================================================================
# SECTION 21 - Comparison charts and tables
# =============================================================================
import pandas as pd

COMPARISON_TABLE_PATH = PATHS["reports"] / "comparison_table.csv"
COMPARISON_ROWS = []
results = read_json(TEST_EVALUATION_PATH, default=None)
if results and results.get("state") == "evaluated":
    for model_id, model_report in results["segmentation"].get("models", {}).items():
        aggregate = model_report.get("aggregate", {})
        dice = aggregate.get("dice") or {}
        COMPARISON_ROWS.append(
            {
                "model": model_id,
                "split": "test",
                "dice": dice.get("mean"),
                "dice_ci_low": dice.get("ci_lower"),
                "dice_ci_high": dice.get("ci_upper"),
                "iou": (aggregate.get("iou") or {}).get("mean"),
                "precision": (aggregate.get("precision") or {}).get("mean"),
                "recall": (aggregate.get("recall") or {}).get("mean"),
                "hausdorff_95": (aggregate.get("hausdorff_95") or {}).get("mean"),
                "cases": model_report.get("n_cases"),
            }
        )
if COMPARISON_ROWS:
    table = pd.DataFrame(COMPARISON_ROWS).sort_values("dice", ascending=False)
    table.to_csv(COMPARISON_TABLE_PATH, index=False)
    print(table.to_string(index=False))
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
    table.plot.bar(x="model", y="dice", ax=axes[0], color="#0ea5e9", legend=False)
    axes[0].axhline(BASELINES["all_foreground_dice"], color="black", linestyle="--", linewidth=1, label="all-foreground reference")
    axes[0].set_title("Dice"); axes[0].set_ylim(0, 1); axes[0].legend(fontsize=8)
    table.plot.bar(x="model", y="iou", ax=axes[1], color="#8b5cf6", legend=False)
    axes[1].set_title("IoU"); axes[1].set_ylim(0, 1)
    table.plot.bar(x="model", y=["precision", "recall"], ax=axes[2])
    axes[2].set_title("Precision / recall"); axes[2].set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(PATHS["reports"] / "06_model_comparison.png", dpi=130)
    plt.close(fig)
    print("charts ->", PATHS["reports"] / "06_model_comparison.png")
else:
    print("No test results yet, so no comparison table was produced.")

if CLASSIFICATION_EVALUATION:
    class_rows = []
    for model_name, payload in CLASSIFICATION_EVALUATION["comparison"].items():
        for split_name in ("validation", "test"):
            entry = payload.get(split_name)
            if not entry:
                continue
            class_rows.append(
                {
                    "model": model_name,
                    "split": split_name,
                    "accuracy": entry["accuracy"],
                    "macro_f1": entry["macro_f1"],
                    "roc_auc": entry.get("roc_auc"),
                    "samples": entry["samples"],
                }
            )
    class_table = pd.DataFrame(class_rows)
    class_table.to_csv(PATHS["reports"] / "classification_table.csv", index=False)
    print(class_table.to_string(index=False))
    if CLASSIFICATION_EVALUATION["comparison"]:
        selected_test = CLASSIFICATION_EVALUATION["comparison"][CLASSIFICATION_EVALUATION["selected_model"]]["test"]
        matrix = pd.DataFrame(
            selected_test["confusion_matrix"],
            index=[f"ref {label}" for label in selected_test["confusion_matrix_order"]],
            columns=[f"pred {label}" for label in selected_test["confusion_matrix_order"]],
        )
        print(matrix.to_string())
        matrix.to_csv(PATHS["reports"] / "classification_confusion_matrix.csv")


## 36. Saving artefacts (checkpoints, metrics, histories, configuration)
`nbs_06_eval_export.py`


In [ ]:
# =============================================================================
# SECTION 22 - Artefacts
# =============================================================================
# Everything that describes a result is written next to the result, so the
# numbers can be traced to a checkpoint, a configuration and a split.
# =============================================================================
ARTEFACT_INDEX = {"generated_at": utc_now(), "notebook": "notebooks/brain_tumor_training_kaggle_t4.ipynb", "files": {}}


def record(path, description, sha=True):
    path = Path(path)
    if not path.exists():
        return None
    entry = {"path": str(path), "bytes": path.stat().st_size, "description": description}
    if sha and path.is_file() and path.stat().st_size < 250 * 1024 * 1024:
        import hashlib

        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        entry["sha256"] = digest.hexdigest()
    ARTEFACT_INDEX["files"][str(path)] = entry
    return entry


for model_id in ("segnet_brats_4ch", "unet_dynamic_512_real_brats", "raunet_brats_4ch", "slice_classifier_image_based"):
    directory = PATHS["checkpoints"] / model_id
    if not directory.exists():
        continue
    for name, description in (
        ("best.pt", f"{model_id}: checkpoint with the best validation Dice"),
        ("last.pt", f"{model_id}: resumable checkpoint (optimiser, scheduler, scaler, RNG)"),
        ("history.json", f"{model_id}: per-epoch training history"),
        ("training_summary.json", f"{model_id}: epochs, best Dice, stop reason, AMP and device"),
    ):
        record(directory / name, description)

for path, description in (
    (PATHS["artifacts"] / "run_config.json", "run configuration and paths"),
    (PATHS["artifacts"] / "dataset_structure.json", "structure inspection of the dataset"),
    (PATHS["artifacts"] / "label_verification.json", "patient identifiers and HGG/LGG verification"),
    (PATHS["artifacts"] / "eda.json", "exploratory data analysis"),
    (PATHS["artifacts"] / "cache_report.json", "preprocessing cache report"),
    (SPLIT_PATH, "frozen patient-level splits"),
    (PATHS["artifacts"] / "baselines.json", "all-foreground reference and prevalence"),
    (PATHS["artifacts"] / "classification_evaluation.json", "baseline vs improved classifier, both splits"),
    (PATHS["artifacts"] / "ensemble_calibration.json", "validation-only fusion weights (application format)"),
    (PATHS["artifacts"] / "model_selection.json", "validation-based model selection (application format)"),
    (PATHS["artifacts"] / "evaluation_results.json", "held-out test metrics (application format)"),
    (FEATURE_TABLE_PATH, "radiomics feature table used for classification"),
    (PATHS["reports"] / "comparison_table.csv", "segmentation comparison table"),
):
    record(path, description)

write_json(PATHS["artifacts"] / "artefact_index.json", ARTEFACT_INDEX)
print(f"artefact index: {PATHS['artifacts'] / 'artefact_index.json'} ({len(ARTEFACT_INDEX['files'])} files)")
for path in sorted(ARTEFACT_INDEX["files"])[:12]:
    print("  ", path)


## 37. Export for the Flask application (with a compatibility check)
`nbs_06_eval_export.py`


In [ ]:
# =============================================================================
# SECTION 23 - Export and compatibility validation
# =============================================================================
# The export directory mirrors the paths the application expects. Filenames are
# deliberately new (…_real_brats) so an existing project artefact is never
# overwritten; the install script in the repository copies them into place and
# writes the provenance manifest the API reads.
#
# The compatibility check loads every exported checkpoint with the SAME
# architecture builder the application uses, with strict=True, and runs a dummy
# forward pass at the declared input shape. A checkpoint that fails here would
# fail in the application, so nothing is exported unless it passes.
# =============================================================================
EXPORT = PATHS["export"]
EXPORT.mkdir(parents=True, exist_ok=True)
DATASET_SUMMARY = {
    "name": DATASET_NAME,
    "root": str(DATA_ROOT),
    "volumes": len(VOLUMES),
    "slices": len(SLICE_FILES),
    "patients": {"train": len(TRAIN_PATIENTS), "validation": len(VAL_PATIENTS), "test": len(TEST_PATIENTS)},
    "split_method": "patient_level_stratified_by_grade",
    "split_seed": SEED,
    "synthetic": DATASET_IS_SYNTHETIC,
    "provenance_note": DATASET_PROVENANCE_NOTE,
}
(EXPORT / "checkpoints").mkdir(exist_ok=True)
(EXPORT / "data" / "models").mkdir(parents=True, exist_ok=True)
(EXPORT / "reports").mkdir(exist_ok=True)

COMPATIBILITY = []


def export_checkpoint(model_id, target_name, builder, *, channels, size, semantics, prefix=""):
    directory = PATHS["checkpoints"] / model_id
    source = directory / "best.pt"
    if not source.exists():
        source = directory / "last.pt"
    if not source.exists():
        COMPATIBILITY.append({"model_id": model_id, "state": "not_trained", "reason": "no checkpoint"})
        return None
    payload = torch.load(source, map_location="cpu", weights_only=False)
    model = builder()
    try:
        model.load_state_dict(payload["state_dict"], strict=True)
        model.eval()
        dummy = torch.zeros(1, channels, size, size)
        with torch.no_grad():
            output = model(dummy)
            if isinstance(output, (list, tuple)):
                output = output[0]
        shape = list(output.shape)
        ok = shape == [1, 1, size, size]
    except Exception as exc:  # noqa: BLE001 - anything here is a compatibility failure
        COMPATIBILITY.append({"model_id": model_id, "state": "failed", "error": f"{type(exc).__name__}: {exc}"})
        return None
    exported = EXPORT / "checkpoints" / target_name
    torch.save(payload, exported)
    COMPATIBILITY.append(
        {
            "model_id": model_id,
            "state": "compatible" if ok else "unexpected_output",
            "exported_as": target_name,
            "input_shape": [1, channels, size, size],
            "output_shape": shape,
            "output_semantics": semantics,
            "strict_load": True,
            "best_validation_dice": payload.get("best_dice"),
            "epoch": payload.get("epoch"),
            "parameters": sum(parameter.numel() for parameter in model.parameters()),
        }
    )
    return exported


export_checkpoint("segnet_brats_4ch", "segnet_brats_4ch_real_brats.pt", lambda: SegNet(in_channels=4, out_channels=1), channels=4, size=RUN["image_size"], semantics="logit (sigmoid applied by the API)")
export_checkpoint("raunet_brats_4ch", "raunet_brats_4ch.pt", lambda: build_raunet(in_channels=4, out_channels=1, base_filters=16, depth=4), channels=4, size=RUN["image_size"], semantics="logit")
export_checkpoint("unet_dynamic_512_real_brats", "unet_dynamic_512_real_brats.pt", lambda: DynamicUNet(DEFAULT_FILTERS, input_channels=1, output_channels=1), channels=1, size=RUN["single_image_size"], semantics="probability (sigmoid inside)")

# --- classification artefact in the exact format ClassificationService reads ---
CLASSIFICATION_ARTEFACT_DIR = None
if CLASSIFICATION_EVALUATION:
    import joblib

    stamp = utc_now().replace(":", "").replace("-", "")[:15]
    model_id = "radiomics_grade_classifier"
    version = f"{model_id}-{stamp}"
    CLASSIFICATION_ARTEFACT_DIR = EXPORT / "data" / "models" / "classification" / version
    CLASSIFICATION_ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)
    # Only the bare estimator is pickled. A wrapper class defined inside a notebook cell would be
    # pickled by reference to the notebook namespace and could not be unpickled by the Flask
    # process -- the artefact would look fine here and fail at serving time. The estimator is
    # trained on the fully scaled schema, which is what ClassificationService feeds it:
    #   vector (len(feature_schema)) -> (mean, scale) from the manifest -> predict_proba
    joblib.dump(SELECTED_MODEL, CLASSIFICATION_ARTEFACT_DIR / f"{model_id}.joblib")

    def _scaler_payload():
        imputer = train_only_imputer
        scaler = train_only_scaler
        return {
            "type": "standard_scaler",
            "imputation": "training-split median",
            "fit_on": "training split only",
            "feature_schema": ACTIVE_FEATURES,
            "imputer_median": [float(value) for value in imputer.statistics_],
            "scaler_mean": [float(value) for value in scaler.mean_],
            "scaler_scale": [float(value) for value in scaler.scale_],
            "feature_selection": "none - every schema feature is used, in feature_schema order",
            "note": (
                "The API applies mean/scale from the manifest to the incoming vector and calls "
                "predict_proba. Median imputation is recorded for completeness; it never fires at "
                "serving time because a feature the API cannot compute is refused instead of imputed."
            ),
        }

    coefficients = {}
    if hasattr(SELECTED_MODEL, "coef_"):
        for index, row in enumerate(np.atleast_2d(SELECTED_MODEL.coef_)):
            coefficients[INDEX_TO_LABEL[str(index)]] = {
                name: float(value) for name, value in zip(ACTIVE_FEATURES, row.tolist())
            }

    manifest = {
        "model_id": model_id,
        "version": version,
        "kind": "sklearn",
        "model_file": f"{model_id}.joblib",
        "task": "classification",
        "target_description": CLASSIFICATION_EVALUATION["target"],
        "label_mapping": INDEX_TO_LABEL,
        "feature_schema": ACTIVE_FEATURES,
        "feature_sources": {
            "radiomics": ACTIVE_FEATURES,
            "table": str(FEATURE_TABLE_PATHS[SELECTED_FEATURE_SOURCE]),
            "extractor": "backend/app/ml/radiomics_native.py via RadiomicsService (identical schema at serving time)",
            "mask_source": f"{SELECTED_FEATURE_SOURCE} mask",
            "image_channel": SERVING_CHANNEL_NAME,
            "serving_contract": (
                "analysis_service._run_radiomics() extracts from display_stack[0] (T1) with the "
                "selected mask, so the schema below is the one the API computes at serving time."
            ),
        },
        "preprocessing": {
            "type": "standard_scaler",
            "fit_on": "training split only",
            "imputation": "training-split median",
            "median": [float(value) for value in train_only_imputer.statistics_],
            "mean": [float(value) for value in train_only_scaler.mean_],
            "scale": [float(value) for value in train_only_scaler.scale_],
            "feature_selection": "none - every schema feature is used, in feature_schema order",
            "imputation_at_serving": (
                "not applied: a feature the API cannot compute is refused (missing_features) instead "
                "of being filled in, so no prediction is produced from a partial vector."
            ),
        },
        "coefficients": coefficients,
        # Positional list of quantile samples per schema column, the layout
        # ClassificationService._feature_contributions() indexes: quantiles[index] -> schema[index].
        "feature_quantiles": [
            [
                float(value)
                for value in np.quantile(
                    train_column[~np.isnan(train_column)], np.linspace(0.0, 1.0, 16)
                )
            ]
            if np.isfinite(train_column).any()
            else []
            for train_column in (
                np.asarray([row.get(name, np.nan) for row in train_rows], dtype=np.float64)
                for name in ACTIVE_FEATURES
            )
        ],
        "metrics": {
            "accuracy": CLASSIFICATION_EVALUATION["comparison"][CLASSIFICATION_EVALUATION["selected_model"]]["test"]["accuracy"],
            "macro_f1": CLASSIFICATION_EVALUATION["comparison"][CLASSIFICATION_EVALUATION["selected_model"]]["test"]["macro_f1"],
            "weighted_f1": CLASSIFICATION_EVALUATION["comparison"][CLASSIFICATION_EVALUATION["selected_model"]]["test"]["weighted_f1"],
            "roc_auc": CLASSIFICATION_EVALUATION["comparison"][CLASSIFICATION_EVALUATION["selected_model"]]["test"]["roc_auc"],
            "per_class": CLASSIFICATION_EVALUATION["comparison"][CLASSIFICATION_EVALUATION["selected_model"]]["test"]["per_class"],
            "confusion_matrix": CLASSIFICATION_EVALUATION["comparison"][CLASSIFICATION_EVALUATION["selected_model"]]["test"]["confusion_matrix"],
            "confusion_matrix_order": CLASSIFICATION_EVALUATION["comparison"][CLASSIFICATION_EVALUATION["selected_model"]]["test"]["confusion_matrix_order"],
            "samples": CLASSIFICATION_EVALUATION["comparison"][CLASSIFICATION_EVALUATION["selected_model"]]["test"]["samples"],
            "class_distribution": CLASSIFICATION_EVALUATION["comparison"][CLASSIFICATION_EVALUATION["selected_model"]]["test"]["class_distribution"],
            "evaluated_model": CLASSIFICATION_EVALUATION["selected_model"],
            "evaluation_split": "held-out test split (patient level)",
            "validation_accuracy": CLASSIFICATION_EVALUATION["comparison"][CLASSIFICATION_EVALUATION["selected_model"]]["validation"]["accuracy"],
            "validation_macro_f1": CLASSIFICATION_EVALUATION["comparison"][CLASSIFICATION_EVALUATION["selected_model"]]["validation"]["macro_f1"],
            "validation_roc_auc": CLASSIFICATION_EVALUATION["comparison"][CLASSIFICATION_EVALUATION["selected_model"]]["validation"]["roc_auc"],
        },
        "model_comparison": CLASSIFICATION_EVALUATION["comparison"],
        "selected_model": CLASSIFICATION_EVALUATION["selected_model"],
        "splits": CLASSIFICATION_EVALUATION["class_balance"],
        "leakage_check": {"method": "patient_level_stratified_by_grade", "patient_overlap": [], "passed": True},
        "seed": SEED,
        "trained_at": utc_now(),
        "trainer": "notebooks/brain_tumor_training_kaggle_t4.ipynb",
        "dataset": DATASET_SUMMARY,
        "caveats": CLASSIFICATION_EVALUATION["caveats"],
    }
    write_json(CLASSIFICATION_ARTEFACT_DIR / "manifest.json", manifest)
    write_json(CLASSIFICATION_ARTEFACT_DIR / "classification_results.json", CLASSIFICATION_EVALUATION)

    # --- inference compatibility check: load it back the way the API does -----------------
    classifier_probe = {"state": "not_checked", "detail": None}
    try:
        reloaded = joblib.load(CLASSIFICATION_ARTEFACT_DIR / f"{model_id}.joblib")
        preprocessing = manifest["preprocessing"]
        vector = np.asarray(
            [preprocessing["median"][index] + preprocessing["scale"][index] for index in range(len(ACTIVE_FEATURES))],
            dtype=np.float32,
        ).reshape(1, -1)
        scaled = (vector - np.asarray(preprocessing["mean"], dtype=np.float32)) / np.asarray(
            preprocessing["scale"], dtype=np.float32
        )
        scores = np.asarray(reloaded.predict_proba(scaled)).ravel()
        classifier_probe = {
            "state": "compatible",
            "estimator": type(reloaded).__name__,
            "input_features": int(scaled.shape[1]),
            "probabilities": [float(value) for value in scores],
            "probability_sum": float(scores.sum()),
            "classes": list(INDEX_TO_LABEL.values()),
            "note": "Loaded with joblib and scored on a schema-length vector after manifest preprocessing.",
        }
    except Exception as exc:  # pragma: no cover - a failing probe means the export is unusable
        classifier_probe = {"state": "incompatible", "detail": f"{type(exc).__name__}: {exc}"}
        raise SystemExit(
            f"the exported classifier could not be loaded back ({classifier_probe['detail']}); it would "
            "not be servable, so the export is stopped rather than shipped"
        )
    print("classifier compatibility:", classifier_probe)

    COMPATIBILITY.append(
        {
            "model_id": model_id,
            "state": "compatible",
            "exported_as": str(CLASSIFICATION_ARTEFACT_DIR.relative_to(EXPORT)),
            "feature_count": len(ACTIVE_FEATURES),
            "classes": list(INDEX_TO_LABEL.values()),
            "probe": classifier_probe,
            "note": "manifest.json carries the feature schema, the scaler and the metrics the API requires",
        }
    )

# --- multiclass classification artefacts (second dataset) ---------------------
# Separate directory, separate manifest, and a `task` field on every entry: the application can
# tell a HGG/LGG grade model trained on BraTS apart from a tumour-type model trained on the
# Brain Tumor MRI Dataset, and neither is silently served in place of the other.
MULTICLASS_ARTEFACT_DIR = None
MULTICLASS_COMPATIBILITY: dict = {"state": "not_available", "reason": "the multiclass pipeline did not run"}

if MULTICLASS_EVALUATION.get("state") == "evaluated":
    import joblib

    stamp = utc_now().replace(":", "").replace("-", "")[:15]
    MULTICLASS_MODEL_ID = "brain_tumor_mri_classifier"
    MULTICLASS_VERSION = f"{MULTICLASS_MODEL_ID}-{stamp}"
    MULTICLASS_ARTEFACT_DIR = EXPORT / "data" / "models" / "classification_multiclass" / MULTICLASS_VERSION
    MULTICLASS_ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)

    member_files = {}
    for member_name, estimator in MULTICLASS_FEATURE_MODELS.items():
        filename = f"{member_name}.joblib"
        joblib.dump(estimator, MULTICLASS_ARTEFACT_DIR / filename)
        member_files[member_name] = filename
    joblib.dump(MULTICLASS_STACK_RESULT["stacker"], MULTICLASS_ARTEFACT_DIR / "stacked_ensemble.joblib")

    multiclass_manifest = {
        "model_id": MULTICLASS_MODEL_ID,
        "version": MULTICLASS_VERSION,
        "task": "multiclass_classification",
        "target_description": f"{RUN['classification_task_name']} -- {', '.join(CLASSIFICATION_CLASSES)}",
        "label_mapping": MULTICLASS_INDEX_TO_LABEL,
        "classes": CLASSIFICATION_CLASSES,
        "kind": "sklearn",
        "model_file": "stacked_ensemble.joblib",
        "member_files": member_files,
        "members": MULTICLASS_STACK_RESULT["members"],
        "feature_schema": MULTICLASS_FEATURE_NAMES,
        "provenance_contract": {
            "feature_schema": "the application's own extractor, bare names, unordered input is refused",
            "feature_sources": {
                "extractor": "backend/app/ml/radiomics_native.py via RadiomicsService",
                "mask_source": "auto_roi (Otsu threshold, largest connected component, morphology)",
                "image": f"grayscale, {RUN['radiomics_image_size']}px, no voxel spacing -> pixel-domain units",
                "serving_contract": (
                    "The application computes radiomics from the *segmentation mask of the upload*, not from an "
                    "auto-ROI on a single grayscale slice. This artefact is therefore exported for identification "
                    "and reuse in research code, not as a drop-in replacement for the served classifier: the ROI "
                    "definition differs, and pretending otherwise would change the input distribution silently."
                ),
            },
            "preprocessing": {
                "type": "standard_scaler",
                "fit_on": "training split only",
                "imputation": "training-split median",
                "median": [float(value) for value in MULTICLASS_PREPROCESSING["imputer"].statistics_],
                "mean": [float(value) for value in MULTICLASS_PREPROCESSING["scaler"].mean_],
                "scale": [float(value) for value in MULTICLASS_PREPROCESSING["scaler"].scale_],
                "feature_selection": "none - every schema feature is used, in feature_schema order",
            },
            "stacking_layer": {
                "estimator": type(MULTICLASS_STACK_RESULT["stacker"]).__name__,
                "fitted_on": MULTICLASS_STACK_RESULT["fitted_on"],
                "members": MULTICLASS_STACK_RESULT["members"],
                "coefficients": (
                    np.asarray(MULTICLASS_STACK_RESULT["stacker"].coef_).tolist()
                    if hasattr(MULTICLASS_STACK_RESULT["stacker"], "coef_")
                    else None
                ),
            },
            "cnn": {
                "checkpoint": "checkpoints/brain_tumor_mri_cnn.pt",
                "architecture": "classifier_architecture(channels, num_classes, dropout)",
                "channels": list(RUN["cnn_channels"]),
                "dropout": RUN["cnn_dropout"],
                "input_size": RUN["classification_image_size"],
                "normalisation": "per-image standardisation (mean/std of the image itself)",
            },
            "caveats": MULTICLASS_EVALUATION["caveats"],
        },
        "metrics": {
            name: {
                key: value
                for key, value in entry.items()
                if key in {"accuracy", "macro_f1", "weighted_f1", "macro_precision", "macro_recall", "roc_auc", "roc_auc_average", "samples", "per_class", "confusion_matrix", "confusion_matrix_order"}
            }
            for name, entry in MULTICLASS_EVALUATION["models"].items()
        },
        "evaluation_split": "the dataset's own Testing directory, minus near-duplicates of training images",
        "selected_model": "stacked_logistic_regression",
        "best_single_member": MULTICLASS_STACK_RESULT["best_single_member"],
        "model_comparison": MULTICLASS_EVALUATION["stacked_ensemble"],
        "splits": MULTICLASS_EVALUATION["protocol"],
        "dataset": {
            "name": CLASSIFICATION_DATASET_NAME,
            "root": CLASSIFICATION_DATASET.get("root"),
            "classes": CLASSIFICATION_CLASSES,
            "synthetic": CLASSIFICATION_IS_SYNTHETIC,
            "provenance_note": CLASSIFICATION_PROVENANCE_NOTE,
            "split_source": CLASSIFICATION_SPLIT_SOURCE,
        },
        "seed": SEED,
        "trained_at": utc_now(),
        "trainer": "notebooks/brain_tumor_training_kaggle_t4.ipynb",
    }
    write_json(MULTICLASS_ARTEFACT_DIR / "manifest.json", multiclass_manifest)
    write_json(MULTICLASS_ARTEFACT_DIR / "classification_results.json", MULTICLASS_EVALUATION)

    # --- the CNN checkpoint, with a load-back check of its own -----------------
    cnn_directory = PATHS["checkpoints"] / "brain_tumor_mri_cnn"
    cnn_source = next((path for path in (cnn_directory / "best.pt", cnn_directory / "last.pt") if path.exists()), None)
    if cnn_source is not None:
        payload = torch.load(cnn_source, map_location="cpu", weights_only=False)
        try:
            probe_model = classifier_architecture(tuple(RUN["cnn_channels"]), len(CLASSIFICATION_CLASSES), float(RUN["cnn_dropout"]))
            probe_model.load_state_dict(payload["state_dict"], strict=True)
            probe_model.eval()
            with torch.no_grad():
                probe_output = probe_model(torch.zeros(1, 1, int(RUN["classification_image_size"]), int(RUN["classification_image_size"])))
            cnn_probe = {
                "state": "compatible",
                "strict_load": True,
                "input_shape": [1, 1, int(RUN["classification_image_size"]), int(RUN["classification_image_size"])],
                "output_shape": list(probe_output.shape),
                "classes": list(CLASSIFICATION_CLASSES),
                "parameters": sum(parameter.numel() for parameter in probe_model.parameters()),
            }
        except Exception as exc:  # noqa: BLE001 - a checkpoint that cannot be reloaded is not exported
            raise SystemExit(f"the multiclass CNN checkpoint could not be reloaded ({type(exc).__name__}: {exc})")
        torch.save(payload, EXPORT / "checkpoints" / "brain_tumor_mri_cnn.pt")
        MULTICLASS_COMPATIBILITY = {
            "state": "compatible",
            "model_id": "brain_tumor_mri_cnn",
            "exported_as": "checkpoints/brain_tumor_mri_cnn.pt",
            "probe": cnn_probe,
            "note": "reloaded with the notebook's own builder and a strict state-dict load",
        }
    else:
        MULTICLASS_COMPATIBILITY = {"state": "not_trained", "model_id": "brain_tumor_mri_cnn", "reason": "no checkpoint"}

    # --- stacking layer probe: the exported stacker must score a real vector -----
    try:
        reloaded_stacker = joblib.load(MULTICLASS_ARTEFACT_DIR / "stacked_ensemble.joblib")
        member_count = len(MULTICLASS_STACK_RESULT["members"])
        probe_vector = np.full((1, member_count * len(CLASSIFICATION_CLASSES)), 1.0 / len(CLASSIFICATION_CLASSES), dtype=np.float32)
        probe_scores = np.asarray(reloaded_stacker.predict_proba(probe_vector)).ravel()
        MULTICLASS_COMPATIBILITY["stacker_probe"] = {
            "state": "compatible",
            "estimator": type(reloaded_stacker).__name__,
            "input_features": int(probe_vector.shape[1]),
            "probability_sum": float(probe_scores.sum()),
            "classes": list(CLASSIFICATION_CLASSES),
        }
    except Exception as exc:  # pragma: no cover
        raise SystemExit(f"the exported stacking layer could not be reloaded ({type(exc).__name__}: {exc})")

    print("multiclass artefact :", MULTICLASS_ARTEFACT_DIR)
    print("multiclass probe    :", MULTICLASS_COMPATIBILITY)

# --- application-format metric files -----------------------------------------
if results:
    write_json(EXPORT / "data" / "models" / "evaluation_results.json", results)
if SELECTION["state"] == "selected":
    write_json(EXPORT / "data" / "models" / "model_selection.json", SELECTION)
if ENSEMBLE_REPORT.get("state") == "calibrated":
    write_json(EXPORT / "data" / "models" / "ensemble_calibration.json", ENSEMBLE_REPORT)

# --- provenance manifest read by the application -----------------------------
PROVENANCE = {
    "generated_at": utc_now(),
    "produced_by": "notebooks/brain_tumor_training_kaggle_t4.ipynb",
    "dataset": {
        "name": DATASET_NAME,
        "root": str(DATA_ROOT),
        "volumes": len(VOLUMES),
        "slices": len(SLICE_FILES),
        "labelled_patients": len(LABELLED),
        "class_counts": CLASS_COUNTS,
        "splits": {name: SPLIT_SUMMARY[name] for name in ("train", "validation", "test")},
        "split_method": "patient_level_stratified_by_grade",
        "split_seed": SEED,
        "synthetic": DATASET_IS_SYNTHETIC,
        "provenance_note": DATASET_PROVENANCE_NOTE,
    },
    "classification_dataset": {
        "name": CLASSIFICATION_DATASET_NAME,
        "root": CLASSIFICATION_DATASET.get("root"),
        "classes": CLASSIFICATION_CLASSES,
        "images": {name: sum(len(paths) for paths in groups.values()) for name, groups in CLASSIFICATION_IMAGES.items()},
        "split_source": CLASSIFICATION_SPLIT_SOURCE,
        "split_method": "stratified by class (no patient identifiers exist in this dataset)",
        "patient_level_split_possible": False,
        "near_duplicate_audit": CLASSIFICATION_AUDIT,
        "synthetic": CLASSIFICATION_IS_SYNTHETIC,
        "provenance_note": CLASSIFICATION_PROVENANCE_NOTE,
    },
    "environment": LOCK["environment"],
    "models": {},
}
for entry in COMPATIBILITY:
    if entry["state"] != "compatible":
        PROVENANCE["models"][entry["model_id"]] = {"state": entry["state"], "reason": entry.get("reason") or entry.get("error")}
        continue
    PROVENANCE["models"][entry["model_id"]] = {
        "state": (
            "not_trained"
            if DATASET_IS_SYNTHETIC is None and not results
            else "trained_and_evaluated"
            if results
            else "trained_not_evaluated"
        ),
        # Derived from the data, never assumed: a fixture run records `synthetic_fixture` and the
        # application shows that next to every prediction instead of "trained on real BraTS".
        "trained_on": (
            "real_brats2020"
            if DATASET_IS_SYNTHETIC is False
            else "synthetic_fixture"
            if DATASET_IS_SYNTHETIC is True
            else "unverified_dataset"
        ),
        "task": (
            "classification"
            if entry["model_id"] == "radiomics_grade_classifier"
            else "segmentation"
        ),
        "label_space": (
            "HGG / LGG (BraTS grade)"
            if entry["model_id"] == "radiomics_grade_classifier"
            else "tumour region (binary mask)"
        ),
        "dataset": DATASET_NAME,
        "evaluated_on": "held-out patient-level test split" if results else None,
        "exported_file": entry.get("exported_as"),
        "best_validation_dice": entry.get("best_validation_dice"),
        "parameters": entry.get("parameters"),
    }
for entry in ([MULTICLASS_COMPATIBILITY] if MULTICLASS_ARTEFACT_DIR is not None else []):
    PROVENANCE["models"]["brain_tumor_mri_cnn"] = {
        "state": "trained_and_evaluated" if entry.get("state") == "compatible" else "not_trained",
        "task": "multiclass_classification",
        "classes": CLASSIFICATION_CLASSES,
        "trained_on": (
            "real_brain_tumor_mri"
            if CLASSIFICATION_IS_SYNTHETIC is False
            else "synthetic_fixture"
            if CLASSIFICATION_IS_SYNTHETIC is True
            else "unverified_dataset"
        ),
        "dataset": CLASSIFICATION_DATASET_NAME,
        "evaluated_on": "the dataset's own Testing directory, minus near-duplicates of training images",
        "exported_file": entry.get("exported_as"),
        "probe": entry.get("probe"),
    }
    PROVENANCE["models"][MULTICLASS_MODEL_ID] = {
        "state": "trained_and_evaluated",
        "task": "multiclass_classification",
        "classes": CLASSIFICATION_CLASSES,
        "trained_on": (
            "real_brain_tumor_mri"
            if CLASSIFICATION_IS_SYNTHETIC is False
            else "synthetic_fixture"
            if CLASSIFICATION_IS_SYNTHETIC is True
            else "unverified_dataset"
        ),
        "dataset": CLASSIFICATION_DATASET_NAME,
        "evaluated_on": "the dataset's own Testing directory, minus near-duplicates of training images",
        "exported_file": str(MULTICLASS_ARTEFACT_DIR.relative_to(EXPORT)),
        "members": MULTICLASS_STACK_RESULT["members"],
        "best_single_member": MULTICLASS_STACK_RESULT["best_single_member"],
        "statement": MULTICLASS_STACK_RESULT["statement"],
    }
write_json(EXPORT / "data" / "models" / "artifacts_manifest.json", PROVENANCE)

# A flat index of every exported artefact with its task and dataset, so the application (and a
# human reading the archive) can tell which dataset and which task each file belongs to without
# opening the checkpoints themselves.
MODELS_INDEX = {
    "generated_at": utc_now(),
    "produced_by": "notebooks/brain_tumor_training_kaggle_t4.ipynb",
    "artefacts": [],
}
for entry in COMPATIBILITY:
    MODELS_INDEX["artefacts"].append(
        {
            "name": entry["model_id"],
            "task": "segmentation" if entry["model_id"] in {"segnet_brats_4ch", "raunet_brats_4ch", "unet_dynamic_512_real_brats"} else "classification",
            "label_space": "tumour region (binary mask)" if "brats" in entry["model_id"] or "unet" in entry["model_id"] else "HGG / LGG (BraTS grade)",
            "dataset": DATASET_NAME,
            "dataset_synthetic": DATASET_IS_SYNTHETIC,
            "state": entry["state"],
            "file": entry.get("exported_as"),
            "served_by_application": entry["model_id"] in {"segnet_brats_4ch", "raunet_brats_4ch", "unet_dynamic_512_real_brats", "radiomics_grade_classifier"},
        }
    )
if MULTICLASS_ARTEFACT_DIR is not None:
    MODELS_INDEX["artefacts"].append(
        {
            "name": "brain_tumor_mri_cnn",
            "task": "multiclass_classification",
            "label_space": " / ".join(CLASSIFICATION_CLASSES),
            "dataset": CLASSIFICATION_DATASET_NAME,
            "dataset_synthetic": CLASSIFICATION_IS_SYNTHETIC,
            "state": MULTICLASS_COMPATIBILITY.get("state"),
            "file": "checkpoints/brain_tumor_mri_cnn.pt",
            "served_by_application": False,
            "serving_note": "the API serves the radiomics classifier; this CNN is exported for identification and reuse",
        }
    )
    MODELS_INDEX["artefacts"].append(
        {
            "name": MULTICLASS_MODEL_ID,
            "task": "multiclass_classification",
            "label_space": " / ".join(CLASSIFICATION_CLASSES),
            "dataset": CLASSIFICATION_DATASET_NAME,
            "dataset_synthetic": CLASSIFICATION_IS_SYNTHETIC,
            "state": MULTICLASS_EVALUATION.get("state"),
            "file": str(MULTICLASS_ARTEFACT_DIR.relative_to(EXPORT)),
            "served_by_application": False,
            "serving_note": "different ROI definition from the served pipeline; see provenance_contract in its manifest",
        }
    )
write_json(EXPORT / "data" / "models" / "models_index.json", MODELS_INDEX)

# --- how to install the artefacts -------------------------------------------
INSTALL_NOTES = f"""# BraTS 2020 training artefacts

Produced by `notebooks/brain_tumor_training_kaggle_t4.ipynb` on {utc_now()}.
Dataset: {len(VOLUMES)} volumes / {len(SLICE_FILES)} slices of the BraTS 2020 training data,
patient-level split (train {SPLIT_SUMMARY['train']['patients']} / validation {SPLIT_SUMMARY['validation']['patients']} / test {SPLIT_SUMMARY['test']['patients']} patients), seed {SEED}.

## Contents

| File | What it is |
| --- | --- |
| `checkpoints/segnet_brats_4ch_real_brats.pt` | retrained SegNet, 4x{RUN['image_size']}x{RUN['image_size']}, logits |
| `checkpoints/raunet_brats_4ch.pt` | RAUNet, 4x{RUN['image_size']}x{RUN['image_size']}, logits |
| `checkpoints/unet_dynamic_512_real_brats.pt` | single-channel U-Net, 1x{RUN['single_image_size']}x{RUN['single_image_size']}, probability |
| `data/models/classification/<version>/` | classifier `.joblib` + `manifest.json` (schema, scaler, metrics) |
| `data/models/evaluation_results.json` | held-out test metrics for every evaluated model |
| `data/models/model_selection.json` | which model was chosen, on validation data only |
| `data/models/ensemble_calibration.json` | fusion weights estimated on the validation split |
| `checkpoints/brain_tumor_mri_cnn.pt` | multiclass CNN for the Brain Tumor MRI Dataset (from scratch, no pretrained weights) |
| `data/models/classification_multiclass/<version>/` | multiclass artefact: stacking layer, both radiomics members, `manifest.json` with the ordered schema, scaler, class mapping and metrics |
| `data/models/artifacts_manifest.json` | provenance the API reads to label each model |
| `data/models/models_index.json` | every artefact with its task, label space, dataset and whether the application serves it |
| `reports/` | comparison tables, confusion matrix, EDA and Grad-CAM figures |

## Installing into the project

```bash
# 1. copy this whole directory into the project root, then
python scripts/install_trained_artifacts.py --source brain_mri_artifacts --data-dir data
# 2. verify what the application now sees
python scripts/check_models.py
python -m pytest tests -q
```

The install script copies checkpoints into place **without overwriting** the shipped
files: it writes them next to the originals and prints the environment variables
(`BRAIN_MODEL_SEGNET`, `BRAIN_MODEL_RAUNET`, `BRAIN_MODEL_UNET`) that point the API at the
new artefacts, and it records everything in `data/models/artifacts_manifest.json`.

## Honest status

* Metrics in `evaluation_results.json` were measured on the held-out patient-level test
  split of the split recorded above, with threshold 0.5 and the project's own metric code.
* The classification label is the dataset's HGG/LGG grade annotation -- not a malignancy
  verdict, not a stage, not a prognosis.
* The multiclass metrics come from the second dataset's own Testing directory -- minus any image
  whose perceptual hash matches a training image -- and the stacking layer was fitted on the
  validation split only. The dataset carries no patient identifiers, so a patient-level split is
  not possible and is not claimed.
* The classes of the multiclass model are the dataset's own (glioma, meningioma, pituitary,
  no tumour). They are not a malignancy verdict, a stage or a prognosis.
* Anything missing from `checkpoints/` was not trained in that session; rerun the notebook
  and it resumes from its own checkpoints.

## Two datasets, two tasks

| Task | Dataset | Artefacts |
| --- | --- | --- |
| Segmentation (and the HGG/LGG grade experiment) | BraTS 2020 (`awsaf49/brats2020-training-data`) | `checkpoints/*`, `data/models/classification/<version>/`, `evaluation_results.json` |
| Multiclass tumour type | Brain Tumor MRI Dataset (`masoudnickparvar/brain-tumor-mri-dataset`) | `checkpoints/brain_tumor_mri_cnn.pt`, `data/models/classification_multiclass/<version>/` |

The two label spaces never mix: the stacked ensemble accepts a member only when its dataset and
its class list match, so the BraTS grade model is rejected by construction (the notebook prints
the rejection).
"""
(EXPORT / "README.md").write_text(INSTALL_NOTES, encoding="utf-8")

import shutil

archive = shutil.make_archive(str(PATHS["work"] / "brain_mri_artifacts"), "zip", root_dir=EXPORT)
print("=" * 78)
print("EXPORT")
print("=" * 78)
for entry in COMPATIBILITY:
    print(f"{entry['model_id']:32s} {entry['state']:12s} {entry.get('exported_as') or entry.get('reason') or entry.get('error')}")
print("export directory:", EXPORT)
print("archive         :", archive, f"({Path(archive).stat().st_size / 1024 ** 2:.1f} MiB)")
print("provenance      :", EXPORT / "data" / "models" / "artifacts_manifest.json")
print(
    "\nDownload the archive from the notebook output panel (or /kaggle/working) and follow "
    "the README inside it. The compatibility check above loaded every checkpoint with the "
    "application's own architecture builders and a strict state-dict load, so a checkpoint "
    "listed as 'compatible' will load in the API."
)


# Runbook, resume and export

## Session planning (Tesla T4, 15 GB)

The workload is dominated by three things: how many slices an epoch sees, the input size, and
the batch size. Measure it on your first run instead of trusting a guess -- the trainer prints
`seconds/epoch` and every checkpoint carries `elapsed_minutes`:

| Model | Input | Typical step time on a T4 (fp16) | Notes |
| --- | --- | --- | --- |
| SegNet | 4 x 256 x 256 | fast, small network | epoch time scales linearly with slices per epoch |
| RAUNet | 4 x 256 x 256 | several times SegNet per step | the deep-supervision heads add memory; reduce `batch_size` first |
| DynamicUNet | 1 x 512 x 512 | heaviest per step | keep `batch_size_single` at 2-4 on a T4 |

Set `RUN["max_steps_per_epoch"]` to time-box an epoch (the index is reshuffled every epoch, so a
capped epoch still sees different slices). `RUN["session_budget_minutes"]` stops training cleanly
before Kaggle ends the session, so the next session resumes instead of losing the work.

Cell 18 prints what a step actually costs before the epoch loop starts, and the first thing to
check is `batch preparation`: it is milliseconds when the cache is memory-mappable and seconds
when it is still compressed (see "The data pipeline" above -- cell 9 reports and converts it).

## Resume

* `RUN["smoke_test"] = True` runs the whole pipeline on a handful of patients: use it to verify
  the environment before spending GPU quota.
* Re-run the training cells. `checkpoints/<model_id>/last.pt` holds the model, optimiser,
  scheduler, AMP scaler, epoch, step, best metric and the RNG state; training continues from
  there. `best.pt` keeps the checkpoint with the best validation Dice.
* `history.json` and `training_summary.json` accumulate across sessions, so the final curves
  cover everything that was trained.
* If a session dies mid-epoch, only the work since the last checkpoint is lost
  (`checkpoint_every_steps`, `checkpoint_every_minutes`).

## Export and install

```bash
# 1. download brain_mri_artifacts.zip from the notebook output and unpack it in the project root
unzip brain_mri_artifacts.zip -d /path/to/Brain-segmentation-and-classification/

# 2. copy the artefacts into place (never overwriting the shipped checkpoints)
python scripts/install_trained_artifacts.py --source brain_mri_artifacts

# 3. what the application now sees
python scripts/check_models.py
python -m pytest tests -q
```

`install_trained_artifacts.py` also prints the `BRAIN_MODEL_*` environment variables that point
the API at the newly trained checkpoints, and writes `data/models/artifacts_manifest.json` so the
interface can label each model *trained on real BraTS data* / *trained on a synthetic fixture* /
*implemented but not trained*.

## Honest reporting

* Metrics come from the reserved patient-level test split, with the project's own metric code.
* The classification label is the dataset's HGG/LGG grade annotation. It is not a malignancy
  verdict, not a stage and not a prognosis.
* If a model is missing from `checkpoints/`, it was not trained in that session -- the export
  marks it `not_trained` and the API keeps reporting it as unavailable rather than pretending.
